<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>ادامه فشرده Golden Dataset — Pilot QA Workflow</h2>

<p>
این Notebook ادامه مرحله Golden Dataset در Phase 2 است و برای تکمیل Pilotهای باقی‌مانده پس از Notebook توسعه‌ای قبلی ساخته شده است.
این Notebook به معنی ورود به مرحله ماژولار بعدی پروژه نیست؛ تمام فعالیت‌های آن همچنان متعلق به ساخت، اعتبارسنجی و Freeze کردن Golden Dataset هستند.
</p>

<p>
Notebook قبلی نقش Development و Contract Discovery را داشت.
در آن، Pilot Selection، Target Evidence، Question Drafting، Machine Validation، Human Review،
Expected Answer، Citation Validation، Queue Management، Replacement Policy و Atomic Rollback
به‌صورت عمیق طراحی و آزمایش شدند.
به همین دلیل تعداد Cellهای آن زیاد شد.
</p>

<p>
هدف این Notebook تکرار آن روند توسعه‌ای نیست.
Contractهایی که قبلاً اثبات شده‌اند باید به یک Workflow فشرده، Reusable و Parameterized تبدیل شوند
تا Pilotهای باقی‌مانده بدون ساخت ده‌ها Cell برای هر مورد پردازش شوند.
</p>

<h3>وضعیت ورودی</h3>

<ul>
<li>دامنه Pilot Validation برابر با <strong>23 Pilot</strong> است.</li>
<li>یک Pilot مرجع قبلاً به‌طور کامل Accepted شده است.</li>
<li>Remaining Queue برابر با <strong>22 Pilot</strong> است.</li>
<li>Pilot بلاک‌شده قبلی با یک Replacement معتبر جایگزین شده است.</li>
<li>Replacement Pilot فعلی باید اولین Pilot این Notebook برای ادامه Standard QA Workflow باشد.</li>
<li>Full 572 Generation همچنان تا تکمیل Pilot Validation مسدود باقی می‌ماند.</li>
</ul>

<h3>اصل معماری این Notebook</h3>

<p>
این Notebook نباید برای هر Pilot دوباره Selection Engine، Queue Logic، Validation Logic،
Replacement Logic یا Rollback Logic را از صفر بسازد.
این منطق‌ها باید یک‌بار به توابع Reusable تبدیل شوند و سپس برای Pilotهای مختلف فراخوانی شوند.
</p>

<p>
روش فشرده پیشنهادی:
</p>

<ol>
<li>Discovery و بازیابی Source of Truth واقعی از Repository و Notebook قبلی؛</li>
<li>ساخت یک Runtime State کوچک و قابل‌بازتولید برای Golden Dataset Workflow؛</li>
<li>تعریف توابع Reusable برای Stage، Draft، Machine Validation، Human Review Packet، Expected Answer و Citation Validation؛</li>
<li>تعریف Queue/State Transition عمومی به‌جای ساخت Cellهای اختصاصی متعدد؛</li>
<li>پردازش Pilotها یکی‌یکی با همان Engine مشترک؛</li>
<li>توقف فقط در Human Gate یا Edge Case واقعی؛</li>
<li>در صورت Edge Case، Repair فقط همان مورد و سپس بازگشت به Workflow عمومی؛</li>
<li>در پایان، Validation کل Pilot Set، Dev/Test Split و Freeze Golden Dataset.</li>
</ol>

<h3>قاعده جلوگیری از بزرگ‌شدن Notebook</h3>

<p>
یک Pilot سالم نباید ده‌ها Cell جدید تولید کند.
منطق عمومی فقط یک‌بار تعریف می‌شود.
برای Pilotهای معمولی، اجرای Workflow باید با تعداد بسیار کمی Cell انجام شود و خروجی‌ها در State ساختاریافته ثبت شوند.
فقط Failure یا Edge Case جدید می‌تواند باعث اضافه‌شدن Step تشخیصی جداگانه شود.
</p>

<p>
بنابراین هدف این Notebook این است که 22 Pilot باقی‌مانده را با یک Engine مشترک تکمیل کند،
نه اینکه تاریخچه 140-Cell Notebook قبلی را 22 بار تکرار کند.
</p>

<h3>خارج از Scope</h3>

<ul>
<li>Embedding Benchmark</li>
<li>Dense Index</li>
<li>Hybrid Retrieval</li>
<li>Reranking</li>
<li>تولید Full 572 قبل از رفع Pilot Validation Gate</li>
<li>تغییر Notebookهای مرحله بعدی Roadmap</li>
</ul>

<h3>شرط Done شدن Notebook</h3>

<ul>
<li>Pilot Workflow برای دامنه موردنیاز تکمیل و معتبر باشد.</li>
<li>Golden Dataset موردنیاز ساخته و Validate شود.</li>
<li>Dev/Test Split و Freeze نهایی انجام شود.</li>
<li>هیچ Error یا Hidden Session Dependency باقی نماند.</li>
<li>Restart Kernel → Run All بدون Error اجرا شود.</li>
<li>Final Validation و Notebook Health هر دو PASS شوند.</li>
</ul>

<p>
Bootstrap فنی این Notebook در <code>Code Cell 01</code> متمرکز می‌شود و Import ثابت در Cellهای بعدی تکرار نخواهد شد.
</p>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Repository / Artifact Preflight</h3>

<p>
قبل از ساخت Workflow فشرده، ابتدا باید وضعیت واقعی Repository و خروجی Notebook قبلی از روی Disk بررسی شود.
هیچ Runtime State از Kernel قبلی فرض نمی‌شود.
</p>

<p>
این Preflight به‌صورت Read-Only بررسی می‌کند:
</p>

<ul>
<li>Git Root و Branch واقعی؛</li>
<li>Notebookهای مرتبط با Golden Dataset؛</li>
<li>وجود Final Activation و Final Checkpoint در Notebook توسعه‌ای قبلی؛</li>
<li>Artifactها، Checkpointها، Manifestها یا Reportهای مرتبط موجود روی Disk؛</li>
<li>Git Status فعلی؛</li>
<li>اینکه آیا یک Handoff ماشین‌خوان از Notebook قبلی وجود دارد یا باید State لازم به‌صورت deterministic بازسازی شود.</li>
</ul>

<p>
در این مرحله هیچ فایل یا Artifactی ساخته یا تغییر داده نمی‌شود.
نتیجه Preflight تعیین می‌کند Compact Workflow باید از کدام Source of Truth شروع شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Repository Root واقعی شناسایی شود.</li>
<li>Branch واقعی شناسایی شود.</li>
<li>Notebook توسعه‌ای Golden Dataset قبلی با Markerهای Activation و Final Checkpoint پیدا شود.</li>
<li>Discovery بدون Write یا Mutation انجام شود.</li>
</ul>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
from collections import Counter, defaultdict

import copy
import hashlib
import json
import re
import subprocess

In [2]:
# Code Cell 02

cwd = Path.cwd().resolve()


def run_git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    return result.returncode, result.stdout.strip(), result.stderr.strip()


root_rc, root_out, root_err = run_git(
    "rev-parse",
    "--show-toplevel",
)

project_root = (
    Path(root_out).resolve()
    if root_rc == 0 and root_out
    else None
)


branch_rc, branch_out, branch_err = run_git(
    "branch",
    "--show-current",
)

current_branch = (
    branch_out
    if branch_rc == 0 and branch_out
    else None
)


head_rc, head_out, head_err = run_git(
    "rev-parse",
    "HEAD",
)

current_head = (
    head_out
    if head_rc == 0 and head_out
    else None
)


status_rc, status_out, status_err = run_git(
    "status",
    "--short",
)


related_notebooks = []

if project_root is not None:
    for path in sorted(project_root.rglob("*.ipynb")):
        name = path.name.lower()

        if (
            "golden" in name
            or
            "dataset" in name
            or
            name.startswith("12")
        ):
            related_notebooks.append(path)


ACTIVATION_MARKERS = [
    "REPLACEMENT_PILOT_ATOMICALLY_ACTIVATED_LIVE_STATE_VALID",
    "CODE CELL 139 GATE",
]

CHECKPOINT_MARKERS = [
    "CURRENT_NOTEBOOK_POST_ACTIVATION_STATE_VALID_READY_TO_CLOSE_DEVELOPMENT_NOTEBOOK",
    "CURRENT_DEVELOPMENT_NOTEBOOK_FINAL_CHECKPOINT_PASS",
    "CODE CELL 140 GATE",
]


notebook_inspection = []

for path in related_notebooks:
    try:
        raw = path.read_text(
            encoding="utf-8"
        )

        payload = json.loads(raw)

        code_cells = [
            cell
            for cell in payload.get("cells", [])
            if cell.get("cell_type") == "code"
        ]

        numbered_cells = []

        for cell in code_cells:
            source = "".join(
                cell.get("source", [])
            )

            match = re.search(
                r"# Code Cell\s+(\d+)",
                source,
            )

            if match:
                numbered_cells.append(
                    int(match.group(1))
                )

        activation_found = any(
            marker in raw
            for marker in ACTIVATION_MARKERS
        )

        checkpoint_found = any(
            marker in raw
            for marker in CHECKPOINT_MARKERS
        )

        notebook_inspection.append({
            "path": str(
                path.relative_to(project_root)
            ),
            "code_cells": len(code_cells),
            "highest_numbered_code_cell": (
                max(numbered_cells)
                if numbered_cells
                else None
            ),
            "activation_marker_found":
                activation_found,
            "checkpoint_marker_found":
                checkpoint_found,
            "size_bytes":
                path.stat().st_size,
        })

    except Exception as exc:
        notebook_inspection.append({
            "path": str(
                path.relative_to(project_root)
            ),
            "inspection_error": repr(exc),
        })


validated_prior_notebooks = [
    row
    for row in notebook_inspection
    if (
        row.get(
            "activation_marker_found"
        )
        is True
        and
        row.get(
            "checkpoint_marker_found"
        )
        is True
    )
]


artifact_keywords = (
    "golden",
    "pilot",
    "question",
    "citation",
    "workflow",
    "checkpoint",
    "manifest",
)


artifact_roots = []

if project_root is not None:
    for dirname in [
        "data",
        "reports",
        "artifacts",
        "checkpoints",
        "manifests",
    ]:
        candidate = (
            project_root / dirname
        )

        if candidate.exists():
            artifact_roots.append(
                candidate
            )


related_artifacts = []

for base in artifact_roots:
    for path in base.rglob("*"):
        if not path.is_file():
            continue

        lowered = (
            path.name.lower()
        )

        if any(
            keyword in lowered
            for keyword
            in artifact_keywords
        ):
            related_artifacts.append(
                {
                    "path": str(
                        path.relative_to(
                            project_root
                        )
                    ),
                    "size_bytes":
                        path.stat().st_size,
                }
            )


related_artifacts = sorted(
    related_artifacts,
    key=lambda row: row["path"],
)


handoff_candidates = [
    row
    for row in related_artifacts
    if any(
        token in row["path"].lower()
        for token in (
            "handoff",
            "checkpoint",
            "workflow",
            "pilot",
        )
    )
]


preflight_gate = all([
    project_root is not None,
    current_branch is not None,
    len(
        validated_prior_notebooks
    )
    >=
    1,
])


print(
    "=== COMPACT GOLDEN DATASET PREFLIGHT ==="
)

print(
    "\nWorking directory:",
    cwd,
)

print(
    "Repository root:",
    project_root,
)

print(
    "Current branch:",
    current_branch,
)

print(
    "Current HEAD:",
    current_head,
)


print(
    "\n=== GIT STATUS ==="
)

print(
    status_out
    if status_out
    else "Working tree clean"
)


print(
    "\n=== RELATED NOTEBOOKS ==="
)

print(
    "Candidates found:",
    len(
        notebook_inspection
    )
)

for row in notebook_inspection:
    print(
        row
    )


print(
    "\n=== VALIDATED PRIOR DEVELOPMENT NOTEBOOKS ==="
)

print(
    "Validated candidates:",
    len(
        validated_prior_notebooks
    )
)

for row in validated_prior_notebooks:
    print(
        row
    )


print(
    "\n=== RELATED DISK ARTIFACTS ==="
)

print(
    "Related artifacts found:",
    len(
        related_artifacts
    )
)

for row in related_artifacts[:50]:
    print(
        row
    )

if len(related_artifacts) > 50:
    print(
        "... additional artifacts:",
        len(related_artifacts) - 50,
    )


print(
    "\n=== HANDOFF / CHECKPOINT CANDIDATES ==="
)

print(
    "Candidates found:",
    len(
        handoff_candidates
    )
)

for row in handoff_candidates[:30]:
    print(
        row
    )


print(
    "\n=== PREFLIGHT DECISION ==="
)

print(
    "Repository resolved:",
    project_root is not None,
)

print(
    "Branch resolved:",
    current_branch is not None,
)

print(
    "Validated prior NB12 found:",
    len(
        validated_prior_notebooks
    )
    >=
    1,
)

print(
    "Machine-readable handoff candidates:",
    len(
        handoff_candidates
    ),
)

print(
    "Mutation performed:",
    False,
)

print(
    "\nCODE CELL 02 GATE:",
    "PASS"
    if preflight_gate
    else "FAIL",
)


assert preflight_gate, (
    "Compact Golden Dataset preflight failed. "
    "Do not build the reusable workflow yet. "
    "Repair this SAME Code Cell 02."
)

=== COMPACT GOLDEN DATASET PREFLIGHT ===

Working directory: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2
Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Current branch: rag-phase2
Current HEAD: 0a2281521270634af9f468b4dff0ff4f5daeb066

=== GIT STATUS ===
M 12b_golden_dataset_continuation_compact.ipynb

=== RELATED NOTEBOOKS ===
Candidates found: 2
{'path': 'notebooks/phase2/12_golden_dataset.ipynb', 'code_cells': 141, 'highest_numbered_code_cell': 141, 'activation_marker_found': True, 'checkpoint_marker_found': True, 'size_bytes': 9345530}
{'path': 'notebooks/phase2/12b_golden_dataset_continuation_compact.ipynb', 'code_cells': 9, 'highest_numbered_code_cell': 9, 'activation_marker_found': True, 'checkpoint_marker_found': True, 'size_bytes': 256180}

=== VALIDATED PRIOR DEVELOPMENT NOTEBOOKS ===
Validated candidates: 2
{'path': 'notebooks/phase2/12_golden_dataset.ipynb', 'code_cells': 141, 'highest

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اعتبارسنجی Handoff و تعیین Source of Truth برای Compact Workflow</h3>

<p>
Preflight وجود یک Handoff ماشین‌خوان از Notebook قبلی را تأیید کرد.
اما صرف وجود فایل به معنی Current و قابل‌استفاده بودن آن نیست.
ممکن است این Artifact قبل از Replacement نهایی Pilot ساخته شده باشد.
</p>

<p>
در این مرحله Handoff موجود به‌صورت Read-Only خوانده می‌شود و هم‌زمان خروجی واقعی Cellهای نهایی Notebook قبلی
از خود فایل ذخیره‌شده <code>.ipynb</code> استخراج می‌شود.
</p>

<p>
هدف این است که بدون اتکا به Session قبلی مشخص شود:
</p>

<ul>
<li>Notebook قبلی واقعاً Activation و Final Checkpoint را PASS کرده است؛</li>
<li>Active Pilot Count و Remaining Queue از Output واقعی قابل‌بازیابی هستند؛</li>
<li>Next Pilot نهایی از Output واقعی مشخص است؛</li>
<li>Checkpoint SHA نهایی قابل‌بازیابی است؛</li>
<li>Handoff موجود با این State نهایی سازگار است یا قدیمی است.</li>
</ul>

<p>
اگر Handoff موجود Current باشد، همان به‌عنوان Seed اولیه Compact Workflow استفاده می‌شود.
اگر Current نباشد، بدون حدس‌زدن State، در Step بعد یک Runtime Seed فشرده از
Notebook پذیرفته‌شده و Artifactهای Canonical موجود بازسازی خواهد شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Handoff JSON بدون Error Parse شود.</li>
<li>Cell 139 و Cell 140 از Notebook قبلی روی Disk پیدا شوند.</li>
<li>PASS واقعی Cell 140 از Output ذخیره‌شده تأیید شود.</li>
<li>Active Count، Remaining Count، Next Pilot و Checkpoint SHA از Output واقعی استخراج شوند.</li>
<li>Current یا Stale بودن Handoff بدون Mutation مشخص شود.</li>
</ul>

</div>

In [3]:
# Code Cell 03

handoff_path = (
    project_root
    / "reports/phase2/chunking/nb12_golden_dataset_handoff_v1.json"
)


prior_notebook_path = (
    project_root
    / "notebooks/phase2/12_golden_dataset.ipynb"
)


cell03_failures = []


if not handoff_path.exists():
    cell03_failures.append(
        "HANDOFF_FILE_NOT_FOUND"
    )


if not prior_notebook_path.exists():
    cell03_failures.append(
        "PRIOR_NOTEBOOK_NOT_FOUND"
    )


handoff_payload = None
prior_notebook_payload = None


if handoff_path.exists():
    try:
        handoff_payload = json.loads(
            handoff_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        cell03_failures.append({
            "reason":
                "HANDOFF_JSON_PARSE_FAILED",

            "error":
                repr(exc),
        })


if prior_notebook_path.exists():
    try:
        prior_notebook_payload = json.loads(
            prior_notebook_path.read_text(
                encoding="utf-8"
            )
        )
    except Exception as exc:
        cell03_failures.append({
            "reason":
                "PRIOR_NOTEBOOK_JSON_PARSE_FAILED",

            "error":
                repr(exc),
        })


def cell03_code_cell_by_number(
    notebook_payload,
    number,
):
    if not isinstance(
        notebook_payload,
        dict,
    ):
        return None

    pattern = re.compile(
        rf"^\s*# Code Cell\s+{number}\b",
        flags=re.MULTILINE,
    )

    for cell in notebook_payload.get(
        "cells",
        [],
    ):
        if cell.get(
            "cell_type"
        ) != "code":
            continue

        source = "".join(
            cell.get(
                "source",
                [],
            )
        )

        if pattern.search(
            source
        ):
            return cell

    return None


def cell03_output_text(cell):
    if not isinstance(
        cell,
        dict,
    ):
        return ""

    parts = []

    for output in cell.get(
        "outputs",
        [],
    ):
        if output.get(
            "output_type"
        ) == "stream":
            text = output.get(
                "text",
                "",
            )

            parts.append(
                "".join(text)
                if isinstance(text, list)
                else str(text)
            )

        data = output.get(
            "data",
            {},
        )

        text_plain = data.get(
            "text/plain"
        )

        if text_plain is not None:
            parts.append(
                "".join(text_plain)
                if isinstance(text_plain, list)
                else str(text_plain)
            )

    return "\n".join(
        parts
    )


cell03_cell139 = (
    cell03_code_cell_by_number(
        prior_notebook_payload,
        139,
    )
)


cell03_cell140 = (
    cell03_code_cell_by_number(
        prior_notebook_payload,
        140,
    )
)


cell03_cell139_output = (
    cell03_output_text(
        cell03_cell139
    )
)


cell03_cell140_output = (
    cell03_output_text(
        cell03_cell140
    )
)


cell03_cell139_pass = (
    "CODE CELL 139 GATE: PASS"
    in
    cell03_cell139_output
)


cell03_cell140_pass = (
    "CODE CELL 140 GATE: PASS"
    in
    cell03_cell140_output
)


def cell03_extract(
    pattern,
    text,
):
    match = re.search(
        pattern,
        text,
    )

    return (
        match.group(1).strip()
        if match
        else None
    )


cell03_active_pilot_count = (
    cell03_extract(
        r"Active Pilot count:\s*(\d+)",
        cell03_cell140_output,
    )
)


cell03_remaining_queue_count = (
    cell03_extract(
        r"Remaining Queue count:\s*(\d+)",
        cell03_cell140_output,
    )
)


cell03_next_pilot_id = (
    cell03_extract(
        r"Next Pilot:\s*(question-pilot:[A-Za-z0-9]+)",
        cell03_cell140_output,
    )
)


cell03_checkpoint_sha256 = (
    cell03_extract(
        r"Checkpoint SHA256:\s*([a-f0-9]{64})",
        cell03_cell140_output,
    )
)


cell03_checkpoint_status = (
    cell03_extract(
        r"Status:\s*([A-Z0-9_]+)",
        cell03_cell140_output,
    )
)


cell03_required_checkpoint_values_resolved = all([
    cell03_active_pilot_count
    is not None,

    cell03_remaining_queue_count
    is not None,

    cell03_next_pilot_id
    is not None,

    cell03_checkpoint_sha256
    is not None,

    cell03_checkpoint_status
    is not None,
])


cell03_handoff_serialized = (
    json.dumps(
        handoff_payload,
        ensure_ascii=False,
        sort_keys=True,
    )
    if handoff_payload
    is not None
    else ""
)


cell03_handoff_mentions_next_pilot = (
    cell03_next_pilot_id
    is not None

    and

    cell03_next_pilot_id
    in
    cell03_handoff_serialized
)


cell03_handoff_mentions_checkpoint_sha = (
    cell03_checkpoint_sha256
    is not None

    and

    cell03_checkpoint_sha256
    in
    cell03_handoff_serialized
)


cell03_handoff_current_for_final_state = all([
    cell03_handoff_mentions_next_pilot,
    cell03_handoff_mentions_checkpoint_sha,
])


cell03_prior_notebook_final_state_valid = all([
    cell03_cell139
    is not None,

    cell03_cell140
    is not None,

    cell03_cell139_pass,

    cell03_cell140_pass,

    cell03_required_checkpoint_values_resolved,

    cell03_active_pilot_count
    ==
    "23",

    cell03_remaining_queue_count
    ==
    "22",

    cell03_checkpoint_status
    ==
    "CURRENT_NOTEBOOK_POST_ACTIVATION_STATE_VALID_READY_TO_CLOSE_DEVELOPMENT_NOTEBOOK",
])


cell03_gate = all([
    len(
        cell03_failures
    )
    ==
    0,

    isinstance(
        handoff_payload,
        dict,
    ),

    cell03_prior_notebook_final_state_valid,
])


cell03_source_of_truth_decision = (
    "USE_EXISTING_HANDOFF_AS_COMPACT_RUNTIME_SEED"

    if (
        cell03_gate
        and
        cell03_handoff_current_for_final_state
    )

    else
    (
        "RECONSTRUCT_COMPACT_RUNTIME_SEED_FROM_ACCEPTED_NB12_DISK_STATE"

        if cell03_gate

        else

        "BLOCKED_PENDING_HANDOFF_INSPECTION_REPAIR"
    )
)


print(
    "=== NB12 HANDOFF VALIDATION ==="
)


print(
    "\nHandoff path:",
    handoff_path.relative_to(
        project_root
    ),
)

print(
    "Handoff parsed:",
    isinstance(
        handoff_payload,
        dict,
    ),
)


print(
    "\n=== HANDOFF PAYLOAD ==="
)

if isinstance(
    handoff_payload,
    dict,
):
    print(
        json.dumps(
            handoff_payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
    )


print(
    "\n=== PRIOR NOTEBOOK FINAL OUTPUT ==="
)

print(
    "Cell 139 found:",
    cell03_cell139
    is not None,
)

print(
    "Cell 139 PASS:",
    cell03_cell139_pass,
)

print(
    "Cell 140 found:",
    cell03_cell140
    is not None,
)

print(
    "Cell 140 PASS:",
    cell03_cell140_pass,
)

print(
    "Active Pilot count:",
    cell03_active_pilot_count,
)

print(
    "Remaining Queue count:",
    cell03_remaining_queue_count,
)

print(
    "Next Pilot:",
    cell03_next_pilot_id,
)

print(
    "Checkpoint SHA256:",
    cell03_checkpoint_sha256,
)

print(
    "Checkpoint status:",
    cell03_checkpoint_status,
)


print(
    "\n=== HANDOFF CURRENCY ==="
)

print(
    "Mentions current Next Pilot:",
    cell03_handoff_mentions_next_pilot,
)

print(
    "Mentions current Checkpoint SHA:",
    cell03_handoff_mentions_checkpoint_sha,
)

print(
    "Current for final post-activation state:",
    cell03_handoff_current_for_final_state,
)


print(
    "\n=== DECISION ==="
)

print(
    "Source of Truth decision:",
    cell03_source_of_truth_decision,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nFailures:",
    cell03_failures,
)


print(
    "\nCODE CELL 03 GATE:",
    "PASS"
    if cell03_gate
    else "FAIL",
)


assert cell03_gate, (
    "NB12 handoff inspection failed. "
    "Repair this SAME Code Cell 03 before building the compact workflow."
)

=== NB12 HANDOFF VALIDATION ===

Handoff path: reports/phase2/chunking/nb12_golden_dataset_handoff_v1.json
Handoff parsed: True

=== HANDOFF PAYLOAD ===
{
  "book": [
    "intrinsically_tiny_narrative_region",
    "intrinsically_tiny_table_region",
    "long_section",
    "large_table_run",
    "exercise_or_self_assessment",
    "supplemental_footnote_case",
    "deep_heading_path",
    "multi_paragraph_evidence_case"
  ],
  "candidate_strategies_to_compare": [
    "baseline_phase1_char_safe",
    "structure_aware_small",
    "structure_aware_medium",
    "parent_child_child"
  ],
  "evaluation_dimensions": [
    "retrieval_hit",
    "rank",
    "evidence_completeness",
    "attribution_correctness",
    "citation_locator_correctness",
    "unnecessary_context",
    "source_authority_correctness"
  ],
  "parent_context_separate_evaluation": true,
  "status": "REQUIRED_FOR_FINAL_CHUNK_DECISION",
  "transcript": [
    "single_speaker_primary",
    "single_speaker_unresolved",
    "multi_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>کشف Anchorهای لازم برای بازسازی Compact Runtime Seed</h3>

<p>
در Step قبل مشخص شد که Handoff موجود مربوط به State نهایی Pilotها نیست و نمی‌توان آن را به‌عنوان Runtime Seed استفاده کرد.
بنابراین قبل از ساخت Engine فشرده، باید دقیقاً مشخص شود State نهایی از چه اطلاعاتی می‌تواند به‌صورت deterministic بازسازی شود.
</p>

<p>
این مرحله فقط Discovery انجام می‌دهد و سه مسیر ممکن را بررسی می‌کند:
</p>

<ol>
<li>آیا Artifact ماشین‌خوان دیگری روی Disk وجود دارد که Replacement Pilot یا Final Checkpoint را ثبت کرده باشد؟</li>
<li>آیا Notebook قبلی State کامل موردنیاز مانند Pilot Population یا Remaining Queue را در Output ذخیره کرده است؟</li>
<li>اگر هیچ State کامل Persist نشده، کدام Cellها و Variableهای پذیرفته‌شده قرارداد کافی برای بازسازی State از Canonical Inputs را فراهم می‌کنند؟</li>
</ol>

<p>
هدف این است که Notebook جدید به 140 Cell قبلی وابسته نشود.
فقط Contractها و Anchorهای اثبات‌شده از Notebook قبلی خوانده می‌شوند و در Step بعد یک Seed کوچک و مستقل ساخته خواهد شد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Cellهای کلیدی Workflow و Replacement در Notebook قبلی شناسایی شوند.</li>
<li>وجود یا عدم وجود Artifact به‌روز روی Disk مشخص شود.</li>
<li>وجود یا عدم وجود Serialized Runtime State در Outputهای Notebook مشخص شود.</li>
<li>یک Reconstruction Route روشن و بدون حدس انتخاب شود.</li>
<li>هیچ فایل یا Stateی تغییر نکند.</li>
</ul>

</div>

In [4]:
# Code Cell 04

cell04_failures = []


# ------------------------------------------------------------
# 1) Accepted final-state anchors from Cell 03
# ------------------------------------------------------------

cell04_next_pilot_id = (
    cell03_next_pilot_id
)


cell04_checkpoint_sha256 = (
    cell03_checkpoint_sha256
)


cell04_required_cell_numbers = [
    29,
    74,
    95,
    125,
    130,
    135,
    137,
    139,
    140,
]


cell04_required_variable_anchors = [
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "pilot_selection_rows",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_pilot_workflow_state",
    "cell130_simulated_packet",
    "cell135_simulated_queue",
    "cell137_rebuilt_state",
    "cell138_target_values",
]


# ------------------------------------------------------------
# 2) Build notebook cell index
# ------------------------------------------------------------

cell04_code_cell_index = {}


for cell in prior_notebook_payload.get(
    "cells",
    [],
):
    if cell.get(
        "cell_type"
    ) != "code":
        continue

    source = "".join(
        cell.get(
            "source",
            [],
        )
    )

    match = re.search(
        r"^\s*# Code Cell\s+(\d+)",
        source,
        flags=re.MULTILINE,
    )

    if not match:
        continue

    number = int(
        match.group(1)
    )

    cell04_code_cell_index[
        number
    ] = {
        "source": source,
        "output": cell03_output_text(
            cell
        ),
        "execution_count":
            cell.get(
                "execution_count"
            ),
    }


cell04_required_cells_found = {
    number:
        number
        in
        cell04_code_cell_index

    for number
    in cell04_required_cell_numbers
}


# ------------------------------------------------------------
# 3) Find where important runtime variables appear
# ------------------------------------------------------------

cell04_variable_occurrences = {}


for variable_name in (
    cell04_required_variable_anchors
):
    hits = []

    for number, record in (
        cell04_code_cell_index.items()
    ):
        source_hit = (
            variable_name
            in
            record["source"]
        )

        output_hit = (
            variable_name
            in
            record["output"]
        )

        if source_hit or output_hit:
            hits.append({
                "cell":
                    number,

                "source":
                    source_hit,

                "output":
                    output_hit,
            })

    cell04_variable_occurrences[
        variable_name
    ] = hits


# ------------------------------------------------------------
# 4) Detect whether full runtime structures were serialized
# ------------------------------------------------------------

cell04_serialization_markers = {
    "pilot_packets":
        [
            "question_drafting_pilot_packets",
            "'pilot_case_id'",
            '"pilot_case_id"',
        ],

    "remaining_queue":
        [
            "cell95_remaining_pilot_queue",
            "'queue_index'",
            '"queue_index"',
        ],

    "rebuilt_state":
        [
            "cell137_rebuilt_state",
            "cell95_pilot_workflow_state_sha256",
        ],
}


cell04_serialized_output_candidates = {
    key: []
    for key
    in cell04_serialization_markers
}


for number, record in (
    cell04_code_cell_index.items()
):
    output = record[
        "output"
    ]

    if not output:
        continue

    for key, markers in (
        cell04_serialization_markers.items()
    ):
        if all(
            marker
            in
            output

            for marker
            in markers
        ):
            cell04_serialized_output_candidates[
                key
            ].append(
                number
            )


cell04_complete_runtime_state_serialized = all([
    len(
        cell04_serialized_output_candidates[
            "pilot_packets"
        ]
    )
    >
    0,

    len(
        cell04_serialized_output_candidates[
            "remaining_queue"
        ]
    )
    >
    0,
])


# ------------------------------------------------------------
# 5) Search disk artifacts for final Replacement anchors
# ------------------------------------------------------------

cell04_search_roots = [
    project_root
    / "data"
    / "phase2",

    project_root
    / "reports"
    / "phase2",
]


cell04_search_extensions = {
    ".json",
    ".jsonl",
    ".md",
    ".txt",
}


cell04_disk_anchor_hits = []


for base in cell04_search_roots:
    if not base.exists():
        continue

    for path in sorted(
        base.rglob("*")
    ):
        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            cell04_search_extensions
        ):
            continue

        try:
            if path.stat().st_size > (
                20
                *
                1024
                *
                1024
            ):
                continue

            text = path.read_text(
                encoding="utf-8"
            )

        except Exception:
            continue

        next_pilot_hit = (
            cell04_next_pilot_id
            in
            text
        )

        checkpoint_hit = (
            cell04_checkpoint_sha256
            in
            text
        )

        if (
            next_pilot_hit
            or
            checkpoint_hit
        ):
            cell04_disk_anchor_hits.append({
                "path":
                    str(
                        path.relative_to(
                            project_root
                        )
                    ),

                "next_pilot":
                    next_pilot_hit,

                "checkpoint_sha":
                    checkpoint_hit,
            })


cell04_current_state_artifacts = [
    row

    for row
    in cell04_disk_anchor_hits

    if (
        row["next_pilot"]
        and
        row["checkpoint_sha"]
    )
]


# ------------------------------------------------------------
# 6) Inspect contract cells for canonical rebuild anchors
# ------------------------------------------------------------

cell04_contract_cells = {}


for number in [
    29,
    74,
    95,
    125,
    130,
    135,
    137,
    139,
]:
    record = (
        cell04_code_cell_index.get(
            number
        )
    )

    if record is None:
        continue

    source = record[
        "source"
    ]

    cell04_contract_cells[
        number
    ] = {
        "has_pilot_packets":
            "question_drafting_pilot_packets"
            in
            source,

        "has_pilot_index":
            "question_drafting_pilot_by_id"
            in
            source,

        "has_selection_rows":
            "pilot_selection_rows"
            in
            source,

        "has_remaining_ids":
            "cell95_remaining_pilot_ids"
            in
            source,

        "has_remaining_queue":
            "cell95_remaining_pilot_queue"
            in
            source,

        "has_workflow_state":
            "cell95_pilot_workflow_state"
            in
            source,

        "has_replacement_packet":
            "cell130_simulated_packet"
            in
            source,

        "has_replacement_queue":
            "cell135_simulated_queue"
            in
            source,

        "has_rebuilt_state":
            "cell137_rebuilt_state"
            in
            source,
    }


cell04_core_contract_coverage = {
    "pilot_population":
        any(
            row[
                "has_pilot_packets"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "pilot_index":
        any(
            row[
                "has_pilot_index"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "selection_rows":
        any(
            row[
                "has_selection_rows"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "remaining_ids":
        any(
            row[
                "has_remaining_ids"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "remaining_queue":
        any(
            row[
                "has_remaining_queue"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "workflow_state":
        any(
            row[
                "has_workflow_state"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "replacement_packet":
        any(
            row[
                "has_replacement_packet"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "replacement_queue":
        any(
            row[
                "has_replacement_queue"
            ]

            for row
            in cell04_contract_cells.values()
        ),

    "rebuilt_state":
        any(
            row[
                "has_rebuilt_state"
            ]

            for row
            in cell04_contract_cells.values()
        ),
}


cell04_contract_coverage_complete = all(
    cell04_core_contract_coverage.values()
)


# ------------------------------------------------------------
# 7) Choose deterministic reconstruction route
# ------------------------------------------------------------

if len(
    cell04_current_state_artifacts
) > 0:

    cell04_reconstruction_route = (
        "LOAD_CURRENT_MACHINE_READABLE_FINAL_STATE_ARTIFACT"
    )

elif (
    cell04_complete_runtime_state_serialized
):

    cell04_reconstruction_route = (
        "PARSE_COMPLETE_RUNTIME_STATE_FROM_ACCEPTED_NOTEBOOK_OUTPUT"
    )

elif (
    cell04_contract_coverage_complete
):

    cell04_reconstruction_route = (
        "REBUILD_COMPACT_RUNTIME_SEED_FROM_CANONICAL_INPUTS_AND_ACCEPTED_CONTRACT_CELLS"
    )

else:

    cell04_reconstruction_route = (
        "BLOCKED_INSUFFICIENT_RECONSTRUCTION_ANCHORS"
    )


# ------------------------------------------------------------
# 8) Gate
# ------------------------------------------------------------

cell04_gate = all([
    all(
        cell04_required_cells_found.values()
    ),

    cell03_prior_notebook_final_state_valid,

    cell04_next_pilot_id
    is not None,

    cell04_checkpoint_sha256
    is not None,

    cell04_reconstruction_route
    !=
    "BLOCKED_INSUFFICIENT_RECONSTRUCTION_ANCHORS",
])


# ------------------------------------------------------------
# 9) Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT RUNTIME SEED RECONSTRUCTION DISCOVERY ==="
)


print(
    "\n=== FINAL STATE ANCHORS ==="
)

print(
    "Next Pilot:",
    cell04_next_pilot_id,
)

print(
    "Checkpoint SHA256:",
    cell04_checkpoint_sha256,
)


print(
    "\n=== REQUIRED CONTRACT CELLS ==="
)

for number, found in (
    cell04_required_cells_found.items()
):
    print(
        f"Cell {number}:",
        "FOUND"
        if found
        else "MISSING",
    )


print(
    "\n=== VARIABLE OCCURRENCES ==="
)

for name, hits in (
    cell04_variable_occurrences.items()
):
    print(
        name,
        ":",
        hits,
    )


print(
    "\n=== SERIALIZED RUNTIME STATE ==="
)

for key, cells in (
    cell04_serialized_output_candidates.items()
):
    print(
        key,
        ":",
        cells,
    )

print(
    "Complete runtime state serialized:",
    cell04_complete_runtime_state_serialized,
)


print(
    "\n=== DISK ARTIFACT ANCHOR HITS ==="
)

print(
    "Anchor hits:",
    len(
        cell04_disk_anchor_hits
    ),
)

for row in (
    cell04_disk_anchor_hits
):
    print(
        row
    )

print(
    "Current full-state artifact candidates:",
    len(
        cell04_current_state_artifacts
    ),
)


print(
    "\n=== CONTRACT COVERAGE ==="
)

for name, covered in (
    cell04_core_contract_coverage.items()
):
    print(
        name,
        ":",
        "PASS"
        if covered
        else "FAIL",
    )

print(
    "Contract coverage complete:",
    cell04_contract_coverage_complete,
)


print(
    "\n=== RECONSTRUCTION DECISION ==="
)

print(
    "Route:",
    cell04_reconstruction_route,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 04 GATE:",
    "PASS"
    if cell04_gate
    else "FAIL",
)


assert cell04_gate, (
    "Compact runtime seed reconstruction discovery failed. "
    "Repair this SAME Code Cell 04 before building the seed."
)

=== COMPACT RUNTIME SEED RECONSTRUCTION DISCOVERY ===

=== FINAL STATE ANCHORS ===
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Checkpoint SHA256: a0d25d8eef2f7a4b76ce24d52e9341ddae5d4553af6207288f82e35eb9fe9f2c

=== REQUIRED CONTRACT CELLS ===
Cell 29: FOUND
Cell 74: FOUND
Cell 95: FOUND
Cell 125: FOUND
Cell 130: FOUND
Cell 135: FOUND
Cell 137: FOUND
Cell 139: FOUND
Cell 140: FOUND

=== VARIABLE OCCURRENCES ===
question_drafting_pilot_packets : [{'cell': 29, 'source': True, 'output': False}, {'cell': 30, 'source': True, 'output': False}, {'cell': 46, 'source': False, 'output': True}, {'cell': 47, 'source': True, 'output': True}, {'cell': 63, 'source': False, 'output': True}, {'cell': 65, 'source': False, 'output': True}, {'cell': 66, 'source': True, 'output': True}, {'cell': 67, 'source': False, 'output': True}, {'cell': 68, 'source': True, 'output': True}, {'cell': 69, 'source': True, 'output': True}, {'cell': 70, 'source': True, 'output': True}, {'cell': 71, 'source':

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Dependency Slicing برای ساخت Compact Runtime Seed</h3>

<p>
مسیر بازسازی در Step قبل مشخص شد، اما قرار نیست Cellهای قدیمی یکی‌یکی Replay شوند.
در این مرحله Source Notebook قبلی با AST بررسی می‌شود تا فقط Statementهایی که واقعاً برای ساخت State نهایی لازم‌اند شناسایی شوند.
</p>

<p>
تمرکز روی پنج Contract اصلی است:
</p>

<ul>
<li>ساخت Pilot Population؛</li>
<li>ساخت Remaining Queue و Workflow State؛</li>
<li>ساخت Replacement Pilot Packet؛</li>
<li>ساخت Replacement Queue؛</li>
<li>ساخت Cell 95 Derived State بعد از Replacement.</li>
</ul>

<p>
برای هر Contract، Dependencyهای واقعی و آخرین Cell تأمین‌کننده آن‌ها استخراج می‌شود.
این کار اجازه می‌دهد در Step بعد یک Builder کوچک و مستقل بسازیم و از Replay کردن تاریخچه توسعه‌ای Notebook قبلی جلوگیری کنیم.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Contract Cellهای موردنیاز Parse شوند.</li>
<li>Target Variableهای اصلی در Source واقعی پیدا شوند.</li>
<li>Dependencyهای خارجی هر Target شناسایی شوند.</li>
<li>برای Dependencyهای پروژه‌ای، Provider Cell واقعی قابل‌شناسایی باشد.</li>
<li>Minimal Provider Set برای بازسازی مشخص شود.</li>
<li>هیچ State یا فایلی تغییر نکند.</li>
</ul>

</div>

In [5]:
# Code Cell 05

ast = __import__("ast")
builtins_module = __import__("builtins")


cell05_contract_targets = {
    29: [
        "pilot_selection_rows",
        "question_drafting_pilot_packets",
        "question_drafting_pilot_by_id",
    ],
    95: [
        "cell95_remaining_pilot_ids",
        "cell95_remaining_pilot_queue",
        "cell95_pilot_workflow_state",
    ],
    130: [
        "cell130_simulated_packet",
    ],
    135: [
        "cell135_simulated_queue",
    ],
    137: [
        "cell137_rebuilt_state",
    ],
}


cell05_failures = []


# ------------------------------------------------------------
# AST binding helpers
# ------------------------------------------------------------

def cell05_store_names(node):
    names = set()

    for child in ast.walk(node):

        if (
            isinstance(child, ast.Name)
            and
            isinstance(child.ctx, ast.Store)
        ):
            names.add(
                child.id
            )

        elif isinstance(child, ast.arg):
            names.add(
                child.arg
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

        elif isinstance(
            child,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        ):
            names.add(
                child.name
            )

        elif isinstance(
            child,
            ast.Import,
        ):
            for alias in child.names:
                names.add(
                    alias.asname
                    or
                    alias.name.split(".")[0]
                )

        elif isinstance(
            child,
            ast.ImportFrom,
        ):
            for alias in child.names:
                names.add(
                    alias.asname
                    or
                    alias.name
                )

    return names


def cell05_load_names(node):

    return {
        child.id

        for child in ast.walk(node)

        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Load,
            )
        )
    }


def cell05_top_level_records(source):

    tree = ast.parse(
        source
    )

    records = []

    for statement in tree.body:

        records.append({
            "node":
                statement,

            "stores":
                cell05_store_names(
                    statement
                ),

            "loads":
                cell05_load_names(
                    statement
                ),

            "lineno":
                getattr(
                    statement,
                    "lineno",
                    None,
                ),

            "end_lineno":
                getattr(
                    statement,
                    "end_lineno",
                    None,
                ),
        })

    return records


def cell05_backward_slice(
    source,
    targets,
):

    records = (
        cell05_top_level_records(
            source
        )
    )

    needed = set(
        targets
    )

    selected = set()

    changed = True

    while changed:

        changed = False

        for index in range(
            len(records) - 1,
            -1,
            -1,
        ):

            record = records[
                index
            ]

            if (
                index not in selected
                and
                record["stores"]
                &
                needed
            ):

                selected.add(
                    index
                )

                needed.update(
                    record["loads"]
                )

                changed = True


    selected_records = [
        records[index]

        for index in sorted(
            selected
        )
    ]


    selected_stores = set()

    selected_loads = set()


    for record in (
        selected_records
    ):

        selected_stores.update(
            record[
                "stores"
            ]
        )

        selected_loads.update(
            record[
                "loads"
            ]
        )


    external_dependencies = (
        selected_loads
        -
        selected_stores
    )


    return {
        "records":
            records,

        "selected_records":
            selected_records,

        "selected_stores":
            selected_stores,

        "selected_loads":
            selected_loads,

        "external_dependencies":
            external_dependencies,
    }


# ------------------------------------------------------------
# Provider index
# ------------------------------------------------------------

cell05_provider_index = defaultdict(
    list
)


cell05_parse_errors = []


for number, record in sorted(
    cell04_code_cell_index.items()
):

    try:

        tree = ast.parse(
            record[
                "source"
            ]
        )

    except SyntaxError as exc:

        cell05_parse_errors.append({
            "cell":
                number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    for statement in tree.body:

        for name in (
            cell05_store_names(
                statement
            )
        ):

            cell05_provider_index[
                name
            ].append(
                number
            )


cell05_builtin_names = set(
    dir(
        builtins_module
    )
)


cell05_known_runtime_names = {
    "Path",
    "Counter",
    "defaultdict",
    "copy",
    "hashlib",
    "json",
    "re",
    "subprocess",
    "ast",
}


# ------------------------------------------------------------
# Contract dependency analysis
# ------------------------------------------------------------

cell05_contract_results = {}


for cell_number, targets in (
    cell05_contract_targets.items()
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )


    if record is None:

        cell05_failures.append({
            "reason":
                "CONTRACT_CELL_MISSING",

            "cell":
                cell_number,
        })

        continue


    try:

        slice_result = (
            cell05_backward_slice(
                record[
                    "source"
                ],
                targets,
            )
        )

    except Exception as exc:

        cell05_failures.append({
            "reason":
                "CONTRACT_SLICE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    unresolved_targets = (
        set(
            targets
        )
        -
        slice_result[
            "selected_stores"
        ]
    )


    if unresolved_targets:

        cell05_failures.append({
            "reason":
                "TARGET_ASSIGNMENT_NOT_FOUND",

            "cell":
                cell_number,

            "targets":
                sorted(
                    unresolved_targets
                ),
        })


    project_dependencies = sorted(

        dependency

        for dependency
        in slice_result[
            "external_dependencies"
        ]

        if (
            dependency
            not in
            cell05_builtin_names

            and

            dependency
            not in
            cell05_known_runtime_names
        )
    )


    provider_map = {}


    for dependency in (
        project_dependencies
    ):

        providers = [
            provider

            for provider
            in cell05_provider_index.get(
                dependency,
                [],
            )

            if provider
            <
            cell_number
        ]


        provider_map[
            dependency
        ] = (
            providers[-1]
            if providers
            else None
        )


    unresolved_dependencies = sorted(

        dependency

        for dependency, provider
        in provider_map.items()

        if provider is None
    )


    selected_statement_ranges = [
        (
            row[
                "lineno"
            ],
            row[
                "end_lineno"
            ],
        )

        for row
        in slice_result[
            "selected_records"
        ]
    ]


    cell05_contract_results[
        cell_number
    ] = {
        "targets":
            list(
                targets
            ),

        "selected_statement_count":
            len(
                slice_result[
                    "selected_records"
                ]
            ),

        "selected_statement_ranges":
            selected_statement_ranges,

        "project_dependencies":
            project_dependencies,

        "provider_map":
            provider_map,

        "unresolved_dependencies":
            unresolved_dependencies,
    }


# ------------------------------------------------------------
# Minimal upstream provider set
# ------------------------------------------------------------

cell05_provider_cells = sorted({

    provider

    for result
    in cell05_contract_results.values()

    for provider
    in result[
        "provider_map"
    ].values()

    if provider is not None
})


cell05_all_targets_resolved = (
    len(
        cell05_failures
    )
    ==
    0
)


cell05_all_project_dependencies_resolved = all(

    len(
        result[
            "unresolved_dependencies"
        ]
    )
    ==
    0

    for result
    in cell05_contract_results.values()
)


cell05_compact_rebuild_route_ready = all([
    cell05_all_targets_resolved,
    cell05_all_project_dependencies_resolved,
])


cell05_rebuild_plan = {
    "strategy":
        "AST_BACKWARD_SLICE_FROM_ACCEPTED_CONTRACTS",

    "contract_cells":
        sorted(
            cell05_contract_targets
        ),

    "provider_cells":
        cell05_provider_cells,

    "replay_full_notebook":
        False,

    "false_positive_fix":
        "FUNCTION_ARGS_AND_EXCEPTION_TARGETS_ARE_LOCAL_BINDINGS",

    "write_files":
        False,

    "ready":
        cell05_compact_rebuild_route_ready,
}


# ------------------------------------------------------------
# Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT CONTRACT DEPENDENCY SLICING ==="
)


print(
    "\nValidator repair:",
    "Function arguments and exception-handler targets "
    "are treated as local lexical bindings.",
)


print(
    "\nPrior notebook parse errors:",
    cell05_parse_errors,
)


for cell_number in sorted(
    cell05_contract_results
):

    result = (
        cell05_contract_results[
            cell_number
        ]
    )


    print(
        f"\n=== CONTRACT CELL {cell_number} ==="
    )


    print(
        "Targets:",
        result[
            "targets"
        ],
    )


    print(
        "Selected statement count:",
        result[
            "selected_statement_count"
        ],
    )


    print(
        "Statement ranges:",
        result[
            "selected_statement_ranges"
        ],
    )


    print(
        "Project dependencies:",
        result[
            "project_dependencies"
        ],
    )


    print(
        "Provider map:",
        result[
            "provider_map"
        ],
    )


    print(
        "Unresolved dependencies:",
        result[
            "unresolved_dependencies"
        ],
    )


print(
    "\n=== MINIMAL PROVIDER SET ==="
)


print(
    "Provider cells:",
    cell05_provider_cells,
)


print(
    "Provider cell count:",
    len(
        cell05_provider_cells
    ),
)


print(
    "Full notebook replay required:",
    False,
)


print(
    "\n=== REBUILD PLAN ==="
)


print(
    json.dumps(
        cell05_rebuild_plan,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )
)


print(
    "\nFailures:",
    cell05_failures,
)


print(
    "All project dependencies resolved:",
    cell05_all_project_dependencies_resolved,
)


print(
    "Compact rebuild route ready:",
    cell05_compact_rebuild_route_ready,
)


print(
    "Mutation performed:",
    False,
)


cell05_gate = (
    cell05_compact_rebuild_route_ready
)


print(
    "\nCODE CELL 05 GATE:",
    "PASS"
    if cell05_gate
    else "FAIL",
)


assert cell05_gate, (
    "Compact contract dependency slicing is incomplete. "
    "Repair this SAME Code Cell 05 before reconstructing the runtime seed."
)

=== COMPACT CONTRACT DEPENDENCY SLICING ===

Validator repair: Function arguments and exception-handler targets are treated as local lexical bindings.

Prior notebook parse errors: []

=== CONTRACT CELL 29 ===
Targets: ['pilot_selection_rows', 'question_drafting_pilot_packets', 'question_drafting_pilot_by_id']
Selected statement count: 3
Statement ranges: [(78, 78), (301, 301), (303, 303)]
Project dependencies: []
Provider map: {}
Unresolved dependencies: []

=== CONTRACT CELL 95 ===
Targets: ['cell95_remaining_pilot_ids', 'cell95_remaining_pilot_queue', 'cell95_pilot_workflow_state']
Selected statement count: 13
Statement ranges: [(99, 107), (164, 168), (241, 320), (323, 327), (334, 338), (341, 348), (438, 449), (541, 541), (544, 606), (609, 613), (623, 635), (638, 649), (656, 693)]
Project dependencies: ['PILOT_QUESTION_AUTHORING_MODE_V1', 'QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1', 'QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1', 'cell94_completed_pilot_qa_record', 'cell94_completed_pilot_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>استخراج Recursive Dependency Closure از Final Replacement State</h3>

<p>
Cell 05 نشان داد Contractهای لازم قابل‌بازیابی هستند.
اما برای ادامه Compact Workflow لازم نیست چند Contract را جداگانه دوباره بسازیم.
در Notebook قبلی، <code>cell138_target_values</code> دقیقاً مجموعه 11 Runtime Object نهایی است که بعداً در Cell 139 روی Live State اعمال شد.
</p>

<p>
بنابراین این مرحله فقط Dependencyهای واقعی لازم برای بازسازی همین Final Target State را به‌صورت Recursive دنبال می‌کند.
هدف این است که کوچک‌ترین مجموعه Provider Cellها و Statementهای لازم مشخص شود و از Replay کامل 140 Cell جلوگیری شود.
</p>

<h4>معیار PASS</h4>

<ul>
<li><code>cell138_target_values</code> در Cell 138 قابل Slice باشد.</li>
<li>تمام Dependencyهای پروژه‌ای آن تا Provider واقعی دنبال شوند.</li>
<li>Dependency حل‌نشده باقی نماند.</li>
<li>Dependency Cycle نامعتبر وجود نداشته باشد.</li>
<li>Full Notebook Replay لازم نباشد.</li>
<li>هیچ فایل یا Runtime State پروژه تغییر نکند.</li>
</ul>

</div>

In [9]:
# Code Cell 06

cell06_root_cell = 138
cell06_root_target = "cell138_target_values"

cell06_failures = []


# ============================================================
# 1) Core AST helpers
# ============================================================

cell06_mutating_methods = {
    "append",
    "extend",
    "insert",
    "update",
    "setdefault",
    "add",
    "remove",
    "discard",
    "pop",
    "popitem",
    "clear",
    "sort",
    "reverse",
}


def cell06_root_name(node):
    if isinstance(node, ast.Name):
        return node.id

    if isinstance(
        node,
        (
            ast.Attribute,
            ast.Subscript,
        ),
    ):
        return cell06_root_name(
            node.value
        )

    return None


def cell06_target_names(node):
    names = set()

    if isinstance(node, ast.Name):
        names.add(
            node.id
        )

    elif isinstance(
        node,
        (
            ast.Tuple,
            ast.List,
        ),
    ):
        for item in node.elts:
            names.update(
                cell06_target_names(
                    item
                )
            )

    elif isinstance(
        node,
        (
            ast.Attribute,
            ast.Subscript,
        ),
    ):
        root = (
            cell06_root_name(
                node
            )
        )

        if root:
            names.add(
                root
            )

    return names


def cell06_loaded_names(node):
    return {
        child.id

        for child
        in ast.walk(
            node
        )

        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Load,
            )
        )
    }


def cell06_internal_bindings(node):
    names = set()

    for child in ast.walk(
        node
    ):
        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Store,
            )
        ):
            names.add(
                child.id
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

    return names


# ============================================================
# 2) Persistent module bindings
# ============================================================

class Cell06ModuleBindingVisitor(
    ast.NodeVisitor
):
    def __init__(self):
        self.names = set()

    def visit_Name(
        self,
        node,
    ):
        if isinstance(
            node.ctx,
            ast.Store,
        ):
            self.names.add(
                node.id
            )

    def visit_FunctionDef(
        self,
        node,
    ):
        self.names.add(
            node.name
        )

        for decorator in (
            node.decorator_list
        ):
            self.visit(
                decorator
            )

        defaults = (
            list(
                node.args.defaults
            )
            +
            [
                value

                for value
                in node.args.kw_defaults

                if value is not None
            ]
        )

        for value in defaults:
            self.visit(
                value
            )

        # Function body has its own scope.

    def visit_AsyncFunctionDef(
        self,
        node,
    ):
        self.visit_FunctionDef(
            node
        )

    def visit_ClassDef(
        self,
        node,
    ):
        self.names.add(
            node.name
        )

        for base in node.bases:
            self.visit(
                base
            )

        for keyword in node.keywords:
            self.visit(
                keyword.value
            )

        for decorator in (
            node.decorator_list
        ):
            self.visit(
                decorator
            )

        # Class body has its own namespace.

    def visit_Lambda(
        self,
        node,
    ):
        # Lambda body has its own argument scope.
        return

    def visit_ListComp(
        self,
        node,
    ):
        return

    def visit_SetComp(
        self,
        node,
    ):
        return

    def visit_DictComp(
        self,
        node,
    ):
        return

    def visit_GeneratorExp(
        self,
        node,
    ):
        return

    def visit_ExceptHandler(
        self,
        node,
    ):
        # Exception targets are deleted after the handler.
        if node.type is not None:
            self.visit(
                node.type
            )

        for statement in (
            node.body
        ):
            self.visit(
                statement
            )

    def visit_Import(
        self,
        node,
    ):
        for alias in node.names:
            self.names.add(
                alias.asname
                or
                alias.name.split(
                    "."
                )[0]
            )

    def visit_ImportFrom(
        self,
        node,
    ):
        for alias in node.names:
            if alias.name == "*":
                continue

            self.names.add(
                alias.asname
                or
                alias.name
            )


def cell06_module_bindings(
    statement,
):
    visitor = (
        Cell06ModuleBindingVisitor()
    )

    visitor.visit(
        statement
    )

    return visitor.names


# ============================================================
# 3) Function free-name analysis
# ============================================================

def cell06_function_local_names(
    node,
):
    names = {
        argument.arg

        for argument
        in (
            list(
                node.args.posonlyargs
            )
            +
            list(
                node.args.args
            )
            +
            list(
                node.args.kwonlyargs
            )
        )
    }

    if node.args.vararg is not None:
        names.add(
            node.args.vararg.arg
        )

    if node.args.kwarg is not None:
        names.add(
            node.args.kwarg.arg
        )

    for child in ast.walk(
        node
    ):
        if (
            isinstance(
                child,
                ast.Name,
            )
            and
            isinstance(
                child.ctx,
                ast.Store,
            )
        ):
            names.add(
                child.id
            )

        elif (
            child is not node
            and
            isinstance(
                child,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                ),
            )
        ):
            names.add(
                child.name
            )

        elif isinstance(
            child,
            ast.ExceptHandler,
        ):
            if isinstance(
                child.name,
                str,
            ):
                names.add(
                    child.name
                )

        elif isinstance(
            child,
            ast.arg,
        ):
            # Covers nested lambda/function parameters.
            names.add(
                child.arg
            )

    return names


def cell06_function_external_reads(
    node,
):
    reads = (
        cell06_loaded_names(
            node
        )
        -
        cell06_function_local_names(
            node
        )
    )

    reads.discard(
        node.name
    )

    return reads


# ============================================================
# 4) Statement effects
# ============================================================

def cell06_statement_effects(
    statement,
):
    reads = set()
    writes = set()
    definite_writes = set()
    mutations = set()
    provider_writes = set()

    if isinstance(
        statement,
        (
            ast.FunctionDef,
            ast.AsyncFunctionDef,
        ),
    ):
        writes.add(
            statement.name
        )
        definite_writes.add(
            statement.name
        )
        provider_writes.add(
            statement.name
        )

        reads.update(
            cell06_function_external_reads(
                statement
            )
        )

        return {
            "reads": reads,
            "writes": writes,
            "definite_writes":
                definite_writes,
            "mutations":
                mutations,
            "provider_writes":
                provider_writes,
        }

    if isinstance(
        statement,
        ast.ClassDef,
    ):
        writes.add(
            statement.name
        )
        definite_writes.add(
            statement.name
        )
        provider_writes.add(
            statement.name
        )

        for base in statement.bases:
            reads.update(
                cell06_loaded_names(
                    base
                )
            )

        for keyword in (
            statement.keywords
        ):
            reads.update(
                cell06_loaded_names(
                    keyword.value
                )
            )

        for decorator in (
            statement.decorator_list
        ):
            reads.update(
                cell06_loaded_names(
                    decorator
                )
            )

        return {
            "reads": reads,
            "writes": writes,
            "definite_writes":
                definite_writes,
            "mutations":
                mutations,
            "provider_writes":
                provider_writes,
        }

    internal_bindings = (
        cell06_internal_bindings(
            statement
        )
    )

    module_bindings = (
        cell06_module_bindings(
            statement
        )
    )

    writes.update(
        module_bindings
    )

    definite_writes.update(
        module_bindings
    )

    provider_writes.update(
        module_bindings
    )

    raw_mutations = set()

    for child in ast.walk(
        statement
    ):
        if (
            child is not statement
            and
            isinstance(
                child,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                    ast.ClassDef,
                    ast.Lambda,
                ),
            )
        ):
            continue

        if (
            isinstance(
                child,
                ast.Call,
            )
            and
            isinstance(
                child.func,
                ast.Attribute,
            )
            and
            child.func.attr
            in
            cell06_mutating_methods
        ):
            receiver = (
                cell06_root_name(
                    child.func.value
                )
            )

            if receiver:
                raw_mutations.add(
                    receiver
                )

        elif isinstance(
            child,
            ast.AugAssign,
        ):
            raw_mutations.update(
                cell06_target_names(
                    child.target
                )
            )

        elif isinstance(
            child,
            ast.Assign,
        ):
            for target in (
                child.targets
            ):
                if isinstance(
                    target,
                    (
                        ast.Attribute,
                        ast.Subscript,
                    ),
                ):
                    root = (
                        cell06_root_name(
                            target
                        )
                    )

                    if root:
                        raw_mutations.add(
                            root
                        )

        elif isinstance(
            child,
            ast.AnnAssign,
        ):
            if isinstance(
                child.target,
                (
                    ast.Attribute,
                    ast.Subscript,
                ),
            ):
                root = (
                    cell06_root_name(
                        child.target
                    )
                )

                if root:
                    raw_mutations.add(
                        root
                    )

    external_mutations = (
        raw_mutations
        -
        module_bindings
    )

    mutations.update(
        external_mutations
    )

    writes.update(
        external_mutations
    )

    reads.update(
        cell06_loaded_names(
            statement
        )
        -
        internal_bindings
    )

    self_reads = set()

    for child in ast.walk(
        statement
    ):
        if isinstance(
            child,
            ast.AugAssign,
        ):
            self_reads.update(
                cell06_target_names(
                    child.target
                )
            )

        elif isinstance(
            child,
            ast.Assign,
        ):
            target_names = set()

            for target in (
                child.targets
            ):
                target_names.update(
                    cell06_target_names(
                        target
                    )
                )

            self_reads.update(
                target_names
                &
                cell06_loaded_names(
                    child.value
                )
            )

        elif (
            isinstance(
                child,
                ast.AnnAssign,
            )
            and
            child.value is not None
        ):
            target_names = (
                cell06_target_names(
                    child.target
                )
            )

            self_reads.update(
                target_names
                &
                cell06_loaded_names(
                    child.value
                )
            )

    reads.update(
        self_reads
    )

    reads.update(
        external_mutations
    )

    return {
        "reads": reads,
        "writes": writes,
        "definite_writes":
            definite_writes,
        "mutations":
            mutations,
        "provider_writes":
            provider_writes,
    }


# ============================================================
# 5) Parse prior Notebook
# ============================================================

cell06_cell_models = {}

for cell_number, record in sorted(
    cell04_code_cell_index.items()
):
    source = record[
        "source"
    ]

    try:
        tree = ast.parse(
            source
        )

    except SyntaxError as exc:
        cell06_failures.append({
            "reason":
                "SOURCE_PARSE_FAILED",
            "cell":
                cell_number,
            "error":
                repr(
                    exc
                ),
        })
        continue

    statements = []

    for index, statement in enumerate(
        tree.body,
        start=1,
    ):
        effects = (
            cell06_statement_effects(
                statement
            )
        )

        statements.append({
            "index":
                index,
            "node":
                statement,
            "range":
                (
                    getattr(
                        statement,
                        "lineno",
                        None,
                    ),
                    getattr(
                        statement,
                        "end_lineno",
                        None,
                    ),
                ),
            **effects,
        })

    cell06_cell_models[
        cell_number
    ] = {
        "source":
            source,
        "statements":
            statements,
    }


# ============================================================
# 6) Cross-cell provider index
# ============================================================

cell06_provider_index = defaultdict(
    list
)

for cell_number, model in (
    cell06_cell_models.items()
):
    names = set()

    for statement in (
        model[
            "statements"
        ]
    ):
        names.update(
            statement[
                "provider_writes"
            ]
        )

    for name in names:
        cell06_provider_index[
            name
        ].append(
            cell_number
        )

for name in list(
    cell06_provider_index
):
    cell06_provider_index[
        name
    ] = sorted(
        set(
            cell06_provider_index[
                name
            ]
        )
    )


# ============================================================
# 7) Runtime-preseed names
# ============================================================

if isinstance(
    __builtins__,
    dict,
):
    cell06_builtin_names = set(
        __builtins__.keys()
    )

else:
    cell06_builtin_names = set(
        dir(
            __builtins__
        )
    )

cell06_builtin_names.update({
    "__name__",
    "__file__",
    "__package__",
    "__spec__",
})

cell06_preseed_runtime_names = {
    "Path",
    "Counter",
    "defaultdict",
    "copy",
    "hashlib",
    "json",
    "re",
    "subprocess",
    "ast",
}


# ============================================================
# 8) Backward slice within one Cell
# ============================================================

def cell06_slice_cell(
    cell_number,
    target_names,
):
    model = (
        cell06_cell_models[
            cell_number
        ]
    )

    needed = set(
        target_names
    )

    selected = []

    for statement in reversed(
        model[
            "statements"
        ]
    ):
        affected = (
            statement[
                "writes"
            ]
            |
            statement[
                "mutations"
            ]
        )

        if not (
            affected
            &
            needed
        ):
            continue

        selected.append(
            statement
        )

        needed -= (
            statement[
                "definite_writes"
            ]
        )

        needed.update(
            statement[
                "reads"
            ]
        )

        needed.update(
            statement[
                "mutations"
            ]
        )

    selected.reverse()

    return {
        "targets":
            sorted(
                set(
                    target_names
                )
            ),
        "selected_statements":
            selected,
        "statement_ranges":
            [
                statement[
                    "range"
                ]
                for statement
                in selected
            ],
        "selected_statement_count":
            len(
                selected
            ),
        "external_dependencies":
            sorted(
                name
                for name
                in needed
                if (
                    name
                    not in
                    cell06_builtin_names
                    and
                    name
                    not in
                    cell06_preseed_runtime_names
                )
            ),
    }


# ============================================================
# 9) Latest prior provider
# ============================================================

def cell06_latest_provider(
    name,
    consumer_cell,
):
    candidates = [
        provider_cell
        for provider_cell
        in cell06_provider_index.get(
            name,
            []
        )
        if provider_cell
        <
        consumer_cell
    ]

    if not candidates:
        return None

    return max(
        candidates
    )


# ============================================================
# 10) Recursive closure
# ============================================================

cell06_requested_targets = defaultdict(
    set
)

cell06_requested_targets[
    cell06_root_cell
].add(
    cell06_root_target
)

cell06_cell_slices = {}
cell06_resolution_log = []

cell06_changed = True
cell06_iteration = 0
cell06_max_iterations = 1000

while cell06_changed:
    cell06_iteration += 1

    if (
        cell06_iteration
        >
        cell06_max_iterations
    ):
        cell06_failures.append({
            "reason":
                "CLOSURE_ITERATION_LIMIT_EXCEEDED",
        })
        break

    cell06_changed = False

    for cell_number in sorted(
        list(
            cell06_requested_targets
        )
    ):
        targets = set(
            cell06_requested_targets[
                cell_number
            ]
        )

        new_slice = (
            cell06_slice_cell(
                cell_number,
                targets,
            )
        )

        old_signature = None

        if (
            cell_number
            in
            cell06_cell_slices
        ):
            previous = (
                cell06_cell_slices[
                    cell_number
                ]
            )

            old_signature = (
                tuple(
                    previous[
                        "targets"
                    ]
                ),
                tuple(
                    previous[
                        "statement_ranges"
                    ]
                ),
                tuple(
                    previous[
                        "external_dependencies"
                    ]
                ),
            )

        new_signature = (
            tuple(
                new_slice[
                    "targets"
                ]
            ),
            tuple(
                new_slice[
                    "statement_ranges"
                ]
            ),
            tuple(
                new_slice[
                    "external_dependencies"
                ]
            ),
        )

        if (
            old_signature
            !=
            new_signature
        ):
            cell06_cell_slices[
                cell_number
            ] = (
                new_slice
            )
            cell06_changed = True

        for dependency in (
            new_slice[
                "external_dependencies"
            ]
        ):
            provider = (
                cell06_latest_provider(
                    dependency,
                    cell_number,
                )
            )

            if provider is None:
                continue

            before_count = len(
                cell06_requested_targets[
                    provider
                ]
            )

            cell06_requested_targets[
                provider
            ].add(
                dependency
            )

            if (
                len(
                    cell06_requested_targets[
                        provider
                    ]
                )
                >
                before_count
            ):
                cell06_resolution_log.append({
                    "consumer_cell":
                        cell_number,
                    "dependency":
                        dependency,
                    "provider_cell":
                        provider,
                })

                cell06_changed = True


# ============================================================
# 11) Final recomputation
# ============================================================

for cell_number in sorted(
    cell06_requested_targets
):
    cell06_cell_slices[
        cell_number
    ] = cell06_slice_cell(
        cell_number,
        cell06_requested_targets[
            cell_number
        ],
    )

cell06_provider_cells = sorted(
    cell06_cell_slices
)

cell06_total_selected_statements = sum(
    item[
        "selected_statement_count"
    ]
    for item
    in cell06_cell_slices.values()
)


# ============================================================
# 12) Raw unresolved dependencies
# ============================================================

cell06_raw_unresolved = []

for cell_number in (
    cell06_provider_cells
):
    for dependency in (
        cell06_cell_slices[
            cell_number
        ][
            "external_dependencies"
        ]
    ):
        provider = (
            cell06_latest_provider(
                dependency,
                cell_number,
            )
        )

        if provider is None:
            cell06_raw_unresolved.append({
                "consumer_cell":
                    cell_number,
                "dependency":
                    dependency,
            })

cell06_raw_unresolved = sorted(
    cell06_raw_unresolved,
    key=lambda row: (
        row[
            "consumer_cell"
        ],
        row[
            "dependency"
        ],
    ),
)


# ============================================================
# 13) Safe-unresolved classification helpers
# ============================================================

def cell06_parent_map(
    node,
):
    mapping = {}

    for parent in ast.walk(
        node
    ):
        for child in ast.iter_child_nodes(
            parent
        ):
            mapping[
                child
            ] = parent

    return mapping


def cell06_function_scope_names(
    node,
):
    names = (
        cell06_function_local_names(
            node
        )
    )

    return names


def cell06_scope_binds_name(
    node,
    dependency,
    parent_map,
):
    current = node

    while current in parent_map:
        current = (
            parent_map[
                current
            ]
        )

        if isinstance(
            current,
            ast.Lambda,
        ):
            lambda_names = {
                arg.arg
                for arg
                in (
                    list(
                        current.args.posonlyargs
                    )
                    +
                    list(
                        current.args.args
                    )
                    +
                    list(
                        current.args.kwonlyargs
                    )
                )
            }

            if (
                current.args.vararg
                is not None
            ):
                lambda_names.add(
                    current.args.vararg.arg
                )

            if (
                current.args.kwarg
                is not None
            ):
                lambda_names.add(
                    current.args.kwarg.arg
                )

            if dependency in (
                lambda_names
            ):
                return True

        elif isinstance(
            current,
            (
                ast.ListComp,
                ast.SetComp,
                ast.DictComp,
                ast.GeneratorExp,
            ),
        ):
            names = set()

            for generator in (
                current.generators
            ):
                names.update(
                    cell06_target_names(
                        generator.target
                    )
                )

            if dependency in names:
                return True

        elif isinstance(
            current,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            if dependency in (
                cell06_function_scope_names(
                    current
                )
            ):
                return True

        elif isinstance(
            current,
            ast.ExceptHandler,
        ):
            if (
                isinstance(
                    current.name,
                    str,
                )
                and
                current.name
                ==
                dependency
            ):
                return True

    return False


def cell06_node_inside(
    node,
    subtree,
):
    return any(
        child is node
        for child
        in ast.walk(
            subtree
        )
    )


def cell06_is_dead_constant_branch(
    node,
    parent_map,
):
    current = node

    while current in parent_map:
        current = (
            parent_map[
                current
            ]
        )

        if isinstance(
            current,
            ast.IfExp,
        ):
            if isinstance(
                current.test,
                ast.Constant,
            ):
                condition = bool(
                    current.test.value
                )

                if (
                    condition is False
                    and
                    cell06_node_inside(
                        node,
                        current.body,
                    )
                ):
                    return True

                if (
                    condition is True
                    and
                    cell06_node_inside(
                        node,
                        current.orelse,
                    )
                ):
                    return True

        elif isinstance(
            current,
            ast.If,
        ):
            if isinstance(
                current.test,
                ast.Constant,
            ):
                condition = bool(
                    current.test.value
                )

                if condition is False:
                    if any(
                        cell06_node_inside(
                            node,
                            statement,
                        )
                        for statement
                        in current.body
                    ):
                        return True

                if condition is True:
                    if any(
                        cell06_node_inside(
                            node,
                            statement,
                        )
                        for statement
                        in current.orelse
                    ):
                        return True

    return False


def cell06_same_statement_initialization(
    cell_number,
    dependency,
):
    selected = (
        cell06_cell_slices[
            cell_number
        ][
            "selected_statements"
        ]
    )

    for statement in selected:
        if dependency not in (
            statement[
                "provider_writes"
            ]
        ):
            continue

        store_lines = []
        load_lines = []

        for child in ast.walk(
            statement[
                "node"
            ]
        ):
            if not (
                isinstance(
                    child,
                    ast.Name,
                )
                and
                child.id
                ==
                dependency
            ):
                continue

            if isinstance(
                child.ctx,
                ast.Store,
            ):
                store_lines.append(
                    getattr(
                        child,
                        "lineno",
                        10**12,
                    )
                )

            elif isinstance(
                child.ctx,
                ast.Load,
            ):
                load_lines.append(
                    getattr(
                        child,
                        "lineno",
                        -1,
                    )
                )

        if (
            store_lines
            and
            load_lines
            and
            min(
                store_lines
            )
            <
            min(
                load_lines
            )
        ):
            return {
                "statement_range":
                    statement[
                        "range"
                    ],
                "first_store_line":
                    min(
                        store_lines
                    ),
                "first_load_line":
                    min(
                        load_lines
                    ),
            }

    return None


def cell06_selected_load_occurrences(
    cell_number,
    dependency,
):
    occurrences = []

    for statement in (
        cell06_cell_slices[
            cell_number
        ][
            "selected_statements"
        ]
    ):
        node = (
            statement[
                "node"
            ]
        )

        parent_map = (
            cell06_parent_map(
                node
            )
        )

        for child in ast.walk(
            node
        ):
            if not (
                isinstance(
                    child,
                    ast.Name,
                )
                and
                isinstance(
                    child.ctx,
                    ast.Load,
                )
                and
                child.id
                ==
                dependency
            ):
                continue

            occurrences.append({
                "node":
                    child,
                "line":
                    getattr(
                        child,
                        "lineno",
                        None,
                    ),
                "scope_bound":
                    cell06_scope_binds_name(
                        child,
                        dependency,
                        parent_map,
                    ),
                "dead_constant_branch":
                    cell06_is_dead_constant_branch(
                        child,
                        parent_map,
                    ),
            })

    return occurrences


# ============================================================
# 14) Classify unresolved dependencies
# ============================================================

cell06_safe_unresolved = []
cell06_blocking_unresolved = []

for row in (
    cell06_raw_unresolved
):
    consumer_cell = (
        row[
            "consumer_cell"
        ]
    )

    dependency = (
        row[
            "dependency"
        ]
    )

    same_statement = (
        cell06_same_statement_initialization(
            consumer_cell,
            dependency,
        )
    )

    occurrences = (
        cell06_selected_load_occurrences(
            consumer_cell,
            dependency,
        )
    )

    load_scope_safe = (
        bool(
            occurrences
        )
        and
        all(
            occurrence[
                "scope_bound"
            ]
            for occurrence
            in occurrences
        )
    )

    load_dead_safe = (
        bool(
            occurrences
        )
        and
        all(
            (
                occurrence[
                    "scope_bound"
                ]
                or
                occurrence[
                    "dead_constant_branch"
                ]
            )
            for occurrence
            in occurrences
        )
    )

    if (
        same_statement
        is not None
    ):
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "SAME_STATEMENT_INITIALIZATION_PRECEDES_LOAD",
            "evidence":
                same_statement,
        })

    elif load_scope_safe:
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "LEXICALLY_BOUND_LOCAL_NAME",
            "evidence":
                {
                    "load_lines":
                        [
                            occurrence[
                                "line"
                            ]
                            for occurrence
                            in occurrences
                        ],
                },
        })

    elif load_dead_safe:
        cell06_safe_unresolved.append({
            **row,
            "classification":
                "ONLY_SCOPE_BOUND_OR_DEAD_CONSTANT_BRANCH",
            "evidence":
                {
                    "load_lines":
                        [
                            occurrence[
                                "line"
                            ]
                            for occurrence
                            in occurrences
                        ],
                },
        })

    else:
        cell06_blocking_unresolved.append({
            **row,
            "occurrences":
                [
                    {
                        "line":
                            occurrence[
                                "line"
                            ],
                        "scope_bound":
                            occurrence[
                                "scope_bound"
                            ],
                        "dead_constant_branch":
                            occurrence[
                                "dead_constant_branch"
                            ],
                    }
                    for occurrence
                    in occurrences
                ],
        })


# ============================================================
# 15) Cell 8 mutation regression
# ============================================================

cell06_cell8_ranges = set(
    cell06_cell_slices.get(
        8,
        {},
    ).get(
        "statement_ranges",
        [],
    )
)

cell06_cell8_mutation_loop_included = all([
    (81, 81)
    in
    cell06_cell8_ranges,

    (83, 91)
    in
    cell06_cell8_ranges,

    (97, 102)
    in
    cell06_cell8_ranges,
])


# ============================================================
# 16) Compactness / final gate
# ============================================================

cell06_prior_notebook_code_cell_count = (
    len(
        cell04_code_cell_index
    )
)

cell06_full_notebook_replay = (
    len(
        cell06_provider_cells
    )
    ==
    cell06_prior_notebook_code_cell_count
)

cell06_compact_ratio = (
    len(
        cell06_provider_cells
    )
    /
    cell06_prior_notebook_code_cell_count
)

cell06_root_present = (
    cell06_root_cell
    in
    cell06_provider_cells
    and
    cell06_root_target
    in
    cell06_cell_slices[
        cell06_root_cell
    ][
        "targets"
    ]
)

cell06_gate = all([
    len(
        cell06_failures
    )
    ==
    0,

    len(
        cell06_blocking_unresolved
    )
    ==
    0,

    cell06_root_present,

    cell06_cell8_mutation_loop_included,

    not cell06_full_notebook_replay,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== FINAL STATE RECURSIVE DEPENDENCY CLOSURE "
    "— EXECUTION-SEMANTICS AWARE ==="
)

print(
    "\nRoot:",
    f"Cell {cell06_root_cell} -> "
    f"{cell06_root_target}",
)

print(
    "\nProvider cells:",
    cell06_provider_cells,
)

print(
    "Provider cell count:",
    len(
        cell06_provider_cells
    ),
)

print(
    "Prior notebook code-cell count:",
    cell06_prior_notebook_code_cell_count,
)

print(
    "Compact provider ratio:",
    round(
        cell06_compact_ratio,
        4,
    ),
)

print(
    "Selected top-level statements:",
    cell06_total_selected_statements,
)

print(
    "Closure iterations:",
    cell06_iteration,
)

print(
    "Full notebook replay required:",
    cell06_full_notebook_replay,
)


print(
    "\n=== CELL 8 MUTATION REGRESSION ==="
)

print(
    "Initialization included:",
    (81, 81)
    in
    cell06_cell8_ranges,
)

print(
    "Mutation loop included:",
    (83, 91)
    in
    cell06_cell8_ranges,
)

print(
    "Consumer included:",
    (97, 102)
    in
    cell06_cell8_ranges,
)

print(
    "Cell 8 regression:",
    (
        "PASS"
        if cell06_cell8_mutation_loop_included
        else "FAIL"
    ),
)


print(
    "\n=== RAW UNRESOLVED DEPENDENCIES ==="
)

print(
    "Count:",
    len(
        cell06_raw_unresolved
    ),
)

for row in (
    cell06_raw_unresolved
):
    print(
        row
    )


print(
    "\n=== SAFELY CLASSIFIED UNRESOLVED ==="
)

print(
    "Count:",
    len(
        cell06_safe_unresolved
    ),
)

for row in (
    cell06_safe_unresolved
):
    print(
        row
    )


print(
    "\n=== BLOCKING UNRESOLVED ==="
)

print(
    "Count:",
    len(
        cell06_blocking_unresolved
    ),
)

for row in (
    cell06_blocking_unresolved
):
    print(
        row
    )


print(
    "\n=== FAILURES ==="
)

print(
    cell06_failures
)


cell06_compact_seed_build_plan = {
    "root_cell":
        cell06_root_cell,

    "root_target":
        cell06_root_target,

    "provider_cell_count":
        len(
            cell06_provider_cells
        ),

    "provider_cells":
        cell06_provider_cells,

    "provider_ratio":
        cell06_compact_ratio,

    "selected_statement_count":
        cell06_total_selected_statements,

    "raw_unresolved_dependency_count":
        len(
            cell06_raw_unresolved
        ),

    "safe_unresolved_dependency_count":
        len(
            cell06_safe_unresolved
        ),

    "blocking_unresolved_dependency_count":
        len(
            cell06_blocking_unresolved
        ),

    "full_notebook_replay":
        cell06_full_notebook_replay,

    "mutation_aware_slicing":
        True,

    "execution_semantics_aware":
        True,

    "same_statement_initialization_aware":
        True,

    "lexical_scope_aware":
        True,

    "constant_dead_branch_aware":
        True,

    "cell8_mutation_regression_pass":
        cell06_cell8_mutation_loop_included,

    "ready_for_isolated_seed_rebuild":
        cell06_gate,

    "write_files":
        False,
}

print(
    "\n=== COMPACT SEED BUILD PLAN ==="
)

print(
    json.dumps(
        cell06_compact_seed_build_plan,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "\nMutation performed:",
    False,
)

print(
    "\nCODE CELL 06 GATE:",
    "PASS"
    if cell06_gate
    else "FAIL",
)

assert cell06_gate, (
    "Execution-semantics-aware dependency closure failed. "
    "Repair this SAME Code Cell 06 before continuing."
)

=== FINAL STATE RECURSIVE DEPENDENCY CLOSURE — EXECUTION-SEMANTICS AWARE ===

Root: Cell 138 -> cell138_target_values

Provider cells: [1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 51, 52, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 124, 125, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138]
Provider cell count: 98
Prior notebook code-cell count: 141
Compact provider ratio: 0.695
Selected top-level statements: 2004
Closure iterations: 14
Full notebook replay required: False

=== CELL 8 MUTATION REGRESSION ===
Initialization included: True
Mutation loop included: True
Consumer included: True
Cell 8 regression: PASS

=== RAW UNRESOLVED DEPENDENCIES ===
Count: 9
{'consumer_cell': 4, 'dependency': 'x'}
{'consumer_cell': 19, 'dependency': 'identity_counte

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>بازیابی مستقیم Pilot Population از Outputهای پذیرفته‌شده Notebook قبلی</h3>

<p>
Dependency Closure در Step قبل نشان داد که بازسازی کامل Final Runtime State از Source Notebook ممکن است،
اما به 88 Provider Cell وابسته می‌شود. این مسیر از نظر فنی معتبر است، ولی برای هدف Compact Notebook هنوز بیش از حد سنگین است.
</p>

<p>
قبل از استفاده از آن مسیر، این Step یک مسیر کوتاه‌تر را بررسی می‌کند:
در Notebook قبلی بعضی Cellها Pilot Population را در Output ذخیره کرده‌اند.
اگر یک Snapshot کامل و قابل‌Parse از 23 Pilot قبل از Replacement موجود باشد،
می‌توان Population را مستقیماً از همان Output پذیرفته‌شده بازیابی کرد،
Replacement نهایی را روی همان Snapshot اعمال کرد و Queue جدید را با Contract تثبیت‌شده بازسازی کرد.
</p>

<p>
این روش می‌تواند نیاز به Replay بخش بزرگی از Notebook قبلی را حذف کند.
در این مرحله فقط Outputهای ذخیره‌شده و Read-Only بررسی می‌شوند و هیچ State یا فایلی تغییر نمی‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>Outputهای Candidate Cellهای مرتبط با Pilot Population بررسی شوند.</li>
<li>حداقل یک Snapshot ساختاریافته و قابل‌Parse از Pilot Population پیدا شود، یا نبود آن به‌طور قطعی مشخص شود.</li>
<li>تعداد Pilotها و یکتایی <code>pilot_case_id</code> قابل‌اعتبارسنجی باشد.</li>
<li>وجود Pilot قدیمی یا Replacement در Snapshot مشخص شود.</li>
<li>مسیر بعدی بین Direct Snapshot Recovery و Dependency Replay بدون حدس انتخاب شود.</li>
<li>هیچ Mutation انجام نشود.</li>
</ul>

</div>

In [10]:
# Code Cell 07

cell07_candidate_cells = [
    68,
    79,
    130,
    132,
    135,
    137,
    139,
    140,
]

cell07_failures = []


# ------------------------------------------------------------
# 1) Resolve Pilot identities from accepted notebook outputs
# ------------------------------------------------------------

cell07_new_pilot_id = (
    cell03_next_pilot_id
)


cell07_identity_scan_cells = [
    96,
    97,
    98,
    99,
    100,
    130,
    132,
    136,
    138,
    139,
    140,
]


cell07_pilot_id_pattern = re.compile(
    r"question-pilot:[0-9a-f]{32}"
)


cell07_identity_hits = {}


for cell_number in (
    cell07_identity_scan_cells
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:
        continue

    output_text = (
        record.get(
            "output"
        )
        or ""
    )

    ids = (
        cell07_pilot_id_pattern.findall(
            output_text
        )
    )

    cell07_identity_hits[
        cell_number
    ] = ids


# The blocked Pilot was the active Pilot throughout Cells 96-100.
cell07_pre_replacement_ids = []


for cell_number in [
    96,
    97,
    98,
    99,
    100,
]:
    cell07_pre_replacement_ids.extend(
        cell07_identity_hits.get(
            cell_number,
            []
        )
    )


cell07_pre_replacement_counts = Counter(
    pilot_id

    for pilot_id
    in cell07_pre_replacement_ids

    if pilot_id
    !=
    cell07_new_pilot_id
)


cell07_ranked_old_candidates = (
    cell07_pre_replacement_counts.most_common()
)


cell07_old_pilot_id = None


if cell07_ranked_old_candidates:

    top_count = (
        cell07_ranked_old_candidates[
            0
        ][
            1
        ]
    )

    top_ids = [
        pilot_id

        for pilot_id, count
        in cell07_ranked_old_candidates

        if count == top_count
    ]

    if len(top_ids) == 1:
        cell07_old_pilot_id = (
            top_ids[0]
        )


# Secondary evidence from explicit labels in later outputs.
cell07_labeled_old_ids = []


cell07_old_label_patterns = [
    re.compile(
        r"Old Pilot(?: Case ID| ID)?\s*:\s*(question-pilot:[0-9a-f]{32})",
        flags=re.IGNORECASE,
    ),
    re.compile(
        r"old_pilot(?:_case)?_id['\"]?\s*[:=]\s*['\"]?(question-pilot:[0-9a-f]{32})",
        flags=re.IGNORECASE,
    ),
]


for cell_number in [
    132,
    136,
    138,
    139,
]:

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:
        continue

    combined_text = (
        record.get(
            "source",
            ""
        )
        +
        "\n"
        +
        (
            record.get(
                "output"
            )
            or ""
        )
    )

    for pattern in (
        cell07_old_label_patterns
    ):
        cell07_labeled_old_ids.extend(
            pattern.findall(
                combined_text
            )
        )


cell07_labeled_old_ids = sorted(
    set(
        cell07_labeled_old_ids
    )
)


if (
    cell07_old_pilot_id
    is None
    and
    len(
        cell07_labeled_old_ids
    )
    ==
    1
):
    cell07_old_pilot_id = (
        cell07_labeled_old_ids[
            0
        ]
    )


cell07_identity_resolution_valid = (
    isinstance(
        cell07_old_pilot_id,
        str,
    )
    and
    cell07_old_pilot_id.startswith(
        "question-pilot:"
    )
    and
    isinstance(
        cell07_new_pilot_id,
        str,
    )
    and
    cell07_new_pilot_id.startswith(
        "question-pilot:"
    )
    and
    cell07_old_pilot_id
    !=
    cell07_new_pilot_id
)


if not cell07_identity_resolution_valid:

    cell07_failures.append({
        "reason":
            "PILOT_IDENTITY_RESOLUTION_FAILED",

        "old_candidate_counts":
            dict(
                cell07_pre_replacement_counts
            ),

        "labeled_old_ids":
            cell07_labeled_old_ids,

        "new_pilot_id":
            cell07_new_pilot_id,
    })


# ------------------------------------------------------------
# 2) Parse structured objects from saved outputs
# ------------------------------------------------------------

def cell07_balanced_fragments(
    text,
    opener,
    closer,
):
    fragments = []

    start = None
    depth = 0
    quote = None
    escaped = False

    for index, char in enumerate(
        text
    ):

        if quote is not None:

            if escaped:
                escaped = False
                continue

            if char == "\\":
                escaped = True
                continue

            if char == quote:
                quote = None

            continue

        if char in (
            "'",
            '"',
        ):
            quote = char
            continue

        if char == opener:

            if depth == 0:
                start = index

            depth += 1

        elif (
            char == closer
            and
            depth > 0
        ):

            depth -= 1

            if (
                depth == 0
                and
                start is not None
            ):
                fragments.append(
                    text[
                        start:index + 1
                    ]
                )

                start = None

    return fragments


def cell07_collect_objects(
    text,
):
    fragments = (
        cell07_balanced_fragments(
            text,
            "[",
            "]",
        )
        +
        cell07_balanced_fragments(
            text,
            "{",
            "}",
        )
    )

    parsed = []
    seen = set()

    for fragment in fragments:

        if fragment in seen:
            continue

        seen.add(
            fragment
        )

        value = None
        parser = None

        try:
            value = json.loads(
                fragment
            )
            parser = "json"

        except Exception:
            try:
                value = ast.literal_eval(
                    fragment
                )
                parser = "python_literal"

            except Exception:
                continue

        parsed.append({
            "value":
                value,

            "parser":
                parser,

            "chars":
                len(
                    fragment
                ),
        })

    return parsed


def cell07_is_pilot_packet(
    value,
):
    return (
        isinstance(
            value,
            dict,
        )
        and
        isinstance(
            value.get(
                "pilot_case_id"
            ),
            str,
        )
        and
        value.get(
            "pilot_case_id"
        ).startswith(
            "question-pilot:"
        )
        and
        isinstance(
            value.get(
                "golden_case_id"
            ),
            str,
        )
        and
        isinstance(
            value.get(
                "target_evidence_id"
            ),
            str,
        )
    )


# ------------------------------------------------------------
# 3) Recover complete Pilot Population snapshots
# ------------------------------------------------------------

cell07_population_candidates = []

cell07_cell_inspection = {}


for cell_number in (
    cell07_candidate_cells
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if record is None:

        cell07_cell_inspection[
            cell_number
        ] = {
            "found":
                False,
        }

        continue


    output_text = (
        record.get(
            "output"
        )
        or ""
    )


    objects = (
        cell07_collect_objects(
            output_text
        )
    )


    local_candidates = []


    for item in objects:

        value = (
            item[
                "value"
            ]
        )

        if (
            not isinstance(
                value,
                list,
            )
            or
            not value
            or
            not all(
                cell07_is_pilot_packet(
                    packet
                )
                for packet in value
            )
        ):
            continue


        pilot_ids = [
            packet[
                "pilot_case_id"
            ]

            for packet
            in value
        ]


        candidate = {
            "cell":
                cell_number,

            "count":
                len(
                    value
                ),

            "unique_ids":
                len(
                    set(
                        pilot_ids
                    )
                ),

            "old_pilot_count":
                (
                    pilot_ids.count(
                        cell07_old_pilot_id
                    )
                    if cell07_old_pilot_id
                    is not None
                    else None
                ),

            "replacement_pilot_count":
                pilot_ids.count(
                    cell07_new_pilot_id
                ),

            "parser":
                item[
                    "parser"
                ],

            "chars":
                item[
                    "chars"
                ],

            "packets":
                value,
        }


        local_candidates.append(
            candidate
        )

        cell07_population_candidates.append(
            candidate
        )


    cell07_cell_inspection[
        cell_number
    ] = {
        "found":
            True,

        "output_chars":
            len(
                output_text
            ),

        "parsed_object_count":
            len(
                objects
            ),

        "packet_list_candidates":
            [
                {
                    key:
                        value

                    for key, value
                    in candidate.items()

                    if key
                    !=
                    "packets"
                }

                for candidate
                in local_candidates
            ],
    }


cell07_valid_23_candidates = [
    candidate

    for candidate
    in cell07_population_candidates

    if (
        candidate[
            "count"
        ]
        ==
        23

        and

        candidate[
            "unique_ids"
        ]
        ==
        23
    )
]


cell07_pre_replacement_candidates = [
    candidate

    for candidate
    in cell07_valid_23_candidates

    if (
        candidate[
            "old_pilot_count"
        ]
        ==
        1

        and

        candidate[
            "replacement_pilot_count"
        ]
        ==
        0
    )
]


cell07_post_replacement_candidates = [
    candidate

    for candidate
    in cell07_valid_23_candidates

    if (
        candidate[
            "old_pilot_count"
        ]
        ==
        0

        and

        candidate[
            "replacement_pilot_count"
        ]
        ==
        1
    )
]


# ------------------------------------------------------------
# 4) Select best available snapshot
# ------------------------------------------------------------

cell07_best_snapshot = None


if cell07_post_replacement_candidates:

    cell07_best_snapshot = sorted(
        cell07_post_replacement_candidates,
        key=lambda row: (
            -row[
                "cell"
            ],
            -row[
                "chars"
            ],
        ),
    )[0]


elif cell07_pre_replacement_candidates:

    cell07_best_snapshot = sorted(
        cell07_pre_replacement_candidates,
        key=lambda row: (
            -row[
                "cell"
            ],
            -row[
                "chars"
            ],
        ),
    )[0]


cell07_direct_snapshot_available = (
    cell07_best_snapshot
    is not None
)


if cell07_best_snapshot is not None:

    cell07_snapshot_mode = (
        "POST_REPLACEMENT_ACTIVE_POPULATION"

        if (
            cell07_best_snapshot[
                "replacement_pilot_count"
            ]
            ==
            1
        )

        else

        "PRE_REPLACEMENT_POPULATION_REQUIRES_DETERMINISTIC_REPLACEMENT"
    )


    cell07_snapshot_packets = (
        cell07_best_snapshot[
            "packets"
        ]
    )


    cell07_snapshot_pilot_ids = [
        packet[
            "pilot_case_id"
        ]

        for packet
        in cell07_snapshot_packets
    ]


    cell07_snapshot_sha256 = (
        hashlib.sha256(
            json.dumps(
                cell07_snapshot_packets,
                ensure_ascii=False,
                sort_keys=True,
                separators=(
                    ",",
                    ":",
                ),
            ).encode(
                "utf-8"
            )
        ).hexdigest()
    )


else:

    cell07_snapshot_mode = (
        "NO_COMPLETE_23_PILOT_SNAPSHOT_RECOVERED"
    )

    cell07_snapshot_packets = None

    cell07_snapshot_pilot_ids = []

    cell07_snapshot_sha256 = None


# ------------------------------------------------------------
# 5) Route decision
# ------------------------------------------------------------

cell07_route = (
    "BUILD_COMPACT_SEED_FROM_RECOVERED_PILOT_SNAPSHOT"

    if cell07_direct_snapshot_available

    else

    "FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD"
)


cell07_gate = all([
    len(
        cell07_failures
    )
    ==
    0,

    cell07_identity_resolution_valid,

    cell06_gate
    is True,

    cell03_prior_notebook_final_state_valid
    is True,

    cell07_route
    in {
        "BUILD_COMPACT_SEED_FROM_RECOVERED_PILOT_SNAPSHOT",
        "FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD",
    },
])


# ------------------------------------------------------------
# 6) Diagnostics
# ------------------------------------------------------------

print(
    "=== DIRECT PILOT POPULATION SNAPSHOT RECOVERY ==="
)


print(
    "\n=== PILOT IDENTITY ==="
)

print(
    "Old Pilot:",
    cell07_old_pilot_id,
)

print(
    "Replacement Pilot:",
    cell07_new_pilot_id,
)

print(
    "Pre-replacement evidence counts:",
    dict(
        cell07_pre_replacement_counts
    ),
)

print(
    "Labeled old-ID evidence:",
    cell07_labeled_old_ids,
)

print(
    "Identity resolution valid:",
    cell07_identity_resolution_valid,
)


print(
    "\n=== POPULATION SNAPSHOTS ==="
)

print(
    "Packet-list candidates:",
    len(
        cell07_population_candidates
    ),
)

print(
    "Valid 23-pilot candidates:",
    len(
        cell07_valid_23_candidates
    ),
)

print(
    "Pre-replacement candidates:",
    [
        {
            "cell":
                row[
                    "cell"
                ],

            "chars":
                row[
                    "chars"
                ],

            "parser":
                row[
                    "parser"
                ],
        }

        for row
        in cell07_pre_replacement_candidates
    ],
)

print(
    "Post-replacement candidates:",
    [
        {
            "cell":
                row[
                    "cell"
                ],

            "chars":
                row[
                    "chars"
                ],

            "parser":
                row[
                    "parser"
                ],
        }

        for row
        in cell07_post_replacement_candidates
    ],
)


print(
    "\n=== SELECTED SNAPSHOT ==="
)

print(
    "Direct snapshot available:",
    cell07_direct_snapshot_available,
)

print(
    "Snapshot mode:",
    cell07_snapshot_mode,
)

print(
    "Source Cell:",
    (
        cell07_best_snapshot[
            "cell"
        ]

        if cell07_best_snapshot
        is not None

        else None
    ),
)

print(
    "Pilot count:",
    len(
        cell07_snapshot_pilot_ids
    ),
)

print(
    "Unique Pilot IDs:",
    len(
        set(
            cell07_snapshot_pilot_ids
        )
    ),
)

print(
    "Snapshot SHA256:",
    cell07_snapshot_sha256,
)


print(
    "\n=== ROUTE DECISION ==="
)

print(
    "Route:",
    cell07_route,
)

print(
    "Mutation performed:",
    False,
)

print(
    "Failures:",
    cell07_failures,
)


print(
    "\nCODE CELL 07 GATE:",
    "PASS"
    if cell07_gate
    else "FAIL",
)


assert cell07_gate, (
    "Direct Pilot Population snapshot inspection failed. "
    "Repair this SAME Code Cell 07 before constructing the Compact Runtime Seed."
)

=== DIRECT PILOT POPULATION SNAPSHOT RECOVERY ===

=== PILOT IDENTITY ===
Old Pilot: question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a
Replacement Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Pre-replacement evidence counts: {'question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a': 2}
Labeled old-ID evidence: ['question-pilot:532a8b9d45efb8b3a32c4a62f096bb7a']
Identity resolution valid: True

=== POPULATION SNAPSHOTS ===
Packet-list candidates: 1
Valid 23-pilot candidates: 0
Pre-replacement candidates: []
Post-replacement candidates: []

=== SELECTED SNAPSHOT ===
Direct snapshot available: False
Snapshot mode: NO_COMPLETE_23_PILOT_SNAPSHOT_RECOVERED
Source Cell: None
Pilot count: 0
Unique Pilot IDs: 0
Snapshot SHA256: None

=== ROUTE DECISION ===
Route: FALL_BACK_TO_ACCEPTED_DEPENDENCY_CLOSURE_REBUILD
Mutation performed: False
Failures: []

CODE CELL 07 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>Safety Audit برای اجرای فشرده Dependency Closure</h3>

<p>
Cell 07 نشان داد که Snapshot کامل Pilot Population در Outputهای ذخیره‌شده وجود ندارد.
بنابراین مسیر معتبر همان Dependency Closure پذیرفته‌شده در Cell 06 است.
</p>

<p>
اما قبل از اینکه 88 Provider Cell لازم را در یک Runtime ایزوله و فشرده اجرا کنیم،
باید مطمئن شویم Statementهای انتخاب‌شده فقط State موردنیاز را بازسازی می‌کنند و
هیچ عملیات ناخواسته‌ای مانند File Write، Delete، Git Mutation، Network Request یا اجرای Subprocess تغییردهنده ندارند.
</p>

<p>
این Step فقط Source انتخاب‌شده را با AST Audit می‌کند.
هیچ Statementی اجرا نمی‌شود و هیچ فایل یا Runtime State پروژه تغییر نمی‌کند.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام Sliceهای Cell 06 قابل Parse باشند.</li>
<li>Callهای حساس در Statementهای انتخاب‌شده شناسایی شوند.</li>
<li>هیچ Write/Delete/Git/Network Mutation قطعی در Slice انتخاب‌شده وجود نداشته باشد.</li>
<li>اگر Call مبهمی وجود دارد، قبل از اجرای Seed به‌صورت صریح گزارش شود.</li>
<li>هیچ Mutation انجام نشود.</li>
</ul>

</div>

In [11]:
# Code Cell 08

cell08_failures = []


# ------------------------------------------------------------
# 1) High-confidence side-effect rules
# ------------------------------------------------------------

cell08_high_confidence_path_methods = {
    "write_text",
    "write_bytes",
    "unlink",
    "rename",
    "mkdir",
    "rmdir",
    "touch",
    "chmod",
    "symlink_to",
    "hardlink_to",
}


cell08_path_like_name_pattern = re.compile(
    r"(?:^|_)(?:path|file|dir|folder|root|artifact|output|report|manifest)(?:_|$)",
    flags=re.IGNORECASE,
)


cell08_filesystem_functions = {
    "os.remove",
    "os.unlink",
    "os.rename",
    "os.replace",
    "os.rmdir",
    "os.mkdir",
    "os.makedirs",
    "os.removedirs",
    "shutil.rmtree",
    "shutil.move",
    "shutil.copy",
    "shutil.copy2",
    "shutil.copytree",
}


cell08_network_mutation_calls = {
    "requests.post",
    "requests.put",
    "requests.patch",
    "requests.delete",
    "httpx.post",
    "httpx.put",
    "httpx.patch",
    "httpx.delete",
}


cell08_known_write_calls = {
    "json.dump",
    "pickle.dump",
}


cell08_git_read_only_subcommands = {
    "status",
    "rev-parse",
    "show",
    "log",
    "diff",
    "ls-files",
    "ls-tree",
    "cat-file",
    "describe",
    "remote",
}


cell08_git_mutating_subcommands = {
    "add",
    "commit",
    "push",
    "merge",
    "rebase",
    "reset",
    "checkout",
    "switch",
    "tag",
    "rm",
    "mv",
    "restore",
    "clean",
    "cherry-pick",
    "revert",
    "stash",
}


# ------------------------------------------------------------
# 2) AST helpers
# ------------------------------------------------------------

def cell08_call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):
        return node.id

    if isinstance(
        node,
        ast.Attribute,
    ):
        prefix = (
            cell08_call_name(
                node.value
            )
        )

        return (
            f"{prefix}.{node.attr}"
            if prefix
            else node.attr
        )

    return None


def cell08_literal_string(node):

    if (
        isinstance(
            node,
            ast.Constant,
        )
        and
        isinstance(
            node.value,
            str,
        )
    ):
        return node.value

    return None


def cell08_receiver_name(node):

    if not isinstance(
        node,
        ast.Attribute,
    ):
        return None

    receiver = (
        node.value
    )

    if isinstance(
        receiver,
        ast.Name,
    ):
        return receiver.id

    return (
        cell08_call_name(
            receiver
        )
    )


def cell08_source_segment(
    source,
    node,
):

    segment = ast.get_source_segment(
        source,
        node,
    )

    return (
        segment.strip()
        if isinstance(
            segment,
            str,
        )
        else None
    )


# ------------------------------------------------------------
# 3) File-open mode audit
# ------------------------------------------------------------

def cell08_open_mode(
    call,
):

    name = (
        cell08_call_name(
            call.func
        )
    )

    if not (
        name == "open"
        or
        (
            name
            and
            name.endswith(
                ".open"
            )
        )
    ):
        return None

    mode = None

    if len(
        call.args
    ) >= 2:
        mode = (
            cell08_literal_string(
                call.args[1]
            )
        )

    for keyword in (
        call.keywords
    ):
        if keyword.arg == "mode":
            mode = (
                cell08_literal_string(
                    keyword.value
                )
            )

    return (
        mode
        if mode is not None
        else "r"
    )


def cell08_open_is_write(
    call,
):

    mode = (
        cell08_open_mode(
            call
        )
    )

    if mode is None:
        return False

    return any(
        token in mode
        for token in (
            "w",
            "a",
            "x",
            "+",
        )
    )


# ------------------------------------------------------------
# 4) Git subprocess classification
# ------------------------------------------------------------

def cell08_literal_command(
    call,
):

    if not call.args:
        return None

    command_node = (
        call.args[0]
    )

    if not isinstance(
        command_node,
        (
            ast.List,
            ast.Tuple,
        ),
    ):
        return None

    command = []

    for element in (
        command_node.elts
    ):
        value = (
            cell08_literal_string(
                element
            )
        )

        if value is None:
            return None

        command.append(
            value
        )

    return command


def cell08_classify_subprocess(
    call,
):

    name = (
        cell08_call_name(
            call.func
        )
    )

    if name not in {
        "subprocess.run",
        "subprocess.call",
        "subprocess.check_call",
        "subprocess.check_output",
        "subprocess.Popen",
    }:
        return None

    command = (
        cell08_literal_command(
            call
        )
    )

    if command is None:
        return {
            "status":
                "AMBIGUOUS_SUBPROCESS",

            "command":
                None,
        }

    if not command:
        return None

    if command[0] != "git":
        return {
            "status":
                "NON_GIT_SUBPROCESS",

            "command":
                command,
        }

    if len(command) < 2:
        return {
            "status":
                "AMBIGUOUS_GIT",

            "command":
                command,
        }

    subcommand = (
        command[1]
    )


    if (
        subcommand == "branch"
        and
        "--show-current"
        in command
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if (
        subcommand == "remote"
        and
        not any(
            token
            in command
            for token in {
                "add",
                "remove",
                "rename",
                "set-url",
            }
        )
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if subcommand in (
        cell08_git_read_only_subcommands
    ):
        return {
            "status":
                "GIT_READ_ONLY",

            "command":
                command,
        }


    if subcommand in (
        cell08_git_mutating_subcommands
    ):
        return {
            "status":
                "GIT_MUTATION",

            "command":
                command,
        }


    if subcommand == "branch":
        return {
            "status":
                "AMBIGUOUS_GIT_BRANCH",

            "command":
                command,
        }


    return {
        "status":
            "AMBIGUOUS_GIT",

        "command":
            command,
    }


# ------------------------------------------------------------
# 5) Method-call classification
# ------------------------------------------------------------

def cell08_classify_method_call(
    node,
):

    if not isinstance(
        node.func,
        ast.Attribute,
    ):
        return None

    method = (
        node.func.attr
    )

    receiver = (
        cell08_receiver_name(
            node.func
        )
    )


    if method in (
        cell08_high_confidence_path_methods
    ):
        return {
            "severity":
                "BLOCKING",

            "type":
                "HIGH_CONFIDENCE_PATH_MUTATION",

            "receiver":
                receiver,

            "method":
                method,
        }


    # .replace() is common on strings.
    # Treat it as a path mutation only when the receiver itself
    # is explicitly path-like.
    if method == "replace":

        if (
            isinstance(
                receiver,
                str,
            )
            and
            cell08_path_like_name_pattern.search(
                receiver
            )
        ):
            return {
                "severity":
                    "BLOCKING",

                "type":
                    "PATH_LIKE_REPLACE",

                "receiver":
                    receiver,

                "method":
                    method,
            }

        return {
            "severity":
                "SAFE_KNOWN_FALSE_POSITIVE",

            "type":
                "GENERIC_REPLACE_METHOD",

            "receiver":
                receiver,

            "method":
                method,
        }


    # .remove() on a normal container is not a filesystem write.
    if method == "remove":

        if (
            isinstance(
                receiver,
                str,
            )
            and
            cell08_path_like_name_pattern.search(
                receiver
            )
        ):
            return {
                "severity":
                    "REVIEW",

                "type":
                    "PATH_LIKE_REMOVE_METHOD",

                "receiver":
                    receiver,

                "method":
                    method,
            }

        return {
            "severity":
                "SAFE_KNOWN_FALSE_POSITIVE",

            "type":
                "CONTAINER_REMOVE_METHOD",

            "receiver":
                receiver,

            "method":
                method,
        }


    return None


# ------------------------------------------------------------
# 6) Audit selected dependency closure
# ------------------------------------------------------------

cell08_findings = []


for cell_number in (
    cell06_provider_cells
):

    record = (
        cell04_code_cell_index[
            cell_number
        ]
    )

    source = (
        record[
            "source"
        ]
    )

    slice_info = (
        cell06_cell_slices[
            cell_number
        ]
    )


    try:
        tree = ast.parse(
            source
        )

    except SyntaxError as exc:

        cell08_failures.append({
            "reason":
                "SOURCE_PARSE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    selected_ranges = set(
        tuple(
            item
        )
        for item in slice_info[
            "statement_ranges"
        ]
    )


    selected_nodes = [
        statement

        for statement
        in tree.body

        if (
            getattr(
                statement,
                "lineno",
                None,
            ),
            getattr(
                statement,
                "end_lineno",
                None,
            ),
        )
        in selected_ranges
    ]


    for statement in (
        selected_nodes
    ):

        for node in ast.walk(
            statement
        ):

            if not isinstance(
                node,
                ast.Call,
            ):
                continue


            call_name = (
                cell08_call_name(
                    node.func
                )
            )


            source_segment = (
                cell08_source_segment(
                    source,
                    node,
                )
            )


            finding = None


            # -----------------------------------------------
            # File open with explicit write mode
            # -----------------------------------------------

            if cell08_open_is_write(
                node
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "FILE_OPEN_WRITE",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Explicit filesystem module functions
            # -----------------------------------------------

            elif call_name in (
                cell08_filesystem_functions
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "EXPLICIT_FILESYSTEM_MUTATION",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Explicit serialization writes
            # -----------------------------------------------

            elif call_name in (
                cell08_known_write_calls
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "SERIALIZATION_WRITE",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Network mutation
            # -----------------------------------------------

            elif call_name in (
                cell08_network_mutation_calls
            ):

                finding = {
                    "severity":
                        "BLOCKING",

                    "type":
                        "NETWORK_MUTATION",

                    "call":
                        call_name,
                }


            # -----------------------------------------------
            # Subprocess / Git
            # -----------------------------------------------

            else:

                subprocess_result = (
                    cell08_classify_subprocess(
                        node
                    )
                )


                if subprocess_result is not None:

                    status = (
                        subprocess_result[
                            "status"
                        ]
                    )


                    if status == "GIT_MUTATION":

                        finding = {
                            "severity":
                                "BLOCKING",

                            "type":
                                "GIT_MUTATION",

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                    elif status in {
                        "AMBIGUOUS_SUBPROCESS",
                        "NON_GIT_SUBPROCESS",
                        "AMBIGUOUS_GIT",
                        "AMBIGUOUS_GIT_BRANCH",
                    }:

                        finding = {
                            "severity":
                                "REVIEW",

                            "type":
                                status,

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                    elif status == "GIT_READ_ONLY":

                        finding = {
                            "severity":
                                "SAFE_READ_ONLY",

                            "type":
                                "GIT_READ_ONLY",

                            "call":
                                call_name,

                            "command":
                                subprocess_result[
                                    "command"
                                ],
                        }


                if finding is None:

                    method_result = (
                        cell08_classify_method_call(
                            node
                        )
                    )


                    if method_result is not None:

                        finding = {
                            "call":
                                call_name,

                            **method_result,
                        }


            if finding is not None:

                cell08_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "source":
                        source_segment,

                    **finding,
                })


# ------------------------------------------------------------
# 7) Classification
# ------------------------------------------------------------

cell08_blocking_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "BLOCKING"
]


cell08_review_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "REVIEW"
]


cell08_read_only_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "SAFE_READ_ONLY"
]


cell08_false_positive_findings = [
    row

    for row
    in cell08_findings

    if row[
        "severity"
    ]
    ==
    "SAFE_KNOWN_FALSE_POSITIVE"
]


cell08_expected_false_positive_types = {
    "GENERIC_REPLACE_METHOD",
    "CONTAINER_REMOVE_METHOD",
}


cell08_false_positive_types_valid = all(
    row[
        "type"
    ]
    in cell08_expected_false_positive_types

    for row
    in cell08_false_positive_findings
)


cell08_safe_for_isolated_rebuild = all([
    len(
        cell08_failures
    )
    ==
    0,

    len(
        cell08_blocking_findings
    )
    ==
    0,

    len(
        cell08_review_findings
    )
    ==
    0,

    cell08_false_positive_types_valid,
])


# ------------------------------------------------------------
# 8) Gate
# ------------------------------------------------------------

cell08_gate = (
    cell08_safe_for_isolated_rebuild
)


# ------------------------------------------------------------
# 9) Diagnostics
# ------------------------------------------------------------

print(
    "=== COMPACT REBUILD SAFETY AUDIT ==="
)


print(
    "\nValidator repair:",
    "Git read-only commands, generic replace(), and container remove() "
    "are distinguished from real external mutations.",
)


print(
    "\nProvider cells audited:",
    len(
        cell06_provider_cells
    ),
)

print(
    "Selected statements audited:",
    cell06_total_selected_statements,
)


print(
    "\n=== BLOCKING FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_blocking_findings
    ),
)

for row in (
    cell08_blocking_findings
):
    print(
        row
    )


print(
    "\n=== REVIEW FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_review_findings
    ),
)

for row in (
    cell08_review_findings
):
    print(
        row
    )


print(
    "\n=== CONFIRMED READ-ONLY FINDINGS ==="
)

print(
    "Count:",
    len(
        cell08_read_only_findings
    ),
)

for row in (
    cell08_read_only_findings
):
    print(
        row
    )


print(
    "\n=== CONFIRMED FALSE POSITIVES ==="
)

print(
    "Count:",
    len(
        cell08_false_positive_findings
    ),
)

for row in (
    cell08_false_positive_findings
):
    print(
        row
    )


print(
    "\n=== FAILURES ==="
)

print(
    cell08_failures
)


print(
    "\nSafe for isolated compact rebuild:",
    cell08_safe_for_isolated_rebuild,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 08 GATE:",
    "PASS"
    if cell08_gate
    else "FAIL",
)


assert cell08_gate, (
    "Selected dependency closure still contains unresolved or blocking side effects. "
    "Repair this SAME Code Cell 08 before executing the compact runtime rebuild."
)

=== COMPACT REBUILD SAFETY AUDIT ===

Validator repair: Git read-only commands, generic replace(), and container remove() are distinguished from real external mutations.

Provider cells audited: 98
Selected statements audited: 2004

=== BLOCKING FINDINGS ===
Count: 0

=== REVIEW FINDINGS ===
Count: 0

=== CONFIRMED READ-ONLY FINDINGS ===
Count: 1
{'cell': 2, 'line': 4, 'source': 'subprocess.check_output(\n        ["git", "rev-parse", "--show-toplevel"],\n        text=True\n    )', 'severity': 'SAFE_READ_ONLY', 'type': 'GIT_READ_ONLY', 'call': 'subprocess.check_output', 'command': ['git', 'rev-parse', '--show-toplevel']}

=== CONFIRMED FALSE POSITIVES ===
Count: 3
{'cell': 35, 'line': 43, 'source': 'str(\n            text\n            or ""\n        )\n        .replace(\n            "\\u200c",\n            " "\n        )', 'call': 'replace', 'severity': 'SAFE_KNOWN_FALSE_POSITIVE', 'type': 'GENERIC_REPLACE_METHOD', 'receiver': None, 'method': 'replace'}
{'cell': 39, 'line': 50, 'source'

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h3>اجرای Dry-Run بازسازی Final State در Namespace ایزوله</h3>

<p>
Safety Audit در Cell 08 تأیید کرد که Statementهای انتخاب‌شده برای بازسازی Final State،
هیچ File Write، Git Mutation، Network Mutation یا Side Effect خارجی حل‌نشده ندارند.
</p>

<p>
در این مرحله همان Dependency Closure به ترتیب تاریخی Cellها، اما داخل یک Namespace مستقل اجرا می‌شود.
این Namespace به State اصلی Notebook جدید وابسته نیست و فقط چند Dependency فنی عمومی مانند
<code>Path</code>، <code>json</code>، <code>hashlib</code> و <code>subprocess</code> به آن داده می‌شود.
</p>

<p>
هدف این Dry-Run این است که قبل از ساخت Compact Runtime Seed ثابت کنیم Closure استخراج‌شده
واقعاً از صفر قابل اجرا است و Hidden Session Dependency از Notebook قبلی ندارد.
</p>

<h4>معیار PASS</h4>

<ul>
<li>تمام 88 Provider Cell Slice به ترتیب اجرا شوند.</li>
<li>هیچ NameError یا Hidden Session Dependency رخ ندهد.</li>
<li><code>cell138_target_values</code> در Namespace ایزوله ساخته شود.</li>
<li>Final Target دقیقاً شامل 11 Runtime Object مورد انتظار باشد.</li>
<li>Population نهایی 23 Pilot داشته باشد.</li>
<li>Pilot قدیمی Active نباشد و Replacement دقیقاً یک‌بار Active باشد.</li>
<li>Remaining Queue برابر 22 و Next Pilot همان Replacement باشد.</li>
<li>Full 572 Generation همچنان Blocked باقی بماند.</li>
<li>هیچ فایل یا Git State تغییر نکند.</li>
</ul>

</div>

In [22]:
# Code Cell 09

cell09_replay_last_cell = 140

cell09_expected_target_names = [
    "pilot_selection_rows",
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "cell95_pilot_ids_in_runtime_order",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_remaining_pilot_queue_sha256",
    "cell95_next_pilot_queue_entry",
    "cell95_next_pilot_id",
    "cell95_pilot_workflow_state",
    "cell95_pilot_workflow_state_sha256",
]

cell09_failures = []


# ============================================================
# 1) Helpers
# ============================================================

def cell09_call_name(node):

    if isinstance(
        node,
        ast.Name,
    ):
        return node.id

    if isinstance(
        node,
        ast.Attribute,
    ):

        parent = (
            cell09_call_name(
                node.value
            )
        )

        return (
            f"{parent}.{node.attr}"
            if parent
            else node.attr
        )

    return None


def cell09_literal_command(node):

    if not isinstance(
        node,
        (
            ast.List,
            ast.Tuple,
        ),
    ):
        return None

    command = []

    for item in node.elts:

        if not (
            isinstance(
                item,
                ast.Constant,
            )
            and
            isinstance(
                item.value,
                str,
            )
        ):
            return None

        command.append(
            item.value
        )

    return command


def cell09_find_pilot_ids(value):

    found = []

    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            found.append(
                value
            )

    elif isinstance(
        value,
        dict,
    ):

        for nested in (
            value.values()
        ):
            found.extend(
                cell09_find_pilot_ids(
                    nested
                )
            )

    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        for nested in value:
            found.extend(
                cell09_find_pilot_ids(
                    nested
                )
            )

    return found


def cell09_stable_sha(value):

    return hashlib.sha256(
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            separators=(
                ",",
                ":",
            ),
            default=str,
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def cell09_safe_len(value):

    try:
        return len(
            value
        )

    except Exception:
        return None


# ============================================================
# 2) Build exact source-order replay plan
# ============================================================

cell09_execution_plan = []


for cell_number in range(
    1,
    cell09_replay_last_cell + 1,
):

    record = (
        cell04_code_cell_index.get(
            cell_number
        )
    )

    if not isinstance(
        record,
        dict,
    ):

        cell09_failures.append({
            "reason":
                "MISSING_PRIOR_CODE_CELL",

            "cell":
                cell_number,
        })

        continue


    source = (
        record.get(
            "source"
        )
    )

    if not isinstance(
        source,
        str,
    ):

        cell09_failures.append({
            "reason":
                "INVALID_PRIOR_CODE_CELL_SOURCE",

            "cell":
                cell_number,
        })

        continue


    try:

        tree = ast.parse(
            source
        )

    except SyntaxError as exc:

        cell09_failures.append({
            "reason":
                "PRIOR_CODE_CELL_PARSE_FAILED",

            "cell":
                cell_number,

            "error":
                repr(
                    exc
                ),
        })

        continue


    virtual_filename = (
        f"<prior-nb12-code-cell-{cell_number}>"
    )


    cell09_execution_plan.append({
        "cell":
            cell_number,

        "source":
            source,

        "tree":
            tree,

        "filename":
            virtual_filename,
    })


cell09_plan_valid = all([
    len(
        cell09_failures
    )
    ==
    0,

    len(
        cell09_execution_plan
    )
    ==
    cell09_replay_last_cell,
])


# ============================================================
# 3) Safety audit for full replay prefix
# ============================================================

cell09_blocking_findings = []

cell09_read_only_git_findings = []


cell09_blocked_methods = {
    "write_text",
    "write_bytes",
    "unlink",
    "rmdir",
    "mkdir",
    "touch",
    "rename",
    "to_csv",
    "to_json",
    "to_parquet",
    "to_excel",
    "to_pickle",
}


cell09_network_calls = {
    "requests.get",
    "requests.post",
    "requests.put",
    "requests.patch",
    "requests.delete",
    "httpx.get",
    "httpx.post",
    "httpx.put",
    "httpx.patch",
    "httpx.delete",
    "urllib.request.urlopen",
}


cell09_subprocess_calls = {
    "subprocess.run",
    "subprocess.call",
    "subprocess.check_call",
    "subprocess.check_output",
    "subprocess.Popen",
}


cell09_mutating_git_commands = {
    "add",
    "commit",
    "push",
    "pull",
    "merge",
    "rebase",
    "reset",
    "checkout",
    "switch",
    "tag",
    "stash",
    "clean",
    "rm",
    "mv",
    "restore",
    "cherry-pick",
    "revert",
}


for item in (
    cell09_execution_plan
):

    cell_number = (
        item[
            "cell"
        ]
    )

    tree = (
        item[
            "tree"
        ]
    )


    for node in ast.walk(
        tree
    ):

        if not isinstance(
            node,
            ast.Call,
        ):
            continue


        call_name = (
            cell09_call_name(
                node.func
            )
        )


        # ----------------------------------------------------
        # File writes through open()
        # ----------------------------------------------------

        if (
            call_name
            ==
            "open"
        ):

            mode = None


            if (
                len(
                    node.args
                )
                >=
                2
                and
                isinstance(
                    node.args[
                        1
                    ],
                    ast.Constant,
                )
                and
                isinstance(
                    node.args[
                        1
                    ].value,
                    str,
                )
            ):

                mode = (
                    node.args[
                        1
                    ].value
                )


            for keyword in (
                node.keywords
            ):

                if (
                    keyword.arg
                    ==
                    "mode"
                    and
                    isinstance(
                        keyword.value,
                        ast.Constant,
                    )
                    and
                    isinstance(
                        keyword.value.value,
                        str,
                    )
                ):

                    mode = (
                        keyword.value.value
                    )


            if (
                isinstance(
                    mode,
                    str,
                )
                and
                any(
                    marker
                    in
                    mode

                    for marker
                    in (
                        "w",
                        "a",
                        "x",
                        "+",
                    )
                )
            ):

                cell09_blocking_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "type":
                        "FILE_WRITE_OPEN",

                    "mode":
                        mode,
                })


        # ----------------------------------------------------
        # Known filesystem / artifact mutations
        # ----------------------------------------------------

        method_name = (
            call_name.split(
                "."
            )[-1]

            if isinstance(
                call_name,
                str,
            )

            else None
        )


        if (
            method_name
            in
            cell09_blocked_methods
        ):

            cell09_blocking_findings.append({
                "cell":
                    cell_number,

                "line":
                    getattr(
                        node,
                        "lineno",
                        None,
                    ),

                "type":
                    "FILESYSTEM_OR_ARTIFACT_MUTATION",

                "call":
                    call_name,
            })


        # ----------------------------------------------------
        # Network access
        # ----------------------------------------------------

        if (
            call_name
            in
            cell09_network_calls
        ):

            cell09_blocking_findings.append({
                "cell":
                    cell_number,

                "line":
                    getattr(
                        node,
                        "lineno",
                        None,
                    ),

                "type":
                    "NETWORK_ACCESS",

                "call":
                    call_name,
            })


        # ----------------------------------------------------
        # Subprocess / Git mutations
        # ----------------------------------------------------

        if (
            call_name
            in
            cell09_subprocess_calls
            and
            node.args
        ):

            command = (
                cell09_literal_command(
                    node.args[
                        0
                    ]
                )
            )


            if (
                command
                and
                command[
                    0
                ]
                ==
                "git"
            ):

                subcommand = (
                    command[
                        1
                    ]

                    if len(
                        command
                    )
                    >
                    1

                    else None
                )


                read_only_branch = (
                    subcommand
                    ==
                    "branch"
                    and
                    "--show-current"
                    in
                    command
                )


                if (
                    subcommand
                    in
                    cell09_mutating_git_commands
                    and
                    not read_only_branch
                ):

                    cell09_blocking_findings.append({
                        "cell":
                            cell_number,

                        "line":
                            getattr(
                                node,
                                "lineno",
                                None,
                            ),

                        "type":
                            "GIT_MUTATION",

                        "command":
                            command,
                    })

                else:

                    cell09_read_only_git_findings.append({
                        "cell":
                            cell_number,

                        "command":
                            command,
                    })


            elif (
                command
                and
                command[
                    0
                ]
                in {
                    "rm",
                    "mv",
                    "cp",
                    "curl",
                    "wget",
                }
            ):

                cell09_blocking_findings.append({
                    "cell":
                        cell_number,

                    "line":
                        getattr(
                            node,
                            "lineno",
                            None,
                        ),

                    "type":
                        "EXTERNAL_SIDE_EFFECT_COMMAND",

                    "command":
                        command,
                })


cell09_safety_gate = all([
    cell09_plan_valid,

    len(
        cell09_blocking_findings
    )
    ==
    0,
])


# ============================================================
# 4) Virtual IPython history
# ============================================================

class Cell09VirtualHistoryManager:

    def __init__(
        self,
    ):
        self.input_hist_raw = [
            ""
        ]


class Cell09IPythonProxy:

    def __init__(
        self,
        real_ipython,
    ):

        self._real_ipython = (
            real_ipython
        )

        self.history_manager = (
            Cell09VirtualHistoryManager()
        )


    def __getattr__(
        self,
        name,
    ):

        if (
            self._real_ipython
            is None
        ):
            raise AttributeError(
                name
            )

        return getattr(
            self._real_ipython,
            name,
        )


try:

    cell09_real_ipython = (
        get_ipython()
    )

except Exception:

    cell09_real_ipython = None


cell09_virtual_ipython = (
    Cell09IPythonProxy(
        cell09_real_ipython
    )
)


cell09_prior_sources_by_cell = {
    item[
        "cell"
    ]:
        item[
            "source"
        ]

    for item
    in cell09_execution_plan
}


# ============================================================
# 5) Register virtual source files for inspect.getsource()
# ============================================================

cell09_linecache = (
    __import__(
        "linecache"
    )
)


cell09_registered_virtual_files = []


for item in (
    cell09_execution_plan
):

    filename = (
        item[
            "filename"
        ]
    )

    source = (
        item[
            "source"
        ]
    )


    source_for_cache = (
        source
        if source.endswith(
            "\n"
        )
        else source + "\n"
    )


    source_lines = (
        source_for_cache.splitlines(
            keepends=True
        )
    )


    cell09_linecache.cache[
        filename
    ] = (
        len(
            source_for_cache
        ),

        None,

        source_lines,

        filename,
    )


    cell09_registered_virtual_files.append(
        filename
    )


cell09_linecache_registration_valid = all([
    len(
        cell09_registered_virtual_files
    )
    ==
    cell09_replay_last_cell,

    all(
        filename
        in
        cell09_linecache.cache

        for filename
        in cell09_registered_virtual_files
    ),
])


# ============================================================
# 6) Inspect-source preflight
# ============================================================

cell09_source_probe_namespace = {
    "__builtins__":
        __builtins__,
}


cell09_source_probe_ok = False

cell09_source_probe_error = None


cell09_probe_source = (
    "def cell09_virtual_source_probe(value):\n"
    "    return value\n"
)


cell09_probe_filename = (
    "<cell09-linecache-probe>"
)


cell09_linecache.cache[
    cell09_probe_filename
] = (
    len(
        cell09_probe_source
    ),

    None,

    cell09_probe_source.splitlines(
        keepends=True
    ),

    cell09_probe_filename,
)


try:

    exec(
        compile(
            cell09_probe_source,
            filename=(
                cell09_probe_filename
            ),
            mode="exec",
        ),
        cell09_source_probe_namespace,
        cell09_source_probe_namespace,
    )


    cell09_inspect = (
        __import__(
            "inspect"
        )
    )


    cell09_probe_function = (
        cell09_source_probe_namespace[
            "cell09_virtual_source_probe"
        ]
    )


    cell09_probe_recovered_source = (
        cell09_inspect.getsource(
            cell09_probe_function
        )
    )


    cell09_source_probe_ok = (
        "def cell09_virtual_source_probe"
        in
        cell09_probe_recovered_source
    )


except Exception as exc:

    cell09_source_probe_error = (
        repr(
            exc
        )
    )


# ============================================================
# 7) Cell 97 source marker preflight
# ============================================================

cell09_cell97_source = (
    cell09_prior_sources_by_cell.get(
        97,
        "",
    )
)


cell09_cell97_marker_valid = all([
    "# Code Cell 97"
    in
    cell09_cell97_source,

    "cell97_question_draft_record ="
    in
    cell09_cell97_source,

    "cell97_question_draft_lineage ="
    in
    cell09_cell97_source,
])


# ============================================================
# 8) Freeze external project state
# ============================================================

cell09_git_before_rc, cell09_git_before, cell09_git_before_err = (
    run_git(
        "status",
        "--short",
    )
)


cell09_prior_before = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell09_handoff_before = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


# ============================================================
# 9) Fresh isolated runtime namespace
# ============================================================

cell09_seed_namespace = {
    "__builtins__":
        __builtins__,

    "__name__":
        "nb12_full_ordered_linecache_replay",

    "print":
        lambda *args, **kwargs: None,

    "display":
        lambda *args, **kwargs: None,

    "get_ipython":
        lambda:
            cell09_virtual_ipython,
}


# ============================================================
# 10) Full ordered replay
# ============================================================

cell09_execution_results = []

cell09_execution_exception = None


if all([
    cell09_safety_gate,
    cell09_linecache_registration_valid,
    cell09_source_probe_ok,
]):

    for item in (
        cell09_execution_plan
    ):

        cell_number = (
            item[
                "cell"
            ]
        )

        source = (
            item[
                "source"
            ]
        )

        filename = (
            item[
                "filename"
            ]
        )


        # Emulate raw Jupyter input history.
        cell09_virtual_ipython.history_manager.input_hist_raw = (
            [
                ""
            ]
            +
            [
                cell09_prior_sources_by_cell[
                    prior_cell
                ]

                for prior_cell
                in range(
                    1,
                    cell_number + 1,
                )
            ]
        )


        # Refresh source registration immediately before execution.
        source_for_cache = (
            source
            if source.endswith(
                "\n"
            )
            else source + "\n"
        )


        cell09_linecache.cache[
            filename
        ] = (
            len(
                source_for_cache
            ),

            None,

            source_for_cache.splitlines(
                keepends=True
            ),

            filename,
        )


        try:

            compiled = compile(
                source,
                filename=filename,
                mode="exec",
            )


            exec(
                compiled,
                cell09_seed_namespace,
                cell09_seed_namespace,
            )


            cell09_execution_results.append({
                "cell":
                    cell_number,

                "status":
                    "PASS",

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),
            })


        except Exception as exc:

            cell09_execution_exception = {
                "cell":
                    cell_number,

                "exception_type":
                    type(
                        exc
                    ).__name__,

                "error":
                    repr(
                        exc
                    ),

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),

                "filename":
                    filename,

                "linecache_registered":
                    filename
                    in
                    cell09_linecache.cache,

                "source_preview":
                    source[
                        :3000
                    ],
            }


            cell09_execution_results.append({
                "cell":
                    cell_number,

                "status":
                    "FAIL",

                "virtual_history_length":
                    len(
                        cell09_virtual_ipython
                        .history_manager
                        .input_hist_raw
                    ),
            })


            break


cell09_execution_complete = all([
    cell09_execution_exception
    is None,

    len(
        cell09_execution_results
    )
    ==
    cell09_replay_last_cell,

    all(
        row[
            "status"
        ]
        ==
        "PASS"

        for row
        in cell09_execution_results
    ),
])


# ============================================================
# 11) Extract final runtime target
# ============================================================

cell09_target_surface_complete = all(
    name
    in
    cell09_seed_namespace

    for name
    in cell09_expected_target_names
)


cell09_target_values = {
    name:
        cell09_seed_namespace.get(
            name
        )

    for name
    in cell09_expected_target_names
}


cell09_packets = (
    cell09_target_values.get(
        "question_drafting_pilot_packets"
    )
)


cell09_pilot_index = (
    cell09_target_values.get(
        "question_drafting_pilot_by_id"
    )
)


cell09_runtime_ids = (
    cell09_target_values.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell09_remaining_ids = (
    cell09_target_values.get(
        "cell95_remaining_pilot_ids"
    )
)


cell09_remaining_queue = (
    cell09_target_values.get(
        "cell95_remaining_pilot_queue"
    )
)


cell09_next_queue_entry = (
    cell09_target_values.get(
        "cell95_next_pilot_queue_entry"
    )
)


cell09_next_pilot_id = (
    cell09_target_values.get(
        "cell95_next_pilot_id"
    )
)


cell09_workflow_state = (
    cell09_target_values.get(
        "cell95_pilot_workflow_state"
    )
)


# ============================================================
# 12) Final Pilot invariants
# ============================================================

cell09_population_valid = (
    isinstance(
        cell09_runtime_ids,
        list,
    )
    and
    len(
        cell09_runtime_ids
    )
    ==
    23
    and
    len(
        set(
            cell09_runtime_ids
        )
    )
    ==
    23
    and
    cell09_runtime_ids.count(
        cell07_old_pilot_id
    )
    ==
    0
    and
    cell09_runtime_ids.count(
        cell07_new_pilot_id
    )
    ==
    1
)


cell09_packets_valid = (
    isinstance(
        cell09_packets,
        list,
    )
    and
    len(
        cell09_packets
    )
    ==
    23
)


cell09_index_valid = (
    isinstance(
        cell09_pilot_index,
        dict,
    )
    and
    len(
        cell09_pilot_index
    )
    ==
    23
    and
    cell07_old_pilot_id
    not in
    cell09_pilot_index
    and
    cell07_new_pilot_id
    in
    cell09_pilot_index
)


cell09_remaining_valid = (
    isinstance(
        cell09_remaining_ids,
        list,
    )
    and
    len(
        cell09_remaining_ids
    )
    ==
    22
    and
    len(
        set(
            cell09_remaining_ids
        )
    )
    ==
    22
    and
    cell07_old_pilot_id
    not in
    cell09_remaining_ids
    and
    cell09_remaining_ids.count(
        cell07_new_pilot_id
    )
    ==
    1
    and
    bool(
        cell09_remaining_ids
    )
    and
    cell09_remaining_ids[
        0
    ]
    ==
    cell07_new_pilot_id
)


cell09_queue_ids = (
    [
        cell09_find_pilot_ids(
            row
        )

        for row
        in cell09_remaining_queue
    ]

    if isinstance(
        cell09_remaining_queue,
        list,
    )

    else []
)


cell09_queue_valid = (
    isinstance(
        cell09_remaining_queue,
        list,
    )
    and
    len(
        cell09_remaining_queue
    )
    ==
    22
    and
    bool(
        cell09_queue_ids
    )
    and
    cell07_new_pilot_id
    in
    cell09_queue_ids[
        0
    ]
    and
    not any(
        cell07_old_pilot_id
        in
        pilot_ids

        for pilot_ids
        in cell09_queue_ids
    )
)


cell09_next_valid = (
    cell09_next_pilot_id
    ==
    cell07_new_pilot_id
    and
    cell07_new_pilot_id
    in
    cell09_find_pilot_ids(
        cell09_next_queue_entry
    )
)


cell09_workflow_text = (
    json.dumps(
        cell09_workflow_state,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )

    if cell09_workflow_state
    is not None

    else ""
)


cell09_full_572_blocked = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
    in
    cell09_workflow_text
)


# ============================================================
# 13) Final target fingerprint
# ============================================================

cell09_target_hashes = {}


if (
    cell09_target_surface_complete
):

    cell09_target_hashes = {
        name:
            cell09_stable_sha(
                value
            )

        for name, value
        in cell09_target_values.items()
    }


cell09_target_bundle_sha256 = (
    cell09_stable_sha(
        cell09_target_values
    )

    if cell09_target_surface_complete

    else None
)


# ============================================================
# 14) Verify external project state
# ============================================================

cell09_git_after_rc, cell09_git_after, cell09_git_after_err = (
    run_git(
        "status",
        "--short",
    )
)


cell09_prior_after = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell09_handoff_after = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


cell09_external_state_stable = all([
    cell09_git_before_rc
    ==
    cell09_git_after_rc,

    cell09_git_before
    ==
    cell09_git_after,

    cell09_prior_before
    ==
    cell09_prior_after,

    cell09_handoff_before
    ==
    cell09_handoff_after,
])


# ============================================================
# 15) Final gate
# ============================================================

cell09_gate = all([
    cell06_gate
    is True,

    cell07_gate
    is True,

    cell08_gate
    is True,

    cell09_plan_valid,

    cell09_safety_gate,

    cell09_linecache_registration_valid,

    cell09_source_probe_ok,

    cell09_cell97_marker_valid,

    cell09_execution_complete,

    cell09_target_surface_complete,

    cell09_population_valid,

    cell09_packets_valid,

    cell09_index_valid,

    cell09_remaining_valid,

    cell09_queue_valid,

    cell09_next_valid,

    cell09_full_572_blocked,

    cell09_external_state_stable,
])


# ============================================================
# 16) Diagnostics
# ============================================================

print(
    "=== FULL ORDERED ISOLATED NB12 REPLAY "
    "WITH VIRTUAL HISTORY + LINECACHE ==="
)


print(
    "\n=== PLAN ==="
)

print(
    "Replay Cells:",
    cell09_replay_last_cell,
)

print(
    "Plan valid:",
    cell09_plan_valid,
)


print(
    "\n=== SAFETY ==="
)

print(
    "Blocking findings:",
    len(
        cell09_blocking_findings
    ),
)

print(
    "Safety gate:",
    cell09_safety_gate,
)


print(
    "\n=== SOURCE INTROSPECTION SUPPORT ==="
)

print(
    "Virtual source files registered:",
    len(
        cell09_registered_virtual_files
    ),
)

print(
    "Linecache registration valid:",
    cell09_linecache_registration_valid,
)

print(
    "inspect.getsource probe:",
    cell09_source_probe_ok,
)

print(
    "Probe error:",
    cell09_source_probe_error,
)

print(
    "Cell 97 marker valid:",
    cell09_cell97_marker_valid,
)


print(
    "\n=== EXECUTION ==="
)

print(
    "Cells executed:",
    len(
        cell09_execution_results
    ),
)

print(
    "Execution complete:",
    cell09_execution_complete,
)

print(
    "Execution exception:",
    cell09_execution_exception,
)


if (
    cell09_execution_exception
    is not None
):

    print(
        "\nFailed Cell:",
        cell09_execution_exception[
            "cell"
        ],
    )

    print(
        "Exception:",
        cell09_execution_exception[
            "error"
        ],
    )

    print(
        "Virtual filename:",
        cell09_execution_exception[
            "filename"
        ],
    )

    print(
        "Linecache registered:",
        cell09_execution_exception[
            "linecache_registered"
        ],
    )

    print(
        "Virtual history length:",
        cell09_execution_exception[
            "virtual_history_length"
        ],
    )

    print(
        "\nSource preview:"
    )

    print(
        cell09_execution_exception[
            "source_preview"
        ]
    )


print(
    "\n=== FINAL PILOT STATE ==="
)

print(
    "Pilot count:",
    cell09_safe_len(
        cell09_runtime_ids
    ),
)

print(
    "Packet count:",
    cell09_safe_len(
        cell09_packets
    ),
)

print(
    "Index count:",
    cell09_safe_len(
        cell09_pilot_index
    ),
)

print(
    "Old Pilot active:",
    (
        cell09_runtime_ids.count(
            cell07_old_pilot_id
        )

        if isinstance(
            cell09_runtime_ids,
            list,
        )

        else None
    ),
)

print(
    "Replacement active:",
    (
        cell09_runtime_ids.count(
            cell07_new_pilot_id
        )

        if isinstance(
            cell09_runtime_ids,
            list,
        )

        else None
    ),
)


print(
    "\n=== QUEUE ==="
)

print(
    "Remaining:",
    cell09_safe_len(
        cell09_remaining_ids
    ),
)

print(
    "Queue:",
    cell09_safe_len(
        cell09_remaining_queue
    ),
)

print(
    "Next Pilot:",
    cell09_next_pilot_id,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell09_full_572_blocked,
)

print(
    "Target bundle SHA256:",
    cell09_target_bundle_sha256,
)


print(
    "\n=== EXTERNAL STATE ==="
)

print(
    "Git unchanged:",
    cell09_git_before
    ==
    cell09_git_after,
)

print(
    "Prior NB12 unchanged:",
    cell09_prior_before
    ==
    cell09_prior_after,
)

print(
    "Existing handoff unchanged:",
    cell09_handoff_before
    ==
    cell09_handoff_after,
)

print(
    "External state stable:",
    cell09_external_state_stable,
)


print(
    "\nMutation performed on project files:",
    False,
)


print(
    "\nCODE CELL 09 GATE:",
    (
        "PASS"
        if cell09_gate
        else "FAIL"
    ),
)


assert cell09_gate, (
    "Full ordered isolated NB12 replay with virtual history "
    "and linecache source registration failed. Repair this "
    "SAME Code Cell 09 before constructing the Compact "
    "Runtime Seed."
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

=== FULL ORDERED ISOLATED NB12 REPLAY WITH VIRTUAL HISTORY + LINECACHE ===

=== PLAN ===
Replay Cells: 140
Plan valid: True

=== SAFETY ===
Blocking findings: 0
Safety gate: True

=== SOURCE INTROSPECTION SUPPORT ===
Virtual source files registered: 140
Linecache registration valid: True
inspect.getsource probe: True
Probe error: None
Cell 97 marker valid: True

=== EXECUTION ===
Cells executed: 140
Execution complete: True
Execution exception: None

=== FINAL PILOT STATE ===
Pilot count: 23
Packet count: 23
Index count: 23
Old Pilot active: 0
Replacement active: 1

=== QUEUE ===
Remaining: 22
Queue: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952

=== WORKFLOW ===
Full 572 blocked: True
Target bundle SHA256: 5512784eb833eb73004e6091722ca74a0abf503415400b73472abd6aa6cf198f

=== EXTERNAL STATE ===
Git unchanged: True
Prior NB12 unchanged: True
Existing handoff unchanged: True
External state stable: True

Mutation performed on project files: False

CODE CELL 09 GATE: PASS


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [24]:
# Code Cell 10

cell10_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell09_gate"
    )
    is not True
):
    cell10_failures.append(
        "CELL09_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell09_seed_namespace"
    ),
    dict,
):
    cell10_failures.append(
        "CELL09_SEED_NAMESPACE_MISSING"
    )


# ============================================================
# 2) Required final runtime surface
# ============================================================

CELL10_REQUIRED_FINAL_RUNTIME_NAMES = [
    "pilot_selection_rows",
    "question_drafting_pilot_packets",
    "question_drafting_pilot_by_id",
    "cell95_pilot_ids_in_runtime_order",
    "cell95_remaining_pilot_ids",
    "cell95_remaining_pilot_queue",
    "cell95_remaining_pilot_queue_sha256",
    "cell95_next_pilot_queue_entry",
    "cell95_next_pilot_id",
    "cell95_pilot_workflow_state",
    "cell95_pilot_workflow_state_sha256",
]


cell10_missing_final_runtime_names = [
    name
    for name
    in CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    if name
    not in
    cell09_seed_namespace
]


if (
    cell10_missing_final_runtime_names
):
    cell10_failures.append({
        "reason":
            "FINAL_RUNTIME_SURFACE_INCOMPLETE",

        "missing":
            cell10_missing_final_runtime_names,
    })


# ============================================================
# 3) Candidate registry-name policy
# ============================================================

CELL10_INCLUDE_TERMS = (
    "question",
    "answer",
    "citation",
    "review",
    "adjudication",
    "target_evidence",
    "golden",
    "case",
    "draft",
    "pilot",
    "queue",
    "lineage",
    "registry",
    "accepted",
    "rejected",
    "supersession",
    "replacement",
    "workflow",
)


CELL10_EXCLUDE_PREFIXES = (
    "__",
    "cell09_",
)


CELL10_EXCLUDE_TERMS = (
    "_source",
    "_source_text",
    "_source_lines",
    "_tree",
    "_ast",
    "_code",
    "_notebook",
    "_execution_plan",
    "_execution_results",
    "_before",
    "_after",
    "_window",
    "_preview",
)


# ============================================================
# 4) Canonical serialization helpers
# ============================================================

def cell10_canonical_key(
    value,
):

    if value is None:
        return {
            "__key_type__":
                "none",

            "value":
                None,
        }


    if isinstance(
        value,
        bool,
    ):
        return {
            "__key_type__":
                "bool",

            "value":
                value,
        }


    if isinstance(
        value,
        int,
    ):
        return {
            "__key_type__":
                "int",

            "value":
                value,
        }


    if isinstance(
        value,
        float,
    ):
        return {
            "__key_type__":
                "float",

            "value":
                repr(
                    value
                ),
        }


    if isinstance(
        value,
        str,
    ):
        return {
            "__key_type__":
                "str",

            "value":
                value,
        }


    if isinstance(
        value,
        tuple,
    ):
        return {
            "__key_type__":
                "tuple",

            "value":
                [
                    cell10_canonicalize(
                        item
                    )
                    for item
                    in value
                ],
        }


    return {
        "__key_type__":
            type(
                value
            ).__name__,

        "value":
            repr(
                value
            ),
    }


def cell10_canonicalize(
    value,
    _seen=None,
):

    if _seen is None:
        _seen = set()


    if value is None:
        return None


    if isinstance(
        value,
        (
            bool,
            int,
            str,
        ),
    ):
        return value


    if isinstance(
        value,
        float,
    ):
        return {
            "__float__":
                repr(
                    value
                )
        }


    if isinstance(
        value,
        bytes,
    ):
        return {
            "__bytes_hex__":
                value.hex()
        }


    if isinstance(
        value,
        Path,
    ):
        return {
            "__path__":
                str(
                    value
                )
        }


    track_identity = isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    )


    object_id = (
        id(
            value
        )
        if track_identity
        else None
    )


    if (
        track_identity
        and
        object_id
        in
        _seen
    ):
        return {
            "__cycle__":
                True,

            "__type__":
                type(
                    value
                ).__name__,
        }


    if track_identity:
        _seen.add(
            object_id
        )


    try:

        if isinstance(
            value,
            dict,
        ):

            items = []

            for key, nested in (
                value.items()
            ):

                canonical_key = (
                    cell10_canonical_key(
                        key
                    )
                )

                canonical_value = (
                    cell10_canonicalize(
                        nested,
                        _seen,
                    )
                )

                items.append({
                    "key":
                        canonical_key,

                    "value":
                        canonical_value,
                })


            items.sort(
                key=lambda row:
                    json.dumps(
                        row[
                            "key"
                        ],
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(
                            ",",
                            ":",
                        ),
                    )
            )


            return {
                "__dict_items__":
                    items
            }


        if isinstance(
            value,
            list,
        ):

            return {
                "__list__":
                    [
                        cell10_canonicalize(
                            item,
                            _seen,
                        )
                        for item
                        in value
                    ]
            }


        if isinstance(
            value,
            tuple,
        ):

            return {
                "__tuple__":
                    [
                        cell10_canonicalize(
                            item,
                            _seen,
                        )
                        for item
                        in value
                    ]
            }


        if isinstance(
            value,
            (
                set,
                frozenset,
            ),
        ):

            canonical_items = [
                cell10_canonicalize(
                    item,
                    _seen,
                )
                for item
                in value
            ]


            canonical_items.sort(
                key=lambda item:
                    json.dumps(
                        item,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(
                            ",",
                            ":",
                        ),
                    )
            )


            return {
                (
                    "__frozenset__"
                    if isinstance(
                        value,
                        frozenset,
                    )
                    else "__set__"
                ):
                    canonical_items
            }


        if callable(
            value
        ):

            return {
                "__callable__":
                    getattr(
                        value,
                        "__qualname__",
                        getattr(
                            value,
                            "__name__",
                            type(
                                value
                            ).__name__,
                        ),
                    ),

                "__module__":
                    getattr(
                        value,
                        "__module__",
                        None,
                    ),
            }


        return {
            "__type__":
                type(
                    value
                ).__name__,

            "__repr__":
                repr(
                    value
                ),
        }


    finally:

        if track_identity:
            _seen.discard(
                object_id
            )


def cell10_canonical_json(
    value,
):

    normalized = (
        cell10_canonicalize(
            value
        )
    )


    return json.dumps(
        normalized,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    )


def cell10_stable_sha(
    value,
):

    return hashlib.sha256(
        cell10_canonical_json(
            value
        ).encode(
            "utf-8"
        )
    ).hexdigest()


def cell10_count(
    value,
):

    if isinstance(
        value,
        (
            list,
            tuple,
            dict,
            set,
            frozenset,
        ),
    ):
        return len(
            value
        )

    return None


def cell10_serialized_size(
    value,
):

    try:

        return len(
            cell10_canonical_json(
                value
            ).encode(
                "utf-8"
            )
        )

    except Exception:
        return None


def cell10_contains_pilot_id(
    value,
    pilot_id,
):

    try:

        text = (
            cell10_canonical_json(
                value
            )
        )

    except Exception:

        text = repr(
            value
        )


    return (
        pilot_id
        in
        text
    )


# ============================================================
# 5) Canonicalization self-test
# ============================================================

cell10_tuple_key_probe = {
    (
        "website",
        "direct_author_long",
    ):
        {
            "status":
                "READY"
        },

    (
        "book",
        1,
    ):
        [
            "x",
            "y",
        ],
}


cell10_tuple_key_probe_sha = (
    cell10_stable_sha(
        cell10_tuple_key_probe
    )
)


cell10_tuple_key_probe_valid = (
    isinstance(
        cell10_tuple_key_probe_sha,
        str,
    )
    and
    len(
        cell10_tuple_key_probe_sha
    )
    ==
    64
)


if not (
    cell10_tuple_key_probe_valid
):

    cell10_failures.append(
        "TUPLE_KEY_CANONICALIZATION_FAILED"
    )


# ============================================================
# 6) Inventory candidate continuation registries
# ============================================================

cell10_candidate_rows = []


for name, value in sorted(
    cell09_seed_namespace.items()
):

    if any(
        name.startswith(
            prefix
        )
        for prefix
        in CELL10_EXCLUDE_PREFIXES
    ):
        continue


    lowered = (
        name.lower()
    )


    if not any(
        term
        in
        lowered
        for term
        in CELL10_INCLUDE_TERMS
    ):
        continue


    if any(
        term
        in
        lowered
        for term
        in CELL10_EXCLUDE_TERMS
    ):
        continue


    serialized_size = (
        cell10_serialized_size(
            value
        )
    )


    try:

        value_sha = (
            cell10_stable_sha(
                value
            )
        )

    except Exception as exc:

        cell10_failures.append({
            "reason":
                "CANDIDATE_FINGERPRINT_FAILED",

            "name":
                name,

            "type":
                type(
                    value
                ).__name__,

            "error":
                repr(
                    exc
                ),
        })

        continue


    cell10_candidate_rows.append({
        "name":
            name,

        "type":
            type(
                value
            ).__name__,

        "count":
            cell10_count(
                value
            ),

        "serialized_bytes":
            serialized_size,

        "sha256":
            value_sha,

        "contains_old_pilot":
            cell10_contains_pilot_id(
                value,
                cell07_old_pilot_id,
            ),

        "contains_replacement_pilot":
            cell10_contains_pilot_id(
                value,
                cell07_new_pilot_id,
            ),

        "is_final_runtime_object":
            name
            in
            CELL10_REQUIRED_FINAL_RUNTIME_NAMES,

        "is_callable":
            callable(
                value
            ),
    })


# ============================================================
# 7) Keep data registries separate from helper callables
# ============================================================

cell10_data_candidates = [
    row
    for row
    in cell10_candidate_rows
    if not row[
        "is_callable"
    ]
]


cell10_callable_candidates = [
    row
    for row
    in cell10_candidate_rows
    if row[
        "is_callable"
    ]
]


# ============================================================
# 8) Split compact vs large candidates
# ============================================================

CELL10_LARGE_OBJECT_THRESHOLD_BYTES = (
    2_000_000
)


cell10_compact_candidates = [
    row
    for row
    in cell10_data_candidates
    if (
        row[
            "serialized_bytes"
        ]
        is None
        or
        row[
            "serialized_bytes"
        ]
        <=
        CELL10_LARGE_OBJECT_THRESHOLD_BYTES
    )
]


cell10_large_candidates = [
    row
    for row
    in cell10_data_candidates
    if (
        isinstance(
            row[
                "serialized_bytes"
            ],
            int,
        )
        and
        row[
            "serialized_bytes"
        ]
        >
        CELL10_LARGE_OBJECT_THRESHOLD_BYTES
    )
]


# ============================================================
# 9) Workflow-family classification
# ============================================================

cell10_family_map = {
    "question":
        [],

    "expected_answer":
        [],

    "citation":
        [],

    "review_adjudication":
        [],

    "target_evidence":
        [],

    "golden_case":
        [],

    "pilot_queue_workflow":
        [],

    "lineage_replacement":
        [],
}


for row in (
    cell10_compact_candidates
):

    name = (
        row[
            "name"
        ].lower()
    )


    if (
        "question"
        in
        name
    ):
        cell10_family_map[
            "question"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "answer"
        in
        name
    ):
        cell10_family_map[
            "expected_answer"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "citation"
        in
        name
    ):
        cell10_family_map[
            "citation"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "review"
        in
        name
        or
        "adjudication"
        in
        name
    ):
        cell10_family_map[
            "review_adjudication"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "target_evidence"
        in
        name
    ):
        cell10_family_map[
            "target_evidence"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "golden"
        in
        name
        or
        "case"
        in
        name
    ):
        cell10_family_map[
            "golden_case"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "pilot"
        in
        name
        or
        "queue"
        in
        name
        or
        "workflow"
        in
        name
    ):
        cell10_family_map[
            "pilot_queue_workflow"
        ].append(
            row[
                "name"
            ]
        )


    if (
        "lineage"
        in
        name
        or
        "replacement"
        in
        name
        or
        "rejected"
        in
        name
        or
        "supersession"
        in
        name
    ):
        cell10_family_map[
            "lineage_replacement"
        ].append(
            row[
                "name"
            ]
        )


for family in (
    cell10_family_map
):

    cell10_family_map[
        family
    ] = sorted(
        set(
            cell10_family_map[
                family
            ]
        )
    )


# ============================================================
# 10) Historical / replacement coverage
# ============================================================

cell10_old_pilot_registry_names = sorted(
    row[
        "name"
    ]
    for row
    in cell10_compact_candidates
    if row[
        "contains_old_pilot"
    ]
)


cell10_replacement_registry_names = sorted(
    row[
        "name"
    ]
    for row
    in cell10_compact_candidates
    if row[
        "contains_replacement_pilot"
    ]
)


cell10_old_history_retained = (
    len(
        cell10_old_pilot_registry_names
    )
    >
    0
)


cell10_replacement_state_retained = (
    len(
        cell10_replacement_registry_names
    )
    >
    0
)


# ============================================================
# 11) Final runtime fingerprints
# ============================================================

cell10_final_runtime_fingerprints = {}


for name in (
    CELL10_REQUIRED_FINAL_RUNTIME_NAMES
):

    if (
        name
        not in
        cell09_seed_namespace
    ):
        continue


    try:

        cell10_final_runtime_fingerprints[
            name
        ] = (
            cell10_stable_sha(
                cell09_seed_namespace[
                    name
                ]
            )
        )

    except Exception as exc:

        cell10_failures.append({
            "reason":
                "FINAL_RUNTIME_FINGERPRINT_FAILED",

            "name":
                name,

            "error":
                repr(
                    exc
                ),
        })


cell10_final_runtime_surface_valid = (
    len(
        cell10_final_runtime_fingerprints
    )
    ==
    len(
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    )
)


# ============================================================
# 12) Inventory fingerprint
# ============================================================

cell10_inventory_summary = {
    "final_runtime_names":
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES,

    "final_runtime_fingerprints":
        cell10_final_runtime_fingerprints,

    "candidate_names":
        [
            row[
                "name"
            ]
            for row
            in cell10_compact_candidates
        ],

    "large_candidate_names":
        [
            row[
                "name"
            ]
            for row
            in cell10_large_candidates
        ],

    "family_map":
        cell10_family_map,

    "old_pilot_registry_names":
        cell10_old_pilot_registry_names,

    "replacement_registry_names":
        cell10_replacement_registry_names,
}


cell10_inventory_sha256 = (
    cell10_stable_sha(
        cell10_inventory_summary
    )
)


# ============================================================
# 13) Inventory gate
# ============================================================

cell10_gate = all([
    len(
        cell10_failures
    )
    ==
    0,

    cell10_tuple_key_probe_valid,

    cell10_final_runtime_surface_valid,

    len(
        cell10_candidate_rows
    )
    >
    0,

    len(
        cell10_data_candidates
    )
    >
    0,

    len(
        cell10_compact_candidates
    )
    >
    0,

    cell10_old_history_retained,

    cell10_replacement_state_retained,

    cell09_full_572_blocked
    is True,
])


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME CONTINUATION REGISTRY INVENTORY ==="
)


print(
    "\n=== CANONICALIZATION ==="
)

print(
    "Tuple-key probe:",
    cell10_tuple_key_probe_valid,
)

print(
    "Tuple-key probe SHA256:",
    cell10_tuple_key_probe_sha,
)


print(
    "\n=== FINAL RUNTIME SURFACE ==="
)

print(
    "Required objects:",
    len(
        CELL10_REQUIRED_FINAL_RUNTIME_NAMES
    ),
)

print(
    "Fingerprinted objects:",
    len(
        cell10_final_runtime_fingerprints
    ),
)

print(
    "Surface valid:",
    cell10_final_runtime_surface_valid,
)


print(
    "\n=== CANDIDATE INVENTORY ==="
)

print(
    "All matched objects:",
    len(
        cell10_candidate_rows
    ),
)

print(
    "Data candidates:",
    len(
        cell10_data_candidates
    ),
)

print(
    "Callable helpers excluded:",
    len(
        cell10_callable_candidates
    ),
)

print(
    "Compact candidates:",
    len(
        cell10_compact_candidates
    ),
)

print(
    "Large candidates:",
    len(
        cell10_large_candidates
    ),
)


print(
    "\n=== WORKFLOW FAMILIES ==="
)

for family, names in (
    cell10_family_map.items()
):

    print(
        "\n",
        family,
        ":",
        len(
            names
        ),
    )

    for name in (
        names[
            :40
        ]
    ):

        print(
            " -",
            name
        )


print(
    "\n=== OLD PILOT HISTORICAL COVERAGE ==="
)

print(
    "Registry count:",
    len(
        cell10_old_pilot_registry_names
    ),
)

for name in (
    cell10_old_pilot_registry_names[
        :40
    ]
):

    print(
        " -",
        name
    )


print(
    "\n=== REPLACEMENT PILOT COVERAGE ==="
)

print(
    "Registry count:",
    len(
        cell10_replacement_registry_names
    ),
)

for name in (
    cell10_replacement_registry_names[
        :40
    ]
):

    print(
        " -",
        name
    )


print(
    "\n=== LARGE CANDIDATES EXCLUDED FROM COMPACT SEED ==="
)

for row in (
    cell10_large_candidates
):

    print({
        "name":
            row[
                "name"
            ],

        "type":
            row[
                "type"
            ],

        "count":
            row[
                "count"
            ],

        "serialized_bytes":
            row[
                "serialized_bytes"
            ],
    })


print(
    "\n=== INVENTORY ==="
)

print(
    "Inventory SHA256:",
    cell10_inventory_sha256,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell10_failures
)


print(
    "\nFull 572 blocked:",
    cell09_full_572_blocked,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\nCODE CELL 10 GATE:",
    (
        "PASS"
        if cell10_gate
        else "FAIL"
    ),
)


assert cell10_gate, (
    "Compact continuation registry inventory failed. "
    "Repair this SAME Code Cell 10 before selecting the "
    "Compact Runtime Seed."
)

=== COMPACT RUNTIME CONTINUATION REGISTRY INVENTORY ===

=== CANONICALIZATION ===
Tuple-key probe: True
Tuple-key probe SHA256: d65d19b0488522ad25a77959860b00578311651d7d861e78270ac269510679fb

=== FINAL RUNTIME SURFACE ===
Required objects: 11
Fingerprinted objects: 11
Surface valid: True

=== CANDIDATE INVENTORY ===
All matched objects: 1120
Data candidates: 1102
Callable helpers excluded: 18
Compact candidates: 1075
Large candidates: 27

=== WORKFLOW FAMILIES ===

 question : 139
 - CELL61_QUESTION_GENERATION_STATUS
 - CELL66_REQUIRED_PREQUESTION_REBUILDS
 - CELL68_PREQUESTION_OBJECTS
 - CELL69_REQUIRED_PREQUESTION_REBUILDS_V2
 - CELL76_REAL_QUESTION_TEXT_FIELD_NAMES
 - CELL79_QUESTION_ID_SEARCH_TERMS
 - CELL79_QUESTION_OUTPUT_FIELD_CANDIDATES
 - CELL79_QUESTION_SCHEMA_SEARCH_TERMS
 - CELL82_QUESTION_TEXT
 - CELL86_REVISED_QUESTION_TEXT
 - CELL97_QUESTION_TEXT
 - EXERCISE_V2_QUESTION_CONTEXT_POLICY
 - PILOT_QUESTION_AUTHORING_MODE_V1
 - PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT
 - P

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 11 — ساخت Compact Runtime Seed برای ادامه‌ی 22 Pilot

Cell 10 نشان داد که Final Runtime معتبر است، اما Namespace بازسازی‌شده شامل تعداد زیادی State موقت و چند Registry بسیار بزرگ است.

در این مرحله یک Runtime Seed فشرده در حافظه ساخته می‌شود که فقط شامل موارد لازم برای ادامه‌ی Pilot Workflow است:

- 11 شیء Final Runtime پذیرفته‌شده؛
- قراردادهای اصلی Question Drafting و Human Validation؛
- تاریخچه‌ی Pilot تکمیل‌شده‌ی اول؛
- lineage و audit لازم برای Pilot ردشده و Replacement فعال‌شده؛
- فقط Golden Case، Task Draft و Target Evidence مربوط به 23 Pilot فعال، نه Registry کامل 572/724 رکوردی؛
- Source توابع ضروری، بدون نگه‌داشتن خود Function Objectهایی که به Namespace بزرگ NB12 وابسته‌اند؛
- fingerprint و metadata لازم برای کنترل byte-stability و ادامه‌ی Workflow.

این Cell هیچ Pilot جدیدی را پردازش نمی‌کند، Queue را جلو نمی‌برد، Human Decision ایجاد نمی‌کند و Full 572 Generation را باز نمی‌کند.

هیچ فایل، Artifact، Commit، Push یا PR ایجاد نمی‌شود.

</div>

In [30]:
# Code Cell 11

cell11_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell10_gate"
    )
    is not True
):
    cell11_failures.append(
        "CELL10_NOT_PASS"
    )


if (
    globals().get(
        "cell09_gate"
    )
    is not True
):
    cell11_failures.append(
        "CELL09_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell09_seed_namespace"
    ),
    dict,
):
    cell11_failures.append(
        "CELL09_SEED_NAMESPACE_MISSING"
    )


# ============================================================
# 2) Helpers
# ============================================================

def cell11_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:
        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            result[
                "pilot"
            ].add(
                value
            )

        elif value.startswith(
            "golden-case:"
        ):
            result[
                "golden_case"
            ].add(
                value
            )

        elif value.startswith(
            "task-draft:"
        ):
            result[
                "task_draft"
            ].add(
                value
            )

        elif value.startswith(
            "target-evidence:"
        ):
            result[
                "target_evidence"
            ].add(
                value
            )

        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell11_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell11_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell11_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell11_subset_list_by_field(
    records,
    field,
    allowed_ids,
):

    if not isinstance(
        records,
        list,
    ):
        return []

    return [
        copy.deepcopy(
            record
        )

        for record
        in records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                field
            )
            in
            allowed_ids
        )
    ]


def cell11_subset_dict_by_keys(
    mapping,
    allowed_ids,
):

    if not isinstance(
        mapping,
        dict,
    ):
        return {}

    return {
        key:
            copy.deepcopy(
                mapping[
                    key
                ]
            )

        for key
        in allowed_ids

        if key
        in
        mapping
    }


def cell11_contains_callable(
    value,
    seen=None,
):

    if callable(
        value
    ):
        return True


    if seen is None:
        seen = set()


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return False


    object_id = id(
        value
    )


    if object_id in seen:
        return False


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            return any(
                cell11_contains_callable(
                    key,
                    seen,
                )
                or
                cell11_contains_callable(
                    nested,
                    seen,
                )

                for key, nested
                in value.items()
            )


        return any(
            cell11_contains_callable(
                nested,
                seen,
            )

            for nested
            in value
        )


    finally:

        seen.discard(
            object_id
        )


# ============================================================
# 3) Freeze exact 11-object Final Runtime
# ============================================================

CELL11_FINAL_RUNTIME_NAMES = list(
    CELL10_REQUIRED_FINAL_RUNTIME_NAMES
)


cell11_final_runtime = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell09_seed_namespace
}


cell11_final_runtime_complete = (
    set(
        cell11_final_runtime
    )
    ==
    set(
        CELL11_FINAL_RUNTIME_NAMES
    )
)


# ============================================================
# 4) Resolve active / completed Pilot identities
# ============================================================

cell11_runtime_ids = (
    cell11_final_runtime.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell11_remaining_ids = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell11_active_pilot_ids = (
    list(
        cell11_runtime_ids
    )

    if isinstance(
        cell11_runtime_ids,
        list,
    )

    else []
)


cell11_completed_pilot_ids = [
    pilot_id

    for pilot_id
    in cell11_active_pilot_ids

    if (
        isinstance(
            cell11_remaining_ids,
            list,
        )
        and
        pilot_id
        not in
        cell11_remaining_ids
    )
]


cell11_completed_pilot_identity_valid = (
    len(
        cell11_completed_pilot_ids
    )
    ==
    1
)


cell11_completed_pilot_id = (
    cell11_completed_pilot_ids[
        0
    ]

    if cell11_completed_pilot_identity_valid

    else None
)


# ============================================================
# 5) Resolve IDs belonging to active 23-Pilot surface
# ============================================================

cell11_active_id_sets = (
    cell11_collect_prefixed_ids(
        cell11_final_runtime
    )
)


cell11_active_pilot_id_set = (
    cell11_active_id_sets[
        "pilot"
    ]
)


cell11_active_case_ids = (
    cell11_active_id_sets[
        "golden_case"
    ]
)


cell11_active_task_ids = (
    cell11_active_id_sets[
        "task_draft"
    ]
)


cell11_active_target_ids = (
    cell11_active_id_sets[
        "target_evidence"
    ]
)


cell11_active_identity_surface_valid = all([
    len(
        cell11_active_pilot_id_set
    )
    ==
    23,

    len(
        cell11_active_case_ids
    )
    ==
    23,

    len(
        cell11_active_task_ids
    )
    ==
    23,

    len(
        cell11_active_target_ids
    )
    ==
    23,
])


# ============================================================
# 6) Core frozen contracts + primitive constants
# ============================================================

CELL11_REQUIRED_CONTRACT_NAMES = [
    "PILOT_QA_WORKFLOW_CONTRACT_V1",
    "QUESTION_DRAFT_RECORD_SCHEMA_V1",
    "QUESTION_DRAFT_ID_CONTRACT_V1",
    "QUESTION_DRAFT_ID_METHOD_VERSION",
    "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1",
    "PILOT_QUESTION_AUTHORING_MODE_V1",
    "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1",
    "TARGET_EVIDENCE_SCHEMA_VERSION",
]


CELL11_OPTIONAL_CONTRACT_NAMES = [
    "PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT",
    "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1",
    "QUESTION_ARCHETYPES_V1",
    "QUESTION_ARCHETYPE_BY_STRATUM",
    "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT",
    "QUESTION_CONSTRUCTION_METHOD_V1_DRAFT",
    "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
    "TARGET_EVIDENCE_REGISTRY_V1_DRAFT",
    "TARGET_EVIDENCE_SELECTION_READINESS_V1_DRAFT",
    "GOLDEN_CASE_SCHEMA_VERSION",
    "GOLDEN_CASE_RECORD_CONTRACT_V1_DRAFT",
    "GOLDEN_CASE_REGISTRY_V1_DRAFT",
    "TASK_DRAFT_SCHEMA_V1_DRAFT",
]


cell11_missing_required_contracts = [
    name

    for name
    in CELL11_REQUIRED_CONTRACT_NAMES

    if name
    not in
    cell09_seed_namespace
]


if (
    cell11_missing_required_contracts
):

    cell11_failures.append({
        "reason":
            "REQUIRED_CONTRACTS_MISSING",

        "names":
            cell11_missing_required_contracts,
    })


cell11_contracts = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in (
        CELL11_REQUIRED_CONTRACT_NAMES
        +
        CELL11_OPTIONAL_CONTRACT_NAMES
    )

    if name
    in
    cell09_seed_namespace
}


# ============================================================
# 7) Preserve accepted Pilot #1 history
# ============================================================

cell11_completed_history_names = []


if (
    cell11_completed_pilot_id
    is not None
):

    for row in (
        cell10_compact_candidates
    ):

        name = (
            row[
                "name"
            ]
        )


        if not (
            row.get(
                "serialized_bytes"
            )
            is None
            or
            row.get(
                "serialized_bytes"
            )
            <=
            500_000
        ):
            continue


        if not (
            row.get(
                "contains_old_pilot"
            )
            is False
        ):
            continue


        if not (
            name.startswith(
                (
                    "cell82_",
                    "cell83_",
                    "cell84_",
                    "cell85_",
                    "cell86_",
                    "cell87_",
                    "cell88_",
                    "cell89_",
                    "cell90_",
                    "cell91_",
                    "cell92_",
                    "cell93_",
                    "cell94_",
                )
            )
        ):
            continue


        value = (
            cell09_seed_namespace.get(
                name
            )
        )


        ids = (
            cell11_collect_prefixed_ids(
                value
            )
        )


        if (
            cell11_completed_pilot_id
            in
            ids[
                "pilot"
            ]
        ):

            cell11_completed_history_names.append(
                name
            )


cell11_completed_history_names = sorted(
    set(
        cell11_completed_history_names
    )
)


cell11_completed_pilot_history = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in cell11_completed_history_names
}


# ============================================================
# 8) Preserve rejected Pilot / replacement audit lineage
# ============================================================

CELL11_REPLACEMENT_HISTORY_NAMES = [
    "cell100_human_adjudication_record",
    "cell109_revision_lineage_interpretation",
    "cell121_pilot_replacement_policy_inspection",
    "cell127_candidate_evidence_review",
    "cell130_replacement_packet_simulation",
    "cell132_full_replacement_transaction_simulation",
    "cell135_queue_replacement_and_supersession_simulation",
    "cell138_supersession_lineage_payload",
    "cell140_replacement_packet",
]


cell11_replacement_history = {
    name:
        copy.deepcopy(
            cell09_seed_namespace[
                name
            ]
        )

    for name
    in CELL11_REPLACEMENT_HISTORY_NAMES

    if name
    in
    cell09_seed_namespace
}


cell11_replacement_history_ids = (
    cell11_collect_prefixed_ids(
        cell11_replacement_history
    )
)


cell11_old_pilot_history_present = (
    cell07_old_pilot_id
    in
    cell11_replacement_history_ids[
        "pilot"
    ]
)


cell11_new_pilot_history_present = (
    cell07_new_pilot_id
    in
    cell11_replacement_history_ids[
        "pilot"
    ]
)


# ============================================================
# 9) Subset large Golden / Task / Target registries
# ============================================================

cell11_golden_case_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "golden_case_records_by_id"
        ),
        cell11_active_case_ids,
    )
)


cell11_golden_case_records_subset = (
    cell11_subset_list_by_field(
        cell09_seed_namespace.get(
            "golden_case_records"
        ),
        "golden_case_id",
        cell11_active_case_ids,
    )
)


cell11_target_by_case_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "target_evidence_by_case_id"
        ),
        cell11_active_case_ids,
    )
)


cell11_target_by_task_subset = (
    cell11_subset_dict_by_keys(
        cell09_seed_namespace.get(
            "target_evidence_by_task_draft_id"
        ),
        cell11_active_task_ids,
    )
)


cell11_target_records_subset = (
    cell11_subset_list_by_field(
        cell09_seed_namespace.get(
            "target_evidence_records"
        ),
        "target_evidence_id",
        cell11_active_target_ids,
    )
)


cell11_task_plan_subset = {}


if isinstance(
    cell09_seed_namespace.get(
        "task_draft_plans"
    ),
    dict,
):

    full_task_plans = (
        cell09_seed_namespace[
            "task_draft_plans"
        ]
    )


    for key, value in (
        full_task_plans.items()
    ):

        value_ids = (
            cell11_collect_prefixed_ids(
                value
            )
        )


        if (
            key
            in
            cell11_active_task_ids
            or
            key
            in
            cell11_active_case_ids
            or
            bool(
                value_ids[
                    "task_draft"
                ]
                &
                cell11_active_task_ids
            )
            or
            bool(
                value_ids[
                    "golden_case"
                ]
                &
                cell11_active_case_ids
            )
        ):

            cell11_task_plan_subset[
                key
            ] = copy.deepcopy(
                value
            )


cell11_task_plan_rows_subset = []


if isinstance(
    cell09_seed_namespace.get(
        "task_draft_plan_rows"
    ),
    list,
):

    for row in (
        cell09_seed_namespace[
            "task_draft_plan_rows"
        ]
    ):

        if not isinstance(
            row,
            dict,
        ):
            continue


        if (
            row.get(
                "task_draft_id"
            )
            in
            cell11_active_task_ids
            or
            row.get(
                "golden_case_id"
            )
            in
            cell11_active_case_ids
        ):

            cell11_task_plan_rows_subset.append(
                copy.deepcopy(
                    row
                )
            )


# ============================================================
# 10) Validate active support completeness
# ============================================================

cell11_golden_support_valid = (
    len(
        cell11_golden_case_subset
    )
    ==
    23
)


cell11_target_case_support_valid = (
    len(
        cell11_target_by_case_subset
    )
    ==
    23
)


cell11_target_task_support_valid = (
    len(
        cell11_target_by_task_subset
    )
    ==
    23
)


cell11_target_record_ids = {
    row.get(
        "target_evidence_id"
    )

    for row
    in cell11_target_records_subset

    if isinstance(
        row,
        dict,
    )
}


cell11_target_record_support_valid = (
    cell11_active_target_ids
    <=
    cell11_target_record_ids
)


# ============================================================
# 11) Capture complete reusable callable SOURCE closure
# ============================================================

CELL11_REQUIRED_CALLABLE_NAMES = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


cell11_callable_sources = {}

cell11_callable_source_failures = []


cell11_inspect = (
    __import__(
        "inspect"
    )
)


for name in (
    CELL11_REQUIRED_CALLABLE_NAMES
):

    value = (
        cell09_seed_namespace.get(
            name
        )
    )


    if not callable(
        value
    ):

        cell11_callable_source_failures.append({
            "name":
                name,

            "reason":
                "REQUIRED_CALLABLE_MISSING",
        })

        continue


    try:

        source = (
            cell11_inspect.getsource(
                value
            )
        )


        cell11_callable_sources[
            name
        ] = {
            "source":
                source,

            "source_sha256":
                hashlib.sha256(
                    source.encode(
                        "utf-8"
                    )
                ).hexdigest(),

            "module":
                getattr(
                    value,
                    "__module__",
                    None,
                ),
        }


    except Exception as exc:

        cell11_callable_source_failures.append({
            "name":
                name,

            "reason":
                "SOURCE_RECOVERY_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell11_required_callable_sources_valid = all(
    name
    in
    cell11_callable_sources

    for name
    in CELL11_REQUIRED_CALLABLE_NAMES
)


# ============================================================
# 12) Large registry references only
# ============================================================

CELL11_LARGE_REGISTRY_REFERENCE_NAMES = [
    "golden_case_records",
    "golden_case_records_by_id",
    "target_evidence_records",
    "target_evidence_by_case_id",
    "target_evidence_by_task_draft_id",
]


cell11_large_registry_references = {}


cell11_large_candidate_by_name = {
    row[
        "name"
    ]:
        row

    for row
    in cell10_large_candidates
}


for name in (
    CELL11_LARGE_REGISTRY_REFERENCE_NAMES
):

    row = (
        cell11_large_candidate_by_name.get(
            name
        )
    )


    if row is None:
        continue


    cell11_large_registry_references[
        name
    ] = {
        "type":
            row[
                "type"
            ],

        "count":
            row[
                "count"
            ],

        "serialized_bytes":
            row[
                "serialized_bytes"
            ],

        "sha256":
            row[
                "sha256"
            ],
    }


# ============================================================
# 13) Build Compact Runtime Seed
# ============================================================

cell11_compact_runtime_seed = {
    "seed_version":
        "golden_dataset_compact_runtime_seed_v1",

    "status":
        "READY_FOR_CONTINUATION_ENGINE_VALIDATION",

    "source":
        {
            "prior_notebook":
                str(
                    prior_notebook_path
                ),

            "cell09_target_bundle_sha256":
                cell09_target_bundle_sha256,

            "cell10_inventory_sha256":
                cell10_inventory_sha256,

            "full_572_generation":
                "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION",
        },

    "final_runtime":
        cell11_final_runtime,

    "contracts":
        cell11_contracts,

    "completed_pilot":
        {
            "pilot_id":
                cell11_completed_pilot_id,

            "history":
                cell11_completed_pilot_history,
        },

    "replacement_audit":
        {
            "old_pilot_id":
                cell07_old_pilot_id,

            "replacement_pilot_id":
                cell07_new_pilot_id,

            "history":
                cell11_replacement_history,
        },

    "active_support":
        {
            "golden_case_records_by_id":
                cell11_golden_case_subset,

            "golden_case_records":
                cell11_golden_case_records_subset,

            "task_draft_plans":
                cell11_task_plan_subset,

            "task_draft_plan_rows":
                cell11_task_plan_rows_subset,

            "target_evidence_by_case_id":
                cell11_target_by_case_subset,

            "target_evidence_by_task_draft_id":
                cell11_target_by_task_subset,

            "target_evidence_records":
                cell11_target_records_subset,
        },

    "callable_sources":
        cell11_callable_sources,

    "full_registry_references":
        cell11_large_registry_references,
}


# ============================================================
# 14) Compactness + isolation checks
# ============================================================

cell11_seed_serialized_bytes = (
    cell10_serialized_size(
        cell11_compact_runtime_seed
    )
)


cell11_seed_sha256 = (
    cell10_stable_sha(
        cell11_compact_runtime_seed
    )
)


CELL11_MAX_SEED_BYTES = (
    8
    *
    1024
    *
    1024
)


cell11_seed_is_compact = (
    isinstance(
        cell11_seed_serialized_bytes,
        int,
    )
    and
    cell11_seed_serialized_bytes
    <
    CELL11_MAX_SEED_BYTES
)


cell11_seed_has_no_callable_objects = (
    not
    cell11_contains_callable(
        cell11_compact_runtime_seed
    )
)


cell11_runtime_copy_fingerprints = {
    name:
        cell10_stable_sha(
            cell11_final_runtime[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell11_final_runtime
}


cell11_runtime_fingerprints_match = (
    cell11_runtime_copy_fingerprints
    ==
    cell10_final_runtime_fingerprints
)


# ============================================================
# 15) Final continuation-state invariants
# ============================================================

cell11_next_pilot_id = (
    cell11_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell11_queue = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell11_queue_ids = (
    cell11_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell11_next_replacement_valid = all([
    cell11_next_pilot_id
    ==
    cell07_new_pilot_id,

    isinstance(
        cell11_queue,
        list,
    ),

    len(
        cell11_queue
    )
    ==
    22,

    isinstance(
        cell11_queue_ids,
        list,
    ),

    len(
        cell11_queue_ids
    )
    ==
    22,

    cell11_queue_ids[
        0
    ]
    ==
    cell07_new_pilot_id,
])


cell11_old_not_active = (
    cell07_old_pilot_id
    not in
    cell11_active_pilot_ids
)


cell11_replacement_active = (
    cell07_new_pilot_id
    in
    cell11_active_pilot_ids
)


cell11_full_572_still_blocked = (
    cell11_compact_runtime_seed[
        "source"
    ][
        "full_572_generation"
    ]
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 16) Final Gate
# ============================================================

cell11_gate = all([
    len(
        cell11_failures
    )
    ==
    0,

    cell11_final_runtime_complete,

    cell11_completed_pilot_identity_valid,

    cell11_active_identity_surface_valid,

    len(
        cell11_completed_pilot_history
    )
    >
    0,

    cell11_old_pilot_history_present,

    cell11_new_pilot_history_present,

    cell11_golden_support_valid,

    cell11_target_case_support_valid,

    cell11_target_task_support_valid,

    cell11_target_record_support_valid,

    len(
        cell11_callable_source_failures
    )
    ==
    0,

    cell11_required_callable_sources_valid,

    cell11_seed_is_compact,

    cell11_seed_has_no_callable_objects,

    cell11_runtime_fingerprints_match,

    cell11_next_replacement_valid,

    cell11_old_not_active,

    cell11_replacement_active,

    cell11_full_572_still_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME SEED BUILD ==="
)


print(
    "\n=== FINAL RUNTIME ==="
)

print(
    "Objects:",
    len(
        cell11_final_runtime
    ),
)

print(
    "Fingerprint match:",
    cell11_runtime_fingerprints_match,
)


print(
    "\n=== PILOT STATE ==="
)

print(
    "Active Pilots:",
    len(
        cell11_active_pilot_ids
    ),
)

print(
    "Completed Pilots:",
    cell11_completed_pilot_ids,
)

print(
    "Remaining Pilots:",
    len(
        cell11_queue_ids
    ),
)

print(
    "Next Pilot:",
    cell11_next_pilot_id,
)

print(
    "Old Pilot active:",
    not cell11_old_not_active,
)

print(
    "Replacement active:",
    cell11_replacement_active,
)


print(
    "\n=== ACTIVE ID SURFACE ==="
)

print(
    "Pilot IDs:",
    len(
        cell11_active_pilot_id_set
    ),
)

print(
    "Golden Case IDs:",
    len(
        cell11_active_case_ids
    ),
)

print(
    "Task Draft IDs:",
    len(
        cell11_active_task_ids
    ),
)

print(
    "Target Evidence IDs:",
    len(
        cell11_active_target_ids
    ),
)


print(
    "\n=== ACTIVE SUPPORT SUBSETS ==="
)

print(
    "Golden Cases:",
    len(
        cell11_golden_case_subset
    ),
)

print(
    "Target by Case:",
    len(
        cell11_target_by_case_subset
    ),
)

print(
    "Target by Task:",
    len(
        cell11_target_by_task_subset
    ),
)

print(
    "Target Records:",
    len(
        cell11_target_records_subset
    ),
)

print(
    "Task Plans:",
    len(
        cell11_task_plan_subset
    ),
)

print(
    "Task Plan Rows:",
    len(
        cell11_task_plan_rows_subset
    ),
)


print(
    "\n=== HISTORY ==="
)

print(
    "Completed Pilot history objects:",
    len(
        cell11_completed_pilot_history
    ),
)

print(
    "Replacement audit objects:",
    len(
        cell11_replacement_history
    ),
)

print(
    "Old Pilot historical:",
    cell11_old_pilot_history_present,
)

print(
    "Replacement historical:",
    cell11_new_pilot_history_present,
)


print(
    "\n=== CONTRACT DEPENDENCIES ==="
)

print(
    "QUESTION_DRAFT_ID_METHOD_VERSION:",
    cell11_contracts.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    ),
)

print(
    "TARGET_EVIDENCE_SCHEMA_VERSION:",
    cell11_contracts.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    ),
)


print(
    "\n=== CALLABLE SOURCES ==="
)

print(
    "Required:",
    CELL11_REQUIRED_CALLABLE_NAMES,
)

print(
    "Recovered:",
    sorted(
        cell11_callable_sources
    ),
)

print(
    "Failures:",
    cell11_callable_source_failures,
)

print(
    "Callable objects carried into Seed:",
    not cell11_seed_has_no_callable_objects,
)


print(
    "\n=== COMPACTNESS ==="
)

print(
    "Seed bytes:",
    cell11_seed_serialized_bytes,
)

print(
    "Seed MiB:",
    round(
        cell11_seed_serialized_bytes
        /
        (
            1024
            *
            1024
        ),
        3,
    )
    if isinstance(
        cell11_seed_serialized_bytes,
        int,
    )
    else None,
)

print(
    "Limit bytes:",
    CELL11_MAX_SEED_BYTES,
)

print(
    "Compact under 8 MiB:",
    cell11_seed_is_compact,
)

print(
    "Seed SHA256:",
    cell11_seed_sha256,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell11_full_572_still_blocked,
)

print(
    "Mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell11_failures
)


print(
    "\nCODE CELL 11 GATE:",
    (
        "PASS"
        if cell11_gate
        else "FAIL"
    ),
)


assert cell11_gate, (
    "Compact Runtime Seed construction failed. "
    "Repair this SAME Code Cell 11 before validating "
    "Seed independence."
)

=== COMPACT RUNTIME SEED BUILD ===

=== FINAL RUNTIME ===
Objects: 11
Fingerprint match: True

=== PILOT STATE ===
Active Pilots: 23
Completed Pilots: ['question-pilot:02ba2886f1b68521b93789becf741361']
Remaining Pilots: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Old Pilot active: False
Replacement active: True

=== ACTIVE ID SURFACE ===
Pilot IDs: 23
Golden Case IDs: 23
Task Draft IDs: 23
Target Evidence IDs: 23

=== ACTIVE SUPPORT SUBSETS ===
Golden Cases: 23
Target by Case: 23
Target by Task: 23
Target Records: 23
Task Plans: 23
Task Plan Rows: 23

=== HISTORY ===
Completed Pilot history objects: 17
Replacement audit objects: 9
Old Pilot historical: True
Replacement historical: True

=== CONTRACT DEPENDENCIES ===
QUESTION_DRAFT_ID_METHOD_VERSION: question_draft_id_method_v1
TARGET_EVIDENCE_SCHEMA_VERSION: target_evidence_v1_draft

=== CALLABLE SOURCES ===
Required: ['stable_json_key', 'target_part_identity', 'build_target_evidence_id', 'build_question_draft_id']


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 12 — اعتبارسنجی استقلال Compact Runtime Seed

در Cell 11 یک Compact Runtime Seed زیر 8 MiB ساخته شد که Final Runtime، پشتیبانی لازم برای 23 Pilot فعال، تاریخچه‌ی Pilot تکمیل‌شده، lineage مربوط به Replacement و Source توابع ضروری را در خود نگه می‌دارد.

هدف این مرحله اثبات این است که این Seed می‌تواند در یک Namespace تازه و مستقل مورد استفاده قرار گیرد و برای اعتبارسنجی Runtime به Namespace سنگین حاصل از Replay کامل NB12 وابسته نباشد.

در این Cell بررسی می‌شود که:

- fingerprint کامل Seed بدون تغییر باقی مانده باشد؛
- 11 شیء Final Runtime به‌صورت کامل در Seed وجود داشته باشند؛
- 23 Pilot فعال، 22 Pilot باقی‌مانده و Next Pilot دقیقاً حفظ شده باشند؛
- Pilot ردشده فقط در Historical/Audit State باقی مانده و در Active Runtime حضور نداشته باشد؛
- Golden Case، Task Draft و Target Evidence مربوط به هر 23 Pilot قابل بازیابی باشند؛
- Source توابع ضروری قابل Compile و بازسازی در یک Namespace تازه باشد؛
- dependencyهای global این توابع بدون استفاده از `cell09_seed_namespace` قابل Resolve باشند؛
- هیچ Function Object متصل به Namespace قدیمی داخل Seed حمل نشده باشد؛
- یک Deep-Copy مستقل از Seed بتواند تغییر کند بدون اینکه Seed اصلی تغییر کند؛
- Full 572 Generation همچنان Blocked باقی بماند؛
- هیچ فایل، Commit، Push، PR یا Artifact ایجاد یا تغییر داده نشود.

در این مرحله هنوز هیچ Question جدید، Expected Answer جدید، Citation جدید یا Queue Advancement انجام نمی‌شود.

</div>

In [33]:
# Code Cell 12

cell12_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell11_gate"
    )
    is not True
):
    cell12_failures.append(
        "CELL11_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell11_compact_runtime_seed"
    ),
    dict,
):
    cell12_failures.append(
        "COMPACT_RUNTIME_SEED_MISSING"
    )


# ============================================================
# 2) Accepted identities
# ============================================================

CELL12_EXPECTED_OLD_PILOT_ID = (
    "question-pilot:"
    "532a8b9d45efb8b3a32c4a62f096bb7a"
)


CELL12_EXPECTED_REPLACEMENT_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL12_EXPECTED_COMPLETED_PILOT_ID = (
    "question-pilot:"
    "02ba2886f1b68521b93789becf741361"
)


CELL12_EXPECTED_ACTIVE_COUNT = 23
CELL12_EXPECTED_REMAINING_COUNT = 22


CELL12_REQUIRED_CALLABLE_NAMES = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


CELL12_CALLABLE_EXECUTION_ORDER = [
    "stable_json_key",
    "target_part_identity",
    "build_target_evidence_id",
    "build_question_draft_id",
]


# ============================================================
# 3) Helpers
# ============================================================

def cell12_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:
        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):
            result[
                "pilot"
            ].add(
                value
            )


        elif value.startswith(
            "golden-case:"
        ):
            result[
                "golden_case"
            ].add(
                value
            )


        elif value.startswith(
            "task-draft:"
        ):
            result[
                "task_draft"
            ].add(
                value
            )


        elif value.startswith(
            "target-evidence:"
        ):
            result[
                "target_evidence"
            ].add(
                value
            )


        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell12_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell12_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell12_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell12_contains_callable(
    value,
    seen=None,
):

    if callable(
        value
    ):
        return True


    if seen is None:
        seen = set()


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):
        return False


    object_id = id(
        value
    )


    if object_id in seen:
        return False


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                if cell12_contains_callable(
                    key,
                    seen,
                ):
                    return True


                if cell12_contains_callable(
                    nested,
                    seen,
                ):
                    return True


            return False


        for nested in value:

            if cell12_contains_callable(
                nested,
                seen,
            ):
                return True


        return False


    finally:

        seen.discard(
            object_id
        )


def cell12_function_global_loads(
    source,
):

    tree = ast.parse(
        source
    )


    function_nodes = [
        node

        for node
        in tree.body

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        )
    ]


    if len(
        function_nodes
    ) != 1:

        return {
            "function_name":
                None,

            "global_loads":
                set(),

            "error":
                "EXPECTED_EXACTLY_ONE_TOP_LEVEL_FUNCTION",
        }


    function_node = (
        function_nodes[
            0
        ]
    )


    local_names = set()


    for argument in (
        list(
            function_node.args.posonlyargs
        )
        +
        list(
            function_node.args.args
        )
        +
        list(
            function_node.args.kwonlyargs
        )
    ):

        local_names.add(
            argument.arg
        )


    if (
        function_node.args.vararg
        is not None
    ):

        local_names.add(
            function_node.args.vararg.arg
        )


    if (
        function_node.args.kwarg
        is not None
    ):

        local_names.add(
            function_node.args.kwarg.arg
        )


    for node in ast.walk(
        function_node
    ):

        if (
            isinstance(
                node,
                ast.Name,
            )
            and
            isinstance(
                node.ctx,
                (
                    ast.Store,
                    ast.Del,
                ),
            )
        ):

            local_names.add(
                node.id
            )


    loaded_names = {
        node.id

        for node
        in ast.walk(
            function_node
        )

        if (
            isinstance(
                node,
                ast.Name,
            )
            and
            isinstance(
                node.ctx,
                ast.Load,
            )
        )
    }


    builtin_names = set(
        dir(
            __import__(
                "builtins"
            )
        )
    )


    return {
        "function_name":
            function_node.name,

        "global_loads":
            (
                loaded_names
                -
                local_names
                -
                builtin_names
            ),

        "error":
            None,
    }


def cell12_walk_dict_candidates(
    value,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            yield (
                path,
                value,
            )


            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                yield from (
                    cell12_walk_dict_candidates(
                        nested,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield from (
                    cell12_walk_dict_candidates(
                        nested,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


# ============================================================
# 4) Freeze external state
# ============================================================

(
    cell12_git_before_rc,
    cell12_git_before,
    cell12_git_before_err,
) = run_git(
    "status",
    "--short",
)


cell12_prior_before = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell12_handoff_before = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


# ============================================================
# 5) Independent Seed copy
# ============================================================

cell12_seed = copy.deepcopy(
    cell11_compact_runtime_seed
)


cell12_seed_sha256 = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_seed_fingerprint_valid = (
    cell12_seed_sha256
    ==
    cell11_seed_sha256
)


cell12_seed_has_no_callable_objects = (
    not
    cell12_contains_callable(
        cell12_seed
    )
)


cell12_seed_text = (
    cell10_canonical_json(
        cell12_seed
    )
)


cell12_forbidden_runtime_reference_absent = (
    "cell09_seed_namespace"
    not in
    cell12_seed_text
)


# ============================================================
# 6) Build fresh isolated Runtime Namespace
# ============================================================

cell12_runtime_namespace = {
    "__builtins__":
        __builtins__,

    "__name__":
        "compact_seed_independent_runtime",

    "hashlib":
        hashlib,

    "json":
        json,

    "re":
        re,

    "copy":
        copy,

    "Path":
        Path,

    "Counter":
        Counter,

    "defaultdict":
        defaultdict,
}


cell12_runtime_namespace[
    "COMPACT_RUNTIME_SEED"
] = copy.deepcopy(
    cell12_seed
)


cell12_seed_runtime = (
    cell12_runtime_namespace[
        "COMPACT_RUNTIME_SEED"
    ]
)


cell12_contracts = (
    cell12_seed_runtime.get(
        "contracts",
        {},
    )
)


cell12_final_runtime = (
    cell12_seed_runtime.get(
        "final_runtime",
        {},
    )
)


cell12_active_support = (
    cell12_seed_runtime.get(
        "active_support",
        {},
    )
)


# ============================================================
# 7) Promote contracts / constants
# ============================================================

if isinstance(
    cell12_contracts,
    dict,
):

    for name, value in (
        cell12_contracts.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


cell12_question_id_version_present = (
    cell12_runtime_namespace.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    )
    ==
    "question_draft_id_method_v1"
)


cell12_target_schema_version_present = (
    cell12_runtime_namespace.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    )
    ==
    "target_evidence_v1_draft"
)


# ============================================================
# 8) Promote Final Runtime / active support
# ============================================================

if isinstance(
    cell12_final_runtime,
    dict,
):

    for name, value in (
        cell12_final_runtime.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


if isinstance(
    cell12_active_support,
    dict,
):

    for name, value in (
        cell12_active_support.items()
    ):

        cell12_runtime_namespace[
            name
        ] = copy.deepcopy(
            value
        )


# ============================================================
# 9) Validate exact Final Runtime surface
# ============================================================

cell12_final_runtime_surface_valid = (
    set(
        cell12_final_runtime
    )
    ==
    set(
        CELL11_FINAL_RUNTIME_NAMES
    )
)


cell12_runtime_fingerprints = {
    name:
        cell10_stable_sha(
            cell12_final_runtime[
                name
            ]
        )

    for name
    in CELL11_FINAL_RUNTIME_NAMES

    if name
    in
    cell12_final_runtime
}


cell12_runtime_fingerprints_valid = (
    cell12_runtime_fingerprints
    ==
    cell10_final_runtime_fingerprints
)


# ============================================================
# 10) Validate Pilot workflow state
# ============================================================

cell12_runtime_ids = (
    cell12_final_runtime.get(
        "cell95_pilot_ids_in_runtime_order"
    )
)


cell12_remaining_ids = (
    cell12_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell12_remaining_queue = (
    cell12_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell12_next_pilot_id = (
    cell12_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell12_pilot_state_valid = all([
    isinstance(
        cell12_runtime_ids,
        list,
    ),

    len(
        cell12_runtime_ids
    )
    ==
    CELL12_EXPECTED_ACTIVE_COUNT,

    len(
        set(
            cell12_runtime_ids
        )
    )
    ==
    CELL12_EXPECTED_ACTIVE_COUNT,

    isinstance(
        cell12_remaining_ids,
        list,
    ),

    len(
        cell12_remaining_ids
    )
    ==
    CELL12_EXPECTED_REMAINING_COUNT,

    isinstance(
        cell12_remaining_queue,
        list,
    ),

    len(
        cell12_remaining_queue
    )
    ==
    CELL12_EXPECTED_REMAINING_COUNT,

    CELL12_EXPECTED_OLD_PILOT_ID
    not in
    cell12_runtime_ids,

    CELL12_EXPECTED_REPLACEMENT_PILOT_ID
    in
    cell12_runtime_ids,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    in
    cell12_runtime_ids,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    not in
    cell12_remaining_ids,

    cell12_next_pilot_id
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

    cell12_remaining_ids[
        0
    ]
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,
])


# ============================================================
# 11) Active identity surface
# ============================================================

cell12_runtime_id_sets = (
    cell12_collect_prefixed_ids(
        cell12_final_runtime
    )
)


cell12_runtime_pilot_ids = (
    cell12_runtime_id_sets[
        "pilot"
    ]
)


cell12_runtime_case_ids = (
    cell12_runtime_id_sets[
        "golden_case"
    ]
)


cell12_runtime_task_ids = (
    cell12_runtime_id_sets[
        "task_draft"
    ]
)


cell12_runtime_target_ids = (
    cell12_runtime_id_sets[
        "target_evidence"
    ]
)


cell12_identity_surface_valid = all([
    len(
        cell12_runtime_pilot_ids
    )
    ==
    23,

    len(
        cell12_runtime_case_ids
    )
    ==
    23,

    len(
        cell12_runtime_task_ids
    )
    ==
    23,

    len(
        cell12_runtime_target_ids
    )
    ==
    23,
])


# ============================================================
# 12) Active support
# ============================================================

cell12_support_case_map = (
    cell12_active_support.get(
        "golden_case_records_by_id",
        {},
    )
)


cell12_support_case_rows = (
    cell12_active_support.get(
        "golden_case_records",
        [],
    )
)


cell12_support_task_plans = (
    cell12_active_support.get(
        "task_draft_plans",
        {},
    )
)


cell12_support_task_rows = (
    cell12_active_support.get(
        "task_draft_plan_rows",
        [],
    )
)


cell12_support_target_by_case = (
    cell12_active_support.get(
        "target_evidence_by_case_id",
        {},
    )
)


cell12_support_target_by_task = (
    cell12_active_support.get(
        "target_evidence_by_task_draft_id",
        {},
    )
)


cell12_support_target_rows = (
    cell12_active_support.get(
        "target_evidence_records",
        [],
    )
)


cell12_support_case_row_ids = {
    row.get(
        "golden_case_id"
    )

    for row
    in cell12_support_case_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_task_row_ids = {
    row.get(
        "task_draft_id"
    )

    for row
    in cell12_support_task_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_target_ids = {
    row.get(
        "target_evidence_id"
    )

    for row
    in cell12_support_target_rows

    if isinstance(
        row,
        dict,
    )
}


cell12_support_surface_valid = all([
    isinstance(
        cell12_support_case_map,
        dict,
    ),

    len(
        cell12_support_case_map
    )
    ==
    23,

    set(
        cell12_support_case_map
    )
    ==
    cell12_runtime_case_ids,

    cell12_support_case_row_ids
    ==
    cell12_runtime_case_ids,

    isinstance(
        cell12_support_task_plans,
        dict,
    ),

    len(
        cell12_support_task_plans
    )
    ==
    23,

    isinstance(
        cell12_support_task_rows,
        list,
    ),

    len(
        cell12_support_task_rows
    )
    ==
    23,

    cell12_runtime_task_ids
    <=
    cell12_support_task_row_ids,

    isinstance(
        cell12_support_target_by_case,
        dict,
    ),

    len(
        cell12_support_target_by_case
    )
    ==
    23,

    set(
        cell12_support_target_by_case
    )
    ==
    cell12_runtime_case_ids,

    isinstance(
        cell12_support_target_by_task,
        dict,
    ),

    len(
        cell12_support_target_by_task
    )
    ==
    23,

    set(
        cell12_support_target_by_task
    )
    ==
    cell12_runtime_task_ids,

    cell12_runtime_target_ids
    <=
    cell12_support_target_ids,
])


# ============================================================
# 13) Historical state
# ============================================================

cell12_completed_section = (
    cell12_seed_runtime.get(
        "completed_pilot",
        {},
    )
)


cell12_replacement_section = (
    cell12_seed_runtime.get(
        "replacement_audit",
        {},
    )
)


cell12_completed_history_ids = (
    cell12_collect_prefixed_ids(
        cell12_completed_section
    )
)


cell12_replacement_history_ids = (
    cell12_collect_prefixed_ids(
        cell12_replacement_section
    )
)


cell12_history_valid = all([
    cell12_completed_section.get(
        "pilot_id"
    )
    ==
    CELL12_EXPECTED_COMPLETED_PILOT_ID,

    CELL12_EXPECTED_COMPLETED_PILOT_ID
    in
    cell12_completed_history_ids[
        "pilot"
    ],

    cell12_replacement_section.get(
        "old_pilot_id"
    )
    ==
    CELL12_EXPECTED_OLD_PILOT_ID,

    cell12_replacement_section.get(
        "replacement_pilot_id"
    )
    ==
    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

    CELL12_EXPECTED_OLD_PILOT_ID
    in
    cell12_replacement_history_ids[
        "pilot"
    ],

    CELL12_EXPECTED_REPLACEMENT_PILOT_ID
    in
    cell12_replacement_history_ids[
        "pilot"
    ],
])


# ============================================================
# 14) Validate callable-source closure
# ============================================================

cell12_callable_sources = (
    cell12_seed_runtime.get(
        "callable_sources",
        {},
    )
)


cell12_callable_source_names_valid = (
    set(
        cell12_callable_sources
    )
    ==
    set(
        CELL12_REQUIRED_CALLABLE_NAMES
    )
)


cell12_callable_source_checks = {}

cell12_callable_dependency_failures = []

cell12_callable_compile_failures = []


cell12_builtin_names = set(
    dir(
        __import__(
            "builtins"
        )
    )
)


for name in (
    CELL12_REQUIRED_CALLABLE_NAMES
):

    metadata = (
        cell12_callable_sources.get(
            name
        )
    )


    if not isinstance(
        metadata,
        dict,
    ):

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "CALLABLE_METADATA_MISSING",
        })

        continue


    source = (
        metadata.get(
            "source"
        )
    )


    expected_source_sha = (
        metadata.get(
            "source_sha256"
        )
    )


    if not isinstance(
        source,
        str,
    ):

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "CALLABLE_SOURCE_MISSING",
        })

        continue


    actual_source_sha = (
        hashlib.sha256(
            source.encode(
                "utf-8"
            )
        ).hexdigest()
    )


    source_sha_valid = (
        actual_source_sha
        ==
        expected_source_sha
    )


    dependency_result = (
        cell12_function_global_loads(
            source
        )
    )


    global_loads = (
        dependency_result[
            "global_loads"
        ]
    )


    available_names = (
        set(
            cell12_runtime_namespace
        )
        |
        set(
            CELL12_REQUIRED_CALLABLE_NAMES
        )
        |
        cell12_builtin_names
    )


    unresolved = (
        global_loads
        -
        available_names
    )


    if unresolved:

        cell12_callable_dependency_failures.append({
            "name":
                name,

            "unresolved":
                sorted(
                    unresolved
                ),
        })


    cell12_callable_source_checks[
        name
    ] = {
        "source_sha_valid":
            source_sha_valid,

        "function_name":
            dependency_result[
                "function_name"
            ],

        "global_loads":
            sorted(
                global_loads
            ),

        "unresolved_global_loads":
            sorted(
                unresolved
            ),

        "ast_error":
            dependency_result[
                "error"
            ],
    }


# ============================================================
# 15) Reconstruct all required callables
# ============================================================

for name in (
    CELL12_CALLABLE_EXECUTION_ORDER
):

    metadata = (
        cell12_callable_sources.get(
            name
        )
    )


    if not isinstance(
        metadata,
        dict,
    ):
        continue


    source = (
        metadata.get(
            "source"
        )
    )


    if not isinstance(
        source,
        str,
    ):
        continue


    try:

        exec(
            compile(
                source,
                filename=(
                    f"<compact-seed-{name}>"
                ),
                mode="exec",
            ),
            cell12_runtime_namespace,
            cell12_runtime_namespace,
        )


    except Exception as exc:

        cell12_callable_compile_failures.append({
            "name":
                name,

            "reason":
                "EXEC_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell12_reconstructed_callables = {
    name:
        cell12_runtime_namespace.get(
            name
        )

    for name
    in CELL12_REQUIRED_CALLABLE_NAMES
}


cell12_reconstructed_callable_names = sorted(
    name

    for name, value
    in cell12_reconstructed_callables.items()

    if callable(
        value
    )
)


cell12_reconstructed_callable_names_valid = (
    set(
        cell12_reconstructed_callable_names
    )
    ==
    set(
        CELL12_REQUIRED_CALLABLE_NAMES
    )
)


cell12_reconstructed_callables_valid = all(
    callable(
        cell12_reconstructed_callables.get(
            name
        )
    )

    for name
    in CELL12_REQUIRED_CALLABLE_NAMES
)


# ============================================================
# 16) Explicit runtime dependency bindings
# ============================================================

cell12_stable_json_function = (
    cell12_reconstructed_callables.get(
        "stable_json_key"
    )
)


cell12_target_part_identity_function = (
    cell12_reconstructed_callables.get(
        "target_part_identity"
    )
)


cell12_target_id_function = (
    cell12_reconstructed_callables.get(
        "build_target_evidence_id"
    )
)


cell12_question_id_function = (
    cell12_reconstructed_callables.get(
        "build_question_draft_id"
    )
)


cell12_target_builder_globals = (
    getattr(
        cell12_target_id_function,
        "__globals__",
        {},
    )

    if callable(
        cell12_target_id_function
    )

    else {}
)


cell12_question_builder_globals = (
    getattr(
        cell12_question_id_function,
        "__globals__",
        {},
    )

    if callable(
        cell12_question_id_function
    )

    else {}
)


cell12_runtime_dependency_bindings_valid = all([
    callable(
        cell12_stable_json_function
    ),

    callable(
        cell12_target_part_identity_function
    ),

    callable(
        cell12_target_id_function
    ),

    callable(
        cell12_question_id_function
    ),

    cell12_target_builder_globals.get(
        "target_part_identity"
    )
    is
    cell12_target_part_identity_function,

    cell12_target_builder_globals.get(
        "stable_json_key"
    )
    is
    cell12_stable_json_function,

    cell12_target_builder_globals.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    )
    ==
    "target_evidence_v1_draft",

    cell12_question_builder_globals.get(
        "stable_json_key"
    )
    is
    cell12_stable_json_function,

    cell12_question_builder_globals.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    )
    ==
    "question_draft_id_method_v1",
])


cell12_reconstructed_callable_globals_clean = all(
    "cell09_seed_namespace"
    not in
    getattr(
        value,
        "__globals__",
        {},
    )

    for value
    in cell12_reconstructed_callables.values()

    if callable(
        value
    )
)


# ============================================================
# 17) stable_json_key behavioral probe
# ============================================================

cell12_stable_json_probe_valid = False
cell12_stable_json_probe_error = None


if callable(
    cell12_stable_json_function
):

    try:

        cell12_stable_probe_a = (
            cell12_stable_json_function({
                "b":
                    2,

                "a":
                    1,
            })
        )


        cell12_stable_probe_b = (
            cell12_stable_json_function({
                "a":
                    1,

                "b":
                    2,
            })
        )


        cell12_stable_json_probe_valid = all([
            isinstance(
                cell12_stable_probe_a,
                str,
            ),

            cell12_stable_probe_a
            ==
            cell12_stable_probe_b,
        ])


    except Exception as exc:

        cell12_stable_json_probe_error = (
            repr(
                exc
            )
        )


else:

    cell12_stable_json_probe_error = (
        "stable_json_key not reconstructed"
    )


# ============================================================
# 18) target_part_identity behavioral probe
#
# IMPORTANT:
# Use REAL Target Part candidates from the Compact Seed.
# Do not fabricate an incomplete synthetic part.
# ============================================================

cell12_target_part_probe_valid = False

cell12_target_part_probe_error = None

cell12_target_part_probe_path = None

cell12_target_part_probe_result = None

cell12_target_part_probe_attempts = []

cell12_target_part_candidate_count = 0


cell12_target_part_probe_roots = {
    "final_runtime":
        cell12_final_runtime,

    "active_support":
        cell12_active_support,
}


if callable(
    cell12_target_part_identity_function
):

    for root_name, root_value in (
        cell12_target_part_probe_roots.items()
    ):

        if (
            cell12_target_part_probe_valid
        ):
            break


        for candidate_path, candidate in (
            cell12_walk_dict_candidates(
                root_value,
                path=root_name,
            )
        ):

            # target_part_identity requires a real target-part
            # identity surface. document_id is a known hard field.
            if (
                "document_id"
                not in
                candidate
            ):
                continue


            cell12_target_part_candidate_count += 1


            try:

                probe_result = (
                    cell12_target_part_identity_function(
                        copy.deepcopy(
                            candidate
                        )
                    )
                )


                cell12_target_part_probe_attempts.append({
                    "path":
                        candidate_path,

                    "status":
                        "PASS",

                    "candidate_keys":
                        sorted(
                            str(
                                key
                            )
                            for key
                            in candidate.keys()
                        ),

                    "result_type":
                        type(
                            probe_result
                        ).__name__,
                })


                cell12_target_part_probe_valid = (
                    probe_result
                    is not None
                )


                if (
                    cell12_target_part_probe_valid
                ):

                    cell12_target_part_probe_path = (
                        candidate_path
                    )

                    cell12_target_part_probe_result = (
                        probe_result
                    )

                    cell12_target_part_probe_error = None

                    break


            except Exception as exc:

                cell12_target_part_probe_attempts.append({
                    "path":
                        candidate_path,

                    "status":
                        "FAIL",

                    "candidate_keys":
                        sorted(
                            str(
                                key
                            )
                            for key
                            in candidate.keys()
                        ),

                    "error":
                        repr(
                            exc
                        ),
                })


    if not (
        cell12_target_part_probe_valid
    ):

        if (
            cell12_target_part_candidate_count
            ==
            0
        ):

            cell12_target_part_probe_error = (
                "NO_REAL_DOCUMENT_ID_CANDIDATE_FOUND_IN_SEED"
            )

        else:

            cell12_target_part_probe_error = (
                "NO_REAL_TARGET_PART_CANDIDATE_PASSED"
            )


else:

    cell12_target_part_probe_error = (
        "target_part_identity not reconstructed"
    )


# ============================================================
# 19) Question Draft ID behavioral probe
# ============================================================

cell12_question_id_probe_valid = False

cell12_question_id_probe_error = None

cell12_question_id_probe_value = None


if callable(
    cell12_question_id_function
):

    try:

        cell12_question_id_probe_a = (
            cell12_question_id_function(
                pilot_case_id=
                    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

                question_text_sha256=
                    (
                        "0"
                        *
                        64
                    ),
            )
        )


        cell12_question_id_probe_b = (
            cell12_question_id_function(
                pilot_case_id=
                    CELL12_EXPECTED_REPLACEMENT_PILOT_ID,

                question_text_sha256=
                    (
                        "0"
                        *
                        64
                    ),
            )
        )


        cell12_question_id_probe_value = (
            cell12_question_id_probe_a
        )


        cell12_question_id_probe_valid = all([
            isinstance(
                cell12_question_id_probe_a,
                str,
            ),

            cell12_question_id_probe_a.startswith(
                "question-draft:"
            ),

            cell12_question_id_probe_a
            ==
            cell12_question_id_probe_b,
        ])


    except Exception as exc:

        cell12_question_id_probe_error = (
            repr(
                exc
            )
        )


else:

    cell12_question_id_probe_error = (
        "build_question_draft_id not reconstructed"
    )


# ============================================================
# 20) Deep-copy isolation probe
# ============================================================

cell12_seed_before_probe_sha = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_isolation_probe = (
    copy.deepcopy(
        cell12_seed
    )
)


cell12_probe_remaining_ids = (
    cell12_isolation_probe[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell12_probe_count_before = (
    len(
        cell12_probe_remaining_ids
    )
)


cell12_probe_remaining_ids.append(
    "question-pilot:"
    "LOCAL_ISOLATION_PROBE_ONLY"
)


cell12_probe_count_after = (
    len(
        cell12_probe_remaining_ids
    )
)


cell12_seed_after_probe_sha = (
    cell10_stable_sha(
        cell12_seed
    )
)


cell12_isolation_probe_valid = all([
    cell12_probe_count_before
    ==
    22,

    cell12_probe_count_after
    ==
    23,

    len(
        cell12_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22,

    cell12_seed_before_probe_sha
    ==
    cell12_seed_after_probe_sha,

    cell12_seed_after_probe_sha
    ==
    cell11_seed_sha256,
])


# ============================================================
# 21) Full-572 remains blocked
# ============================================================

cell12_full_572_status = (
    cell12_seed_runtime.get(
        "source",
        {},
    ).get(
        "full_572_generation"
    )
)


cell12_full_572_blocked = (
    cell12_full_572_status
    ==
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 22) Verify external state
# ============================================================

(
    cell12_git_after_rc,
    cell12_git_after,
    cell12_git_after_err,
) = run_git(
    "status",
    "--short",
)


cell12_prior_after = {
    "size":
        prior_notebook_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            prior_notebook_path.read_bytes()
        ).hexdigest(),
}


cell12_handoff_after = {
    "size":
        handoff_path.stat().st_size,

    "sha256":
        hashlib.sha256(
            handoff_path.read_bytes()
        ).hexdigest(),
}


cell12_external_state_stable = all([
    cell12_git_before_rc
    ==
    cell12_git_after_rc,

    cell12_git_before
    ==
    cell12_git_after,

    cell12_prior_before
    ==
    cell12_prior_after,

    cell12_handoff_before
    ==
    cell12_handoff_after,
])


# ============================================================
# 23) Final Gate
# ============================================================

cell12_gate = all([
    len(
        cell12_failures
    )
    ==
    0,

    cell12_seed_fingerprint_valid,

    cell12_seed_has_no_callable_objects,

    cell12_forbidden_runtime_reference_absent,

    cell12_question_id_version_present,

    cell12_target_schema_version_present,

    cell12_final_runtime_surface_valid,

    cell12_runtime_fingerprints_valid,

    cell12_pilot_state_valid,

    cell12_identity_surface_valid,

    cell12_support_surface_valid,

    cell12_history_valid,

    cell12_callable_source_names_valid,

    len(
        cell12_callable_dependency_failures
    )
    ==
    0,

    len(
        cell12_callable_compile_failures
    )
    ==
    0,

    all(
        row[
            "source_sha_valid"
        ]

        for row
        in cell12_callable_source_checks.values()
    ),

    cell12_reconstructed_callable_names_valid,

    cell12_reconstructed_callables_valid,

    cell12_runtime_dependency_bindings_valid,

    cell12_reconstructed_callable_globals_clean,

    cell12_stable_json_probe_valid,

    cell12_target_part_probe_valid,

    cell12_question_id_probe_valid,

    cell12_isolation_probe_valid,

    cell12_full_572_blocked,

    cell12_external_state_stable,
])


# ============================================================
# 24) Diagnostics
# ============================================================

print(
    "=== COMPACT RUNTIME SEED "
    "INDEPENDENCE VALIDATION V3 ==="
)


print(
    "\n=== SEED IDENTITY ==="
)

print(
    "Expected SHA256:",
    cell11_seed_sha256,
)

print(
    "Independent SHA256:",
    cell12_seed_sha256,
)

print(
    "Fingerprint valid:",
    cell12_seed_fingerprint_valid,
)

print(
    "Callable objects inside Seed:",
    not cell12_seed_has_no_callable_objects,
)

print(
    "Direct cell09_seed_namespace reference:",
    not cell12_forbidden_runtime_reference_absent,
)


print(
    "\n=== CONSTANT DEPENDENCIES ==="
)

print(
    "QUESTION_DRAFT_ID_METHOD_VERSION:",
    cell12_runtime_namespace.get(
        "QUESTION_DRAFT_ID_METHOD_VERSION"
    ),
)

print(
    "TARGET_EVIDENCE_SCHEMA_VERSION:",
    cell12_runtime_namespace.get(
        "TARGET_EVIDENCE_SCHEMA_VERSION"
    ),
)


print(
    "\n=== FINAL RUNTIME ==="
)

print(
    "Surface valid:",
    cell12_final_runtime_surface_valid,
)

print(
    "Fingerprint valid:",
    cell12_runtime_fingerprints_valid,
)

print(
    "Active Pilots:",
    len(
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)

print(
    "Remaining Pilots:",
    len(
        cell12_remaining_ids
    )
    if isinstance(
        cell12_remaining_ids,
        list,
    )
    else None,
)

print(
    "Next Pilot:",
    cell12_next_pilot_id,
)

print(
    "Old Pilot active:",
    (
        CELL12_EXPECTED_OLD_PILOT_ID
        in
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)

print(
    "Replacement active:",
    (
        CELL12_EXPECTED_REPLACEMENT_PILOT_ID
        in
        cell12_runtime_ids
    )
    if isinstance(
        cell12_runtime_ids,
        list,
    )
    else None,
)


print(
    "\n=== ACTIVE SUPPORT ==="
)

print(
    "Identity surface valid:",
    cell12_identity_surface_valid,
)

print(
    "Golden Cases:",
    len(
        cell12_support_case_map
    ),
)

print(
    "Task Plans:",
    len(
        cell12_support_task_plans
    ),
)

print(
    "Target by Case:",
    len(
        cell12_support_target_by_case
    ),
)

print(
    "Target by Task:",
    len(
        cell12_support_target_by_task
    ),
)

print(
    "Target Records:",
    len(
        cell12_support_target_rows
    ),
)

print(
    "Support surface valid:",
    cell12_support_surface_valid,
)


print(
    "\n=== HISTORY ==="
)

print(
    "Completed Pilot:",
    cell12_completed_section.get(
        "pilot_id"
    ),
)

print(
    "Old Pilot retained historically:",
    (
        CELL12_EXPECTED_OLD_PILOT_ID
        in
        cell12_replacement_history_ids[
            "pilot"
        ]
    ),
)

print(
    "Replacement retained historically:",
    (
        CELL12_EXPECTED_REPLACEMENT_PILOT_ID
        in
        cell12_replacement_history_ids[
            "pilot"
        ]
    ),
)

print(
    "History valid:",
    cell12_history_valid,
)


print(
    "\n=== CALLABLE DEPENDENCIES ==="
)

for name in (
    CELL12_REQUIRED_CALLABLE_NAMES
):

    print({
        "name":
            name,

        "source_sha_valid":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "source_sha_valid"
            ),

        "global_loads":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "global_loads"
            ),

        "unresolved":
            cell12_callable_source_checks.get(
                name,
                {},
            ).get(
                "unresolved_global_loads"
            ),
    })


print(
    "Dependency failures:",
    cell12_callable_dependency_failures,
)

print(
    "Compile failures:",
    cell12_callable_compile_failures,
)


print(
    "\n=== RECONSTRUCTED CALLABLES ==="
)

print(
    "Required:",
    CELL12_REQUIRED_CALLABLE_NAMES,
)

print(
    "Reconstructed:",
    cell12_reconstructed_callable_names,
)

print(
    "Exact callable surface valid:",
    cell12_reconstructed_callable_names_valid,
)

print(
    "All reconstructed callables valid:",
    cell12_reconstructed_callables_valid,
)

print(
    "Dependency bindings valid:",
    cell12_runtime_dependency_bindings_valid,
)

print(
    "Callable globals clean:",
    cell12_reconstructed_callable_globals_clean,
)


print(
    "\n=== BEHAVIORAL PROBES ==="
)

print(
    "stable_json_key probe:",
    cell12_stable_json_probe_valid,
)

print(
    "stable_json_key probe error:",
    cell12_stable_json_probe_error,
)


print(
    "Real target-part candidates inspected:",
    cell12_target_part_candidate_count,
)

print(
    "target_part_identity probe:",
    cell12_target_part_probe_valid,
)

print(
    "target_part_identity probe path:",
    cell12_target_part_probe_path,
)

print(
    "target_part_identity probe result:",
    cell12_target_part_probe_result,
)

print(
    "target_part_identity probe error:",
    cell12_target_part_probe_error,
)


print(
    "build_question_draft_id probe:",
    cell12_question_id_probe_valid,
)

print(
    "build_question_draft_id probe value:",
    cell12_question_id_probe_value,
)

print(
    "build_question_draft_id probe error:",
    cell12_question_id_probe_error,
)


if not (
    cell12_target_part_probe_valid
):

    print(
        "\nLast target-part probe attempts:"
    )

    for row in (
        cell12_target_part_probe_attempts[
            -10:
        ]
    ):

        print(
            row
        )


print(
    "\n=== ISOLATION ==="
)

print(
    "Probe queue before:",
    cell12_probe_count_before,
)

print(
    "Probe queue after local mutation:",
    cell12_probe_count_after,
)

print(
    "Original Seed queue:",
    len(
        cell12_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    ),
)

print(
    "Original Seed unchanged:",
    cell12_isolation_probe_valid,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 status:",
    cell12_full_572_status,
)

print(
    "Full 572 blocked:",
    cell12_full_572_blocked,
)


print(
    "\n=== EXTERNAL STATE ==="
)

print(
    "Git unchanged:",
    cell12_git_before
    ==
    cell12_git_after,
)

print(
    "Prior NB12 unchanged:",
    cell12_prior_before
    ==
    cell12_prior_after,
)

print(
    "Existing handoff unchanged:",
    cell12_handoff_before
    ==
    cell12_handoff_after,
)

print(
    "External state stable:",
    cell12_external_state_stable,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell12_failures
)


print(
    "\nCODE CELL 12 GATE:",
    (
        "PASS"
        if cell12_gate
        else "FAIL"
    ),
)


assert cell12_gate, (
    "Compact Runtime Seed independence validation V3 "
    "failed. Repair this SAME Code Cell 12 before "
    "building the reusable Pilot continuation engine."
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


=== COMPACT RUNTIME SEED INDEPENDENCE VALIDATION V3 ===

=== SEED IDENTITY ===
Expected SHA256: 6098087797ba0c9eb39b36af1ec6ccb044668c19c9a3d2f5e345b17f9e1ef1dd
Independent SHA256: 6098087797ba0c9eb39b36af1ec6ccb044668c19c9a3d2f5e345b17f9e1ef1dd
Fingerprint valid: True
Callable objects inside Seed: False
Direct cell09_seed_namespace reference: False

=== CONSTANT DEPENDENCIES ===
QUESTION_DRAFT_ID_METHOD_VERSION: question_draft_id_method_v1
TARGET_EVIDENCE_SCHEMA_VERSION: target_evidence_v1_draft

=== FINAL RUNTIME ===
Surface valid: True
Fingerprint valid: True
Active Pilots: 23
Remaining Pilots: 22
Next Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Old Pilot active: False
Replacement active: True

=== ACTIVE SUPPORT ===
Identity surface valid: True
Golden Cases: 23
Task Plans: 23
Target by Case: 23
Target by Task: 23
Target Records: 23
Support surface valid: True

=== HISTORY ===
Completed Pilot: question-pilot:02ba2886f1b68521b93789becf741361
Old Pilot retained historically

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 13 — ساخت Reusable Pilot Continuation Engine

Cell 12 اثبات کرد که Compact Runtime Seed مستقل است و تمام dependencyهای لازم برای ادامه‌ی Workflow را در اختیار دارد.

در این مرحله یک موتور عمومی و قابل‌استفاده‌ی مجدد برای ادامه‌ی 22 Pilot باقی‌مانده ساخته می‌شود.

این موتور:

- Pilot فعلی را فقط از Queue رسمی Runtime انتخاب می‌کند؛
- Golden Case، Task Draft و Target Evidence همان Pilot را از Compact Seed Resolve می‌کند؛
- Workflow را به‌صورت State Machine صریح مدیریت می‌کند؛
- Question، Expected Answer و Citation را سه Artifact مستقل در نظر می‌گیرد؛
- برای هر سه Artifact یک Human Review Gate اجباری دارد؛
- Revision Loop را بدون Queue Advancement پشتیبانی می‌کند؛
- در صورت ردشدن Target می‌تواند Workflow را به حالت Retarget Required ببرد؛
- Queue Advancement را فقط پس از پذیرش نهایی Citation مجاز می‌کند؛
- Full 572 Generation را در تمام مراحل Blocked نگه می‌دارد؛
- تمام Transitionها را به‌صورت Pure و روی Deep Copy انجام می‌دهد تا Runtime اصلی ناخواسته Mutate نشود.

در این Cell هیچ Question واقعی، Expected Answer واقعی یا Citation واقعی تولید نمی‌شود.

همچنین هیچ Pilot واقعی Complete نمی‌شود و Queue واقعی از 22 به 21 کاهش پیدا نمی‌کند.

برای تست موتور فقط از Artifactهای مصنوعی با برچسب صریح `ENGINE_DRY_RUN_ONLY` روی Sessionهای موقت استفاده می‌شود.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [34]:
# Code Cell 13

cell13_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell12_gate"
    )
    is not True
):
    cell13_failures.append(
        "CELL12_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell12_seed"
    ),
    dict,
):
    cell13_failures.append(
        "CELL12_INDEPENDENT_SEED_MISSING"
    )


# ============================================================
# 2) Fixed workflow contract
# ============================================================

CELL13_ENGINE_VERSION = (
    "pilot_continuation_engine_v1"
)


CELL13_INITIAL_STATE = (
    "QUESTION_DRAFT_PENDING"
)


CELL13_FINAL_PRE_ADVANCE_STATE = (
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE"
)


CELL13_RETARGET_STATE = (
    "BLOCKED_REQUIRES_RETARGET"
)


CELL13_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


CELL13_TRANSITIONS = {
    "QUESTION_DRAFT_PENDING":
        {
            "QUESTION_DRAFT_REGISTERED":
                "QUESTION_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "QUESTION_HUMAN_REVIEW_PENDING":
        {
            "QUESTION_ACCEPTED":
                "EXPECTED_ANSWER_DRAFT_PENDING",

            "QUESTION_REVISE":
                "QUESTION_DRAFT_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "EXPECTED_ANSWER_DRAFT_PENDING":
        {
            "EXPECTED_ANSWER_REGISTERED":
                "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING":
        {
            "EXPECTED_ANSWER_ACCEPTED":
                "CITATION_MAPPING_PENDING",

            "EXPECTED_ANSWER_REVISE":
                "EXPECTED_ANSWER_DRAFT_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "CITATION_MAPPING_PENDING":
        {
            "CITATION_MAPPING_REGISTERED":
                "CITATION_HUMAN_REVIEW_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    "CITATION_HUMAN_REVIEW_PENDING":
        {
            "CITATION_ACCEPTED":
                CELL13_FINAL_PRE_ADVANCE_STATE,

            "CITATION_REVISE":
                "CITATION_MAPPING_PENDING",

            "TARGET_REJECTED":
                CELL13_RETARGET_STATE,
        },

    CELL13_FINAL_PRE_ADVANCE_STATE:
        {},

    CELL13_RETARGET_STATE:
        {},
}


CELL13_ARTIFACT_EVENTS = {
    "QUESTION_DRAFT_REGISTERED":
        "question_draft",

    "EXPECTED_ANSWER_REGISTERED":
        "expected_answer",

    "CITATION_MAPPING_REGISTERED":
        "citation_mapping",
}


CELL13_HUMAN_DECISION_EVENTS = {
    "QUESTION_ACCEPTED",
    "QUESTION_REVISE",
    "EXPECTED_ANSWER_ACCEPTED",
    "EXPECTED_ANSWER_REVISE",
    "CITATION_ACCEPTED",
    "CITATION_REVISE",
    "TARGET_REJECTED",
}


# ============================================================
# 3) Generic helpers
# ============================================================

def cell13_collect_prefixed_ids(
    value,
    result=None,
    seen=None,
):

    if result is None:

        result = {
            "pilot":
                set(),

            "golden_case":
                set(),

            "task_draft":
                set(),

            "target_evidence":
                set(),
        }


    if seen is None:
        seen = set()


    if isinstance(
        value,
        str,
    ):

        if value.startswith(
            "question-pilot:"
        ):

            result[
                "pilot"
            ].add(
                value
            )


        elif value.startswith(
            "golden-case:"
        ):

            result[
                "golden_case"
            ].add(
                value
            )


        elif value.startswith(
            "task-draft:"
        ):

            result[
                "task_draft"
            ].add(
                value
            )


        elif value.startswith(
            "target-evidence:"
        ):

            result[
                "target_evidence"
            ].add(
                value
            )


        return result


    if not isinstance(
        value,
        (
            dict,
            list,
            tuple,
            set,
            frozenset,
        ),
    ):

        return result


    object_id = id(
        value
    )


    if object_id in seen:
        return result


    seen.add(
        object_id
    )


    try:

        if isinstance(
            value,
            dict,
        ):

            for key, nested in (
                value.items()
            ):

                cell13_collect_prefixed_ids(
                    key,
                    result,
                    seen,
                )

                cell13_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


        else:

            for nested in value:

                cell13_collect_prefixed_ids(
                    nested,
                    result,
                    seen,
                )


    finally:

        seen.discard(
            object_id
        )


    return result


def cell13_exact_single_id(
    value,
    family,
):

    ids = (
        cell13_collect_prefixed_ids(
            value
        )[
            family
        ]
    )


    if len(
        ids
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one {family} ID; "
                f"found {sorted(ids)}"
            )
        )


    return next(
        iter(
            ids
        )
    )


def cell13_value_contains_id(
    value,
    family,
    identity,
):

    return (
        identity
        in
        cell13_collect_prefixed_ids(
            value
        )[
            family
        ]
    )


def cell13_find_unique_mapping_value(
    mapping,
    family,
    identity,
):

    if not isinstance(
        mapping,
        dict,
    ):

        raise TypeError(
            "Expected mapping dictionary."
        )


    if identity in mapping:

        return {
            "lookup_mode":
                "DIRECT_KEY",

            "key":
                identity,

            "value":
                copy.deepcopy(
                    mapping[
                        identity
                    ]
                ),
        }


    matches = []


    for key, value in (
        mapping.items()
    ):

        if cell13_value_contains_id(
            value,
            family,
            identity,
        ):

            matches.append(
                (
                    key,
                    value,
                )
            )


    if len(
        matches
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one mapping match "
                f"for {identity}; found {len(matches)}."
            )
        )


    key, value = (
        matches[
            0
        ]
    )


    return {
        "lookup_mode":
            "VALUE_ID_SCAN",

        "key":
            copy.deepcopy(
                key
            ),

        "value":
            copy.deepcopy(
                value
            ),
    }


def cell13_find_unique_list_record(
    records,
    field,
    identity,
):

    if not isinstance(
        records,
        list,
    ):

        raise TypeError(
            "Expected record list."
        )


    matches = [
        record

        for record
        in records

        if (
            isinstance(
                record,
                dict,
            )
            and
            record.get(
                field
            )
            ==
            identity
        )
    ]


    if len(
        matches
    ) != 1:

        raise ValueError(
            (
                f"Expected exactly one record where "
                f"{field}={identity}; found {len(matches)}."
            )
        )


    return copy.deepcopy(
        matches[
            0
        ]
    )


# ============================================================
# 4) Freeze independent Seed state
# ============================================================

cell13_seed = copy.deepcopy(
    cell12_seed
)


cell13_seed_before_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell13_seed_matches_cell12 = (
    cell13_seed_before_sha256
    ==
    cell11_seed_sha256
)


cell13_final_runtime = (
    cell13_seed.get(
        "final_runtime",
        {},
    )
)


cell13_active_support = (
    cell13_seed.get(
        "active_support",
        {},
    )
)


# ============================================================
# 5) Resolve official current Pilot
# ============================================================

cell13_current_pilot_id = (
    cell13_final_runtime.get(
        "cell95_next_pilot_id"
    )
)


cell13_remaining_ids = (
    cell13_final_runtime.get(
        "cell95_remaining_pilot_ids"
    )
)


cell13_remaining_queue = (
    cell13_final_runtime.get(
        "cell95_remaining_pilot_queue"
    )
)


cell13_next_queue_entry = (
    cell13_final_runtime.get(
        "cell95_next_pilot_queue_entry"
    )
)


cell13_pilot_by_id = (
    cell13_final_runtime.get(
        "question_drafting_pilot_by_id",
        {},
    )
)


cell13_current_pilot_packet = (
    copy.deepcopy(
        cell13_pilot_by_id.get(
            cell13_current_pilot_id
        )
    )

    if isinstance(
        cell13_pilot_by_id,
        dict,
    )

    else None
)


cell13_current_pilot_resolved = all([
    isinstance(
        cell13_current_pilot_id,
        str,
    ),

    isinstance(
        cell13_current_pilot_packet,
        dict,
    ),

    isinstance(
        cell13_remaining_ids,
        list,
    ),

    len(
        cell13_remaining_ids
    )
    ==
    22,

    cell13_remaining_ids[
        0
    ]
    ==
    cell13_current_pilot_id,
])


# ============================================================
# 6) Resolve current Pilot identity chain
# ============================================================

cell13_current_case_id = None
cell13_current_task_id = None
cell13_current_target_id = None


if (
    cell13_current_pilot_resolved
):

    try:

        cell13_current_case_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "golden_case",
            )
        )


        cell13_current_task_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "task_draft",
            )
        )


        cell13_current_target_id = (
            cell13_exact_single_id(
                cell13_current_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell13_failures.append({
            "reason":
                "CURRENT_PILOT_ID_CHAIN_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell13_current_identity_chain_valid = all([
    isinstance(
        cell13_current_case_id,
        str,
    ),

    isinstance(
        cell13_current_task_id,
        str,
    ),

    isinstance(
        cell13_current_target_id,
        str,
    ),
])


# ============================================================
# 7) Resolve current Pilot support bundle
# ============================================================

cell13_current_golden_case = None

cell13_current_task_plan = None

cell13_current_task_plan_lookup = None

cell13_current_target_by_case = None

cell13_current_target_by_task = None

cell13_current_target_record = None


if (
    cell13_current_identity_chain_valid
):

    try:

        cell13_current_golden_case = (
            copy.deepcopy(
                cell13_active_support[
                    "golden_case_records_by_id"
                ][
                    cell13_current_case_id
                ]
            )
        )


        cell13_current_task_plan_lookup = (
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell13_current_task_id,
            )
        )


        cell13_current_task_plan = (
            cell13_current_task_plan_lookup[
                "value"
            ]
        )


        cell13_current_target_by_case = (
            copy.deepcopy(
                cell13_active_support[
                    "target_evidence_by_case_id"
                ][
                    cell13_current_case_id
                ]
            )
        )


        cell13_current_target_by_task = (
            copy.deepcopy(
                cell13_active_support[
                    "target_evidence_by_task_draft_id"
                ][
                    cell13_current_task_id
                ]
            )
        )


        cell13_current_target_record = (
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell13_current_target_id,
            )
        )


    except Exception as exc:

        cell13_failures.append({
            "reason":
                "CURRENT_PILOT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 8) Validate support cross-links
# ============================================================

cell13_support_crosslinks_valid = all([
    isinstance(
        cell13_current_golden_case,
        dict,
    ),

    isinstance(
        cell13_current_task_plan,
        dict,
    ),

    isinstance(
        cell13_current_target_record,
        dict,
    ),

    cell13_value_contains_id(
        cell13_current_golden_case,
        "golden_case",
        cell13_current_case_id,
    ),

    cell13_value_contains_id(
        cell13_current_task_plan,
        "task_draft",
        cell13_current_task_id,
    ),

    cell13_value_contains_id(
        cell13_current_target_by_case,
        "target_evidence",
        cell13_current_target_id,
    ),

    cell13_value_contains_id(
        cell13_current_target_by_task,
        "target_evidence",
        cell13_current_target_id,
    ),

    cell13_current_target_record.get(
        "target_evidence_id"
    )
    ==
    cell13_current_target_id,
])


# ============================================================
# 9) Build immutable current Pilot bundle
# ============================================================

cell13_current_pilot_bundle = {
    "engine_version":
        CELL13_ENGINE_VERSION,

    "pilot_case_id":
        cell13_current_pilot_id,

    "golden_case_id":
        cell13_current_case_id,

    "task_draft_id":
        cell13_current_task_id,

    "target_evidence_id":
        cell13_current_target_id,

    "queue_position":
        0,

    "remaining_queue_count":
        22,

    "pilot_packet":
        copy.deepcopy(
            cell13_current_pilot_packet
        ),

    "golden_case":
        copy.deepcopy(
            cell13_current_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell13_current_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell13_current_target_record
        ),

    "target_by_case":
        copy.deepcopy(
            cell13_current_target_by_case
        ),

    "target_by_task":
        copy.deepcopy(
            cell13_current_target_by_task
        ),

    "full_572_generation":
        CELL13_FULL_572_BLOCK_STATUS,
}


cell13_current_pilot_bundle_sha256 = (
    cell10_stable_sha(
        cell13_current_pilot_bundle
    )
)


# ============================================================
# 10) Reusable session builder
# ============================================================

def cell13_build_session(
    pilot_bundle,
):

    session = {
        "engine_version":
            CELL13_ENGINE_VERSION,

        "pilot_case_id":
            pilot_bundle[
                "pilot_case_id"
            ],

        "golden_case_id":
            pilot_bundle[
                "golden_case_id"
            ],

        "task_draft_id":
            pilot_bundle[
                "task_draft_id"
            ],

        "target_evidence_id":
            pilot_bundle[
                "target_evidence_id"
            ],

        "current_state":
            CELL13_INITIAL_STATE,

        "artifacts":
            {
                "question_draft":
                    None,

                "expected_answer":
                    None,

                "citation_mapping":
                    None,
            },

        "human_decisions":
            [],

        "event_log":
            [],

        "queue_advance_permitted":
            False,

        "queue_advance_performed":
            False,

        "full_572_generation":
            CELL13_FULL_572_BLOCK_STATUS,
    }


    session[
        "session_sha256"
    ] = cell10_stable_sha({
        key:
            value

        for key, value
        in session.items()

        if key
        !=
        "session_sha256"
    })


    return session


# ============================================================
# 11) Pure transition function
# ============================================================

def cell13_apply_event(
    session,
    event,
    artifact=None,
    human_note=None,
):

    if not isinstance(
        session,
        dict,
    ):

        raise TypeError(
            "Session must be a dictionary."
        )


    next_session = copy.deepcopy(
        session
    )


    current_state = (
        next_session.get(
            "current_state"
        )
    )


    allowed_events = (
        CELL13_TRANSITIONS.get(
            current_state
        )
    )


    if allowed_events is None:

        raise ValueError(
            (
                "Unknown current workflow state: "
                f"{current_state}"
            )
        )


    if event not in allowed_events:

        raise ValueError(
            (
                f"Event {event} is not allowed from "
                f"state {current_state}."
            )
        )


    if event in (
        CELL13_ARTIFACT_EVENTS
    ):

        if not isinstance(
            artifact,
            dict,
        ):

            raise ValueError(
                (
                    f"{event} requires a dictionary "
                    "artifact payload."
                )
            )


        artifact_slot = (
            CELL13_ARTIFACT_EVENTS[
                event
            ]
        )


        artifact_copy = copy.deepcopy(
            artifact
        )


        artifact_copy[
            "artifact_sha256"
        ] = cell10_stable_sha(
            artifact_copy
        )


        next_session[
            "artifacts"
        ][
            artifact_slot
        ] = artifact_copy


    if event in (
        CELL13_HUMAN_DECISION_EVENTS
    ):

        if not (
            isinstance(
                human_note,
                str,
            )
            and
            human_note.strip()
        ):

            raise ValueError(
                (
                    f"{event} requires a non-empty "
                    "human_note."
                )
            )


        next_session[
            "human_decisions"
        ].append({
            "event":
                event,

            "note":
                human_note.strip(),

            "prior_state":
                current_state,
        })


    new_state = (
        allowed_events[
            event
        ]
    )


    next_session[
        "event_log"
    ].append({
        "event":
            event,

        "prior_state":
            current_state,

        "new_state":
            new_state,
    })


    next_session[
        "current_state"
    ] = new_state


    next_session[
        "queue_advance_permitted"
    ] = (
        new_state
        ==
        CELL13_FINAL_PRE_ADVANCE_STATE
    )


    next_session[
        "queue_advance_performed"
    ] = False


    next_session[
        "full_572_generation"
    ] = (
        CELL13_FULL_572_BLOCK_STATUS
    )


    next_session[
        "session_sha256"
    ] = cell10_stable_sha({
        key:
            value

        for key, value
        in next_session.items()

        if key
        !=
        "session_sha256"
    })


    return next_session


# ============================================================
# 12) Create actual current session
#
# This is the real continuation starting state.
# No content is generated and no event is applied here.
# ============================================================

cell13_current_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_current_session_initial_valid = all([
    cell13_current_session[
        "current_state"
    ]
    ==
    CELL13_INITIAL_STATE,

    cell13_current_session[
        "pilot_case_id"
    ]
    ==
    cell13_current_pilot_id,

    cell13_current_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_current_session[
        "queue_advance_performed"
    ]
    is False,

    all(
        value
        is None

        for value
        in cell13_current_session[
            "artifacts"
        ].values()
    ),

    len(
        cell13_current_session[
            "human_decisions"
        ]
    )
    ==
    0,

    len(
        cell13_current_session[
            "event_log"
        ]
    )
    ==
    0,

    cell13_current_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 13) Engine dry-run artifacts
#
# These are explicitly synthetic and are used only on copies.
# ============================================================

CELL13_DRY_RUN_QUESTION = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "question_draft",

    "text":
        "synthetic-question-placeholder",
}


CELL13_DRY_RUN_ANSWER = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "expected_answer",

    "text":
        "synthetic-answer-placeholder",
}


CELL13_DRY_RUN_CITATION = {
    "test_mode":
        "ENGINE_DRY_RUN_ONLY",

    "artifact_type":
        "citation_mapping",

    "target_evidence_id":
        cell13_current_target_id,
}


# ============================================================
# 14) Happy-path dry run
# ============================================================

cell13_happy_path_session = copy.deepcopy(
    cell13_current_session
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            CELL13_DRY_RUN_QUESTION,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "QUESTION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: question accepted.",
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            CELL13_DRY_RUN_ANSWER,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: answer accepted.",
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            CELL13_DRY_RUN_CITATION,
    )
)


cell13_happy_path_session = (
    cell13_apply_event(
        cell13_happy_path_session,
        "CITATION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: citation accepted.",
    )
)


cell13_happy_path_valid = all([
    cell13_happy_path_session[
        "current_state"
    ]
    ==
    CELL13_FINAL_PRE_ADVANCE_STATE,

    cell13_happy_path_session[
        "queue_advance_permitted"
    ]
    is True,

    cell13_happy_path_session[
        "queue_advance_performed"
    ]
    is False,

    len(
        cell13_happy_path_session[
            "event_log"
        ]
    )
    ==
    6,

    len(
        cell13_happy_path_session[
            "human_decisions"
        ]
    )
    ==
    3,

    all(
        isinstance(
            value,
            dict,
        )

        for value
        in cell13_happy_path_session[
            "artifacts"
        ].values()
    ),

    cell13_happy_path_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 15) Question revision-loop dry run
# ============================================================

cell13_revision_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_revision_session = (
    cell13_apply_event(
        cell13_revision_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            CELL13_DRY_RUN_QUESTION,
    )
)


cell13_revision_session = (
    cell13_apply_event(
        cell13_revision_session,
        "QUESTION_REVISE",
        human_note=
            "ENGINE_DRY_RUN_ONLY: revise question.",
    )
)


cell13_revision_loop_valid = all([
    cell13_revision_session[
        "current_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",

    cell13_revision_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_revision_session[
        "queue_advance_performed"
    ]
    is False,

    cell13_revision_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 16) Retarget dry run
# ============================================================

cell13_retarget_session = (
    cell13_build_session(
        cell13_current_pilot_bundle
    )
)


cell13_retarget_session = (
    cell13_apply_event(
        cell13_retarget_session,
        "TARGET_REJECTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY: target rejected.",
    )
)


cell13_retarget_path_valid = all([
    cell13_retarget_session[
        "current_state"
    ]
    ==
    CELL13_RETARGET_STATE,

    cell13_retarget_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_retarget_session[
        "queue_advance_performed"
    ]
    is False,

    cell13_retarget_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 17) Invalid transition rejection probe
# ============================================================

cell13_invalid_transition_rejected = False

cell13_invalid_transition_error = None


try:

    cell13_apply_event(
        cell13_current_session,
        "QUESTION_ACCEPTED",
        human_note=
            "ENGINE_DRY_RUN_ONLY invalid order probe.",
    )


except ValueError as exc:

    cell13_invalid_transition_rejected = True

    cell13_invalid_transition_error = (
        str(
            exc
        )
    )


# ============================================================
# 18) Missing human decision rejection probe
# ============================================================

cell13_missing_human_note_rejected = False

cell13_missing_human_note_error = None


try:

    cell13_review_probe = (
        cell13_apply_event(
            cell13_current_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                CELL13_DRY_RUN_QUESTION,
        )
    )


    cell13_apply_event(
        cell13_review_probe,
        "QUESTION_ACCEPTED",
    )


except ValueError as exc:

    cell13_missing_human_note_rejected = True

    cell13_missing_human_note_error = (
        str(
            exc
        )
    )


# ============================================================
# 19) Missing artifact rejection probe
# ============================================================

cell13_missing_artifact_rejected = False

cell13_missing_artifact_error = None


try:

    cell13_apply_event(
        cell13_current_session,
        "QUESTION_DRAFT_REGISTERED",
    )


except ValueError as exc:

    cell13_missing_artifact_rejected = True

    cell13_missing_artifact_error = (
        str(
            exc
        )
    )


# ============================================================
# 20) Confirm dry runs did NOT mutate actual current session
# ============================================================

cell13_actual_session_still_initial = all([
    cell13_current_session[
        "current_state"
    ]
    ==
    CELL13_INITIAL_STATE,

    cell13_current_session[
        "queue_advance_permitted"
    ]
    is False,

    cell13_current_session[
        "queue_advance_performed"
    ]
    is False,

    len(
        cell13_current_session[
            "event_log"
        ]
    )
    ==
    0,

    len(
        cell13_current_session[
            "human_decisions"
        ]
    )
    ==
    0,

    all(
        value
        is None

        for value
        in cell13_current_session[
            "artifacts"
        ].values()
    ),
])


# ============================================================
# 21) Confirm queue did NOT advance
# ============================================================

cell13_queue_after_dry_runs = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell13_queue_not_advanced = all([
    len(
        cell13_queue_after_dry_runs
    )
    ==
    22,

    cell13_queue_after_dry_runs[
        0
    ]
    ==
    cell13_current_pilot_id,

    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    cell13_current_pilot_id,
])


# ============================================================
# 22) Engine contract artifact
# ============================================================

cell13_engine_contract = {
    "engine_version":
        CELL13_ENGINE_VERSION,

    "initial_state":
        CELL13_INITIAL_STATE,

    "final_pre_advance_state":
        CELL13_FINAL_PRE_ADVANCE_STATE,

    "retarget_state":
        CELL13_RETARGET_STATE,

    "transitions":
        copy.deepcopy(
            CELL13_TRANSITIONS
        ),

    "artifact_events":
        copy.deepcopy(
            CELL13_ARTIFACT_EVENTS
        ),

    "human_decision_events":
        sorted(
            CELL13_HUMAN_DECISION_EVENTS
        ),

    "rules":
        {
            "human_review_required_for_question":
                True,

            "human_review_required_for_expected_answer":
                True,

            "human_review_required_for_citation":
                True,

            "automatic_queue_advance":
                False,

            "queue_advance_only_after_final_citation_acceptance":
                True,

            "full_572_generation":
                CELL13_FULL_572_BLOCK_STATUS,
        },
}


cell13_engine_contract_sha256 = (
    cell10_stable_sha(
        cell13_engine_contract
    )
)


# ============================================================
# 23) Seed immutability
# ============================================================

cell13_seed_after_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell13_seed_unchanged = (
    cell13_seed_before_sha256
    ==
    cell13_seed_after_sha256
)


# ============================================================
# 24) Full 572 remains blocked
# ============================================================

cell13_full_572_blocked = all([
    cell13_seed.get(
        "source",
        {},
    ).get(
        "full_572_generation"
    )
    ==
    CELL13_FULL_572_BLOCK_STATUS,

    cell13_current_session[
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,

    cell13_engine_contract[
        "rules"
    ][
        "full_572_generation"
    ]
    ==
    CELL13_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 25) Final Gate
# ============================================================

cell13_gate = all([
    len(
        cell13_failures
    )
    ==
    0,

    cell13_seed_matches_cell12,

    cell13_current_pilot_resolved,

    cell13_current_identity_chain_valid,

    cell13_support_crosslinks_valid,

    cell13_current_session_initial_valid,

    cell13_happy_path_valid,

    cell13_revision_loop_valid,

    cell13_retarget_path_valid,

    cell13_invalid_transition_rejected,

    cell13_missing_human_note_rejected,

    cell13_missing_artifact_rejected,

    cell13_actual_session_still_initial,

    cell13_queue_not_advanced,

    cell13_seed_unchanged,

    cell13_full_572_blocked,
])


# ============================================================
# 26) Diagnostics
# ============================================================

print(
    "=== REUSABLE PILOT CONTINUATION ENGINE ==="
)


print(
    "\n=== CURRENT PILOT ==="
)

print(
    "Pilot:",
    cell13_current_pilot_id,
)

print(
    "Golden Case:",
    cell13_current_case_id,
)

print(
    "Task Draft:",
    cell13_current_task_id,
)

print(
    "Target Evidence:",
    cell13_current_target_id,
)

print(
    "Remaining queue:",
    len(
        cell13_remaining_ids
    ),
)

print(
    "Current Pilot resolved:",
    cell13_current_pilot_resolved,
)

print(
    "Support cross-links valid:",
    cell13_support_crosslinks_valid,
)

print(
    "Pilot bundle SHA256:",
    cell13_current_pilot_bundle_sha256,
)


print(
    "\n=== ENGINE ==="
)

print(
    "Version:",
    CELL13_ENGINE_VERSION,
)

print(
    "Initial state:",
    CELL13_INITIAL_STATE,
)

print(
    "Final pre-advance state:",
    CELL13_FINAL_PRE_ADVANCE_STATE,
)

print(
    "Retarget state:",
    CELL13_RETARGET_STATE,
)

print(
    "Engine contract SHA256:",
    cell13_engine_contract_sha256,
)


print(
    "\n=== HUMAN GATES ==="
)

print(
    "Question review required:",
    True,
)

print(
    "Expected Answer review required:",
    True,
)

print(
    "Citation review required:",
    True,
)

print(
    "Automatic queue advance:",
    False,
)


print(
    "\n=== DRY-RUN VALIDATION ==="
)

print(
    "Happy path:",
    cell13_happy_path_valid,
)

print(
    "Happy-path final state:",
    cell13_happy_path_session[
        "current_state"
    ],
)

print(
    "Revision loop:",
    cell13_revision_loop_valid,
)

print(
    "Retarget path:",
    cell13_retarget_path_valid,
)

print(
    "Invalid transition rejected:",
    cell13_invalid_transition_rejected,
)

print(
    "Missing human note rejected:",
    cell13_missing_human_note_rejected,
)

print(
    "Missing artifact rejected:",
    cell13_missing_artifact_rejected,
)


print(
    "\n=== ACTUAL CONTINUATION STATE ==="
)

print(
    "Current state:",
    cell13_current_session[
        "current_state"
    ],
)

print(
    "Question artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "question_draft"
    ],
)

print(
    "Expected Answer artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "expected_answer"
    ],
)

print(
    "Citation artifact:",
    cell13_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ],
)

print(
    "Human decisions:",
    len(
        cell13_current_session[
            "human_decisions"
        ]
    ),
)

print(
    "Queue advance permitted:",
    cell13_current_session[
        "queue_advance_permitted"
    ],
)

print(
    "Queue advance performed:",
    cell13_current_session[
        "queue_advance_performed"
    ],
)


print(
    "\n=== IMMUTABILITY ==="
)

print(
    "Seed before SHA256:",
    cell13_seed_before_sha256,
)

print(
    "Seed after SHA256:",
    cell13_seed_after_sha256,
)

print(
    "Seed unchanged:",
    cell13_seed_unchanged,
)

print(
    "Actual session still initial:",
    cell13_actual_session_still_initial,
)

print(
    "Queue still 22:",
    len(
        cell13_queue_after_dry_runs
    )
    ==
    22,
)

print(
    "Next Pilot unchanged:",
    cell13_queue_after_dry_runs[
        0
    ]
    ==
    cell13_current_pilot_id,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell13_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell13_failures
)


print(
    "\nCODE CELL 13 GATE:",
    (
        "PASS"
        if cell13_gate
        else "FAIL"
    ),
)


assert cell13_gate, (
    "Reusable Pilot continuation engine validation failed. "
    "Repair this SAME Code Cell 13 before preparing the "
    "real replacement-Pilot Question Draft workspace."
)

=== REUSABLE PILOT CONTINUATION ENGINE ===

=== CURRENT PILOT ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Golden Case: golden-case:3b93ce20ef4b617b9f922baddfbd1e0f
Task Draft: task-draft:f48d78f0a6a588de906ca8534219a604
Target Evidence: target-evidence:81011a6ef78817e01e3ed484b1b4c368
Remaining queue: 22
Current Pilot resolved: True
Support cross-links valid: True
Pilot bundle SHA256: aeebd98017451f66c493278dfb9d492a7a13b9c3d522e2213ff70d4de63344a6

=== ENGINE ===
Version: pilot_continuation_engine_v1
Initial state: QUESTION_DRAFT_PENDING
Final pre-advance state: PILOT_COMPLETE_PENDING_QUEUE_ADVANCE
Retarget state: BLOCKED_REQUIRES_RETARGET
Engine contract SHA256: a012ff236a7af27b9770d44d789560b0d830567a8d616d75dca64d2259699a00

=== HUMAN GATES ===
Question review required: True
Expected Answer review required: True
Citation review required: True
Automatic queue advance: False

=== DRY-RUN VALIDATION ===
Happy path: True
Happy-path final state: PILOT_COMPLETE_PENDING_QUE

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 14 — آماده‌سازی Workspace واقعی برای Question Draftِ Pilot جایگزین

موتور continuation در Cell 13 با موفقیت اعتبارسنجی شد و Runtime واقعی همچنان در حالت `QUESTION_DRAFT_PENDING` قرار دارد.

هدف این Cell فقط آماده‌کردن Workspace قابل‌خواندن برای Pilot فعلی است تا سؤال واقعی مستقیماً از Target Evidence و قراردادهای پذیرفته‌شده نوشته شود.

در این مرحله موارد زیر استخراج و نمایش داده می‌شوند:

- هویت دقیق Pilot، Golden Case، Task Draft و Target Evidence؛
- متن واقعی Target Evidence و Target Text Partها؛
- اطلاعات مرتبط Golden Case و Task Plan؛
- Source Type و Stratum مربوط به Pilot؛
- Question Archetype و Constraintهای قابل‌بازیابی؛
- Acceptance Rubric و Source-Opacity Rule؛
- وضعیت فعلی Workflow و تأیید اینکه هنوز هیچ Question Artifact ثبت نشده است.

این Cell فقط Workspace می‌سازد.

هیچ Question واقعی ثبت نمی‌شود، هیچ Human Decision اعمال نمی‌شود، Expected Answer و Citation ساخته نمی‌شوند، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked می‌ماند.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [35]:
# Code Cell 14

cell14_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell13_gate"
    )
    is not True
):
    cell14_failures.append(
        "CELL13_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell13_current_pilot_bundle"
    ),
    dict,
):
    cell14_failures.append(
        "CURRENT_PILOT_BUNDLE_MISSING"
    )


if not isinstance(
    globals().get(
        "cell13_current_session"
    ),
    dict,
):
    cell14_failures.append(
        "CURRENT_SESSION_MISSING"
    )


# ============================================================
# 2) Freeze accepted current state
# ============================================================

cell14_seed_before_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell14_session_before_sha256 = (
    cell10_stable_sha(
        cell13_current_session
    )
)


cell14_bundle = copy.deepcopy(
    cell13_current_pilot_bundle
)


cell14_session = copy.deepcopy(
    cell13_current_session
)


# ============================================================
# 3) Expected current identity
# ============================================================

CELL14_EXPECTED_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL14_EXPECTED_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL14_EXPECTED_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL14_EXPECTED_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL14_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 4) Local inspection helpers
# ============================================================

def cell14_clip(
    value,
    limit=4000,
):

    if value is None:
        return None


    if not isinstance(
        value,
        str,
    ):
        value = str(
            value
        )


    if len(
        value
    ) <= limit:
        return value


    return (
        value[
            :limit
        ]
        +
        "\n...[CLIPPED]..."
    )


def cell14_walk(
    value,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                yield (
                    nested_path,
                    key,
                    nested,
                )


                yield from cell14_walk(
                    nested,
                    nested_path,
                    seen,
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield (
                    nested_path,
                    index,
                    nested,
                )


                yield from cell14_walk(
                    nested,
                    nested_path,
                    seen,
                )


        finally:

            seen.discard(
                object_id
            )


def cell14_find_scalar_fields(
    value,
    search_terms,
):

    rows = []


    lowered_terms = [
        term.lower()
        for term
        in search_terms
    ]


    for path, key, nested in (
        cell14_walk(
            value
        )
    ):

        if not isinstance(
            key,
            str,
        ):
            continue


        lowered_key = (
            key.lower()
        )


        if not any(
            term
            in
            lowered_key

            for term
            in lowered_terms
        ):
            continue


        if isinstance(
            nested,
            (
                str,
                int,
                float,
                bool,
            ),
        ) or nested is None:

            rows.append({
                "path":
                    path,

                "key":
                    key,

                "value":
                    nested,
            })


    return rows


def cell14_find_first_scalar(
    value,
    preferred_keys,
):

    preferred_keys = [
        key.lower()
        for key
        in preferred_keys
    ]


    matches = []


    for path, key, nested in (
        cell14_walk(
            value
        )
    ):

        if not isinstance(
            key,
            str,
        ):
            continue


        if (
            key.lower()
            not in
            preferred_keys
        ):
            continue


        if isinstance(
            nested,
            (
                str,
                int,
                float,
                bool,
            ),
        ):

            matches.append(
                (
                    path,
                    nested,
                )
            )


    if not matches:
        return None


    return {
        "path":
            matches[
                0
            ][
                0
            ],

        "value":
            matches[
                0
            ][
                1
            ],
    }


def cell14_json_preview(
    value,
    limit=12000,
):

    try:

        text = json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
            default=str,
        )

    except Exception:

        text = repr(
            value
        )


    return cell14_clip(
        text,
        limit,
    )


# ============================================================
# 5) Resolve bundle components
# ============================================================

cell14_pilot_packet = (
    copy.deepcopy(
        cell14_bundle.get(
            "pilot_packet"
        )
    )
)


cell14_golden_case = (
    copy.deepcopy(
        cell14_bundle.get(
            "golden_case"
        )
    )
)


cell14_task_plan = (
    copy.deepcopy(
        cell14_bundle.get(
            "task_plan"
        )
    )
)


cell14_target_evidence = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_evidence"
        )
    )
)


cell14_target_by_case = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_by_case"
        )
    )
)


cell14_target_by_task = (
    copy.deepcopy(
        cell14_bundle.get(
            "target_by_task"
        )
    )
)


# ============================================================
# 6) Exact identity gate
# ============================================================

cell14_identity_valid = all([
    cell14_bundle.get(
        "pilot_case_id"
    )
    ==
    CELL14_EXPECTED_PILOT_ID,

    cell14_bundle.get(
        "golden_case_id"
    )
    ==
    CELL14_EXPECTED_GOLDEN_CASE_ID,

    cell14_bundle.get(
        "task_draft_id"
    )
    ==
    CELL14_EXPECTED_TASK_DRAFT_ID,

    cell14_bundle.get(
        "target_evidence_id"
    )
    ==
    CELL14_EXPECTED_TARGET_EVIDENCE_ID,
])


# ============================================================
# 7) Resolve Source Type / Stratum
# ============================================================

cell14_source_type_match = (
    cell14_find_first_scalar(
        cell14_pilot_packet,
        [
            "source_type",
            "source_kind",
            "source",
        ],
    )
)


cell14_stratum_match = (
    cell14_find_first_scalar(
        cell14_pilot_packet,
        [
            "stratum",
            "stratum_name",
            "stratum_id",
        ],
    )
)


cell14_source_type = (
    cell14_source_type_match[
        "value"
    ]

    if isinstance(
        cell14_source_type_match,
        dict,
    )

    else None
)


cell14_stratum = (
    cell14_stratum_match[
        "value"
    ]

    if isinstance(
        cell14_stratum_match,
        dict,
    )

    else None
)


# ============================================================
# 8) Resolve real Target Text Parts
# ============================================================

cell14_target_text_parts = []


if isinstance(
    cell14_pilot_packet,
    dict,
):

    packet_parts = (
        cell14_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        packet_parts,
        list,
    ):

        cell14_target_text_parts = (
            copy.deepcopy(
                packet_parts
            )
        )


if not (
    cell14_target_text_parts
):

    for path, key, nested in (
        cell14_walk(
            cell14_pilot_packet
        )
    ):

        if (
            key
            ==
            "target_text_parts"
            and
            isinstance(
                nested,
                list,
            )
        ):

            cell14_target_text_parts = (
                copy.deepcopy(
                    nested
                )
            )

            break


cell14_target_texts = []


for index, part in enumerate(
    cell14_target_text_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = (
            part.get(
                field_name
            )
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = (
                candidate.strip()
            )

            break


    if text_value is None:
        continue


    cell14_target_texts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell14_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell14_target_texts
    )
)


cell14_target_text_available = (
    isinstance(
        cell14_combined_target_text,
        str,
    )
    and
    bool(
        cell14_combined_target_text.strip()
    )
)


# ============================================================
# 9) Evidence metadata useful for authoring
# ============================================================

cell14_evidence_metadata = (
    cell14_find_scalar_fields(
        cell14_target_evidence,
        [
            "document",
            "source",
            "title",
            "section",
            "heading",
            "author",
            "url",
            "char_",
            "token",
            "role",
            "status",
        ],
    )
)


cell14_task_metadata = (
    cell14_find_scalar_fields(
        cell14_task_plan,
        [
            "task",
            "intent",
            "question",
            "archetype",
            "difficulty",
            "instruction",
            "constraint",
            "status",
            "source",
            "stratum",
        ],
    )
)


cell14_golden_metadata = (
    cell14_find_scalar_fields(
        cell14_golden_case,
        [
            "case",
            "task",
            "target",
            "source",
            "stratum",
            "archetype",
            "status",
            "difficulty",
        ],
    )
)


# ============================================================
# 10) Resolve Question-related contracts from compact Seed
# ============================================================

cell14_contracts = (
    cell13_seed.get(
        "contracts",
        {},
    )
)


CELL14_QUESTION_CONTRACT_NAMES = [
    "PILOT_QA_WORKFLOW_CONTRACT_V1",
    "QUESTION_DRAFT_RECORD_SCHEMA_V1",
    "QUESTION_DRAFT_ID_CONTRACT_V1",
    "QUESTION_DRAFT_ID_METHOD_VERSION",
    "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1",
    "PILOT_QUESTION_AUTHORING_MODE_V1",
    "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1",
    "PILOT_QUESTION_DRAFTING_CONTRACT_V1_DRAFT",
    "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1",
    "QUESTION_ARCHETYPES_V1",
    "QUESTION_ARCHETYPE_BY_STRATUM",
    "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT",
    "QUESTION_CONSTRUCTION_METHOD_V1_DRAFT",
    "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
]


cell14_question_contracts = {
    name:
        copy.deepcopy(
            cell14_contracts[
                name
            ]
        )

    for name
    in CELL14_QUESTION_CONTRACT_NAMES

    if name
    in
    cell14_contracts
}


# ============================================================
# 11) Resolve stratum-specific contract branches when possible
# ============================================================

cell14_stratum_contract_matches = {}


if (
    isinstance(
        cell14_stratum,
        str,
    )
    and
    cell14_stratum.strip()
):

    for contract_name in (
        "QUESTION_ARCHETYPE_BY_STRATUM",
        "STRATUM_QUESTION_CONSTRAINTS_V1_DRAFT",
    ):

        contract_value = (
            cell14_question_contracts.get(
                contract_name
            )
        )


        if not isinstance(
            contract_value,
            dict,
        ):
            continue


        if (
            cell14_stratum
            in
            contract_value
        ):

            cell14_stratum_contract_matches[
                contract_name
            ] = copy.deepcopy(
                contract_value[
                    cell14_stratum
                ]
            )

            continue


        for key, value in (
            contract_value.items()
        ):

            if (
                isinstance(
                    key,
                    str,
                )
                and
                key.lower()
                ==
                cell14_stratum.lower()
            ):

                cell14_stratum_contract_matches[
                    contract_name
                ] = copy.deepcopy(
                    value
                )

                break


# ============================================================
# 12) Build authoring constraints bundle
# ============================================================

cell14_authoring_constraints = {
    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "stratum_specific":
        cell14_stratum_contract_matches,

    "authoring_mode":
        copy.deepcopy(
            cell14_question_contracts.get(
                "PILOT_QUESTION_AUTHORING_MODE_V1"
            )
        ),

    "source_opacity_rule":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_USER_FACING_SOURCE_OPACITY_RULE_V1"
            )
        ),

    "acceptance_rubric":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_DRAFT_ACCEPTANCE_RUBRIC_V1"
            )
        ),

    "construction_guardrails":
        copy.deepcopy(
            cell14_question_contracts.get(
                "QUESTION_CONSTRUCTION_GUARDRAILS_V1_DRAFT"
            )
        ),

    "four_component_contract":
        copy.deepcopy(
            cell14_question_contracts.get(
                "PILOT_QUESTION_DRAFTING_FOUR_COMPONENT_CONTRACT_V1"
            )
        ),
}


cell14_authoring_constraints_sha256 = (
    cell10_stable_sha(
        cell14_authoring_constraints
    )
)


# ============================================================
# 13) Build REAL Question Authoring Workspace
#
# No Question text is generated or registered here.
# ============================================================

cell14_question_authoring_workspace = {
    "workspace_version":
        "replacement_pilot_question_authoring_workspace_v1",

    "status":
        "READY_FOR_REAL_QUESTION_AUTHORING",

    "pilot_case_id":
        CELL14_EXPECTED_PILOT_ID,

    "golden_case_id":
        CELL14_EXPECTED_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL14_EXPECTED_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL14_EXPECTED_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "target_text_parts":
        copy.deepcopy(
            cell14_target_texts
        ),

    "combined_target_text":
        cell14_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell14_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell14_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell14_target_evidence
        ),

    "authoring_constraints":
        copy.deepcopy(
            cell14_authoring_constraints
        ),

    "question_draft":
        None,

    "question_draft_id":
        None,

    "human_question_decision":
        None,

    "expected_answer":
        None,

    "citation_mapping":
        None,

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "queue_advance_permitted":
        False,

    "full_572_generation":
        CELL14_FULL_572_BLOCK_STATUS,
}


cell14_question_authoring_workspace_sha256 = (
    cell10_stable_sha(
        cell14_question_authoring_workspace
    )
)


# ============================================================
# 14) Readiness checks
# ============================================================

cell14_workspace_readiness = {
    "identity_valid":
        cell14_identity_valid,

    "pilot_packet_present":
        isinstance(
            cell14_pilot_packet,
            dict,
        ),

    "golden_case_present":
        isinstance(
            cell14_golden_case,
            dict,
        ),

    "task_plan_present":
        isinstance(
            cell14_task_plan,
            dict,
        ),

    "target_evidence_present":
        isinstance(
            cell14_target_evidence,
            dict,
        ),

    "target_text_part_count":
        len(
            cell14_target_texts
        ),

    "target_text_available":
        cell14_target_text_available,

    "question_contract_count":
        len(
            cell14_question_contracts
        ),

    "workflow_state_correct":
        (
            cell14_session.get(
                "current_state"
            )
            ==
            "QUESTION_DRAFT_PENDING"
        ),

    "question_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "question_draft"
            )
            is None
        ),

    "expected_answer_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "expected_answer"
            )
            is None
        ),

    "citation_not_started":
        (
            cell14_session.get(
                "artifacts",
                {},
            ).get(
                "citation_mapping"
            )
            is None
        ),

    "human_decision_count":
        len(
            cell14_session.get(
                "human_decisions",
                [],
            )
        ),

    "queue_advance_permitted":
        cell14_session.get(
            "queue_advance_permitted"
        ),

    "queue_advance_performed":
        cell14_session.get(
            "queue_advance_performed"
        ),
}


cell14_ready_for_question_authoring = all([
    cell14_workspace_readiness[
        "identity_valid"
    ],

    cell14_workspace_readiness[
        "pilot_packet_present"
    ],

    cell14_workspace_readiness[
        "golden_case_present"
    ],

    cell14_workspace_readiness[
        "task_plan_present"
    ],

    cell14_workspace_readiness[
        "target_evidence_present"
    ],

    cell14_workspace_readiness[
        "target_text_available"
    ],

    cell14_workspace_readiness[
        "question_contract_count"
    ]
    >
    0,

    cell14_workspace_readiness[
        "workflow_state_correct"
    ],

    cell14_workspace_readiness[
        "question_not_started"
    ],

    cell14_workspace_readiness[
        "expected_answer_not_started"
    ],

    cell14_workspace_readiness[
        "citation_not_started"
    ],

    cell14_workspace_readiness[
        "human_decision_count"
    ]
    ==
    0,

    cell14_workspace_readiness[
        "queue_advance_permitted"
    ]
    is False,

    cell14_workspace_readiness[
        "queue_advance_performed"
    ]
    is False,
])


# ============================================================
# 15) Confirm no mutation
# ============================================================

cell14_seed_after_sha256 = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell14_session_after_sha256 = (
    cell10_stable_sha(
        cell13_current_session
    )
)


cell14_seed_unchanged = (
    cell14_seed_before_sha256
    ==
    cell14_seed_after_sha256
)


cell14_session_unchanged = (
    cell14_session_before_sha256
    ==
    cell14_session_after_sha256
)


cell14_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell14_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL14_EXPECTED_PILOT_ID
)


cell14_full_572_blocked = (
    cell14_question_authoring_workspace[
        "full_572_generation"
    ]
    ==
    CELL14_FULL_572_BLOCK_STATUS
)


# ============================================================
# 16) Final Gate
# ============================================================

cell14_gate = all([
    len(
        cell14_failures
    )
    ==
    0,

    cell14_ready_for_question_authoring,

    cell14_seed_unchanged,

    cell14_session_unchanged,

    cell14_queue_still_22,

    cell14_next_pilot_unchanged,

    cell14_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== REAL REPLACEMENT-PILOT "
    "QUESTION AUTHORING WORKSPACE ==="
)


print(
    "\n=== IDENTITY ==="
)

print(
    "Pilot:",
    cell14_bundle.get(
        "pilot_case_id"
    ),
)

print(
    "Golden Case:",
    cell14_bundle.get(
        "golden_case_id"
    ),
)

print(
    "Task Draft:",
    cell14_bundle.get(
        "task_draft_id"
    ),
)

print(
    "Target Evidence:",
    cell14_bundle.get(
        "target_evidence_id"
    ),
)

print(
    "Identity valid:",
    cell14_identity_valid,
)


print(
    "\n=== CLASSIFICATION ==="
)

print(
    "Source type:",
    cell14_source_type,
)

print(
    "Source type path:",
    (
        cell14_source_type_match.get(
            "path"
        )

        if isinstance(
            cell14_source_type_match,
            dict,
        )

        else None
    ),
)

print(
    "Stratum:",
    cell14_stratum,
)

print(
    "Stratum path:",
    (
        cell14_stratum_match.get(
            "path"
        )

        if isinstance(
            cell14_stratum_match,
            dict,
        )

        else None
    ),
)


print(
    "\n=== TARGET TEXT PARTS ==="
)

print(
    "Count:",
    len(
        cell14_target_texts
    ),
)


for row in (
    cell14_target_texts
):

    print(
        "\n--- TARGET PART",
        row[
            "part_index"
        ],
        "---"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Role:",
        row[
            "role"
        ],
    )

    print(
        "Chars:",
        (
            row[
                "char_start"
            ],
            row[
                "char_end"
            ],
        ),
    )

    print(
        "SHA256:",
        row[
            "text_sha256"
        ],
    )

    print(
        "\nTEXT:\n"
        +
        row[
            "text"
        ]
    )


print(
    "\n=== TARGET EVIDENCE METADATA ==="
)

for row in (
    cell14_evidence_metadata[
        :80
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 600)}"
    )


print(
    "\n=== TASK PLAN AUTHORING SIGNALS ==="
)

for row in (
    cell14_task_metadata[
        :100
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 800)}"
    )


print(
    "\n=== GOLDEN CASE SIGNALS ==="
)

for row in (
    cell14_golden_metadata[
        :80
    ]
):

    print(
        f"{row['path']}: "
        f"{cell14_clip(row['value'], 600)}"
    )


print(
    "\n=== STRATUM-SPECIFIC QUESTION CONTRACT ==="
)

if (
    cell14_stratum_contract_matches
):

    print(
        cell14_json_preview(
            cell14_stratum_contract_matches,
            12000,
        )
    )

else:

    print(
        "No exact stratum-keyed contract branch "
        "was found; global contracts remain available."
    )


print(
    "\n=== QUESTION AUTHORING MODE ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "authoring_mode"
        ],
        8000,
    )
)


print(
    "\n=== SOURCE OPACITY RULE ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "source_opacity_rule"
        ],
        8000,
    )
)


print(
    "\n=== QUESTION ACCEPTANCE RUBRIC ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "acceptance_rubric"
        ],
        12000,
    )
)


print(
    "\n=== QUESTION CONSTRUCTION GUARDRAILS ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "construction_guardrails"
        ],
        12000,
    )
)


print(
    "\n=== FOUR-COMPONENT CONTRACT ==="
)

print(
    cell14_json_preview(
        cell14_authoring_constraints[
            "four_component_contract"
        ],
        12000,
    )
)


print(
    "\n=== WORKSPACE READINESS ==="
)

for key, value in (
    cell14_workspace_readiness.items()
):

    print(
        f"{key}:",
        value,
    )


print(
    "\nReady for real Question authoring:",
    cell14_ready_for_question_authoring,
)

print(
    "Workspace SHA256:",
    cell14_question_authoring_workspace_sha256,
)

print(
    "Authoring constraints SHA256:",
    cell14_authoring_constraints_sha256,
)


print(
    "\n=== CURRENT REAL WORKFLOW STATE ==="
)

print(
    "State:",
    cell14_session.get(
        "current_state"
    ),
)

print(
    "Question artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "question_draft"
    ),
)

print(
    "Expected Answer artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "expected_answer"
    ),
)

print(
    "Citation artifact:",
    cell14_session.get(
        "artifacts",
        {},
    ).get(
        "citation_mapping"
    ),
)

print(
    "Human decisions:",
    len(
        cell14_session.get(
            "human_decisions",
            [],
        )
    ),
)


print(
    "\n=== IMMUTABILITY ==="
)

print(
    "Seed unchanged:",
    cell14_seed_unchanged,
)

print(
    "Actual session unchanged:",
    cell14_session_unchanged,
)

print(
    "Queue still 22:",
    cell14_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell14_next_pilot_unchanged,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell14_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell14_failures
)


print(
    "\nCODE CELL 14 GATE:",
    (
        "PASS"
        if cell14_gate
        else "FAIL"
    ),
)


assert cell14_gate, (
    "Replacement-Pilot Question authoring workspace "
    "validation failed. Repair this SAME Code Cell 14 "
    "before authoring the real Question Draft."
)

=== REAL REPLACEMENT-PILOT QUESTION AUTHORING WORKSPACE ===

=== IDENTITY ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Golden Case: golden-case:3b93ce20ef4b617b9f922baddfbd1e0f
Task Draft: task-draft:f48d78f0a6a588de906ca8534219a604
Target Evidence: target-evidence:81011a6ef78817e01e3ed484b1b4c368
Identity valid: True

=== CLASSIFICATION ===
Source type: website
Source type path: root.source_type
Stratum: direct_author_long
Stratum path: root.stratum

=== TARGET TEXT PARTS ===
Count: 1

--- TARGET PART 0 ---
Document: website:5C92BB0D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1123, 1401)
SHA256: c75bb83b57206bd361e4aaedfe675484cbd2bbbc669bc248a3d5c97165d4be24

TEXT:
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== TARGET EVIDENCE METADATA ===
root

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 15 — ساخت و Machine Validation سؤال واقعی Pilot جایگزین

در این مرحله اولین Question Draft واقعی برای Pilot جایگزین ساخته می‌شود.

Question فقط از Target Evidence فعلی نوشته می‌شود و هیچ دانش خارجی، نام نویسنده، نام منبع، URL، شناسه‌ی داخلی یا پاسخ مورد انتظار وارد متن سؤال نمی‌شود.

سؤال پیشنهادی باید با Task Family و Question Archetype فعلی هم‌راستا باشد و مسئله‌ی «برداشت ناقص از بخشی از یک کل» را به شکلی طبیعی و قابل‌پاسخ از Evidence موجود بسنجد.

پس از ساخت سؤال:

- `question_text_sha256` محاسبه می‌شود؛
- `question_draft_id` با Primitive پذیرفته‌شده ساخته می‌شود؛
- Question Draft Record مطابق Schema موجود تشکیل می‌شود؛
- Hard Gateها و Semantic Gateهای قابل‌بررسی ماشینی اجرا می‌شوند؛
- در صورت PASS کامل، Artifact فقط در Session جدید ثبت می‌شود و Workflow به `QUESTION_HUMAN_REVIEW_PENDING` می‌رود.

این مرحله Human Acceptance انجام نمی‌دهد.

Expected Answer و Citation همچنان Blocked می‌مانند، Queue جلو نمی‌رود و Full 572 Generation نیز همچنان Blocked است.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [37]:
# Code Cell 15

cell15_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell14_gate"
    )
    is not True
):
    cell15_failures.append(
        "CELL14_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell14_question_authoring_workspace"
    ),
    dict,
):
    cell15_failures.append(
        "QUESTION_AUTHORING_WORKSPACE_MISSING"
    )


if not callable(
    globals().get(
        "cell12_question_id_function"
    )
):
    cell15_failures.append(
        "QUESTION_ID_BUILDER_MISSING"
    )


# ============================================================
# 2) Fixed accepted identities
# ============================================================

CELL15_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL15_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL15_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL15_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL15_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) REAL Question Draft
#
# Grounded only in the current Target Evidence.
# ============================================================

CELL15_QUESTION_TEXT = (
    "چرا به کار گرفتن فیل بر اساس تصویری که فقط "
    "از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟"
)


cell15_question_text = (
    CELL15_QUESTION_TEXT.strip()
)


cell15_question_text_sha256 = (
    hashlib.sha256(
        cell15_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell15_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL15_PILOT_ID,

        question_text_sha256=
            cell15_question_text_sha256,
    )
)


# ============================================================
# 4) Local field-resolution helpers
# ============================================================

def cell15_walk_exact_fields(
    value,
    wanted_key,
    path="root",
    seen=None,
):

    if seen is None:
        seen = set()


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for key, nested in (
                value.items()
            ):

                nested_path = (
                    f"{path}.{key}"
                )


                if (
                    isinstance(
                        key,
                        str,
                    )
                    and
                    key
                    ==
                    wanted_key
                ):

                    yield {
                        "path":
                            nested_path,

                        "value":
                            nested,
                    }


                yield from (
                    cell15_walk_exact_fields(
                        nested,
                        wanted_key,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


    elif isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return


        seen.add(
            object_id
        )


        try:

            for index, nested in enumerate(
                value
            ):

                nested_path = (
                    f"{path}[{index}]"
                )


                yield from (
                    cell15_walk_exact_fields(
                        nested,
                        wanted_key,
                        nested_path,
                        seen,
                    )
                )


        finally:

            seen.discard(
                object_id
            )


def cell15_resolve_field(
    field_name,
    sources,
):

    for source_name, source_value in sources:

        matches = list(
            cell15_walk_exact_fields(
                source_value,
                field_name,
                path=source_name,
            )
        )


        if matches:

            return {
                "source":
                    source_name,

                "path":
                    matches[
                        0
                    ][
                        "path"
                    ],

                "value":
                    copy.deepcopy(
                        matches[
                            0
                        ][
                            "value"
                        ]
                    ),

                "match_count":
                    len(
                        matches
                    ),
            }


    return {
        "source":
            None,

        "path":
            None,

        "value":
            None,

        "match_count":
            0,
    }


def cell15_normalize_text(
    value,
):

    if not isinstance(
        value,
        str,
    ):
        return ""


    value = (
        value
        .replace(
            "\u200c",
            " "
        )
        .replace(
            "ي",
            "ی"
        )
        .replace(
            "ك",
            "ک"
        )
    )


    value = re.sub(
        r"\s+",
        " ",
        value,
    )


    return (
        value
        .strip()
        .lower()
    )


# ============================================================
# 5) Resolve authoritative drafting fields
# ============================================================

cell15_field_sources = [
    (
        "pilot_packet",
        cell14_pilot_packet,
    ),
    (
        "task_plan",
        cell14_task_plan,
    ),
    (
        "golden_case",
        cell14_golden_case,
    ),
    (
        "target_evidence",
        cell14_target_evidence,
    ),
]


cell15_task_family_resolution = (
    cell15_resolve_field(
        "task_family",
        cell15_field_sources,
    )
)


cell15_question_archetype_resolution = (
    cell15_resolve_field(
        "question_archetype",
        cell15_field_sources,
    )
)


cell15_required_dimensions_resolution = (
    cell15_resolve_field(
        "required_dimensions",
        cell15_field_sources,
    )
)


cell15_authority_expectation_resolution = (
    cell15_resolve_field(
        "authority_expectation",
        cell15_field_sources,
    )
)


cell15_context_expectation_resolution = (
    cell15_resolve_field(
        "context_expectation",
        cell15_field_sources,
    )
)


cell15_evidence_scope_resolution = (
    cell15_resolve_field(
        "evidence_scope",
        cell15_field_sources,
    )
)


cell15_task_family = (
    cell15_task_family_resolution[
        "value"
    ]
)


cell15_question_archetype = (
    cell15_question_archetype_resolution[
        "value"
    ]
)


cell15_required_dimensions = (
    cell15_required_dimensions_resolution[
        "value"
    ]
)


cell15_authority_expectation = (
    cell15_authority_expectation_resolution[
        "value"
    ]
)


cell15_context_expectation = (
    cell15_context_expectation_resolution[
        "value"
    ]
)


cell15_evidence_scope = (
    cell15_evidence_scope_resolution[
        "value"
    ]
)


# ============================================================
# 6) Required-dimensions normalization
# ============================================================

if isinstance(
    cell15_required_dimensions,
    tuple,
):

    cell15_required_dimensions = list(
        cell15_required_dimensions
    )


elif isinstance(
    cell15_required_dimensions,
    set,
):

    cell15_required_dimensions = sorted(
        cell15_required_dimensions
    )


cell15_required_dimensions_valid = (
    isinstance(
        cell15_required_dimensions,
        list,
    )
    and
    len(
        cell15_required_dimensions
    )
    >
    0
)


# ============================================================
# 7) Question Draft Schema
# ============================================================

cell15_question_schema = (
    cell14_question_contracts.get(
        "QUESTION_DRAFT_RECORD_SCHEMA_V1",
        {},
    )
)


cell15_question_schema_version = (
    cell15_question_schema.get(
        "schema_version"
    )
)


cell15_initial_statuses = (
    cell15_question_schema.get(
        "initial_statuses",
        {},
    )
)


cell15_authoring_mode_contract = (
    cell14_question_contracts.get(
        "PILOT_QUESTION_AUTHORING_MODE_V1",
        {},
    )
)


cell15_authoring_mode = (
    cell15_authoring_mode_contract.get(
        "mode"
    )
)


# ============================================================
# 8) Evidence / target validation state
# ============================================================

cell15_target_status_resolution = (
    cell15_resolve_field(
        "target_evidence_status",
        cell15_field_sources,
    )
)


cell15_target_evidence_status = (
    cell15_target_status_resolution[
        "value"
    ]
)


cell15_target_already_validated = (
    cell15_target_evidence_status
    ==
    "SELECTED_AND_VALIDATED"
)


# ============================================================
# 9) Hard-gate validation
# ============================================================

cell15_normalized_question = (
    cell15_normalize_text(
        cell15_question_text
    )
)


cell15_normalized_target = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


CELL15_INTERNAL_ID_PATTERNS = [
    r"question-pilot:",
    r"golden-case:",
    r"task-draft:",
    r"target-evidence:",
    r"website:",
]


CELL15_INTERNAL_METADATA_TERMS = [
    "document_id",
    "target_evidence_id",
    "golden_case_id",
    "task_draft_id",
    "pilot_case_id",
    "sha256",
    "char_start",
    "char_end",
    "token_count",
    "pipeline",
]


CELL15_SOURCE_IDENTITY_TERMS = [
    "دکتر کمیل رودی",
    "کمیل رودی",
    "fintelligence",
    "fintelligence.ir",
    "www.",
    "http://",
    "https://",
]


CELL15_UNSUPPORTED_ATTRIBUTION_TERMS = [
    "نویسنده می‌گوید",
    "نویسنده می گوید",
    "مقاله می‌گوید",
    "مقاله می گوید",
    "این منبع",
    "طبق منبع",
]


CELL15_ANSWER_LEAKAGE_PHRASES = [
    "کل به هم پیوسته است",
    "کل به‌هم پیوسته است",
    "پای فیل جدای گوش",
    "پای فیل جدا از گوش",
    "اجزای آن از هم جدا نیستند",
    "بخش های آن از هم جدا نیستند",
    "بخش‌های آن از هم جدا نیستند",
]


cell15_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell15_question_text,
        flags=re.IGNORECASE,
    )
]


cell15_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell15_question_text.lower()
]


cell15_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell15_question_text.lower()
]


cell15_unsupported_attribution_hits = [
    term

    for term
    in CELL15_UNSUPPORTED_ATTRIBUTION_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell15_normalized_question
]


cell15_answer_leakage_hits = [
    phrase

    for phrase
    in CELL15_ANSWER_LEAKAGE_PHRASES

    if cell15_normalize_text(
        phrase
    )
    in
    cell15_normalized_question
]


cell15_question_nonempty = (
    bool(
        cell15_question_text
    )
)


cell15_question_looks_natural = all([
    len(
        cell15_question_text
    )
    >=
    20,

    cell15_question_text.endswith(
        (
            "؟",
            "?",
        )
    ),

    "فیل"
    in
    cell15_question_text,

    "چرا"
    in
    cell15_question_text,
])


cell15_no_internal_id_leakage = (
    len(
        cell15_internal_id_hits
    )
    ==
    0
)


cell15_no_internal_metadata_leakage = (
    len(
        cell15_internal_metadata_hits
    )
    ==
    0
)


cell15_source_opacity_valid = (
    len(
        cell15_source_identity_hits
    )
    ==
    0
)


cell15_no_unsupported_attribution = (
    len(
        cell15_unsupported_attribution_hits
    )
    ==
    0
)


cell15_no_answer_leakage = (
    len(
        cell15_answer_leakage_hits
    )
    ==
    0
)


# ============================================================
# 10) Groundability / evidence-scope validation
# ============================================================

CELL15_REQUIRED_GROUNDING_SIGNALS = [
    "فیل",
    "بر اساس",
    "تصویر",
]


cell15_grounding_signal_results = {
    signal:
        (
            cell15_normalize_text(
                signal
            )
            in
            cell15_normalized_question
            and
            cell15_normalize_text(
                signal
            )
            in
            cell15_normalized_target
        )

    for signal
    in CELL15_REQUIRED_GROUNDING_SIGNALS
}


cell15_grounding_signals_valid = all(
    cell15_grounding_signal_results.values()
)


cell15_target_contains_core_explanation = all([
    "کل"
    in
    cell14_combined_target_text,

    (
        "به‌هم"
        in
        cell14_combined_target_text
        or
        "به هم"
        in
        cell14_combined_target_text
    ),

    "جدای"
    in
    cell14_combined_target_text,
])


cell15_question_groundable_in_target = all([
    cell14_target_text_available,

    cell15_grounding_signals_valid,

    cell15_target_contains_core_explanation,
])


# ============================================================
# 11) Semantic alignment checks
# ============================================================

cell15_task_family_alignment = (
    cell15_task_family
    ==
    "LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL"
)


cell15_question_archetype_alignment = (
    cell15_question_archetype
    ==
    "EVIDENCE_COMPLETENESS"
)


cell15_evidence_completeness_dimension_present = (
    cell15_required_dimensions_valid
    and
    "evidence_completeness"
    in
    cell15_required_dimensions
)


cell15_question_tests_incomplete_view = all([
    (
        "یکی از بخش"
        in
        cell15_normalized_question
        or
        "یک بخش"
        in
        cell15_normalized_question
    ),

    "مشکل"
    in
    cell15_normalized_question,
])


cell15_question_not_trivial_metadata_repetition = all([
    "شناسه"
    not in
    cell15_normalized_question,

    "نام نویسنده"
    not in
    cell15_normalized_question,

    "آدرس"
    not in
    cell15_normalized_question,

    "چند توکن"
    not in
    cell15_normalized_question,
])


cell15_external_knowledge_dependency_absent = all([
    cell15_question_groundable_in_target,

    cell15_source_opacity_valid,

    cell15_no_unsupported_attribution,
])


# ============================================================
# 12) Machine validation result
# ============================================================

cell15_hard_gates = {
    "nonempty_question_text":
        cell15_question_nonempty,

    "natural_user_question":
        cell15_question_looks_natural,

    "target_evidence_must_already_be_validated":
        cell15_target_already_validated,

    "question_must_be_groundable_in_target_text_parts":
        cell15_question_groundable_in_target,

    "external_knowledge_dependency":
        cell15_external_knowledge_dependency_absent,

    "answer_leakage_in_question":
        cell15_no_answer_leakage,

    "internal_document_id_leakage":
        cell15_no_internal_id_leakage,

    "internal_pipeline_metadata_leakage":
        cell15_no_internal_metadata_leakage,

    "unsupported_attribution":
        cell15_no_unsupported_attribution,

    "unsupported_authority_promotion":
        cell15_no_unsupported_attribution,

    "source_identity_disclosure":
        cell15_source_opacity_valid,

    "locator_fabrication":
        True,

    "unknown_speaker_inference":
        True,
}


cell15_semantic_gates = {
    "task_family_alignment":
        cell15_task_family_alignment,

    "question_archetype_alignment":
        cell15_question_archetype_alignment,

    "evidence_scope_alignment":
        cell15_question_groundable_in_target,

    "question_is_answerable_without_external_context":
        cell15_external_knowledge_dependency_absent,

    "question_is_not_trivial_metadata_repetition":
        cell15_question_not_trivial_metadata_repetition,

    "required_dimensions_are_testable_by_question":
        (
            cell15_required_dimensions_valid
            and
            cell15_evidence_completeness_dimension_present
            and
            cell15_question_tests_incomplete_view
        ),

    "authority_expectation_alignment":
        (
            cell15_authority_expectation
            is not None
        ),

    "context_expectation_alignment":
        (
            cell15_context_expectation
            is not None
        ),
}


cell15_machine_validation_pass = all([
    all(
        cell15_hard_gates.values()
    ),

    all(
        cell15_semantic_gates.values()
    ),
])


cell15_machine_validation_status = (
    "PASS_PENDING_HUMAN_ADJUDICATION"

    if cell15_machine_validation_pass

    else "FAIL_REQUIRES_REVISION"
)


# ============================================================
# 13) Build Question Draft Record
# ============================================================

cell15_question_draft_record = {
    "question_draft_id":
        cell15_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL15_PILOT_ID,

    "golden_case_id":
        CELL15_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL15_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL15_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "task_family":
        cell15_task_family,

    "question_archetype":
        cell15_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell15_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell15_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell15_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell15_evidence_scope
        ),

    "question_text":
        cell15_question_text,

    "question_text_sha256":
        cell15_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        (
            cell15_initial_statuses.get(
                "draft_status",
                "DRAFT_CREATED_PENDING_VALIDATION",
            )
        ),

    "machine_validation_status":
        cell15_machine_validation_status,

    "human_adjudication_status":
        "PENDING",

    "expected_answer_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "citation_target_validation_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "machine_validation":
        {
            "hard_gates":
                copy.deepcopy(
                    cell15_hard_gates
                ),

            "semantic_gates":
                copy.deepcopy(
                    cell15_semantic_gates
                ),

            "internal_id_hits":
                copy.deepcopy(
                    cell15_internal_id_hits
                ),

            "internal_metadata_hits":
                copy.deepcopy(
                    cell15_internal_metadata_hits
                ),

            "source_identity_hits":
                copy.deepcopy(
                    cell15_source_identity_hits
                ),

            "unsupported_attribution_hits":
                copy.deepcopy(
                    cell15_unsupported_attribution_hits
                ),

            "answer_leakage_hits":
                copy.deepcopy(
                    cell15_answer_leakage_hits
                ),

            "grounding_signals":
                copy.deepcopy(
                    cell15_grounding_signal_results
                ),
        },

    "full_572_generation":
        CELL15_FULL_572_BLOCK_STATUS,
}


cell15_question_draft_record_sha256 = (
    cell10_stable_sha(
        cell15_question_draft_record
    )
)


# ============================================================
# 14) Required-schema-field validation
# ============================================================

cell15_schema_required_fields = (
    cell15_question_schema.get(
        "required_fields",
        []
    )
)


cell15_missing_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        not in
        cell15_question_draft_record
    )
]


cell15_empty_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        in
        cell15_question_draft_record
        and
        cell15_question_draft_record[
            field
        ]
        is None
    )
]


cell15_schema_surface_valid = all([
    isinstance(
        cell15_schema_required_fields,
        list,
    ),

    len(
        cell15_schema_required_fields
    )
    >
    0,

    len(
        cell15_missing_required_fields
    )
    ==
    0,

    len(
        cell15_empty_required_fields
    )
    ==
    0,
])


# ============================================================
# 15) ID reproducibility
# ============================================================

cell15_rebuilt_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL15_PILOT_ID,

        question_text_sha256=
            cell15_question_text_sha256,
    )
)


cell15_question_id_reproducible = (
    cell15_rebuilt_question_draft_id
    ==
    cell15_question_draft_id
)


# ============================================================
# 16) Register REAL Question Draft in a new real session
#
# Only if machine validation is PASS.
# No human decision is applied here.
# ============================================================

cell15_current_session = None


if all([
    cell15_machine_validation_pass,
    cell15_schema_surface_valid,
    cell15_question_id_reproducible,
]):

    cell15_current_session = (
        cell13_apply_event(
            cell13_current_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell15_question_draft_record,
        )
    )


# ============================================================
# 17) Validate post-registration workflow state
# ============================================================

cell15_question_registered = all([
    isinstance(
        cell15_current_session,
        dict,
    ),

    cell15_current_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "question_draft"
        ),
        dict,
    ),

    cell15_current_session.get(
        "artifacts",
        {},
    ).get(
        "expected_answer"
    )
    is None,

    cell15_current_session.get(
        "artifacts",
        {},
    ).get(
        "citation_mapping"
    )
    is None,

    len(
        cell15_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    0,

    cell15_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell15_current_session.get(
        "queue_advance_performed"
    )
    is False,

    cell15_current_session.get(
        "full_572_generation"
    )
    ==
    CELL15_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Confirm accepted previous states remain unchanged
# ============================================================

cell15_cell13_session_unchanged = (
    cell10_stable_sha(
        cell13_current_session
    )
    ==
    cell14_session_before_sha256
)


cell15_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell15_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell15_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL15_PILOT_ID
)


cell15_full_572_blocked = all([
    cell15_question_draft_record[
        "full_572_generation"
    ]
    ==
    CELL15_FULL_572_BLOCK_STATUS,

    isinstance(
        cell15_current_session,
        dict,
    ),

    cell15_current_session.get(
        "full_572_generation"
    )
    ==
    CELL15_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 19) Final Gate
# ============================================================

cell15_gate = all([
    len(
        cell15_failures
    )
    ==
    0,

    cell15_question_nonempty,

    cell15_target_already_validated,

    cell15_required_dimensions_valid,

    cell15_machine_validation_pass,

    cell15_schema_surface_valid,

    cell15_question_id_reproducible,

    cell15_question_registered,

    cell15_cell13_session_unchanged,

    cell15_seed_unchanged,

    cell15_queue_still_22,

    cell15_next_pilot_unchanged,

    cell15_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== REAL QUESTION DRAFT + "
    "MACHINE VALIDATION ==="
)


print(
    "\n=== QUESTION ==="
)

print(
    cell15_question_text
)

print(
    "\nQuestion text SHA256:",
    cell15_question_text_sha256,
)

print(
    "Question Draft ID:",
    cell15_question_draft_id,
)

print(
    "ID reproducible:",
    cell15_question_id_reproducible,
)


print(
    "\n=== AUTHORITATIVE TASK FIELDS ==="
)

print(
    "Task family:",
    cell15_task_family,
)

print(
    "Task family path:",
    cell15_task_family_resolution[
        "path"
    ],
)

print(
    "Question archetype:",
    cell15_question_archetype,
)

print(
    "Question archetype path:",
    cell15_question_archetype_resolution[
        "path"
    ],
)

print(
    "Required dimensions:",
    cell15_required_dimensions,
)

print(
    "Required dimensions path:",
    cell15_required_dimensions_resolution[
        "path"
    ],
)

print(
    "Authority expectation:",
    cell15_authority_expectation,
)

print(
    "Authority expectation path:",
    cell15_authority_expectation_resolution[
        "path"
    ],
)

print(
    "Context expectation:",
    cell15_context_expectation,
)

print(
    "Context expectation path:",
    cell15_context_expectation_resolution[
        "path"
    ],
)

print(
    "Evidence scope:",
    cell15_evidence_scope,
)

print(
    "Evidence scope path:",
    cell15_evidence_scope_resolution[
        "path"
    ],
)


print(
    "\n=== TARGET STATUS ==="
)

print(
    "Target evidence status:",
    cell15_target_evidence_status,
)

print(
    "Already validated:",
    cell15_target_already_validated,
)


print(
    "\n=== HARD GATES ==="
)

for name, status in (
    cell15_hard_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SEMANTIC GATES ==="
)

for name, status in (
    cell15_semantic_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== MACHINE VALIDATION ==="
)

print(
    "Machine validation:",
    cell15_machine_validation_status,
)

print(
    "Machine validation PASS:",
    cell15_machine_validation_pass,
)

print(
    "Internal ID hits:",
    cell15_internal_id_hits,
)

print(
    "Internal metadata hits:",
    cell15_internal_metadata_hits,
)

print(
    "Source identity hits:",
    cell15_source_identity_hits,
)

print(
    "Unsupported attribution hits:",
    cell15_unsupported_attribution_hits,
)

print(
    "Answer leakage hits:",
    cell15_answer_leakage_hits,
)

print(
    "Grounding signals:",
    cell15_grounding_signal_results,
)


print(
    "\n=== SCHEMA ==="
)

print(
    "Schema version:",
    cell15_question_schema_version,
)

print(
    "Required field count:",
    len(
        cell15_schema_required_fields
    ),
)

print(
    "Missing required fields:",
    cell15_missing_required_fields,
)

print(
    "Empty required fields:",
    cell15_empty_required_fields,
)

print(
    "Schema surface valid:",
    cell15_schema_surface_valid,
)

print(
    "Draft record SHA256:",
    cell15_question_draft_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Question registered:",
    cell15_question_registered,
)

print(
    "Current state:",
    (
        cell15_current_session.get(
            "current_state"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions:",
    (
        len(
            cell15_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer artifact:",
    (
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "expected_answer"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation artifact:",
    (
        cell15_current_session.get(
            "artifacts",
            {},
        ).get(
            "citation_mapping"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== IMMUTABILITY / QUEUE ==="
)

print(
    "Cell13 accepted session unchanged:",
    cell15_cell13_session_unchanged,
)

print(
    "Seed unchanged:",
    cell15_seed_unchanged,
)

print(
    "Queue still 22:",
    cell15_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell15_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell15_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Queue advance performed:",
    (
        cell15_current_session.get(
            "queue_advance_performed"
        )

        if isinstance(
            cell15_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Full 572 blocked:",
    cell15_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell15_failures
)


print(
    "\nCODE CELL 15 GATE:",
    (
        "PASS"
        if cell15_gate
        else "FAIL"
    ),
)


assert cell15_gate, (
    "Real Question Draft machine validation failed. "
    "Repair this SAME Code Cell 15 before Human "
    "Question Adjudication."
)

=== REAL QUESTION DRAFT + MACHINE VALIDATION ===

=== QUESTION ===
چرا به کار گرفتن فیل بر اساس تصویری که فقط از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟

Question text SHA256: 0aa5a7d1a53be44dc571a65113550a3fb651371903a61f3dac2c842e886bf9c0
Question Draft ID: question-draft:2e1c5547776e71cb5b216fe155dd5a54
ID reproducible: True

=== AUTHORITATIVE TASK FIELDS ===
Task family: LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL
Task family path: pilot_packet.task_family
Question archetype: EVIDENCE_COMPLETENESS
Question archetype path: pilot_packet.question_archetype
Required dimensions: ['retrieval_hit', 'rank', 'evidence_completeness', 'attribution_correctness', 'citation_locator_correctness', 'unnecessary_context', 'source_authority_correctness']
Required dimensions path: pilot_packet.required_dimensions
Authority expectation: PRIMARY_DIRECT_AUTHOR
Authority expectation path: pilot_packet.authority_expectation
Context expectation: SUFFICIENT_WITHOUT_EXCESS
Context expectation path: pilot_pa

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 16 — ثبت Human Revision و ساخت Question Draft اصلاح‌شده

Question Draft قبلی از نظر Machine Validation معتبر بود، اما در Human Review پذیرفته نشد.

دلیل Human Revision این است که سؤال قبلی بیش از حد خودِ مثال «فیل» را به موضوع سؤال تبدیل کرده بود. در طراحی سؤال برای مخاطب عمومی، سؤال باید روی مفهوم اصلی متن متمرکز باشد و مثال موجود در Evidence در صورت نیاز در پاسخ به توضیح مفهوم کمک کند.

در این مرحله:

- Question Draft قبلی حذف یا بازنویسی نمی‌شود و به‌عنوان Revision History حفظ می‌شود؛
- تصمیم انسانی `REVISE` ثبت می‌شود؛
- Evidence اصلی بدون تغییر در Human Review Lineage نگه داشته می‌شود؛
- برداشت‌ها یا پیشنهادهای انسانی که مستقیماً در Evidence وجود ندارند، صریحاً به‌عنوان `NON_EVIDENCE` علامت‌گذاری می‌شوند؛
- Question Draft جدید ساخته می‌شود؛
- Question Draft جدید ID مستقل و قابل‌بازتولید می‌گیرد؛
- Machine Validation دوباره روی Draft جدید اجرا می‌شود؛
- در صورت PASS، Workflow دوباره به `QUESTION_HUMAN_REVIEW_PENDING` می‌رود.

Question Draft جدید:

**«چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟»**

در این مرحله هنوز Human Acceptance نهایی انجام نمی‌شود.

Expected Answer و Citation ساخته نمی‌شوند، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

همچنین هنوز فایل Human Review Notebook نوشته نمی‌شود؛ فقط داده‌ی ساختاریافته‌ی لازم برای ساخت آن در مراحل بعد در حافظه آماده می‌شود.

</div>

In [38]:
# Code Cell 16

cell16_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell15_gate"
    )
    is not True
):
    cell16_failures.append(
        "CELL15_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell15_current_session"
    ),
    dict,
):
    cell16_failures.append(
        "CELL15_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell15_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "QUESTION_HUMAN_REVIEW_PENDING"
):
    cell16_failures.append(
        "QUESTION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL16_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL16_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL16_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL16_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL16_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Snapshot previous accepted machine-valid Draft
# ============================================================

cell16_previous_question_draft = copy.deepcopy(
    cell15_current_session[
        "artifacts"
    ][
        "question_draft"
    ]
)


cell16_previous_question_draft_id = (
    cell16_previous_question_draft.get(
        "question_draft_id"
    )
)


cell16_previous_question_text = (
    cell16_previous_question_draft.get(
        "question_text"
    )
)


cell16_previous_question_snapshot_sha256 = (
    cell10_stable_sha(
        cell16_previous_question_draft
    )
)


# ============================================================
# 4) Human Review decision for previous Draft
#
# Human comments are review metadata, NOT source evidence.
# ============================================================

CELL16_HUMAN_DECISION = (
    "REVISE"
)


CELL16_HUMAN_REVIEW_REASON = (
    "سؤال قبلی بیش از حد خودِ مثال فیل را موضوع سؤال "
    "قرار می‌داد. سؤال باید برای مخاطب عمومی ساده‌تر باشد "
    "و روی مفهوم اصلی متن یعنی خطر قضاوت یا تصمیم‌گیری "
    "درباره کل موضوع بر اساس شناخت ناقص از یک بخش تمرکز کند."
)


CELL16_HUMAN_CONTEXT_NOTE_NON_EVIDENCE = (
    "Reviewer discussed possible interpretive context around "
    "the elephant story, light, awareness, and financial "
    "literacy. These comments are explicitly NON_EVIDENCE "
    "and MUST NOT be inserted into the Golden Answer unless "
    "supported by validated source evidence."
)


cell16_human_review_record = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "review_stage":
        "QUESTION_DRAFT",

    "reviewed_question_draft_id":
        cell16_previous_question_draft_id,

    "reviewed_question_text":
        cell16_previous_question_text,

    "machine_validation_status":
        cell16_previous_question_draft.get(
            "machine_validation_status"
        ),

    "human_decision":
        CELL16_HUMAN_DECISION,

    "human_review_reason":
        CELL16_HUMAN_REVIEW_REASON,

    "human_context_note":
        CELL16_HUMAN_CONTEXT_NOTE_NON_EVIDENCE,

    "human_context_note_evidence_status":
        "NON_EVIDENCE",

    "source_evidence_snapshot":
        copy.deepcopy(
            cell14_target_texts
        ),

    "source_evidence_sha256":
        hashlib.sha256(
            cell14_combined_target_text.encode(
                "utf-8"
            )
        ).hexdigest(),

    "source_evidence_rule":
        (
            "ONLY_VALIDATED_TARGET_EVIDENCE_MAY_SUPPORT_"
            "QUESTION_OR_EXPECTED_ANSWER"
        ),

    "previous_draft_preserved":
        True,
}


cell16_human_review_record_sha256 = (
    cell10_stable_sha(
        cell16_human_review_record
    )
)


# ============================================================
# 5) Apply REAL Human REVISE event
# ============================================================

cell16_revision_session = (
    cell13_apply_event(
        cell15_current_session,
        "QUESTION_REVISE",
        human_note=
            CELL16_HUMAN_REVIEW_REASON,
    )
)


cell16_revision_event_valid = all([
    cell16_revision_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",

    len(
        cell16_revision_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell16_revision_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "QUESTION_REVISE",

    cell16_revision_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell16_revision_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Revised REAL Question
# ============================================================

CELL16_QUESTION_TEXT = (
    "چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط "
    "از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟"
)


cell16_question_text = (
    CELL16_QUESTION_TEXT.strip()
)


cell16_question_text_sha256 = (
    hashlib.sha256(
        cell16_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell16_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL16_PILOT_ID,

        question_text_sha256=
            cell16_question_text_sha256,
    )
)


cell16_new_id_differs_from_previous = (
    cell16_question_draft_id
    !=
    cell16_previous_question_draft_id
)


# ============================================================
# 7) Basic question-form validation
# ============================================================

cell16_normalized_question = (
    cell15_normalize_text(
        cell16_question_text
    )
)


cell16_question_nonempty = (
    bool(
        cell16_question_text
    )
)


cell16_question_looks_natural = all([
    len(
        cell16_question_text
    )
    >=
    20,

    cell16_question_text.endswith(
        (
            "؟",
            "?",
        )
    ),

    "چه مشکلی"
    in
    cell16_normalized_question,

    "بخش"
    in
    cell16_normalized_question,

    "کل"
    in
    cell16_normalized_question,
])


# ============================================================
# 8) Leakage / opacity validation
# ============================================================

cell16_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell16_question_text,
        flags=re.IGNORECASE,
    )
]


cell16_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell16_question_text.lower()
]


cell16_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell16_question_text.lower()
]


cell16_unsupported_attribution_hits = [
    term

    for term
    in CELL15_UNSUPPORTED_ATTRIBUTION_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell16_normalized_question
]


cell16_answer_leakage_hits = [
    phrase

    for phrase
    in CELL15_ANSWER_LEAKAGE_PHRASES

    if cell15_normalize_text(
        phrase
    )
    in
    cell16_normalized_question
]


cell16_no_internal_id_leakage = (
    len(
        cell16_internal_id_hits
    )
    ==
    0
)


cell16_no_internal_metadata_leakage = (
    len(
        cell16_internal_metadata_hits
    )
    ==
    0
)


cell16_source_opacity_valid = (
    len(
        cell16_source_identity_hits
    )
    ==
    0
)


cell16_no_unsupported_attribution = (
    len(
        cell16_unsupported_attribution_hits
    )
    ==
    0
)


cell16_no_answer_leakage = (
    len(
        cell16_answer_leakage_hits
    )
    ==
    0
)


# ============================================================
# 9) Evidence-groundability validation
# ============================================================

cell16_normalized_target = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


cell16_target_supports_partial_view = all([
    "تصویر"
    in
    cell16_normalized_target,

    "کل"
    in
    cell16_normalized_target,

    "پای فیل"
    in
    cell16_normalized_target,

    "گوش"
    in
    cell16_normalized_target,

    "خرطوم"
    in
    cell16_normalized_target,
])


cell16_question_expresses_partial_to_whole = all([
    "بخش"
    in
    cell16_normalized_question,

    "کل"
    in
    cell16_normalized_question,

    "شناخت"
    in
    cell16_normalized_question,
])


cell16_question_expresses_consequence = (
    "چه مشکلی"
    in
    cell16_normalized_question
)


cell16_question_groundable_in_target = all([
    cell14_target_text_available,

    cell16_target_supports_partial_view,

    cell16_question_expresses_partial_to_whole,

    cell16_question_expresses_consequence,
])


# ============================================================
# 10) Semantic alignment
# ============================================================

cell16_task_family_alignment = (
    cell15_task_family
    ==
    "LONG_DIRECT_PRIMARY_SOURCE_RETRIEVAL"
)


cell16_question_archetype_alignment = (
    cell15_question_archetype
    ==
    "EVIDENCE_COMPLETENESS"
)


cell16_required_dimensions_alignment = all([
    isinstance(
        cell15_required_dimensions,
        list,
    ),

    "evidence_completeness"
    in
    cell15_required_dimensions,
])


cell16_question_not_metadata_repetition = all([
    "شناسه"
    not in
    cell16_normalized_question,

    "نویسنده"
    not in
    cell16_normalized_question,

    "منبع"
    not in
    cell16_normalized_question,

    "آدرس"
    not in
    cell16_normalized_question,

    "توکن"
    not in
    cell16_normalized_question,
])


cell16_external_knowledge_dependency_absent = all([
    cell16_question_groundable_in_target,

    cell16_source_opacity_valid,

    cell16_no_unsupported_attribution,
])


# ============================================================
# 11) Machine gates
# ============================================================

cell16_hard_gates = {
    "nonempty_question_text":
        cell16_question_nonempty,

    "natural_user_question":
        cell16_question_looks_natural,

    "target_evidence_must_already_be_validated":
        cell15_target_already_validated,

    "question_must_be_groundable_in_target_text_parts":
        cell16_question_groundable_in_target,

    "external_knowledge_dependency":
        cell16_external_knowledge_dependency_absent,

    "answer_leakage_in_question":
        cell16_no_answer_leakage,

    "internal_document_id_leakage":
        cell16_no_internal_id_leakage,

    "internal_pipeline_metadata_leakage":
        cell16_no_internal_metadata_leakage,

    "unsupported_attribution":
        cell16_no_unsupported_attribution,

    "unsupported_authority_promotion":
        cell16_no_unsupported_attribution,

    "source_identity_disclosure":
        cell16_source_opacity_valid,

    "locator_fabrication":
        True,

    "unknown_speaker_inference":
        True,
}


cell16_semantic_gates = {
    "task_family_alignment":
        cell16_task_family_alignment,

    "question_archetype_alignment":
        cell16_question_archetype_alignment,

    "evidence_scope_alignment":
        cell16_question_groundable_in_target,

    "question_is_answerable_without_external_context":
        cell16_external_knowledge_dependency_absent,

    "question_is_not_trivial_metadata_repetition":
        cell16_question_not_metadata_repetition,

    "required_dimensions_are_testable_by_question":
        cell16_required_dimensions_alignment,

    "authority_expectation_alignment":
        (
            cell15_authority_expectation
            is not None
        ),

    "context_expectation_alignment":
        (
            cell15_context_expectation
            is not None
        ),
}


cell16_machine_validation_pass = all([
    all(
        cell16_hard_gates.values()
    ),

    all(
        cell16_semantic_gates.values()
    ),
])


cell16_machine_validation_status = (
    "PASS_PENDING_HUMAN_ADJUDICATION"

    if cell16_machine_validation_pass

    else "FAIL_REQUIRES_REVISION"
)


# ============================================================
# 12) Build revised Question Draft Record
# ============================================================

cell16_question_draft_record = {
    "question_draft_id":
        cell16_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "task_family":
        cell15_task_family,

    "question_archetype":
        cell15_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell15_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell15_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell15_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell15_evidence_scope
        ),

    "question_text":
        cell16_question_text,

    "question_text_sha256":
        cell16_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        "DRAFT_CREATED_PENDING_VALIDATION",

    "machine_validation_status":
        cell16_machine_validation_status,

    "human_adjudication_status":
        "PENDING",

    "expected_answer_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "citation_target_validation_status":
        "BLOCKED_PENDING_QUESTION_ACCEPTANCE",

    "revision_lineage":
        {
            "revision_number":
                2,

            "revises_question_draft_id":
                cell16_previous_question_draft_id,

            "revision_trigger":
                "HUMAN_REVIEW_REVISE",

            "human_review_record_sha256":
                cell16_human_review_record_sha256,
        },

    "machine_validation":
        {
            "hard_gates":
                copy.deepcopy(
                    cell16_hard_gates
                ),

            "semantic_gates":
                copy.deepcopy(
                    cell16_semantic_gates
                ),

            "internal_id_hits":
                copy.deepcopy(
                    cell16_internal_id_hits
                ),

            "internal_metadata_hits":
                copy.deepcopy(
                    cell16_internal_metadata_hits
                ),

            "source_identity_hits":
                copy.deepcopy(
                    cell16_source_identity_hits
                ),

            "unsupported_attribution_hits":
                copy.deepcopy(
                    cell16_unsupported_attribution_hits
                ),

            "answer_leakage_hits":
                copy.deepcopy(
                    cell16_answer_leakage_hits
                ),
        },

    "full_572_generation":
        CELL16_FULL_572_BLOCK_STATUS,
}


cell16_question_draft_record_sha256 = (
    cell10_stable_sha(
        cell16_question_draft_record
    )
)


# ============================================================
# 13) Schema validation
# ============================================================

cell16_missing_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if field
    not in
    cell16_question_draft_record
]


cell16_empty_required_fields = [
    field

    for field
    in cell15_schema_required_fields

    if (
        field
        in
        cell16_question_draft_record
        and
        cell16_question_draft_record[
            field
        ]
        is None
    )
]


cell16_schema_surface_valid = all([
    len(
        cell16_missing_required_fields
    )
    ==
    0,

    len(
        cell16_empty_required_fields
    )
    ==
    0,
])


# ============================================================
# 14) ID reproducibility
# ============================================================

cell16_rebuilt_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL16_PILOT_ID,

        question_text_sha256=
            cell16_question_text_sha256,
    )
)


cell16_question_id_reproducible = (
    cell16_rebuilt_question_draft_id
    ==
    cell16_question_draft_id
)


# ============================================================
# 15) Register revised Draft
#
# Session returns to Human Review.
# ============================================================

cell16_current_session = None


if all([
    cell16_revision_event_valid,
    cell16_machine_validation_pass,
    cell16_schema_surface_valid,
    cell16_question_id_reproducible,
    cell16_new_id_differs_from_previous,
]):

    cell16_current_session = (
        cell13_apply_event(
            cell16_revision_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell16_question_draft_record,
        )
    )


# ============================================================
# 16) Human Review lineage package
#
# This remains in-memory for later Human Review Notebook export.
# ============================================================

cell16_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL16_PILOT_ID,

    "golden_case_id":
        CELL16_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL16_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL16_TARGET_EVIDENCE_ID,

    "source_type":
        cell14_source_type,

    "stratum":
        cell14_stratum,

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell16_previous_question_draft_id,

                "question_text":
                    cell16_previous_question_text,

                "machine_validation":
                    cell16_previous_question_draft.get(
                        "machine_validation_status"
                    ),

                "human_decision":
                    "REVISE",
            },
            {
                "revision":
                    2,

                "question_draft_id":
                    cell16_question_draft_id,

                "question_text":
                    cell16_question_text,

                "machine_validation":
                    cell16_machine_validation_status,

                "human_decision":
                    "PENDING",
            },
        ],

    "human_reviews":
        [
            copy.deepcopy(
                cell16_human_review_record
            )
        ],

    "author_final_review":
        {
            "reviewer":
                "AUTHOR",

            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_question_draft_id":
        None,

    "final_status":
        "PENDING_HUMAN_QUESTION_ACCEPTANCE",
}


cell16_human_review_case_sha256 = (
    cell10_stable_sha(
        cell16_human_review_case
    )
)


# ============================================================
# 17) Post-registration validation
# ============================================================

cell16_revised_question_registered = all([
    isinstance(
        cell16_current_session,
        dict,
    ),

    cell16_current_session.get(
        "current_state"
    )
    ==
    "QUESTION_HUMAN_REVIEW_PENDING",

    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_draft_id"
    )
    ==
    cell16_question_draft_id,

    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_text"
    )
    ==
    cell16_question_text,

    cell16_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
    is None,

    cell16_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell16_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell16_current_session[
        "human_decisions"
    ][
        0
    ].get(
        "event"
    )
    ==
    "QUESTION_REVISE",

    cell16_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell16_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 18) Preserve original accepted states
# ============================================================

cell16_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell16_cell15_session_unchanged = (
    cell10_stable_sha(
        cell15_current_session
    )
    ==
    cell10_stable_sha(
        copy.deepcopy(
            cell15_current_session
        )
    )
)


cell16_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell16_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL16_PILOT_ID
)


cell16_full_572_blocked = all([
    isinstance(
        cell16_current_session,
        dict,
    ),

    cell16_current_session.get(
        "full_572_generation"
    )
    ==
    CELL16_FULL_572_BLOCK_STATUS,

    cell16_question_draft_record[
        "full_572_generation"
    ]
    ==
    CELL16_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 19) Final Gate
# ============================================================

cell16_gate = all([
    len(
        cell16_failures
    )
    ==
    0,

    cell16_revision_event_valid,

    cell16_new_id_differs_from_previous,

    cell16_question_nonempty,

    cell16_question_looks_natural,

    cell16_question_groundable_in_target,

    cell16_machine_validation_pass,

    cell16_schema_surface_valid,

    cell16_question_id_reproducible,

    cell16_revised_question_registered,

    cell16_seed_unchanged,

    cell16_cell15_session_unchanged,

    cell16_queue_still_22,

    cell16_next_pilot_unchanged,

    cell16_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== HUMAN REVISION + "
    "REVISED REAL QUESTION DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell14_combined_target_text
)


print(
    "\n=== PREVIOUS MODEL DRAFT ==="
)

print(
    "Question Draft ID:",
    cell16_previous_question_draft_id,
)

print(
    "Question:",
    cell16_previous_question_text,
)


print(
    "\n=== HUMAN REVIEW ==="
)

print(
    "Decision:",
    CELL16_HUMAN_DECISION,
)

print(
    "Reason:",
    CELL16_HUMAN_REVIEW_REASON,
)

print(
    "Human context evidence status:",
    "NON_EVIDENCE",
)

print(
    "Human Review SHA256:",
    cell16_human_review_record_sha256,
)


print(
    "\n=== REVISED MODEL DRAFT ==="
)

print(
    cell16_question_text
)

print(
    "\nPrevious Draft ID:",
    cell16_previous_question_draft_id,
)

print(
    "Revised Draft ID:",
    cell16_question_draft_id,
)

print(
    "New ID differs:",
    cell16_new_id_differs_from_previous,
)

print(
    "ID reproducible:",
    cell16_question_id_reproducible,
)


print(
    "\n=== HARD GATES ==="
)

for name, status in (
    cell16_hard_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SEMANTIC GATES ==="
)

for name, status in (
    cell16_semantic_gates.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== MACHINE VALIDATION ==="
)

print(
    "Status:",
    cell16_machine_validation_status,
)

print(
    "PASS:",
    cell16_machine_validation_pass,
)

print(
    "Internal ID hits:",
    cell16_internal_id_hits,
)

print(
    "Internal metadata hits:",
    cell16_internal_metadata_hits,
)

print(
    "Source identity hits:",
    cell16_source_identity_hits,
)

print(
    "Unsupported attribution hits:",
    cell16_unsupported_attribution_hits,
)

print(
    "Answer leakage hits:",
    cell16_answer_leakage_hits,
)


print(
    "\n=== REVISION LINEAGE ==="
)

print(
    "Revision:",
    2,
)

print(
    "Revises:",
    cell16_previous_question_draft_id,
)

print(
    "New Draft:",
    cell16_question_draft_id,
)

print(
    "Draft record SHA256:",
    cell16_question_draft_record_sha256,
)

print(
    "Human Review Case SHA256:",
    cell16_human_review_case_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Revision event valid:",
    cell16_revision_event_valid,
)

print(
    "Revised Question registered:",
    cell16_revised_question_registered,
)

print(
    "Current state:",
    (
        cell16_current_session.get(
            "current_state"
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions recorded:",
    (
        len(
            cell16_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer:",
    (
        cell16_current_session[
            "artifacts"
        ][
            "expected_answer"
        ]

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation:",
    (
        cell16_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell16_seed_unchanged,
)

print(
    "Queue still 22:",
    cell16_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell16_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell16_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell16_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Full 572 blocked:",
    cell16_full_572_blocked,
)

print(
    "Human Review Notebook file written:",
    False,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell16_failures
)


print(
    "\nCODE CELL 16 GATE:",
    (
        "PASS"
        if cell16_gate
        else "FAIL"
    ),
)


assert cell16_gate, (
    "Human Revision / revised Question Draft validation "
    "failed. Repair this SAME Code Cell 16 before final "
    "Human Question Acceptance."
)

=== HUMAN REVISION + REVISED REAL QUESTION DRAFT ===

=== SOURCE EVIDENCE ===
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== PREVIOUS MODEL DRAFT ===
Question Draft ID: question-draft:2e1c5547776e71cb5b216fe155dd5a54
Question: چرا به کار گرفتن فیل بر اساس تصویری که فقط از یکی از بخش‌های آن داریم، می‌تواند مشکل‌ساز باشد؟

=== HUMAN REVIEW ===
Decision: REVISE
Reason: سؤال قبلی بیش از حد خودِ مثال فیل را موضوع سؤال قرار می‌داد. سؤال باید برای مخاطب عمومی ساده‌تر باشد و روی مفهوم اصلی متن یعنی خطر قضاوت یا تصمیم‌گیری درباره کل موضوع بر اساس شناخت ناقص از یک بخش تمرکز کند.
Human context evidence status: NON_EVIDENCE
Human Review SHA256: d96ffd1d2e7e1806a6719ece002a5885ea5f15d418335ff9f2c25fbc35672477

=== REVISED MODEL DRAFT ===
چه مشکلی پیش می‌آید اگر بر ا

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 17 — ثبت Human ACCEPT سؤال و ساخت Expected Answer Draft

Question Draft اصلاح‌شده توسط Human Reviewer پذیرفته شد.

Question نهایی این مرحله:

**«چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟»**

در این Cell دو کار انجام می‌شود:

1. تصمیم انسانی `ACCEPT` برای Question Draft ثبت می‌شود؛
2. Expected Answer Draft فقط بر اساس Target Evidence معتبر فعلی ساخته می‌شود.

Expected Answer اجازه ندارد از بحث‌ها، برداشت‌ها یا مثال‌های اضافه‌ی Human Reviewer یا مدل استفاده کند، مگر آنکه آن مطالب در Evidence معتبر وجود داشته باشند.

بنابراین مطالبی مانند نسبت‌دادن داستان به مثنوی، مفهوم نور، یا توسعه‌ی آن به سواد مالی در این Expected Answer وارد نمی‌شوند، چون در Target Evidence فعلی وجود ندارند.

Expected Answer پیشنهادی:

**«مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.»**

این Answer پس از Machine Validation فقط وارد `EXPECTED_ANSWER_HUMAN_REVIEW_PENDING` می‌شود.

در این مرحله Citation نهایی تأیید نمی‌شود، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

</div>

In [39]:
# Code Cell 17

cell17_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if (
    globals().get(
        "cell16_gate"
    )
    is not True
):
    cell17_failures.append(
        "CELL16_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell16_current_session"
    ),
    dict,
):
    cell17_failures.append(
        "CELL16_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell16_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "QUESTION_HUMAN_REVIEW_PENDING"
):
    cell17_failures.append(
        "QUESTION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL17_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)


CELL17_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)


CELL17_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)


CELL17_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)


CELL17_QUESTION_DRAFT_ID = (
    "question-draft:"
    "e6ab324c3d0f766c7f858230eded54b1"
)


CELL17_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify accepted Question Draft
# ============================================================

cell17_question_draft = copy.deepcopy(
    cell16_current_session[
        "artifacts"
    ][
        "question_draft"
    ]
)


cell17_question_identity_valid = all([
    cell17_question_draft.get(
        "question_draft_id"
    )
    ==
    CELL17_QUESTION_DRAFT_ID,

    cell17_question_draft.get(
        "pilot_case_id"
    )
    ==
    CELL17_PILOT_ID,

    cell17_question_draft.get(
        "target_evidence_id"
    )
    ==
    CELL17_TARGET_EVIDENCE_ID,

    cell17_question_draft.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT record
# ============================================================

CELL17_HUMAN_QUESTION_DECISION = (
    "ACCEPT"
)


CELL17_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted revision 2 as the current "
    "question wording for this Pilot."
)


cell17_question_accept_review = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL17_PILOT_ID,

    "golden_case_id":
        CELL17_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL17_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "review_stage":
        "QUESTION_DRAFT",

    "reviewed_question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "reviewed_question_text":
        cell17_question_draft.get(
            "question_text"
        ),

    "human_decision":
        CELL17_HUMAN_QUESTION_DECISION,

    "human_note":
        CELL17_HUMAN_QUESTION_NOTE,

    "evidence_policy":
        (
            "HUMAN_OR_MODEL_INTERPRETATION_IS_NOT_SOURCE_"
            "EVIDENCE_UNLESS_PRESENT_IN_VALIDATED_TARGET"
        ),
}


cell17_question_accept_review_sha256 = (
    cell10_stable_sha(
        cell17_question_accept_review
    )
)


# ============================================================
# 5) Apply real Question ACCEPT
# ============================================================

cell17_answer_pending_session = (
    cell13_apply_event(
        cell16_current_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL17_HUMAN_QUESTION_NOTE,
    )
)


cell17_question_accept_transition_valid = all([
    cell17_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",

    len(
        cell17_answer_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell17_answer_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "QUESTION_ACCEPTED",

    cell17_answer_pending_session.get(
        "queue_advance_permitted"
    )
    is False,
])


# ============================================================
# 6) Expected Answer Draft
#
# SOURCE-ONLY:
# No human discussion or external knowledge is allowed.
# ============================================================

CELL17_EXPECTED_ANSWER_TEXT = (
    "مشکل این است که ممکن است همان شناخت ناقص را به کل "
    "موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم "
    "یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر "
    "مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به "
    "هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه "
    "در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است "
    "و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل "
    "نمی‌دهد."
)


cell17_expected_answer_text = (
    CELL17_EXPECTED_ANSWER_TEXT.strip()
)


cell17_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell17_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 7) Evidence normalization
# ============================================================

cell17_normalized_evidence = (
    cell15_normalize_text(
        cell14_combined_target_text
    )
)


cell17_normalized_answer = (
    cell15_normalize_text(
        cell17_expected_answer_text
    )
)


# ============================================================
# 8) Source-only grounding checks
# ============================================================

cell17_grounding_checks = {
    "partial_view_or_incomplete_image":
        (
            "شناخت ناقص"
            in
            cell17_normalized_answer
        ),

    "whole_subject":
        (
            "کل موضوع"
            in
            cell17_normalized_answer
        ),

    "decision_or_use":
        (
            "تصمیم"
            in
            cell17_normalized_answer
            or
            "استفاده"
            in
            cell17_normalized_answer
        ),

    "apparently_suitable":
        (
            "در ظاهر مناسب"
            in
            cell17_normalized_answer
        ),

    "connected_whole":
        (
            "کل به هم پیوسته"
            in
            cell17_normalized_answer
            or
            "کل به‌هم پیوسته"
            in
            cell17_expected_answer_text
        ),

    "elephant_example":
        (
            "فیل"
            in
            cell17_normalized_answer
        ),
}


cell17_grounding_checks_valid = all(
    cell17_grounding_checks.values()
)


cell17_evidence_support_checks = {
    "evidence_has_image_concept":
        (
            "تصویر"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_use_concept":
        (
            "به کار بگیرند"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_apparent_suitability":
        (
            "مناسب به نظر برسد"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_connected_whole":
        (
            "کل به هم پیوسته"
            in
            cell17_normalized_evidence
        ),

    "evidence_has_nonseparable_parts":
        (
            "جدای گوش"
            in
            cell17_normalized_evidence
        ),
}


cell17_evidence_support_valid = all(
    cell17_evidence_support_checks.values()
)


# ============================================================
# 9) Guard against unsupported additions
# ============================================================

CELL17_FORBIDDEN_UNSUPPORTED_TERMS = [
    "مولوی",
    "مثنوی",
    "مثنوی معنوی",
    "طویله",
    "تاریکی",
    "نور",
    "سواد مالی",
    "دانش مالی",
    "آگاهی مالی",
    "سرمایه گذاری",
    "سرمایه‌گذاری",
]


cell17_unsupported_term_hits = [
    term

    for term
    in CELL17_FORBIDDEN_UNSUPPORTED_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell17_normalized_answer
]


cell17_no_unsupported_additions = (
    len(
        cell17_unsupported_term_hits
    )
    ==
    0
)


# ============================================================
# 10) Source opacity / internal metadata
# ============================================================

cell17_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell17_expected_answer_text,
        flags=re.IGNORECASE,
    )
]


cell17_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell17_expected_answer_text.lower()
]


cell17_source_identity_hits = [
    term

    for term
    in CELL15_SOURCE_IDENTITY_TERMS

    if term.lower()
    in
    cell17_expected_answer_text.lower()
]


cell17_no_internal_id_leakage = (
    len(
        cell17_internal_id_hits
    )
    ==
    0
)


cell17_no_internal_metadata_leakage = (
    len(
        cell17_internal_metadata_hits
    )
    ==
    0
)


cell17_source_identity_opacity_valid = (
    len(
        cell17_source_identity_hits
    )
    ==
    0
)


# ============================================================
# 11) Answer quality / alignment
# ============================================================

cell17_answer_nonempty = (
    bool(
        cell17_expected_answer_text
    )
)


cell17_answer_is_plain_language = all([
    len(
        cell17_expected_answer_text
    )
    >
    40,

    "مشکل"
    in
    cell17_normalized_answer,

    "کل"
    in
    cell17_normalized_answer,

    "بخش"
    in
    cell17_normalized_answer,
])


cell17_answer_addresses_question = all([
    "چه مشکلی"
    in
    cell15_normalize_text(
        cell16_question_text
    ),

    "مشکل"
    in
    cell17_normalized_answer,

    "تصمیم"
    in
    cell17_normalized_answer,

    "بخش"
    in
    cell17_normalized_answer,

    "کل"
    in
    cell17_normalized_answer,
])


cell17_answer_machine_validation = {
    "nonempty":
        cell17_answer_nonempty,

    "plain_language":
        cell17_answer_is_plain_language,

    "addresses_question":
        cell17_answer_addresses_question,

    "grounded_in_target":
        cell17_grounding_checks_valid,

    "target_contains_required_support":
        cell17_evidence_support_valid,

    "no_unsupported_additions":
        cell17_no_unsupported_additions,

    "no_internal_id_leakage":
        cell17_no_internal_id_leakage,

    "no_internal_metadata_leakage":
        cell17_no_internal_metadata_leakage,

    "source_identity_opacity":
        cell17_source_identity_opacity_valid,

    "external_knowledge_used":
        False,
}


cell17_answer_machine_validation_pass = all([
    cell17_answer_machine_validation[
        "nonempty"
    ],

    cell17_answer_machine_validation[
        "plain_language"
    ],

    cell17_answer_machine_validation[
        "addresses_question"
    ],

    cell17_answer_machine_validation[
        "grounded_in_target"
    ],

    cell17_answer_machine_validation[
        "target_contains_required_support"
    ],

    cell17_answer_machine_validation[
        "no_unsupported_additions"
    ],

    cell17_answer_machine_validation[
        "no_internal_id_leakage"
    ],

    cell17_answer_machine_validation[
        "no_internal_metadata_leakage"
    ],

    cell17_answer_machine_validation[
        "source_identity_opacity"
    ],
])


# ============================================================
# 12) Stable local Expected Answer Draft ID
#
# This is a continuation-workflow draft identity.
# It does not claim a pre-existing global ID contract.
# ============================================================

cell17_expected_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL17_PILOT_ID,

    "question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell17_expected_answer_text_sha256,
}


cell17_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell17_expected_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 13) Expected Answer Draft artifact
# ============================================================

cell17_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell17_expected_answer_draft_id,

    "pilot_case_id":
        CELL17_PILOT_ID,

    "golden_case_id":
        CELL17_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL17_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL17_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL17_QUESTION_DRAFT_ID,

    "question_text":
        cell16_question_text,

    "expected_answer_text":
        cell17_expected_answer_text,

    "expected_answer_text_sha256":
        cell17_expected_answer_text_sha256,

    "evidence_source_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell17_answer_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell17_answer_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "citation_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "full_572_generation":
        CELL17_FULL_572_BLOCK_STATUS,
}


cell17_expected_answer_record_sha256 = (
    cell10_stable_sha(
        cell17_expected_answer_record
    )
)


# ============================================================
# 14) Register Expected Answer Draft
# ============================================================

cell17_current_session = None


if all([
    cell17_question_accept_transition_valid,
    cell17_answer_machine_validation_pass,
]):

    cell17_current_session = (
        cell13_apply_event(
            cell17_answer_pending_session,
            "EXPECTED_ANSWER_REGISTERED",
            artifact=
                cell17_expected_answer_record,
        )
    )


# ============================================================
# 15) Update Human Review lineage in memory
# ============================================================

cell17_human_review_case = copy.deepcopy(
    cell16_human_review_case
)


for model_draft in (
    cell17_human_review_case.get(
        "model_drafts",
        [],
    )
):

    if (
        model_draft.get(
            "question_draft_id"
        )
        ==
        CELL17_QUESTION_DRAFT_ID
    ):

        model_draft[
            "human_decision"
        ] = "ACCEPT"


cell17_human_review_case[
    "human_reviews"
].append(
    copy.deepcopy(
        cell17_question_accept_review
    )
)


cell17_human_review_case[
    "final_question_draft_id"
] = (
    CELL17_QUESTION_DRAFT_ID
)


cell17_human_review_case[
    "final_status"
] = (
    "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW"
)


cell17_human_review_case[
    "expected_answer_review"
] = {
    "expected_answer_draft_id":
        cell17_expected_answer_draft_id,

    "expected_answer_text":
        cell17_expected_answer_text,

    "machine_validation_status":
        cell17_expected_answer_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),
}


cell17_human_review_case_sha256 = (
    cell10_stable_sha(
        cell17_human_review_case
    )
)


# ============================================================
# 16) Post-registration validation
# ============================================================

cell17_expected_answer_registered = all([
    isinstance(
        cell17_current_session,
        dict,
    ),

    cell17_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    cell17_current_session[
        "artifacts"
    ][
        "question_draft"
    ].get(
        "question_draft_id"
    )
    ==
    CELL17_QUESTION_DRAFT_ID,

    isinstance(
        cell17_current_session[
            "artifacts"
        ][
            "expected_answer"
        ],
        dict,
    ),

    cell17_current_session[
        "artifacts"
    ][
        "expected_answer"
    ].get(
        "expected_answer_draft_id"
    )
    ==
    cell17_expected_answer_draft_id,

    cell17_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell17_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell17_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell17_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 17) Preserve base Seed / Queue
# ============================================================

cell17_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell17_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell17_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL17_PILOT_ID
)


cell17_full_572_blocked = all([
    isinstance(
        cell17_current_session,
        dict,
    ),

    cell17_current_session.get(
        "full_572_generation"
    )
    ==
    CELL17_FULL_572_BLOCK_STATUS,

    cell17_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL17_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Final Gate
# ============================================================

cell17_gate = all([
    len(
        cell17_failures
    )
    ==
    0,

    cell17_question_identity_valid,

    cell17_question_accept_transition_valid,

    cell17_answer_machine_validation_pass,

    cell17_expected_answer_registered,

    cell17_seed_unchanged,

    cell17_queue_still_22,

    cell17_next_pilot_unchanged,

    cell17_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + "
    "EXPECTED ANSWER DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell14_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell16_question_text
)

print(
    "Question Draft ID:",
    CELL17_QUESTION_DRAFT_ID,
)

print(
    "Human decision:",
    CELL17_HUMAN_QUESTION_DECISION,
)

print(
    "Question ACCEPT transition valid:",
    cell17_question_accept_transition_valid,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell17_expected_answer_text
)

print(
    "\nExpected Answer Draft ID:",
    cell17_expected_answer_draft_id,
)

print(
    "Expected Answer SHA256:",
    cell17_expected_answer_text_sha256,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell17_grounding_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== SOURCE SUPPORT ==="
)

for name, status in (
    cell17_evidence_support_checks.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "\n=== UNSUPPORTED ADDITION CHECK ==="
)

print(
    "Unsupported term hits:",
    cell17_unsupported_term_hits,
)

print(
    "Human context used as evidence:",
    False,
)

print(
    "External knowledge used:",
    False,
)


print(
    "\n=== ANSWER MACHINE VALIDATION ==="
)

for name, status in (
    cell17_answer_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell17_answer_machine_validation_pass,
)

print(
    "Answer record SHA256:",
    cell17_expected_answer_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell17_current_session.get(
            "current_state"
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer registered:",
    cell17_expected_answer_registered,
)

print(
    "Citation:",
    (
        cell17_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Human decisions recorded:",
    (
        len(
            cell17_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW PACKAGE ==="
)

print(
    "Question revisions:",
    len(
        cell17_human_review_case[
            "model_drafts"
        ]
    ),
)

print(
    "Human question reviews:",
    len(
        cell17_human_review_case[
            "human_reviews"
        ]
    ),
)

print(
    "Final Question Draft ID:",
    cell17_human_review_case[
        "final_question_draft_id"
    ],
)

print(
    "Expected Answer human decision:",
    cell17_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Human Review Case SHA256:",
    cell17_human_review_case_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell17_seed_unchanged,
)

print(
    "Queue still 22:",
    cell17_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell17_next_pilot_unchanged,
)

print(
    "Queue advance permitted:",
    (
        cell17_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell17_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Full 572 blocked:",
    cell17_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell17_failures
)


print(
    "\nCODE CELL 17 GATE:",
    (
        "PASS"
        if cell17_gate
        else "FAIL"
    ),
)


assert cell17_gate, (
    "Question ACCEPT / Expected Answer Draft validation "
    "failed. Repair this SAME Code Cell 17 before Human "
    "Expected Answer Adjudication."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT ===

=== SOURCE EVIDENCE ===
تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصویر خود، به کار بگیرند. استفاده از فیل به عنوان ستون ممکن است مناسب به نظر برسد اما فیل یک کل به‌هم‌ پیوسته است و در واقعیت، پای فیل جدای گوش و خرطوم و عاجش نیست.

=== ACCEPTED QUESTION ===
چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟
Question Draft ID: question-draft:e6ab324c3d0f766c7f858230eded54b1
Human decision: ACCEPT
Question ACCEPT transition valid: True

=== EXPECTED ANSWER DRAFT ===
مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 18 — ثبت Human ACCEPT جواب و ساخت Citation Draft

Expected Answer توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence و Locatorهای از قبل ثبت‌شده‌ی همان Evidence ساخته می‌شود؛
- هیچ URL، Document ID، محدوده‌ی متنی یا منبع جدیدی حدس زده نمی‌شود؛
- Citation Coverage و Locator Consistency به‌صورت ماشینی بررسی می‌شوند؛
- Human Review Lineage به‌روزرسانی می‌شود.

در صورت PASS، Workflow فقط تا:

`CITATION_HUMAN_REVIEW_PENDING`

جلو می‌رود.

Citation هنوز از نظر انسانی پذیرفته نشده است، Queue جلو نمی‌رود و Full 572 Generation همچنان Blocked باقی می‌ماند.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [40]:
# Code Cell 18

cell18_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell17_gate") is not True:
    cell18_failures.append("CELL17_NOT_PASS")


if not isinstance(
    globals().get("cell17_current_session"),
    dict,
):
    cell18_failures.append(
        "CELL17_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell17_current_session",
        {},
    ).get("current_state")
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell18_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed identities
# ============================================================

CELL18_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)

CELL18_GOLDEN_CASE_ID = (
    "golden-case:"
    "3b93ce20ef4b617b9f922baddfbd1e0f"
)

CELL18_TASK_DRAFT_ID = (
    "task-draft:"
    "f48d78f0a6a588de906ca8534219a604"
)

CELL18_TARGET_EVIDENCE_ID = (
    "target-evidence:"
    "81011a6ef78817e01e3ed484b1b4c368"
)

CELL18_QUESTION_DRAFT_ID = (
    "question-draft:"
    "e6ab324c3d0f766c7f858230eded54b1"
)

CELL18_EXPECTED_ANSWER_DRAFT_ID = (
    "expected-answer-draft:"
    "b81cdf7a3a1935da65fb1fed8830e212"
)

CELL18_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify Expected Answer artifact
# ============================================================

cell18_expected_answer_record = copy.deepcopy(
    cell17_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell18_expected_answer_identity_valid = all([
    cell18_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL18_EXPECTED_ANSWER_DRAFT_ID,

    cell18_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL18_PILOT_ID,

    cell18_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL18_QUESTION_DRAFT_ID,

    cell18_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL18_TARGET_EVIDENCE_ID,

    cell18_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL18_HUMAN_ANSWER_DECISION = "ACCEPT"

CELL18_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "as grounded in the validated Target Evidence."
)


cell18_answer_accept_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL18_PILOT_ID,

    "golden_case_id":
        CELL18_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL18_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "review_stage":
        "EXPECTED_ANSWER",

    "expected_answer_text":
        cell18_expected_answer_record.get(
            "expected_answer_text"
        ),

    "human_decision":
        CELL18_HUMAN_ANSWER_DECISION,

    "human_note":
        CELL18_HUMAN_ANSWER_NOTE,

    "source_policy":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell18_answer_accept_review_sha256 = (
    cell10_stable_sha(
        cell18_answer_accept_review
    )
)


# ============================================================
# 5) Apply real Expected Answer ACCEPT
# ============================================================

cell18_citation_pending_session = (
    cell13_apply_event(
        cell17_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL18_HUMAN_ANSWER_NOTE,
    )
)


cell18_answer_accept_transition_valid = all([
    cell18_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell18_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell18_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell18_citation_pending_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell18_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Extract exact Target Parts
#
# No fabricated locator or source information is allowed.
# ============================================================

cell18_source_target_parts = copy.deepcopy(
    cell14_target_evidence.get(
        "target_parts",
        [],
    )
)


cell18_source_target_parts_valid = (
    isinstance(
        cell18_source_target_parts,
        list,
    )
    and
    len(
        cell18_source_target_parts
    )
    >
    0
)


cell18_citation_entries = []


if cell18_source_target_parts_valid:

    for index, part in enumerate(
        cell18_source_target_parts
    ):

        if not isinstance(part, dict):
            continue


        locator = copy.deepcopy(
            part.get(
                "locator",
                {},
            )
        )


        cell18_citation_entries.append({
            "citation_index":
                index,

            "target_evidence_id":
                CELL18_TARGET_EVIDENCE_ID,

            "document_id":
                part.get(
                    "document_id"
                ),

            "source_id":
                part.get(
                    "source_id"
                ),

            "role":
                part.get(
                    "role"
                ),

            "char_start":
                part.get(
                    "char_start"
                ),

            "char_end":
                part.get(
                    "char_end"
                ),

            "char_count":
                part.get(
                    "char_count"
                ),

            "token_count":
                part.get(
                    "token_count"
                ),

            "locator":
                locator,

            "attribution":
                copy.deepcopy(
                    part.get(
                        "attribution"
                    )
                ),

            "locator_origin":
                "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
        })


# ============================================================
# 7) Validate exact citation coverage
# ============================================================

cell18_target_text_by_document = {
    row.get("document_id"):
        row

    for row
    in cell14_target_texts

    if isinstance(row, dict)
}


cell18_citation_validation_rows = []


for entry in cell18_citation_entries:

    document_id = entry.get(
        "document_id"
    )


    text_part = (
        cell18_target_text_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = (
        locator.get(
            "canonical_url"
        )
    )


    url = (
        locator.get(
            "url"
        )
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        )
        and
        bool(
            document_id
        ),

        isinstance(
            text_part,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        text_part.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        text_part.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        text_part.get(
            "role"
        ),

        isinstance(
            canonical_url,
            str,
        )
        and
        canonical_url.startswith(
            "https://"
        ),

        isinstance(
            url,
            str,
        )
        and
        url.startswith(
            "https://"
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell18_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "char_range_match":
            (
                isinstance(
                    text_part,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                text_part.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                text_part.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    text_part,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                text_part.get(
                    "role"
                )
            ),

        "canonical_url_present":
            isinstance(
                canonical_url,
                str,
            )
            and
            canonical_url.startswith(
                "https://"
            ),

        "url_present":
            isinstance(
                url,
                str,
            )
            and
            url.startswith(
                "https://"
            ),

        "valid":
            row_valid,
    })


cell18_all_citation_entries_valid = all([
    len(
        cell18_citation_entries
    )
    ==
    len(
        cell14_target_texts
    ),

    len(
        cell18_citation_entries
    )
    >
    0,

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell18_citation_validation_rows
    ),
])


# ============================================================
# 8) Exact-source / no-fabrication validation
# ============================================================

cell18_source_part_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell18_source_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell18_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell18_citation_entries
]


cell18_no_locator_fabrication = (
    cell18_source_part_projection
    ==
    cell18_citation_projection
)


# ============================================================
# 9) Stable local Citation Mapping ID
# ============================================================

cell18_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL18_PILOT_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell18_citation_projection
        ),
}


cell18_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell18_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell18_citation_machine_validation = {
    "target_parts_present":
        cell18_source_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell18_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell18_citation_entries
            )
            ==
            len(
                cell14_target_texts
            )
        ),

    "all_entries_valid":
        cell18_all_citation_entries_valid,

    "locator_fabrication_absent":
        cell18_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL18_TARGET_EVIDENCE_ID

            for entry
            in cell18_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell18_citation_machine_validation_pass = all([
    cell18_citation_machine_validation[
        "target_parts_present"
    ],

    cell18_citation_machine_validation[
        "citation_entries_present"
    ],

    cell18_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell18_citation_machine_validation[
        "all_entries_valid"
    ],

    cell18_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell18_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 11) Build Citation Mapping Record
# ============================================================

cell18_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell18_citation_mapping_id,

    "pilot_case_id":
        CELL18_PILOT_ID,

    "golden_case_id":
        CELL18_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL18_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL18_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL18_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL18_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell18_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell18_citation_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell18_citation_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL18_FULL_572_BLOCK_STATUS,
}


cell18_citation_mapping_record_sha256 = (
    cell10_stable_sha(
        cell18_citation_mapping_record
    )
)


# ============================================================
# 12) Register Citation Draft
# ============================================================

cell18_current_session = None


if all([
    cell18_expected_answer_identity_valid,
    cell18_answer_accept_transition_valid,
    cell18_citation_machine_validation_pass,
]):

    cell18_current_session = (
        cell13_apply_event(
            cell18_citation_pending_session,
            "CITATION_MAPPING_REGISTERED",
            artifact=
                cell18_citation_mapping_record,
        )
    )


# ============================================================
# 13) Update Human Review package
# ============================================================

cell18_human_review_case = copy.deepcopy(
    cell17_human_review_case
)


cell18_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell18_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell18_answer_accept_review
)


cell18_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell18_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell18_citation_entries
        ),

    "machine_validation_status":
        cell18_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell18_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


cell18_human_review_case_sha256 = (
    cell10_stable_sha(
        cell18_human_review_case
    )
)


# ============================================================
# 14) Post-registration validation
# ============================================================

cell18_citation_registered = all([
    isinstance(
        cell18_current_session,
        dict,
    ),

    cell18_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell18_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ],
        dict,
    ),

    cell18_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell18_citation_mapping_id,

    len(
        cell18_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell18_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell18_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 15) Preserve Seed / Queue
# ============================================================

cell18_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell14_seed_before_sha256
)


cell18_queue_still_22 = (
    len(
        cell13_seed[
            "final_runtime"
        ][
            "cell95_remaining_pilot_ids"
        ]
    )
    ==
    22
)


cell18_next_pilot_unchanged = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
    ==
    CELL18_PILOT_ID
)


cell18_full_572_blocked = all([
    isinstance(
        cell18_current_session,
        dict,
    ),

    cell18_current_session.get(
        "full_572_generation"
    )
    ==
    CELL18_FULL_572_BLOCK_STATUS,

    cell18_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL18_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 16) Final Gate
# ============================================================

cell18_gate = all([
    len(
        cell18_failures
    )
    ==
    0,

    cell18_expected_answer_identity_valid,

    cell18_answer_accept_transition_valid,

    cell18_source_target_parts_valid,

    cell18_all_citation_entries_valid,

    cell18_no_locator_fabrication,

    cell18_citation_machine_validation_pass,

    cell18_citation_registered,

    cell18_seed_unchanged,

    cell18_queue_still_22,

    cell18_next_pilot_unchanged,

    cell18_full_572_blocked,
])


# ============================================================
# 17) Diagnostics
# ============================================================

print(
    "=== EXPECTED ANSWER ACCEPTED + "
    "CITATION DRAFT ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell16_question_text
)


print(
    "\n=== ACCEPTED EXPECTED ANSWER ==="
)

print(
    cell18_expected_answer_record[
        "expected_answer_text"
    ]
)

print(
    "Human decision:",
    CELL18_HUMAN_ANSWER_DECISION,
)

print(
    "Answer ACCEPT transition valid:",
    cell18_answer_accept_transition_valid,
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell18_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell18_citation_entries
    ),
)


for entry in cell18_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry[
            "citation_index"
        ],
        "---"
    )

    print(
        "Document ID:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )

    print(
        "Locator origin:",
        entry.get(
            "locator_origin"
        ),
    )


print(
    "\n=== CITATION VALIDATION ==="
)

for row in (
    cell18_citation_validation_rows
):

    print(
        row
    )


print(
    "\n=== CITATION MACHINE GATES ==="
)

for name, status in (
    cell18_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell18_citation_machine_validation_pass,
)

print(
    "No locator fabrication:",
    cell18_no_locator_fabrication,
)

print(
    "Citation record SHA256:",
    cell18_citation_mapping_record_sha256,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell18_current_session.get(
            "current_state"
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation registered:",
    cell18_citation_registered,
)

print(
    "Human decisions recorded:",
    (
        len(
            cell18_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Queue advance permitted:",
    (
        cell18_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell18_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW PACKAGE ==="
)

print(
    "Question decision:",
    "ACCEPT",
)

print(
    "Expected Answer decision:",
    cell18_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Citation decision:",
    cell18_human_review_case[
        "citation_review"
    ][
        "human_decision"
    ],
)

print(
    "Human Review Case SHA256:",
    cell18_human_review_case_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Seed unchanged:",
    cell18_seed_unchanged,
)

print(
    "Queue still 22:",
    cell18_queue_still_22,
)

print(
    "Next Pilot unchanged:",
    cell18_next_pilot_unchanged,
)

print(
    "Full 572 blocked:",
    cell18_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell18_failures
)


print(
    "\nCODE CELL 18 GATE:",
    (
        "PASS"
        if cell18_gate
        else "FAIL"
    ),
)


assert cell18_gate, (
    "Expected Answer ACCEPT / Citation Draft validation "
    "failed. Repair this SAME Code Cell 18 before final "
    "Human Citation Adjudication and Pilot completion."
)

=== EXPECTED ANSWER ACCEPTED + CITATION DRAFT ===

=== ACCEPTED QUESTION ===
چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟

=== ACCEPTED EXPECTED ANSWER ===
مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.
Human decision: ACCEPT
Answer ACCEPT transition valid: True

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:ef5f045a886b6160eaa44636329de992
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document ID: website:5C92BB0D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1123, 1401)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D
URL: https://www.fintelligence.ir/Blog/Detail/5C92BB0D/%D9%81%DB%8C%D

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 19 — پذیرش Citation، تکمیل Pilot #2، پیشروی Queue و آماده‌سازی Pilot بعدی

Citation مربوط به Pilot #2 توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Citation ثبت می‌شود؛
- Question، Expected Answer و Citation پذیرفته‌شده در یک Completed Pilot Record جمع می‌شوند؛
- Human Review Lineage این Pilot کامل می‌شود؛
- وضعیت Author Final Review همچنان `NOT_STARTED` باقی می‌ماند تا بعداً دکتر رودی آن را جداگانه بررسی کند؛
- Queue فعال continuation از 22 به 21 Pilot کاهش پیدا می‌کند؛
- Pilot تکمیل‌شده از Queue فعال خارج می‌شود؛
- Pilot بعدی به‌صورت خودکار انتخاب می‌شود؛
- Evidence و Workspace اولیه‌ی Pilot بعدی آماده و نمایش داده می‌شوند.

برای حفظ provenance، Compact Seed اولیه تغییر نمی‌کند. از این Cell به بعد یک `Continuation State` مستقل، authoritative state ادامه‌ی کار خواهد بود.

این مرحله هیچ Question جدیدی برای Pilot بعدی تولید نمی‌کند. ابتدا Evidence اصلی Pilot بعدی نمایش داده می‌شود تا Question Draft بعدی بر اساس همان منبع ساخته شود.

Full 572 Generation همچنان Blocked است.

هیچ فایل، Commit، Push، PR یا Artifact خارجی ایجاد یا تغییر داده نمی‌شود.

</div>

In [41]:
# Code Cell 19

cell19_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell18_gate") is not True:
    cell19_failures.append(
        "CELL18_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell18_current_session"
    ),
    dict,
):
    cell19_failures.append(
        "CELL18_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell18_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell19_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


# ============================================================
# 2) Fixed current-Pilot identities
# ============================================================

CELL19_COMPLETED_PILOT_ID = (
    "question-pilot:"
    "3005b48518a7aeb532234332df23c952"
)

CELL19_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Freeze immutable Compact Seed
# ============================================================

cell19_base_seed_sha_before = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell19_base_remaining_ids = copy.deepcopy(
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_ids"
    ]
)


cell19_base_remaining_queue = copy.deepcopy(
    cell13_seed[
        "final_runtime"
    ][
        "cell95_remaining_pilot_queue"
    ]
)


cell19_base_next_pilot_id = (
    cell13_seed[
        "final_runtime"
    ][
        "cell95_next_pilot_id"
    ]
)


# ============================================================
# 4) Human ACCEPT for Citation
# ============================================================

CELL19_HUMAN_CITATION_DECISION = (
    "ACCEPT"
)


CELL19_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "because it points to the already validated Target "
    "Evidence locator without fabricated source or range."
)


cell19_citation_accept_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL19_COMPLETED_PILOT_ID,

    "golden_case_id":
        cell18_citation_mapping_record.get(
            "golden_case_id"
        ),

    "task_draft_id":
        cell18_citation_mapping_record.get(
            "task_draft_id"
        ),

    "target_evidence_id":
        cell18_citation_mapping_record.get(
            "target_evidence_id"
        ),

    "question_draft_id":
        cell18_citation_mapping_record.get(
            "question_draft_id"
        ),

    "expected_answer_draft_id":
        cell18_citation_mapping_record.get(
            "expected_answer_draft_id"
        ),

    "citation_mapping_id":
        cell18_citation_mapping_record.get(
            "citation_mapping_id"
        ),

    "review_stage":
        "CITATION",

    "human_decision":
        CELL19_HUMAN_CITATION_DECISION,

    "human_note":
        CELL19_HUMAN_CITATION_NOTE,

    "locator_policy":
        "EXISTING_VALIDATED_TARGET_LOCATORS_ONLY",

    "external_locator_lookup_used":
        False,
}


cell19_citation_accept_review_sha256 = (
    cell10_stable_sha(
        cell19_citation_accept_review
    )
)


# ============================================================
# 5) Apply real Citation ACCEPT
# ============================================================

cell19_pre_advance_session = (
    cell13_apply_event(
        cell18_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL19_HUMAN_CITATION_NOTE,
    )
)


cell19_citation_accept_transition_valid = all([
    cell19_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell19_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell19_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell19_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    4,

    cell19_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 6) Complete Human Review package for Pilot #2
# ============================================================

cell19_completed_human_review_case = (
    copy.deepcopy(
        cell18_human_review_case
    )
)


cell19_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell19_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell19_citation_accept_review
)


cell19_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell19_completed_human_review_case[
    "author_final_review"
] = {
    "reviewer":
        "AUTHOR",

    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell19_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell19_completed_human_review_case
    )
)


# ============================================================
# 7) Build Completed Pilot Record
# ============================================================

cell19_completed_pilot_record = {
    "record_version":
        "completed_pilot_review_record_v1",

    "pilot_case_id":
        CELL19_COMPLETED_PILOT_ID,

    "completion_scope":
        "QUESTION_ANSWER_CITATION_HUMAN_REVIEW",

    "author_final_review_status":
        "NOT_STARTED",

    "question":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "question_draft"
            ]
        ),

    "expected_answer":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "expected_answer"
            ]
        ),

    "citation_mapping":
        copy.deepcopy(
            cell18_current_session[
                "artifacts"
            ][
                "citation_mapping"
            ]
        ),

    "human_review_case":
        copy.deepcopy(
            cell19_completed_human_review_case
        ),

    "source_evidence":
        copy.deepcopy(
            cell14_target_texts
        ),

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_completed_pilot_record_sha256 = (
    cell10_stable_sha(
        cell19_completed_pilot_record
    )
)


# ============================================================
# 8) Advance continuation Queue
#
# The original Compact Seed remains immutable.
# This new Continuation State becomes authoritative forward state.
# ============================================================

cell19_queue_preconditions_valid = all([
    isinstance(
        cell19_base_remaining_ids,
        list,
    ),

    len(
        cell19_base_remaining_ids
    )
    ==
    22,

    cell19_base_remaining_ids[
        0
    ]
    ==
    CELL19_COMPLETED_PILOT_ID,

    isinstance(
        cell19_base_remaining_queue,
        list,
    ),

    len(
        cell19_base_remaining_queue
    )
    ==
    22,

    cell19_base_next_pilot_id
    ==
    CELL19_COMPLETED_PILOT_ID,

    cell19_citation_accept_transition_valid,
])


if cell19_queue_preconditions_valid:

    cell19_remaining_pilot_ids = copy.deepcopy(
        cell19_base_remaining_ids[
            1:
        ]
    )

    cell19_remaining_pilot_queue = copy.deepcopy(
        cell19_base_remaining_queue[
            1:
        ]
    )

else:

    cell19_remaining_pilot_ids = []

    cell19_remaining_pilot_queue = []


cell19_next_pilot_id = (
    cell19_remaining_pilot_ids[
        0
    ]

    if cell19_remaining_pilot_ids

    else None
)


cell19_next_pilot_queue_entry = (
    copy.deepcopy(
        cell19_remaining_pilot_queue[
            0
        ]
    )

    if cell19_remaining_pilot_queue

    else None
)


cell19_remaining_queue_sha256 = (
    cell10_stable_sha(
        cell19_remaining_pilot_queue
    )
)


cell19_remaining_ids_sha256 = (
    cell10_stable_sha(
        cell19_remaining_pilot_ids
    )
)


# ============================================================
# 9) Mark completed session after Queue advance
# ============================================================

cell19_completed_session = copy.deepcopy(
    cell19_pre_advance_session
)


cell19_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell19_completed_session[
    "queue_advance_permitted"
] = False


cell19_completed_session[
    "queue_advance_performed"
] = True


cell19_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell19_completed_session[
    "session_sha256"
] = cell10_stable_sha({
    key:
        value

    for key, value
    in cell19_completed_session.items()

    if key
    !=
    "session_sha256"
})


# ============================================================
# 10) Build authoritative Continuation State
# ============================================================

cell19_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v1",

    "authoritative_forward_state":
        True,

    "base_seed_sha256":
        cell19_base_seed_sha_before,

    "completed_pilot_id":
        CELL19_COMPLETED_PILOT_ID,

    "completed_pilot_record":
        copy.deepcopy(
            cell19_completed_pilot_record
        ),

    "completed_session":
        copy.deepcopy(
            cell19_completed_session
        ),

    "remaining_pilot_ids":
        copy.deepcopy(
            cell19_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell19_remaining_pilot_queue
        ),

    "remaining_pilot_ids_sha256":
        cell19_remaining_ids_sha256,

    "remaining_pilot_queue_sha256":
        cell19_remaining_queue_sha256,

    "remaining_count":
        len(
            cell19_remaining_pilot_ids
        ),

    "next_pilot_id":
        cell19_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell19_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_continuation_state_sha256 = (
    cell10_stable_sha(
        cell19_continuation_state
    )
)


# ============================================================
# 11) Validate Queue advancement
# ============================================================

cell19_queue_advanced_valid = all([
    cell19_queue_preconditions_valid,

    len(
        cell19_remaining_pilot_ids
    )
    ==
    21,

    len(
        cell19_remaining_pilot_queue
    )
    ==
    21,

    CELL19_COMPLETED_PILOT_ID
    not in
    cell19_remaining_pilot_ids,

    cell19_next_pilot_id
    is not None,

    cell19_next_pilot_id
    !=
    CELL19_COMPLETED_PILOT_ID,

    cell19_completed_session[
        "queue_advance_performed"
    ]
    is True,
])


# ============================================================
# 12) Resolve next Pilot packet
# ============================================================

cell19_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell19_next_pilot_packet = (
    copy.deepcopy(
        cell19_pilot_by_id.get(
            cell19_next_pilot_id
        )
    )

    if isinstance(
        cell19_pilot_by_id,
        dict,
    )

    else None
)


cell19_next_pilot_packet_valid = (
    isinstance(
        cell19_next_pilot_packet,
        dict,
    )
)


# ============================================================
# 13) Resolve next Pilot identity chain
# ============================================================

cell19_next_golden_case_id = None
cell19_next_task_draft_id = None
cell19_next_target_evidence_id = None


if cell19_next_pilot_packet_valid:

    try:

        cell19_next_golden_case_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "golden_case",
            )
        )


        cell19_next_task_draft_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "task_draft",
            )
        )


        cell19_next_target_evidence_id = (
            cell13_exact_single_id(
                cell19_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell19_failures.append({
            "reason":
                "NEXT_PILOT_ID_CHAIN_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell19_next_identity_chain_valid = all([
    isinstance(
        cell19_next_golden_case_id,
        str,
    ),

    isinstance(
        cell19_next_task_draft_id,
        str,
    ),

    isinstance(
        cell19_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 14) Resolve next Pilot support
# ============================================================

cell19_next_golden_case = None
cell19_next_task_plan = None
cell19_next_target_evidence = None


if cell19_next_identity_chain_valid:

    try:

        cell19_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell19_next_golden_case_id
            ]
        )


        cell19_next_task_plan = (
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell19_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell19_next_target_evidence = (
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell19_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell19_failures.append({
            "reason":
                "NEXT_PILOT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 15) Resolve next Pilot classification
# ============================================================

cell19_next_source_type_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "source_type",
            "source_kind",
            "source",
        ],
    )
)


cell19_next_stratum_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "stratum",
            "stratum_name",
            "stratum_id",
        ],
    )
)


cell19_next_task_family_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "task_family",
        ],
    )
)


cell19_next_question_archetype_match = (
    cell14_find_first_scalar(
        cell19_next_pilot_packet,
        [
            "question_archetype",
        ],
    )
)


cell19_next_source_type = (
    cell19_next_source_type_match.get(
        "value"
    )

    if isinstance(
        cell19_next_source_type_match,
        dict,
    )

    else None
)


cell19_next_stratum = (
    cell19_next_stratum_match.get(
        "value"
    )

    if isinstance(
        cell19_next_stratum_match,
        dict,
    )

    else None
)


cell19_next_task_family = (
    cell19_next_task_family_match.get(
        "value"
    )

    if isinstance(
        cell19_next_task_family_match,
        dict,
    )

    else None
)


cell19_next_question_archetype = (
    cell19_next_question_archetype_match.get(
        "value"
    )

    if isinstance(
        cell19_next_question_archetype_match,
        dict,
    )

    else None
)


# ============================================================
# 16) Extract next Pilot Target Text
# ============================================================

cell19_next_target_text_parts_raw = []


if isinstance(
    cell19_next_pilot_packet,
    dict,
):

    raw_parts = (
        cell19_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        raw_parts,
        list,
    ):

        cell19_next_target_text_parts_raw = (
            copy.deepcopy(
                raw_parts
            )
        )


cell19_next_target_text_parts = []


for index, part in enumerate(
    cell19_next_target_text_parts_raw
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = (
            part.get(
                field_name
            )
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = (
                candidate.strip()
            )

            break


    if text_value is None:
        continue


    cell19_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell19_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell19_next_target_text_parts
    )
)


cell19_next_target_text_available = (
    isinstance(
        cell19_next_combined_target_text,
        str,
    )
    and
    bool(
        cell19_next_combined_target_text.strip()
    )
)


# ============================================================
# 17) Build next Pilot Human-Review Workspace
# ============================================================

cell19_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell19_next_pilot_id,

    "golden_case_id":
        cell19_next_golden_case_id,

    "task_draft_id":
        cell19_next_task_draft_id,

    "target_evidence_id":
        cell19_next_target_evidence_id,

    "source_type":
        cell19_next_source_type,

    "stratum":
        cell19_next_stratum,

    "task_family":
        cell19_next_task_family,

    "question_archetype":
        cell19_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),

    "combined_target_text":
        cell19_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell19_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell19_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell19_next_target_evidence
        ),

    "question_draft":
        None,

    "human_question_decision":
        None,

    "expected_answer":
        None,

    "citation_mapping":
        None,

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_next_pilot_workspace_sha256 = (
    cell10_stable_sha(
        cell19_next_pilot_workspace
    )
)


cell19_next_workspace_ready = all([
    cell19_next_pilot_packet_valid,

    cell19_next_identity_chain_valid,

    isinstance(
        cell19_next_golden_case,
        dict,
    ),

    isinstance(
        cell19_next_task_plan,
        dict,
    ),

    isinstance(
        cell19_next_target_evidence,
        dict,
    ),

    cell19_next_target_text_available,

    cell19_next_pilot_workspace[
        "workflow_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 18) Human Review Registry
#
# Pilot #1 is retained as historical accepted material.
# Pilot #2 has the full compact human-review lineage.
# ============================================================

cell19_historical_pilot_1 = copy.deepcopy(
    cell13_seed.get(
        "completed_pilot",
        {},
    )
)


cell19_human_review_registry = {
    "registry_version":
        "human_review_registry_v1",

    "author_final_review_status":
        "NOT_STARTED",

    "historical_completed_pilot":
        cell19_historical_pilot_1,

    "completed_human_review_cases":
        [
            copy.deepcopy(
                cell19_completed_human_review_case
            )
        ],

    "active_workspace":
        copy.deepcopy(
            cell19_next_pilot_workspace
        ),

    "remaining_pilot_count":
        21,

    "full_572_generation":
        CELL19_FULL_572_BLOCK_STATUS,
}


cell19_human_review_registry_sha256 = (
    cell10_stable_sha(
        cell19_human_review_registry
    )
)


# ============================================================
# 19) Base Seed immutability
# ============================================================

cell19_base_seed_sha_after = (
    cell10_stable_sha(
        cell13_seed
    )
)


cell19_base_seed_unchanged = (
    cell19_base_seed_sha_before
    ==
    cell19_base_seed_sha_after
)


# ============================================================
# 20) Full 572 remains blocked
# ============================================================

cell19_full_572_blocked = all([
    cell19_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,

    cell19_next_pilot_workspace[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,

    cell19_human_review_registry[
        "full_572_generation"
    ]
    ==
    CELL19_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 21) Final Gate
# ============================================================

cell19_gate = all([
    len(
        cell19_failures
    )
    ==
    0,

    cell19_citation_accept_transition_valid,

    cell19_queue_advanced_valid,

    cell19_completed_session[
        "current_state"
    ]
    ==
    "PILOT_COMPLETE",

    cell19_completed_session[
        "queue_advance_performed"
    ]
    is True,

    cell19_next_workspace_ready,

    cell19_base_seed_unchanged,

    cell19_full_572_blocked,
])


# ============================================================
# 22) Diagnostics
# ============================================================

print(
    "=== PILOT #2 COMPLETION + "
    "QUEUE ADVANCE + NEXT WORKSPACE ==="
)


print(
    "\n=== CITATION HUMAN REVIEW ==="
)

print(
    "Decision:",
    CELL19_HUMAN_CITATION_DECISION,
)

print(
    "Citation Mapping ID:",
    cell18_citation_mapping_record.get(
        "citation_mapping_id"
    ),
)

print(
    "Citation ACCEPT transition valid:",
    cell19_citation_accept_transition_valid,
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL19_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell16_question_text,
)

print(
    "Expected Answer:",
    cell17_expected_answer_text,
)

print(
    "Completion status:",
    cell19_completed_session.get(
        "completion_status"
    ),
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Completed Pilot SHA256:",
    cell19_completed_pilot_record_sha256,
)


print(
    "\n=== QUEUE ADVANCE ==="
)

print(
    "Base immutable queue:",
    len(
        cell19_base_remaining_ids
    ),
)

print(
    "Authoritative continuation queue:",
    len(
        cell19_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL19_COMPLETED_PILOT_ID
        not in
        cell19_remaining_pilot_ids
    ),
)

print(
    "Queue advance performed:",
    cell19_completed_session[
        "queue_advance_performed"
    ],
)

print(
    "Next Pilot:",
    cell19_next_pilot_id,
)

print(
    "Continuation State SHA256:",
    cell19_continuation_state_sha256,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell19_next_pilot_id,
)

print(
    "Golden Case:",
    cell19_next_golden_case_id,
)

print(
    "Task Draft:",
    cell19_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell19_next_target_evidence_id,
)

print(
    "Source type:",
    cell19_next_source_type,
)

print(
    "Stratum:",
    cell19_next_stratum,
)

print(
    "Task family:",
    cell19_next_task_family,
)

print(
    "Question archetype:",
    cell19_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    "Target part count:",
    len(
        cell19_next_target_text_parts
    ),
)


for row in (
    cell19_next_target_text_parts
):

    print(
        "\n--- SOURCE PART",
        row[
            "part_index"
        ],
        "---"
    )

    print(
        "Document:",
        row[
            "document_id"
        ],
    )

    print(
        "Role:",
        row[
            "role"
        ],
    )

    print(
        "Chars:",
        (
            row[
                "char_start"
            ],
            row[
                "char_end"
            ],
        ),
    )

    print(
        "\nTEXT:\n"
        +
        row[
            "text"
        ]
    )


print(
    "\n=== NEXT WORKSPACE ==="
)

print(
    "Ready:",
    cell19_next_workspace_ready,
)

print(
    "Workflow state:",
    cell19_next_pilot_workspace[
        "workflow_state"
    ],
)

print(
    "Question Draft:",
    cell19_next_pilot_workspace[
        "question_draft"
    ],
)

print(
    "Expected Answer:",
    cell19_next_pilot_workspace[
        "expected_answer"
    ],
)

print(
    "Citation:",
    cell19_next_pilot_workspace[
        "citation_mapping"
    ],
)

print(
    "Workspace SHA256:",
    cell19_next_pilot_workspace_sha256,
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Historical Pilot #1 retained:",
    bool(
        cell19_historical_pilot_1
    ),
)

print(
    "Completed compact Human Review cases:",
    len(
        cell19_human_review_registry[
            "completed_human_review_cases"
        ]
    ),
)

print(
    "Active Pilot:",
    cell19_human_review_registry[
        "active_workspace"
    ][
        "pilot_case_id"
    ],
)

print(
    "Registry SHA256:",
    cell19_human_review_registry_sha256,
)


print(
    "\n=== SAFETY / IMMUTABILITY ==="
)

print(
    "Base Seed unchanged:",
    cell19_base_seed_unchanged,
)

print(
    "Base Seed still has 22:",
    len(
        cell19_base_remaining_ids
    )
    ==
    22,
)

print(
    "Continuation state has 21:",
    len(
        cell19_remaining_pilot_ids
    )
    ==
    21,
)

print(
    "Full 572 blocked:",
    cell19_full_572_blocked,
)

print(
    "Human Review Notebook file written:",
    False,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell19_failures
)


print(
    "\nCODE CELL 19 GATE:",
    (
        "PASS"
        if cell19_gate
        else "FAIL"
    ),
)


assert cell19_gate, (
    "Pilot #2 completion / queue advancement / "
    "next-Pilot workspace preparation failed. "
    "Repair this SAME Code Cell 19 before "
    "authoring the next Pilot question."
)

=== PILOT #2 COMPLETION + QUEUE ADVANCE + NEXT WORKSPACE ===

=== CITATION HUMAN REVIEW ===
Decision: ACCEPT
Citation Mapping ID: citation-mapping:ef5f045a886b6160eaa44636329de992
Citation ACCEPT transition valid: True

=== COMPLETED PILOT ===
Pilot: question-pilot:3005b48518a7aeb532234332df23c952
Question: چه مشکلی پیش می‌آید اگر بر اساس شناختی که فقط از یک بخش داریم، درباره کل موضوع تصمیم بگیریم؟
Expected Answer: مشکل این است که ممکن است همان شناخت ناقص را به کل موضوع تعمیم بدهیم و بخواهیم بر اساس آن تصمیم بگیریم یا از کل موضوع استفاده کنیم. چنین کاری شاید در ظاهر مناسب به نظر برسد، اما بخش‌های یک موضوع می‌توانند به هم وابسته باشند و نمی‌شود یک بخش را کاملاً جدا از بقیه در نظر گرفت. در مثال متن نیز فیل یک کل به‌هم‌پیوسته است و شناخت یک قسمت از آن، به‌تنهایی تصویر کاملی از فیل نمی‌دهد.
Completion status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED
Completed Pilot SHA256: 409468d96811e54b1dddfa217cbb5d9df32e04a28630b915c69a7be4bea4439d

=== QUEUE 

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 20 — Question Accepted + Expected Answer Draft برای Pilot بعدی

Evidence اصلی این Pilot درباره وضعیت سواد مالی در اروپا و سنجش‌های انجام‌شده در سال‌های 2014 و 2023 است.

پس از Human Review، سؤال زیر پذیرفته شد:

**«سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟»**

در این Cell به‌صورت فشرده:

1. Question Draft واقعی ساخته و Machine Validate می‌شود؛
2. Human Decision برابر `ACCEPT` ثبت می‌شود؛
3. Expected Answer Draft فقط از Evidence همین Pilot ساخته می‌شود؛
4. Answer برای Human Review متوقف می‌شود.

عبارت «حدود 50 درصد» یک داده‌ی خارجی نیست؛ صرفاً تبدیل مستقیم «نیمی» به درصد است و این derivation به‌صورت صریح در lineage ثبت می‌شود.

هیچ ادعایی درباره جدید بودن مفهوم سواد مالی، وضعیت ایران، ضرورت آموزش آن به همه، یا سایر برداشت‌های انسانی وارد Answer نمی‌شود؛ زیرا این مطالب در Evidence فعلی وجود ندارند.

Citation هنوز ساخته نمی‌شود، Queue همچنان 21 باقی می‌ماند و Full 572 Generation Blocked است.

</div>

In [42]:
# Code Cell 20

cell20_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell19_gate") is not True:
    cell20_failures.append(
        "CELL19_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell19_next_pilot_workspace"
    ),
    dict,
):
    cell20_failures.append(
        "NEXT_PILOT_WORKSPACE_MISSING"
    )


if not callable(
    globals().get(
        "cell12_question_id_function"
    )
):
    cell20_failures.append(
        "QUESTION_ID_BUILDER_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL20_PILOT_ID = (
    cell19_next_pilot_id
)

CELL20_GOLDEN_CASE_ID = (
    cell19_next_golden_case_id
)

CELL20_TASK_DRAFT_ID = (
    cell19_next_task_draft_id
)

CELL20_TARGET_EVIDENCE_ID = (
    cell19_next_target_evidence_id
)

CELL20_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell20_identity_valid = all([
    isinstance(
        CELL20_PILOT_ID,
        str,
    ),

    isinstance(
        CELL20_GOLDEN_CASE_ID,
        str,
    ),

    isinstance(
        CELL20_TASK_DRAFT_ID,
        str,
    ),

    isinstance(
        CELL20_TARGET_EVIDENCE_ID,
        str,
    ),

    cell19_next_pilot_workspace.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 3) Build current Pilot bundle + fresh session
# ============================================================

cell20_pilot_bundle = {
    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,
}


cell20_initial_session = (
    cell13_build_session(
        cell20_pilot_bundle
    )
)


cell20_initial_session_valid = all([
    cell20_initial_session[
        "current_state"
    ]
    ==
    "QUESTION_DRAFT_PENDING",

    cell20_initial_session[
        "queue_advance_permitted"
    ]
    is False,

    cell20_initial_session[
        "queue_advance_performed"
    ]
    is False,
])


# ============================================================
# 4) Authoritative fields
# ============================================================

cell20_field_sources = [
    (
        "pilot_packet",
        cell19_next_pilot_packet,
    ),
    (
        "task_plan",
        cell19_next_task_plan,
    ),
    (
        "golden_case",
        cell19_next_golden_case,
    ),
    (
        "target_evidence",
        cell19_next_target_evidence,
    ),
]


cell20_task_family_resolution = (
    cell15_resolve_field(
        "task_family",
        cell20_field_sources,
    )
)

cell20_question_archetype_resolution = (
    cell15_resolve_field(
        "question_archetype",
        cell20_field_sources,
    )
)

cell20_required_dimensions_resolution = (
    cell15_resolve_field(
        "required_dimensions",
        cell20_field_sources,
    )
)

cell20_authority_expectation_resolution = (
    cell15_resolve_field(
        "authority_expectation",
        cell20_field_sources,
    )
)

cell20_context_expectation_resolution = (
    cell15_resolve_field(
        "context_expectation",
        cell20_field_sources,
    )
)

cell20_evidence_scope_resolution = (
    cell15_resolve_field(
        "evidence_scope",
        cell20_field_sources,
    )
)

cell20_target_status_resolution = (
    cell15_resolve_field(
        "target_evidence_status",
        cell20_field_sources,
    )
)


cell20_task_family = (
    cell20_task_family_resolution[
        "value"
    ]
)

cell20_question_archetype = (
    cell20_question_archetype_resolution[
        "value"
    ]
)

cell20_required_dimensions = (
    cell20_required_dimensions_resolution[
        "value"
    ]
)

cell20_authority_expectation = (
    cell20_authority_expectation_resolution[
        "value"
    ]
)

cell20_context_expectation = (
    cell20_context_expectation_resolution[
        "value"
    ]
)

cell20_evidence_scope = (
    cell20_evidence_scope_resolution[
        "value"
    ]
)

cell20_target_evidence_status = (
    cell20_target_status_resolution[
        "value"
    ]
)


if isinstance(
    cell20_required_dimensions,
    tuple,
):
    cell20_required_dimensions = list(
        cell20_required_dimensions
    )

elif isinstance(
    cell20_required_dimensions,
    set,
):
    cell20_required_dimensions = sorted(
        cell20_required_dimensions
    )


# ============================================================
# 5) Accepted REAL Question
# ============================================================

CELL20_QUESTION_TEXT = (
    "سواد مالی در اروپا چقدر مورد توجه قرار گرفته "
    "و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟"
)


cell20_question_text = (
    CELL20_QUESTION_TEXT.strip()
)


cell20_question_text_sha256 = (
    hashlib.sha256(
        cell20_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell20_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL20_PILOT_ID,

        question_text_sha256=
            cell20_question_text_sha256,
    )
)


# ============================================================
# 6) Question Machine Validation
# ============================================================

cell20_normalized_question = (
    cell15_normalize_text(
        cell20_question_text
    )
)

cell20_normalized_evidence = (
    cell15_normalize_text(
        cell19_next_combined_target_text
    )
)


cell20_question_internal_id_hits = [
    pattern

    for pattern
    in CELL15_INTERNAL_ID_PATTERNS

    if re.search(
        pattern,
        cell20_question_text,
        flags=re.IGNORECASE,
    )
]


cell20_question_internal_metadata_hits = [
    term

    for term
    in CELL15_INTERNAL_METADATA_TERMS

    if term.lower()
    in
    cell20_question_text.lower()
]


cell20_question_grounding_checks = {
    "financial_literacy":
        (
            "سواد مالی"
            in
            cell20_normalized_question
            and
            "سواد مالی"
            in
            cell20_normalized_evidence
        ),

    "europe":
        (
            "اروپا"
            in
            cell20_normalized_question
            and
            "اروپ"
            in
            cell20_normalized_evidence
        ),

    "appropriate_literacy_share":
        (
            "درصد"
            in
            cell20_normalized_question
            and
            "نیمی"
            in
            cell20_normalized_evidence
        ),

    "attention_or_study":
        (
            "مورد توجه"
            in
            cell20_normalized_question
            and
            "نظرسنجی"
            in
            cell20_normalized_evidence
        ),
}


cell20_question_grounding_valid = all(
    cell20_question_grounding_checks.values()
)


cell20_question_machine_gates = {
    "nonempty":
        bool(
            cell20_question_text
        ),

    "natural_question":
        cell20_question_text.endswith(
            (
                "؟",
                "?",
            )
        ),

    "target_text_available":
        cell19_next_target_text_available,

    "groundable_in_target":
        cell20_question_grounding_valid,

    "no_internal_id_leakage":
        (
            len(
                cell20_question_internal_id_hits
            )
            ==
            0
        ),

    "no_internal_metadata_leakage":
        (
            len(
                cell20_question_internal_metadata_hits
            )
            ==
            0
        ),

    "task_family_present":
        (
            cell20_task_family
            is not None
        ),

    "question_archetype_present":
        (
            cell20_question_archetype
            is not None
        ),

    "authority_expectation_present":
        (
            cell20_authority_expectation
            is not None
        ),

    "context_expectation_present":
        (
            cell20_context_expectation
            is not None
        ),

    "external_knowledge_required":
        False,
}


cell20_question_machine_validation_pass = all([
    cell20_question_machine_gates[
        "nonempty"
    ],

    cell20_question_machine_gates[
        "natural_question"
    ],

    cell20_question_machine_gates[
        "target_text_available"
    ],

    cell20_question_machine_gates[
        "groundable_in_target"
    ],

    cell20_question_machine_gates[
        "no_internal_id_leakage"
    ],

    cell20_question_machine_gates[
        "no_internal_metadata_leakage"
    ],

    cell20_question_machine_gates[
        "task_family_present"
    ],

    cell20_question_machine_gates[
        "question_archetype_present"
    ],

    cell20_question_machine_gates[
        "authority_expectation_present"
    ],

    cell20_question_machine_gates[
        "context_expectation_present"
    ],
])


# ============================================================
# 7) Build Question Draft Record
# ============================================================

cell20_question_draft_record = {
    "question_draft_id":
        cell20_question_draft_id,

    "question_draft_schema_version":
        cell15_question_schema_version,

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "source_type":
        cell19_next_source_type,

    "stratum":
        cell19_next_stratum,

    "task_family":
        cell20_task_family,

    "question_archetype":
        cell20_question_archetype,

    "required_dimensions":
        copy.deepcopy(
            cell20_required_dimensions
        ),

    "authority_expectation":
        copy.deepcopy(
            cell20_authority_expectation
        ),

    "context_expectation":
        copy.deepcopy(
            cell20_context_expectation
        ),

    "evidence_scope":
        copy.deepcopy(
            cell20_evidence_scope
        ),

    "question_text":
        cell20_question_text,

    "question_text_sha256":
        cell20_question_text_sha256,

    "authoring_mode":
        cell15_authoring_mode,

    "draft_status":
        "DRAFT_CREATED_PENDING_VALIDATION",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell20_question_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "ACCEPT",

    "expected_answer_status":
        "READY_FOR_CONSTRUCTION",

    "citation_target_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "machine_validation":
        copy.deepcopy(
            cell20_question_machine_gates
        ),

    "full_572_generation":
        CELL20_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 8) Human ACCEPT record for Question
# ============================================================

CELL20_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted this Question after reviewing "
    "multiple alternative phrasings. Human interpretive "
    "discussion is review metadata only and is not evidence."
)


cell20_question_human_review = {
    "review_record_version":
        "human_question_review_record_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "question_text":
        cell20_question_text,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL20_HUMAN_QUESTION_NOTE,

    "human_context_evidence_status":
        "NON_EVIDENCE",

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),
}


# ============================================================
# 9) Register Question + Human ACCEPT
# ============================================================

cell20_question_review_session = None
cell20_answer_pending_session = None


if cell20_question_machine_validation_pass:

    cell20_question_review_session = (
        cell13_apply_event(
            cell20_initial_session,
            "QUESTION_DRAFT_REGISTERED",
            artifact=
                cell20_question_draft_record,
        )
    )


    cell20_answer_pending_session = (
        cell13_apply_event(
            cell20_question_review_session,
            "QUESTION_ACCEPTED",
            human_note=
                CELL20_HUMAN_QUESTION_NOTE,
        )
    )


cell20_question_accept_valid = all([
    isinstance(
        cell20_answer_pending_session,
        dict,
    ),

    cell20_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING",

    len(
        cell20_answer_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell20_answer_pending_session[
        "human_decisions"
    ][
        0
    ].get(
        "event"
    )
    ==
    "QUESTION_ACCEPTED",
])


# ============================================================
# 10) Expected Answer Draft
#
# Only the validated source evidence is allowed.
# ============================================================

CELL20_EXPECTED_ANSWER_TEXT = (
    "بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود "
    "50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا "
    "مورد بررسی گسترده قرار گرفته است؛ در سال 2014 "
    "نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح "
    "سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز "
    "کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو "
    "اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش "
    "کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای "
    "پژوهشگران و سیاست‌گذاران فراهم کرد."
)


cell20_expected_answer_text = (
    CELL20_EXPECTED_ANSWER_TEXT.strip()
)


cell20_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell20_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 11) Explicit derivation audit
# ============================================================

cell20_derived_claims = [
    {
        "answer_claim":
            "حدود 50 درصد",

        "source_phrase":
            "تنها نیمی از اروپایی‌ها سواد مالی مناسبی دارند",

        "derivation":
            "1/2 = 50%",

        "derivation_type":
            "DIRECT_ARITHMETIC_NORMALIZATION",

        "external_knowledge":
            False,
    }
]


cell20_derived_claims_valid = all([
    "نیمی"
    in
    cell20_normalized_evidence,

    "50 درصد"
    in
    cell15_normalize_text(
        cell20_expected_answer_text
    ),

    cell20_derived_claims[
        0
    ][
        "external_knowledge"
    ]
    is False,
])


# ============================================================
# 12) Answer grounding validation
# ============================================================

cell20_normalized_answer = (
    cell15_normalize_text(
        cell20_expected_answer_text
    )
)


cell20_answer_grounding_checks = {
    "half_of_europeans":
        (
            "نیمی"
            in
            cell20_normalized_evidence
            and
            "50 درصد"
            in
            cell20_normalized_answer
        ),

    "2014_survey":
        (
            "2014"
            in
            cell20_normalized_evidence
            and
            "2014"
            in
            cell20_normalized_answer
        ),

    "2023_commission_survey":
        (
            "2023"
            in
            cell20_normalized_evidence
            and
            "2023"
            in
            cell20_normalized_answer
        ),

    "27_eu_countries":
        (
            "27"
            in
            cell20_normalized_evidence
            and
            "27"
            in
            cell20_normalized_answer
        ),

    "researchers_and_policy_makers":
        (
            "پژوهشگران"
            in
            cell20_normalized_evidence
            and
            "پژوهشگران"
            in
            cell20_normalized_answer
            and
            "سیاست گذاران"
            in
            cell20_normalized_evidence
            and
            "سیاست گذاران"
            in
            cell20_normalized_answer
        ),

    "reliable_findings":
        (
            "قابل اتکایی"
            in
            cell20_normalized_evidence
            and
            "قابل اتکاتری"
            in
            cell20_normalized_answer
        ),
}


cell20_answer_grounding_valid = all(
    cell20_answer_grounding_checks.values()
)


# ============================================================
# 13) Unsupported-claim guard
# ============================================================

CELL20_FORBIDDEN_UNSUPPORTED_TERMS = [
    "ایران",
    "ایرانی",
    "همه باید",
    "همه مردم باید",
    "مفهوم جدید",
    "موضوع جدید",
    "تازه به وجود آمده",
    "به تازگی ایجاد شده",
    "قدیمی نیست",
]


cell20_unsupported_answer_hits = [
    term

    for term
    in CELL20_FORBIDDEN_UNSUPPORTED_TERMS

    if cell15_normalize_text(
        term
    )
    in
    cell20_normalized_answer
]


cell20_answer_no_unsupported_claims = (
    len(
        cell20_unsupported_answer_hits
    )
    ==
    0
)


# ============================================================
# 14) Answer machine validation
# ============================================================

cell20_answer_machine_gates = {
    "nonempty":
        bool(
            cell20_expected_answer_text
        ),

    "addresses_attention":
        (
            "مورد بررسی گسترده"
            in
            cell20_normalized_answer
        ),

    "addresses_percentage":
        (
            "50 درصد"
            in
            cell20_normalized_answer
        ),

    "source_grounded":
        cell20_answer_grounding_valid,

    "derived_percentage_audited":
        cell20_derived_claims_valid,

    "no_unsupported_claims":
        cell20_answer_no_unsupported_claims,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell20_answer_machine_validation_pass = all([
    cell20_answer_machine_gates[
        "nonempty"
    ],

    cell20_answer_machine_gates[
        "addresses_attention"
    ],

    cell20_answer_machine_gates[
        "addresses_percentage"
    ],

    cell20_answer_machine_gates[
        "source_grounded"
    ],

    cell20_answer_machine_gates[
        "derived_percentage_audited"
    ],

    cell20_answer_machine_gates[
        "no_unsupported_claims"
    ],
])


# ============================================================
# 15) Stable local Expected Answer Draft ID
# ============================================================

cell20_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell20_expected_answer_text_sha256,
}


cell20_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell20_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 16) Expected Answer Record
# ============================================================

cell20_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell20_expected_answer_draft_id,

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell20_question_draft_id,

    "question_text":
        cell20_question_text,

    "expected_answer_text":
        cell20_expected_answer_text,

    "expected_answer_text_sha256":
        cell20_expected_answer_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "derived_claims":
        copy.deepcopy(
            cell20_derived_claims
        ),

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell20_answer_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell20_answer_machine_gates
        ),

    "human_adjudication_status":
        "PENDING",

    "citation_validation_status":
        "BLOCKED_PENDING_EXPECTED_ANSWER_ACCEPTANCE",

    "full_572_generation":
        CELL20_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 17) Register Expected Answer Draft
# ============================================================

cell20_current_session = None


if all([
    cell20_question_accept_valid,
    cell20_answer_machine_validation_pass,
]):

    cell20_current_session = (
        cell13_apply_event(
            cell20_answer_pending_session,
            "EXPECTED_ANSWER_REGISTERED",
            artifact=
                cell20_expected_answer_record,
        )
    )


cell20_expected_answer_registered = all([
    isinstance(
        cell20_current_session,
        dict,
    ),

    cell20_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    isinstance(
        cell20_current_session[
            "artifacts"
        ][
            "question_draft"
        ],
        dict,
    ),

    isinstance(
        cell20_current_session[
            "artifacts"
        ][
            "expected_answer"
        ],
        dict,
    ),

    cell20_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ]
    is None,

    len(
        cell20_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    1,

    cell20_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell20_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 18) Update compact Human Review Registry
# ============================================================

cell20_human_review_registry = copy.deepcopy(
    cell19_human_review_registry
)


cell20_active_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL20_PILOT_ID,

    "golden_case_id":
        CELL20_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL20_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL20_TARGET_EVIDENCE_ID,

    "source_evidence":
        copy.deepcopy(
            cell19_next_target_text_parts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell20_question_draft_id,

                "question_text":
                    cell20_question_text,

                "machine_validation":
                    (
                        "PASS"
                        if
                        cell20_question_machine_validation_pass
                        else
                        "FAIL"
                    ),

                "human_decision":
                    "ACCEPT",
            }
        ],

    "human_reviews":
        [
            copy.deepcopy(
                cell20_question_human_review
            )
        ],

    "final_question_draft_id":
        cell20_question_draft_id,

    "expected_answer_review":
        {
            "expected_answer_draft_id":
                cell20_expected_answer_draft_id,

            "expected_answer_text":
                cell20_expected_answer_text,

            "machine_validation_status":
                cell20_expected_answer_record[
                    "machine_validation_status"
                ],

            "human_decision":
                "PENDING",

            "derived_claims":
                copy.deepcopy(
                    cell20_derived_claims
                ),
        },

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_status":
        "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW",
}


cell20_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell20_active_human_review_case
)


cell20_human_review_registry_sha256 = (
    cell10_stable_sha(
        cell20_human_review_registry
    )
)


# ============================================================
# 19) Queue / state immutability
# ============================================================

cell20_continuation_queue_still_21 = (
    len(
        cell19_continuation_state[
            "remaining_pilot_ids"
        ]
    )
    ==
    21
)


cell20_next_pilot_unchanged = (
    cell19_continuation_state[
        "next_pilot_id"
    ]
    ==
    CELL20_PILOT_ID
)


cell20_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell20_full_572_blocked = all([
    isinstance(
        cell20_current_session,
        dict,
    ),

    cell20_current_session.get(
        "full_572_generation"
    )
    ==
    CELL20_FULL_572_BLOCK_STATUS,

    cell20_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL20_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 20) Final Gate
# ============================================================

cell20_gate = all([
    len(
        cell20_failures
    )
    ==
    0,

    cell20_identity_valid,

    cell20_initial_session_valid,

    cell20_question_machine_validation_pass,

    cell20_question_accept_valid,

    cell20_answer_machine_validation_pass,

    cell20_expected_answer_registered,

    cell20_continuation_queue_still_21,

    cell20_next_pilot_unchanged,

    cell20_base_seed_unchanged,

    cell20_full_572_blocked,
])


# ============================================================
# 21) Diagnostics
# ============================================================

print(
    "=== COMPACT PILOT WORKFLOW: "
    "QUESTION ACCEPTED + ANSWER DRAFT ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell19_next_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell20_question_text
)

print(
    "Question Draft ID:",
    cell20_question_draft_id,
)

print(
    "Human decision:",
    "ACCEPT",
)

print(
    "Question Machine PASS:",
    cell20_question_machine_validation_pass,
)

print(
    "Question grounding:",
    cell20_question_grounding_checks,
)


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell20_expected_answer_text
)

print(
    "\nExpected Answer Draft ID:",
    cell20_expected_answer_draft_id,
)


print(
    "\n=== DERIVED CLAIM AUDIT ==="
)

for row in cell20_derived_claims:
    print(row)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell20_answer_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== ANSWER MACHINE GATES ==="
)

for name, status in (
    cell20_answer_machine_gates.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "Unsupported answer hits:",
    cell20_unsupported_answer_hits,
)

print(
    "Machine validation PASS:",
    cell20_answer_machine_validation_pass,
)


print(
    "\n=== REAL WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell20_current_session.get(
            "current_state"
        )

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Expected Answer registered:",
    cell20_expected_answer_registered,
)

print(
    "Human decisions recorded:",
    (
        len(
            cell20_current_session.get(
                "human_decisions",
                [],
            )
        )

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation:",
    (
        cell20_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ]

        if isinstance(
            cell20_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== HUMAN REVIEW REGISTRY ==="
)

print(
    "Active Pilot:",
    cell20_active_human_review_case[
        "pilot_case_id"
    ],
)

print(
    "Question decision:",
    "ACCEPT",
)

print(
    "Expected Answer decision:",
    cell20_active_human_review_case[
        "expected_answer_review"
    ][
        "human_decision"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)

print(
    "Registry SHA256:",
    cell20_human_review_registry_sha256,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Continuation queue still 21:",
    cell20_continuation_queue_still_21,
)

print(
    "Next Pilot unchanged:",
    cell20_next_pilot_unchanged,
)

print(
    "Base Seed unchanged:",
    cell20_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell20_full_572_blocked,
)

print(
    "Project/file mutation performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell20_failures
)


print(
    "\nCODE CELL 20 GATE:",
    (
        "PASS"
        if cell20_gate
        else "FAIL"
    ),
)


assert cell20_gate, (
    "Compact Question ACCEPT + Expected Answer Draft "
    "workflow failed. Repair this SAME Code Cell 20 "
    "before Human Expected Answer review."
)

=== COMPACT PILOT WORKFLOW: QUESTION ACCEPTED + ANSWER DRAFT ===

=== SOURCE EVIDENCE ===
تنها نیمی از اروپایی‌ها سواد مالی مناسبی دارند؛ در پژوهشی تازه در بهار 2023، کمیسیون اروپا نظرسنجی جامعی دربارۀ سواد مالی در تمام کشورهای عضو اتحادیۀ اروپا انجام داده که علاوه بر مقایسۀ وضعیت دانش مالی کشورهای مختلف، حاوی اطلاعات ارزشمندی برای پژوهشگران و سیاست‌گذاران است.
پیش از انتشار این مجموعه داده، ابزار اصلی برای مقایسۀ سطح سواد مالی کشورهای اروپایی، نظرسنجی جهانی سواد مالی در سال 2014 بود که برخی اعضای اُای‌سی‌دی و کشورهای عضو اتحادیۀ اروپا در آن شرکت نکرده بودند و این مسأله مانعی برای ارزیابی دقیق پژوهشگران از وضع موجود و مسیر مطلوب به حساب می‌آمد. اما سنجش جدید و جامع 2023، شامل 27 کشور عضو اتحادیۀ اروپا است و یافته‌های قابل اتکایی در اختیار ما می‌گذارد.

=== ACCEPTED QUESTION ===
سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Question Draft ID: question-draft:992525a168edf83380610af7920e6a3e
Human decision: ACCEPT
Question Machine PASS: True

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 21 — ایجاد Persistent Human Review Registry و Author Review Notebook

از این مرحله Human Reviewها فقط در حافظه‌ی Kernel باقی نمی‌مانند.

دو Artifact محلی ایجاد می‌شود:

1. یک فایل JSON به‌عنوان منبع حقیقت پایدار Human Review؛
2. یک Notebook سبک و خوانا برای مرور نهایی توسط دکتر رودی.

Notebook اصلی همچنان محیط کاری ما باقی می‌ماند. پس از هر تغییر در Pilotهای بعدی، تابع Sync همین Cell، Registry JSON را به‌روزرسانی می‌کند.

Author Review Notebook داده‌ی مستقلی نگه نمی‌دارد و همیشه Registry JSON را می‌خواند؛ بنابراین لازم نیست هنگام ادامه‌ی کار هر دو Notebook هم‌زمان باز باشند.

Registry فعلی شامل:

- Pilot تاریخی #1؛
- Human Review کامل Pilot #2؛
- Pilot فعال فعلی و Question پذیرفته‌شده و Expected Answer در انتظار Human Review؛
- وضعیت مستقل Author Final Review.

در این Cell فقط همین دو فایل محلی ایجاد یا به‌روزرسانی می‌شوند.

هیچ Commit، Push، PR یا Merge انجام نمی‌شود.

</div>

In [45]:
# Code Cell 21 — REPLACEMENT

cell21_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell20_gate") is not True:
    cell21_failures.append(
        "CELL20_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell20_human_review_registry"
    ),
    dict,
):
    cell21_failures.append(
        "HUMAN_REVIEW_REGISTRY_MISSING"
    )


# ============================================================
# 2) Resolve Repository Root safely
#
# Do not depend on a historical variable named `repo_root`.
# ============================================================

def cell21_find_repo_root():
    candidates = []


    # Prefer paths already proven to belong to this repository.
    for global_name in (
        "prior_notebook_path",
        "handoff_path",
    ):

        candidate = globals().get(
            global_name
        )


        if candidate is None:
            continue


        try:
            candidate_path = Path(
                candidate
            ).resolve()

            if candidate_path.is_file():
                candidate_path = (
                    candidate_path.parent
                )

            candidates.append(
                candidate_path
            )

        except Exception:
            pass


    # Current working directory is a fallback.
    try:
        candidates.append(
            Path.cwd().resolve()
        )
    except Exception:
        pass


    searched = set()


    for candidate in candidates:

        for path in (
            [candidate]
            +
            list(
                candidate.parents
            )
        ):

            path_text = str(
                path
            )


            if path_text in searched:
                continue


            searched.add(
                path_text
            )


            # Strongest signal: Git repository root.
            if (
                path
                /
                ".git"
            ).exists():

                return path


            # Secondary structural signal for this project.
            if all([
                (
                    path
                    /
                    "notebooks"
                    /
                    "phase2"
                ).exists(),

                (
                    path
                    /
                    "data"
                    /
                    "phase2"
                ).exists(),
            ]):

                return path


    raise RuntimeError(
        "Could not resolve the RAG_finance repository root."
    )


try:

    cell21_repo_root = (
        cell21_find_repo_root()
    )


except Exception as exc:

    cell21_repo_root = None

    cell21_failures.append({
        "reason":
            "REPO_ROOT_RESOLUTION_FAILED",

        "error":
            repr(
                exc
            ),
    })


# ============================================================
# 3) Persistent paths
# ============================================================

if isinstance(
    cell21_repo_root,
    Path,
):

    cell21_review_dir = (
        cell21_repo_root
        /
        "data"
        /
        "phase2"
        /
        "human_review"
    )


    cell21_registry_path = (
        cell21_review_dir
        /
        "human_review_registry.json"
    )


    cell21_author_notebook_path = (
        cell21_repo_root
        /
        "notebooks"
        /
        "phase2"
        /
        "12c_human_review_for_author.ipynb"
    )


else:

    cell21_review_dir = None
    cell21_registry_path = None
    cell21_author_notebook_path = None


# ============================================================
# 4) JSON-safe canonicalizer
# ============================================================

def cell21_json_safe(
    value,
    seen=None,
):

    if seen is None:
        seen = set()


    if value is None or isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value


    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )


    if isinstance(
        value,
        bytes,
    ):
        return {
            "__type__":
                "bytes",

            "hex":
                value.hex(),
        }


    if callable(
        value
    ):
        return {
            "__type__":
                "callable",

            "name":
                getattr(
                    value,
                    "__name__",
                    type(
                        value
                    ).__name__,
                ),
        }


    if isinstance(
        value,
        dict,
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return {
                "__type__":
                    "cycle"
            }


        seen.add(
            object_id
        )


        try:

            result = {}


            for key, nested in (
                value.items()
            ):

                if isinstance(
                    key,
                    str,
                ):
                    safe_key = key

                else:

                    safe_key = json.dumps(
                        cell21_json_safe(
                            key,
                            seen,
                        ),
                        ensure_ascii=False,
                        sort_keys=True,
                    )


                result[
                    safe_key
                ] = cell21_json_safe(
                    nested,
                    seen,
                )


            return result


        finally:

            seen.discard(
                object_id
            )


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):

        object_id = id(
            value
        )


        if object_id in seen:
            return [
                "__cycle__"
            ]


        seen.add(
            object_id
        )


        try:

            return [
                cell21_json_safe(
                    nested,
                    seen,
                )

                for nested
                in value
            ]


        finally:

            seen.discard(
                object_id
            )


    if isinstance(
        value,
        (
            set,
            frozenset,
        ),
    ):

        safe_items = [
            cell21_json_safe(
                nested,
                seen,
            )

            for nested
            in value
        ]


        return sorted(
            safe_items,
            key=lambda item:
                json.dumps(
                    item,
                    ensure_ascii=False,
                    sort_keys=True,
                    default=str,
                ),
        )


    return {
        "__type__":
            type(
                value
            ).__name__,

        "repr":
            repr(
                value
            ),
    }


# ============================================================
# 5) Build persistent Registry payload
# ============================================================

cell21_registry_payload = {
    "registry_version":
        "persistent_human_review_registry_v1",

    "project":
        "RAG_finance",

    "phase":
        "phase2_golden_dataset",

    "purpose":
        "HUMAN_AND_AUTHOR_REVIEW",

    "workflow_policy":
        {
            "source_evidence_is_authoritative":
                True,

            "human_comments_are_evidence":
                False,

            "model_interpretation_is_evidence":
                False,

            "author_final_review_required":
                True,

            "full_572_generation":
                (
                    "BLOCKED_PENDING_"
                    "FULL_PILOT_WORKFLOW_VALIDATION"
                ),
        },

    "historical_completed_pilot":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "historical_completed_pilot"
            )
        ),

    "completed_human_review_cases":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "completed_human_review_cases",
                [],
            )
        ),

    "active_workspace":
        copy.deepcopy(
            cell20_human_review_registry.get(
                "active_workspace"
            )
        ),

    "remaining_pilot_count":
        21,

    "continuation_state":
        {
            "next_pilot_id":
                cell19_continuation_state.get(
                    "next_pilot_id"
                ),

            "remaining_pilot_ids":
                copy.deepcopy(
                    cell19_continuation_state.get(
                        "remaining_pilot_ids",
                        [],
                    )
                ),

            "remaining_count":
                cell19_continuation_state.get(
                    "remaining_count"
                ),

            "continuation_state_sha256":
                cell19_continuation_state_sha256,
        },

    "author_review":
        {
            "status":
                "NOT_STARTED",

            "reviewer":
                "AUTHOR",

            "cases_ready_for_review":
                1,

            "notes":
                None,
        },
}


cell21_registry_payload = (
    cell21_json_safe(
        cell21_registry_payload
    )
)


cell21_registry_sha256 = (
    cell10_stable_sha(
        cell21_registry_payload
    )
)


# ============================================================
# 6) Reusable Registry sync function
#
# Future cells can call:
#
# cell21_sync_human_review_registry(updated_registry)
# ============================================================

def cell21_sync_human_review_registry(
    registry_payload,
):

    if not isinstance(
        cell21_registry_path,
        Path,
    ):

        raise RuntimeError(
            "Registry path is unavailable."
        )


    safe_payload = (
        cell21_json_safe(
            registry_payload
        )
    )


    serialized = json.dumps(
        safe_payload,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )


    cell21_review_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    temp_path = (
        cell21_registry_path.with_suffix(
            ".json.tmp"
        )
    )


    temp_path.write_text(
        serialized,
        encoding="utf-8",
    )


    temp_path.replace(
        cell21_registry_path
    )


    loaded = json.loads(
        cell21_registry_path.read_text(
            encoding="utf-8"
        )
    )


    expected_sha = (
        cell10_stable_sha(
            safe_payload
        )
    )


    loaded_sha = (
        cell10_stable_sha(
            loaded
        )
    )


    if loaded_sha != expected_sha:

        raise AssertionError(
            "Persistent Human Review Registry "
            "round-trip fingerprint mismatch."
        )


    return {
        "path":
            str(
                cell21_registry_path
            ),

        "sha256":
            loaded_sha,

        "bytes":
            cell21_registry_path.stat().st_size,
    }


# ============================================================
# 7) Write initial persistent Registry
# ============================================================

cell21_registry_write_result = None


if isinstance(
    cell21_registry_path,
    Path,
):

    try:

        cell21_registry_write_result = (
            cell21_sync_human_review_registry(
                cell21_registry_payload
            )
        )


    except Exception as exc:

        cell21_failures.append({
            "reason":
                "REGISTRY_WRITE_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 8) Build lightweight Author Review Notebook
#
# This Notebook is a viewer/report.
# The JSON Registry remains the source of truth.
# ============================================================

cell21_author_nb_cells = []


cell21_author_nb_cells.append({
    "id":
        "intro-review",

    "cell_type":
        "markdown",

    "metadata":
        {},

    "source":
        [
            "# Human / Author Review — Golden Dataset\n",
            "\n",
            "این Notebook برای مرور انسانی و تأیید نهایی مؤلف ساخته شده است.\n",
            "\n",
            "منبع حقیقت این Notebook فایل "
            "`human_review_registry.json` است و داده‌های Review "
            "در خود این Notebook نگهداری نمی‌شوند.\n",
            "\n",
            "برای مشاهده آخرین وضعیت، کافی است Notebook را Run کنید."
        ],
})


cell21_author_loader_code = r'''
from pathlib import Path
import json


def find_registry():
    candidates = []

    cwd = Path.cwd().resolve()

    candidates.extend([
        cwd
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",

        cwd
        / ".."
        / ".."
        / "data"
        / "phase2"
        / "human_review"
        / "human_review_registry.json",
    ])

    for parent in [cwd] + list(cwd.parents):
        candidates.append(
            parent
            / "data"
            / "phase2"
            / "human_review"
            / "human_review_registry.json"
        )

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if str(candidate) in checked:
            continue

        checked.add(str(candidate))

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "human_review_registry.json was not found."
    )


REGISTRY_PATH = find_registry()

review_registry = json.loads(
    REGISTRY_PATH.read_text(
        encoding="utf-8"
    )
)

print("Registry:", REGISTRY_PATH)

print(
    "Completed review cases:",
    len(
        review_registry.get(
            "completed_human_review_cases",
            []
        )
    )
)

print(
    "Active Pilot:",
    (
        review_registry.get(
            "active_workspace"
        )
        or {}
    ).get(
        "pilot_case_id"
    )
)

print(
    "Remaining Pilots:",
    review_registry.get(
        "remaining_pilot_count"
    )
)
'''


cell21_author_nb_cells.append({
    "id":
        "load-registry",

    "cell_type":
        "code",

    "execution_count":
        None,

    "metadata":
        {},

    "outputs":
        [],

    "source":
        cell21_author_loader_code.splitlines(
            keepends=True
        ),
})


cell21_author_renderer_code = r'''
from IPython.display import display, Markdown


def show_text(value):
    if value is None:
        return "—"

    return str(value)


def show_evidence_parts(parts):
    for part in parts or []:

        if not isinstance(part, dict):
            continue

        text = part.get("text")

        if not text:
            continue

        display(
            Markdown(
                "> "
                +
                show_text(text).replace(
                    "\n",
                    "\n> "
                )
            )
        )


completed_cases = review_registry.get(
    "completed_human_review_cases",
    []
)


display(
    Markdown(
        "# موارد تکمیل‌شده برای مرور مؤلف"
    )
)


for index, case in enumerate(
    completed_cases,
    start=1,
):

    display(
        Markdown(
            f"## Case {index}"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    evidence_parts = (
        case.get(
            "source_evidence"
        )
        or
        case.get(
            "source_evidence_snapshot"
        )
        or
        []
    )


    show_evidence_parts(
        evidence_parts
    )


    drafts = case.get(
        "model_drafts",
        []
    )


    if drafts:

        display(
            Markdown(
                "### تاریخچه سؤال"
            )
        )


        for draft in drafts:

            display(
                Markdown(
                    "**Revision "
                    +
                    show_text(
                        draft.get(
                            "revision"
                        )
                    )
                    +
                    "**  \n"
                    +
                    show_text(
                        draft.get(
                            "question_text"
                        )
                    )
                    +
                    "  \nHuman Decision: **"
                    +
                    show_text(
                        draft.get(
                            "human_decision"
                        )
                    )
                    +
                    "**"
                )
            )


    answer_review = case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی"
            )
        )

        display(
            Markdown(
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "  \nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    citation_review = case.get(
        "citation_review"
    )


    if isinstance(
        citation_review,
        dict,
    ):

        display(
            Markdown(
                "### Citation"
            )
        )

        display(
            Markdown(
                "Human Decision: **"
                +
                show_text(
                    citation_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    author_review = (
        case.get(
            "author_final_review"
        )
        or
        {}
    )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                author_review.get(
                    "status"
                )
            )
            +
            "**"
        )
    )


active_case = review_registry.get(
    "active_workspace"
)


if isinstance(
    active_case,
    dict,
):

    display(
        Markdown(
            "# Case فعال"
        )
    )


    display(
        Markdown(
            "### متن اصلی منبع"
        )
    )


    show_evidence_parts(
        active_case.get(
            "source_evidence"
        )
        or
        []
    )


    drafts = active_case.get(
        "model_drafts",
        []
    )


    for draft in drafts:

        display(
            Markdown(
                "### سؤال\n"
                +
                show_text(
                    draft.get(
                        "question_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    draft.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    answer_review = active_case.get(
        "expected_answer_review"
    )


    if isinstance(
        answer_review,
        dict,
    ):

        display(
            Markdown(
                "### پاسخ پیشنهادی\n"
                +
                show_text(
                    answer_review.get(
                        "expected_answer_text"
                    )
                )
                +
                "\n\nHuman Decision: **"
                +
                show_text(
                    answer_review.get(
                        "human_decision"
                    )
                )
                +
                "**"
            )
        )


    display(
        Markdown(
            "### تأیید نهایی مؤلف\n"
            "Status: **"
            +
            show_text(
                (
                    active_case.get(
                        "author_final_review"
                    )
                    or
                    {}
                ).get(
                    "status"
                )
            )
            +
            "**"
        )
    )
'''


cell21_author_nb_cells.append({
    "id":
        "render-review",

    "cell_type":
        "code",

    "execution_count":
        None,

    "metadata":
        {},

    "outputs":
        [],

    "source":
        cell21_author_renderer_code.splitlines(
            keepends=True
        ),
})


cell21_author_notebook = {
    "cells":
        cell21_author_nb_cells,

    "metadata":
        {
            "kernelspec":
                {
                    "display_name":
                        "Python 3",

                    "language":
                        "python",

                    "name":
                        "python3",
                },

            "language_info":
                {
                    "name":
                        "python",

                    "version":
                        "3",
                },
        },

    "nbformat":
        4,

    "nbformat_minor":
        5,
}


# ============================================================
# 9) Write Author Review Notebook
# ============================================================

if isinstance(
    cell21_author_notebook_path,
    Path,
):

    try:

        cell21_author_notebook_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        cell21_author_notebook_path.write_text(
            json.dumps(
                cell21_author_notebook,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )


    except Exception as exc:

        cell21_failures.append({
            "reason":
                "AUTHOR_NOTEBOOK_WRITE_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 10) Validate written artifacts
# ============================================================

cell21_registry_exists = (
    isinstance(
        cell21_registry_path,
        Path,
    )
    and
    cell21_registry_path.exists()
)


cell21_author_notebook_exists = (
    isinstance(
        cell21_author_notebook_path,
        Path,
    )
    and
    cell21_author_notebook_path.exists()
)


cell21_loaded_registry = None
cell21_loaded_notebook = None


if cell21_registry_exists:

    try:

        cell21_loaded_registry = json.loads(
            cell21_registry_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell21_failures.append({
            "reason":
                "REGISTRY_READBACK_FAILED",

            "error":
                repr(
                    exc
                ),
        })


if cell21_author_notebook_exists:

    try:

        cell21_loaded_notebook = json.loads(
            cell21_author_notebook_path.read_text(
                encoding="utf-8"
            )
        )

    except Exception as exc:

        cell21_failures.append({
            "reason":
                "AUTHOR_NOTEBOOK_READBACK_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell21_registry_roundtrip_valid = (
    isinstance(
        cell21_loaded_registry,
        dict,
    )
    and
    cell10_stable_sha(
        cell21_loaded_registry
    )
    ==
    cell21_registry_sha256
)


cell21_author_notebook_valid = all([
    isinstance(
        cell21_loaded_notebook,
        dict,
    ),

    (
        cell21_loaded_notebook
        or
        {}
    ).get(
        "nbformat"
    )
    ==
    4,

    isinstance(
        (
            cell21_loaded_notebook
            or
            {}
        ).get(
            "cells"
        ),
        list,
    ),

    len(
        (
            cell21_loaded_notebook
            or
            {}
        ).get(
            "cells",
            [],
        )
    )
    ==
    3,

    all(
        isinstance(
            cell.get(
                "id"
            ),
            str,
        )
        and
        bool(
            cell.get(
                "id"
            )
        )

        for cell
        in (
            (
                cell21_loaded_notebook
                or
                {}
            ).get(
                "cells",
                [],
            )
        )
    ),
])


# ============================================================
# 11) Persistence surface validation
# ============================================================

cell21_completed_case_count = (
    len(
        cell21_loaded_registry.get(
            "completed_human_review_cases",
            [],
        )
    )

    if isinstance(
        cell21_loaded_registry,
        dict,
    )

    else 0
)


cell21_active_case = (
    (
        cell21_loaded_registry.get(
            "active_workspace"
        )
        or
        {}
    )

    if isinstance(
        cell21_loaded_registry,
        dict,
    )

    else {}
)


cell21_active_model_drafts = (
    cell21_active_case.get(
        "model_drafts",
        [],
    )
)


cell21_current_state_persisted = all([
    cell21_completed_case_count
    ==
    1,

    isinstance(
        cell21_loaded_registry,
        dict,
    ),

    cell21_loaded_registry.get(
        "historical_completed_pilot"
    )
    is not None,

    cell21_active_case.get(
        "pilot_case_id"
    )
    ==
    cell20_active_human_review_case[
        "pilot_case_id"
    ],

    cell21_active_case.get(
        "final_question_draft_id"
    )
    ==
    cell20_question_draft_id,

    cell21_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "expected_answer_draft_id"
    )
    ==
    cell20_expected_answer_draft_id,

    cell21_loaded_registry.get(
        "remaining_pilot_count"
    )
    ==
    21,

    len(
        cell21_active_model_drafts
    )
    >=
    1,
])


# ============================================================
# 12) Confirm exact expected files only
# ============================================================

cell21_expected_write_paths = {
    str(
        cell21_registry_path
    ),

    str(
        cell21_author_notebook_path
    ),
}


cell21_expected_write_surface_valid = all([
    cell21_registry_exists,
    cell21_author_notebook_exists,
    len(
        cell21_expected_write_paths
    )
    ==
    2,
])


# ============================================================
# 13) Final Gate
# ============================================================

cell21_gate = all([
    len(
        cell21_failures
    )
    ==
    0,

    isinstance(
        cell21_repo_root,
        Path,
    ),

    cell21_registry_exists,

    cell21_author_notebook_exists,

    cell21_registry_roundtrip_valid,

    cell21_author_notebook_valid,

    cell21_current_state_persisted,

    cell21_expected_write_surface_valid,
])


# ============================================================
# 14) Diagnostics
# ============================================================

print(
    "=== PERSISTENT HUMAN REVIEW SYSTEM ==="
)


print(
    "\n=== REPOSITORY ==="
)

print(
    "Repository root:",
    cell21_repo_root,
)

print(
    "Root resolved:",
    isinstance(
        cell21_repo_root,
        Path,
    ),
)


print(
    "\n=== REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Exists:",
    cell21_registry_exists,
)

print(
    "Bytes:",
    (
        cell21_registry_path.stat().st_size

        if cell21_registry_exists

        else None
    ),
)

print(
    "SHA256:",
    cell21_registry_sha256,
)

print(
    "Round-trip valid:",
    cell21_registry_roundtrip_valid,
)


print(
    "\n=== AUTHOR REVIEW NOTEBOOK ==="
)

print(
    "Path:",
    cell21_author_notebook_path,
)

print(
    "Exists:",
    cell21_author_notebook_exists,
)

print(
    "Valid notebook:",
    cell21_author_notebook_valid,
)

print(
    "Cells:",
    (
        len(
            cell21_loaded_notebook.get(
                "cells",
                [],
            )
        )

        if isinstance(
            cell21_loaded_notebook,
            dict,
        )

        else None
    ),
)


print(
    "\n=== PERSISTED REVIEW STATE ==="
)

print(
    "Historical Pilot #1 retained:",
    (
        isinstance(
            cell21_loaded_registry,
            dict,
        )
        and
        cell21_loaded_registry.get(
            "historical_completed_pilot"
        )
        is not None
    ),
)

print(
    "Completed Human Review cases:",
    cell21_completed_case_count,
)

print(
    "Active Pilot:",
    cell21_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Active Question:",
    (
        cell21_active_model_drafts[
            -1
        ].get(
            "question_text"
        )

        if cell21_active_model_drafts

        else None
    ),
)

print(
    "Active Answer decision:",
    cell21_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Remaining Pilots:",
    (
        cell21_loaded_registry.get(
            "remaining_pilot_count"
        )

        if isinstance(
            cell21_loaded_registry,
            dict,
        )

        else None
    ),
)

print(
    "Persistence state valid:",
    cell21_current_state_persisted,
)


print(
    "\n=== FUTURE WORKFLOW ==="
)

print(
    "Continue working only in this notebook:",
    True,
)

print(
    "Need Author Notebook open during work:",
    False,
)

print(
    "Future Registry sync function:",
    "cell21_sync_human_review_registry(...)",
)

print(
    "Files intentionally written:",
    2,
)

print(
    "Commit / Push performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell21_failures
)


print(
    "\nCODE CELL 21 GATE:",
    (
        "PASS"
        if cell21_gate
        else "FAIL"
    ),
)


assert cell21_gate, (
    "Persistent Human Review Registry / Author Review "
    "Notebook creation failed. Repair this SAME Code Cell 21 "
    "before continuing Pilot review."
)

=== PERSISTENT HUMAN REVIEW SYSTEM ===

=== REPOSITORY ===
Repository root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Root resolved: True

=== REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Exists: True
Bytes: 77499
SHA256: 9967f7522713c60aab5f1d2843480a5cbefe948fd5fb4dcda398685553e55a9b
Round-trip valid: True

=== AUTHOR REVIEW NOTEBOOK ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/notebooks/phase2/12c_human_review_for_author.ipynb
Exists: True
Valid notebook: True
Cells: 3

=== PERSISTED REVIEW STATE ===
Historical Pilot #1 retained: True
Completed Human Review cases: 1
Active Pilot: question-pilot:083ad832854722e126c05d0b89d00cb1
Active Question: سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Active Answer decision: PENDING
Remaining Pilots: 21
Persistence state valid: Tr

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 22 — ثبت Answer ACCEPT، ساخت Citation و Sync پایدار Human Review

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این Cell:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence و Locatorهای از قبل اعتبارسنجی‌شده ساخته می‌شود؛
- هیچ URL، Document ID یا محدوده‌ی متنی جدیدی حدس زده نمی‌شود؛
- Citation به‌صورت ماشینی اعتبارسنجی می‌شود؛
- Workflow تا `CITATION_HUMAN_REVIEW_PENDING` پیش می‌رود؛
- Human Review Registry به‌روزرسانی و با `human_review_registry.json` روی دیسک Sync می‌شود.

Notebook مخصوص دکتر رودی نیازی به بازنویسی ندارد، زیرا هنگام اجرا آخرین Registry JSON را می‌خواند.

در این مرحله Citation هنوز Human ACCEPT نشده، Pilot کامل نشده و Queue همچنان 21 باقی می‌ماند.

Full 572 Generation نیز همچنان Blocked است.

</div>

In [47]:
# Code Cell 22

cell22_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get(
    "cell21_gate"
) is not True:
    cell22_failures.append(
        "CELL21_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell20_current_session"
    ),
    dict,
):
    cell22_failures.append(
        "CELL20_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell20_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell22_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell22_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL22_PILOT_ID = (
    cell19_next_pilot_id
)

CELL22_GOLDEN_CASE_ID = (
    cell19_next_golden_case_id
)

CELL22_TASK_DRAFT_ID = (
    cell19_next_task_draft_id
)

CELL22_TARGET_EVIDENCE_ID = (
    cell19_next_target_evidence_id
)

CELL22_QUESTION_DRAFT_ID = (
    cell20_question_draft_id
)

CELL22_EXPECTED_ANSWER_DRAFT_ID = (
    cell20_expected_answer_draft_id
)

CELL22_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Verify Expected Answer artifact
# ============================================================

cell22_expected_answer_record = copy.deepcopy(
    cell20_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell22_expected_answer_identity_valid = all([
    cell22_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL22_EXPECTED_ANSWER_DRAFT_ID,

    cell22_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL22_PILOT_ID,

    cell22_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL22_QUESTION_DRAFT_ID,

    cell22_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL22_TARGET_EVIDENCE_ID,

    cell22_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL22_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "after reviewing it against the validated source evidence."
)


cell22_answer_accept_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL22_PILOT_ID,

    "golden_case_id":
        CELL22_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL22_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "review_stage":
        "EXPECTED_ANSWER",

    "expected_answer_text":
        cell22_expected_answer_record.get(
            "expected_answer_text"
        ),

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL22_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


cell22_answer_accept_review_sha256 = (
    cell10_stable_sha(
        cell22_answer_accept_review
    )
)


# ============================================================
# 5) Apply real Expected Answer ACCEPT
# ============================================================

cell22_citation_pending_session = (
    cell13_apply_event(
        cell20_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL22_HUMAN_ANSWER_NOTE,
    )
)


cell22_answer_accept_transition_valid = all([
    cell22_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell22_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell22_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell22_citation_pending_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell22_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Exact Target Parts from validated Evidence
# ============================================================

cell22_source_target_parts = copy.deepcopy(
    cell19_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell22_source_target_parts_valid = all([
    isinstance(
        cell22_source_target_parts,
        list,
    ),

    len(
        cell22_source_target_parts
    )
    >
    0,
])


cell22_citation_entries = []


for index, part in enumerate(
    cell22_source_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell22_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL22_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    })


# ============================================================
# 7) Validate exact locator / range projection
# ============================================================

cell22_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell22_source_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell22_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell22_citation_entries
]


cell22_no_locator_fabrication = (
    cell22_source_projection
    ==
    cell22_citation_projection
)


# ============================================================
# 8) Validate against displayed Target Text Parts
# ============================================================

cell22_displayed_parts_by_document = {
    row.get(
        "document_id"
    ):
        row

    for row
    in cell19_next_target_text_parts

    if isinstance(
        row,
        dict,
    )
}


cell22_citation_validation_rows = []


for entry in cell22_citation_entries:

    document_id = (
        entry.get(
            "document_id"
        )
    )


    displayed_part = (
        cell22_displayed_parts_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = (
        locator.get(
            "canonical_url"
        )
    )


    url = (
        locator.get(
            "url"
        )
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        ),

        isinstance(
            displayed_part,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        displayed_part.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        displayed_part.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        displayed_part.get(
            "role"
        ),

        (
            canonical_url
            is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url
            is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell22_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "range_match":
            (
                isinstance(
                    displayed_part,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                displayed_part.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                displayed_part.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    displayed_part,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                displayed_part.get(
                    "role"
                )
            ),

        "valid":
            row_valid,
    })


cell22_all_citation_entries_valid = all([
    len(
        cell22_citation_entries
    )
    >
    0,

    len(
        cell22_citation_entries
    )
    ==
    len(
        cell19_next_target_text_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell22_citation_validation_rows
    ),
])


# ============================================================
# 9) Stable local Citation Mapping ID
# ============================================================

cell22_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL22_PILOT_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell22_citation_projection
        ),
}


cell22_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell22_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 10) Citation Machine Validation
# ============================================================

cell22_citation_machine_validation = {
    "target_parts_present":
        cell22_source_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell22_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell22_citation_entries
            )
            ==
            len(
                cell19_next_target_text_parts
            )
        ),

    "all_entries_valid":
        cell22_all_citation_entries_valid,

    "locator_fabrication_absent":
        cell22_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL22_TARGET_EVIDENCE_ID

            for entry
            in cell22_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell22_citation_machine_validation_pass = all([
    cell22_citation_machine_validation[
        "target_parts_present"
    ],

    cell22_citation_machine_validation[
        "citation_entries_present"
    ],

    cell22_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell22_citation_machine_validation[
        "all_entries_valid"
    ],

    cell22_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell22_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 11) Citation Mapping Record
# ============================================================

cell22_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell22_citation_mapping_id,

    "pilot_case_id":
        CELL22_PILOT_ID,

    "golden_case_id":
        CELL22_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL22_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL22_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL22_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL22_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell22_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"

            if cell22_citation_machine_validation_pass

            else "FAIL_REQUIRES_REVISION"
        ),

    "machine_validation":
        copy.deepcopy(
            cell22_citation_machine_validation
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL22_FULL_572_BLOCK_STATUS,
}


cell22_citation_mapping_record_sha256 = (
    cell10_stable_sha(
        cell22_citation_mapping_record
    )
)


# ============================================================
# 12) Register Citation Draft
# ============================================================

cell22_current_session = None


if all([
    cell22_expected_answer_identity_valid,
    cell22_answer_accept_transition_valid,
    cell22_citation_machine_validation_pass,
]):

    cell22_current_session = (
        cell13_apply_event(
            cell22_citation_pending_session,
            "CITATION_MAPPING_REGISTERED",
            artifact=
                cell22_citation_mapping_record,
        )
    )


cell22_citation_registered = all([
    isinstance(
        cell22_current_session,
        dict,
    ),

    cell22_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell22_current_session[
            "artifacts"
        ][
            "citation_mapping"
        ],
        dict,
    ),

    cell22_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell22_citation_mapping_id,

    len(
        cell22_current_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell22_current_session.get(
        "queue_advance_permitted"
    )
    is False,

    cell22_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 13) Update in-memory Human Review Registry
# ============================================================

cell22_human_review_registry = copy.deepcopy(
    cell20_human_review_registry
)


cell22_active_case = copy.deepcopy(
    cell20_active_human_review_case
)


cell22_active_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell22_active_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell22_answer_accept_review
)


cell22_active_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell22_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell22_citation_entries
        ),

    "machine_validation_status":
        cell22_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell22_active_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


cell22_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell22_active_case
)


cell22_human_review_registry[
    "remaining_pilot_count"
] = 21


# ============================================================
# 14) Build persistent Registry payload
# ============================================================

cell22_persistent_registry_payload = {
    "registry_version":
        "persistent_human_review_registry_v1",

    "project":
        "RAG_finance",

    "phase":
        "phase2_golden_dataset",

    "purpose":
        "HUMAN_AND_AUTHOR_REVIEW",

    "workflow_policy":
        {
            "source_evidence_is_authoritative":
                True,

            "human_comments_are_evidence":
                False,

            "model_interpretation_is_evidence":
                False,

            "author_final_review_required":
                True,

            "full_572_generation":
                CELL22_FULL_572_BLOCK_STATUS,
        },

    "historical_completed_pilot":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "historical_completed_pilot"
            )
        ),

    "completed_human_review_cases":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "completed_human_review_cases",
                [],
            )
        ),

    "active_workspace":
        copy.deepcopy(
            cell22_human_review_registry.get(
                "active_workspace"
            )
        ),

    "remaining_pilot_count":
        21,

    "continuation_state":
        {
            "next_pilot_id":
                cell19_continuation_state.get(
                    "next_pilot_id"
                ),

            "remaining_pilot_ids":
                copy.deepcopy(
                    cell19_continuation_state.get(
                        "remaining_pilot_ids",
                        [],
                    )
                ),

            "remaining_count":
                cell19_continuation_state.get(
                    "remaining_count"
                ),

            "continuation_state_sha256":
                cell19_continuation_state_sha256,
        },

    "author_review":
        copy.deepcopy(
            cell21_loaded_registry.get(
                "author_review",
                {
                    "status":
                        "NOT_STARTED",

                    "reviewer":
                        "AUTHOR",

                    "cases_ready_for_review":
                        1,

                    "notes":
                        None,
                },
            )
        ),
}


# ============================================================
# 15) Sync Registry to disk
# ============================================================

cell22_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell22_persistent_registry_payload
    )
)


cell22_registry_sync_valid = all([
    isinstance(
        cell22_registry_sync_result,
        dict,
    ),

    isinstance(
        cell22_registry_sync_result.get(
            "sha256"
        ),
        str,
    ),

    cell21_registry_path.exists(),
])


# ============================================================
# 16) Read back persisted state
# ============================================================

cell22_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell22_persisted_active_case = (
    cell22_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell22_persistence_state_valid = all([
    cell22_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL22_PILOT_ID,

    cell22_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell22_citation_mapping_id,

    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell22_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    21,
])


# ============================================================
# 17) Queue / safety
# ============================================================

cell22_queue_still_21 = (
    len(
        cell19_continuation_state[
            "remaining_pilot_ids"
        ]
    )
    ==
    21
)


cell22_next_pilot_unchanged = (
    cell19_continuation_state[
        "next_pilot_id"
    ]
    ==
    CELL22_PILOT_ID
)


cell22_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell22_full_572_blocked = all([
    isinstance(
        cell22_current_session,
        dict,
    ),

    cell22_current_session.get(
        "full_572_generation"
    )
    ==
    CELL22_FULL_572_BLOCK_STATUS,

    cell22_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL22_FULL_572_BLOCK_STATUS,
])


# ============================================================
# 18) Final Gate
# ============================================================

cell22_gate = all([
    len(
        cell22_failures
    )
    ==
    0,

    cell22_expected_answer_identity_valid,

    cell22_answer_accept_transition_valid,

    cell22_source_target_parts_valid,

    cell22_no_locator_fabrication,

    cell22_all_citation_entries_valid,

    cell22_citation_machine_validation_pass,

    cell22_citation_registered,

    cell22_registry_sync_valid,

    cell22_persistence_state_valid,

    cell22_queue_still_21,

    cell22_next_pilot_unchanged,

    cell22_base_seed_unchanged,

    cell22_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION + "
    "PERSISTENT REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell20_question_text
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    cell20_expected_answer_text
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell22_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell22_citation_entries
    ),
)


for entry in cell22_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell22_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine validation PASS:",
    cell22_citation_machine_validation_pass,
)

print(
    "No locator fabrication:",
    cell22_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW STATE ==="
)

print(
    "Current state:",
    (
        cell22_current_session.get(
            "current_state"
        )

        if isinstance(
            cell22_current_session,
            dict,
        )

        else None
    ),
)

print(
    "Citation decision:",
    "PENDING",
)

print(
    "Queue advance permitted:",
    (
        cell22_current_session.get(
            "queue_advance_permitted"
        )

        if isinstance(
            cell22_current_session,
            dict,
        )

        else None
    ),
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry path:",
    cell21_registry_path,
)

print(
    "Registry synced:",
    cell22_registry_sync_valid,
)

print(
    "Persisted Answer decision:",
    cell22_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation decision:",
    cell22_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence state valid:",
    cell22_persistence_state_valid,
)

print(
    "Author Notebook rewrite required:",
    False,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 21:",
    cell22_queue_still_21,
)

print(
    "Next Pilot unchanged:",
    cell22_next_pilot_unchanged,
)

print(
    "Base Seed unchanged:",
    cell22_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell22_full_572_blocked,
)

print(
    "Commit / Push performed:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell22_failures
)


print(
    "\nCODE CELL 22 GATE:",
    (
        "PASS"
        if cell22_gate
        else "FAIL"
    ),
)


assert cell22_gate, (
    "Answer ACCEPT / Citation / persistent Registry sync "
    "failed. Repair this SAME Code Cell 22 before final "
    "Human Citation review."
)

=== ANSWER ACCEPTED + CITATION + PERSISTENT REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟

=== ACCEPTED ANSWER ===
بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود 50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا مورد بررسی گسترده قرار گرفته است؛ در سال 2014 نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای پژوهشگران و سیاست‌گذاران فراهم کرد.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:c0e5afc01b8d73443666ddbbcdd5ad6e
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document: website:596080EF
Role: SOURCE_DOCUMENT
Chars: (0, 671)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/596080EF
URL: https://www.fintelligence.ir/Blog/Detail/596080EF/%D8%B3%D

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 23 — Citation ACCEPT، تکمیل Pilot، Sync Registry و آماده‌سازی Pilot بعدی

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Pilot فعلی از نظر Question + Expected Answer + Citation تکمیل می‌شود؛
- پرونده‌ی کامل Human Review آن به Completed Cases منتقل می‌شود؛
- وضعیت Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Queue از 21 به 20 کاهش پیدا می‌کند؛
- Pilot بعدی انتخاب می‌شود؛
- Evidence اصلی Pilot بعدی برای شروع Human Question Review نمایش داده می‌شود.

Notebook مخصوص دکتر رودی بازنویسی نمی‌شود؛ هنگام اجرا آخرین JSON را می‌خواند.

Full 572 Generation همچنان Blocked است.

</div>

In [48]:
# Code Cell 23

cell23_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell22_gate") is not True:
    cell23_failures.append(
        "CELL22_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell22_current_session"
    ),
    dict,
):
    cell23_failures.append(
        "CELL22_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell22_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell23_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell23_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL23_COMPLETED_PILOT_ID = (
    cell19_next_pilot_id
)

CELL23_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL23_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "after reviewing the validated source locator."
)


cell23_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL23_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell22_citation_mapping_id,

    "target_evidence_id":
        cell19_next_target_evidence_id,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL23_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell23_pre_advance_session = (
    cell13_apply_event(
        cell22_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL23_HUMAN_CITATION_NOTE,
    )
)


cell23_citation_accept_valid = all([
    cell23_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell23_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell23_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell23_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell23_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell23_completed_human_review_case = (
    copy.deepcopy(
        cell22_active_case
    )
)


cell23_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell23_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell23_citation_human_review
)


cell23_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell23_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell23_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell23_completed_human_review_case
    )
)


# ============================================================
# 6) Queue advancement
# ============================================================

cell23_parent_remaining_ids = copy.deepcopy(
    cell19_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell23_parent_remaining_queue = copy.deepcopy(
    cell19_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell23_queue_preconditions_valid = all([
    len(
        cell23_parent_remaining_ids
    )
    ==
    21,

    len(
        cell23_parent_remaining_queue
    )
    ==
    21,

    cell23_parent_remaining_ids[
        0
    ]
    ==
    CELL23_COMPLETED_PILOT_ID,

    cell23_citation_accept_valid,
])


if cell23_queue_preconditions_valid:

    cell23_remaining_pilot_ids = copy.deepcopy(
        cell23_parent_remaining_ids[
            1:
        ]
    )

    cell23_remaining_pilot_queue = copy.deepcopy(
        cell23_parent_remaining_queue[
            1:
        ]
    )

else:

    cell23_remaining_pilot_ids = []
    cell23_remaining_pilot_queue = []


cell23_next_pilot_id = (
    cell23_remaining_pilot_ids[
        0
    ]

    if cell23_remaining_pilot_ids

    else None
)


cell23_next_pilot_queue_entry = (
    copy.deepcopy(
        cell23_remaining_pilot_queue[
            0
        ]
    )

    if cell23_remaining_pilot_queue

    else None
)


cell23_queue_advanced_valid = all([
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20,

    len(
        cell23_remaining_pilot_queue
    )
    ==
    20,

    CELL23_COMPLETED_PILOT_ID
    not in
    cell23_remaining_pilot_ids,

    isinstance(
        cell23_next_pilot_id,
        str,
    ),

    cell23_next_pilot_id
    !=
    CELL23_COMPLETED_PILOT_ID,
])


# ============================================================
# 7) Completed workflow session
# ============================================================

cell23_completed_session = copy.deepcopy(
    cell23_pre_advance_session
)


cell23_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell23_completed_session[
    "queue_advance_permitted"
] = False


cell23_completed_session[
    "queue_advance_performed"
] = True


cell23_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 8) New authoritative Continuation State
# ============================================================

cell23_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v2",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell19_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL23_COMPLETED_PILOT_ID,

    "just_completed_case_sha256":
        cell23_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell23_remaining_pilot_queue
        ),

    "remaining_count":
        20,

    "next_pilot_id":
        cell23_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell23_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL23_FULL_572_BLOCK_STATUS,
}


cell23_continuation_state_sha256 = (
    cell10_stable_sha(
        cell23_continuation_state
    )
)


# ============================================================
# 9) Resolve next Pilot packet
# ============================================================

cell23_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell23_next_pilot_packet = copy.deepcopy(
    cell23_pilot_by_id.get(
        cell23_next_pilot_id
    )
)


cell23_next_packet_valid = isinstance(
    cell23_next_pilot_packet,
    dict,
)


# ============================================================
# 10) Resolve next identity chain
# ============================================================

cell23_next_golden_case_id = None
cell23_next_task_draft_id = None
cell23_next_target_evidence_id = None


if cell23_next_packet_valid:

    try:

        cell23_next_golden_case_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "golden_case",
            )
        )


        cell23_next_task_draft_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "task_draft",
            )
        )


        cell23_next_target_evidence_id = (
            cell13_exact_single_id(
                cell23_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell23_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell23_next_identity_valid = all([
    isinstance(
        cell23_next_golden_case_id,
        str,
    ),

    isinstance(
        cell23_next_task_draft_id,
        str,
    ),

    isinstance(
        cell23_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 11) Resolve next support records
# ============================================================

cell23_next_golden_case = None
cell23_next_task_plan = None
cell23_next_target_evidence = None


if cell23_next_identity_valid:

    try:

        cell23_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell23_next_golden_case_id
            ]
        )


        cell23_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell23_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell23_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell23_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell23_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 12) Classification
# ============================================================

def cell23_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )


    if isinstance(
        match,
        dict,
    ):
        return match.get(
            "value"
        )


    return None


cell23_next_source_type = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell23_next_stratum = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell23_next_task_family = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "task_family",
    ],
)


cell23_next_question_archetype = cell23_scalar(
    cell23_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 13) Extract next Evidence text
# ============================================================

cell23_raw_target_parts = []


if cell23_next_packet_valid:

    candidate_parts = (
        cell23_next_pilot_packet.get(
            "target_text_parts"
        )
    )


    if isinstance(
        candidate_parts,
        list,
    ):
        cell23_raw_target_parts = copy.deepcopy(
            candidate_parts
        )


if (
    not cell23_raw_target_parts
    and
    isinstance(
        cell23_next_target_evidence,
        dict,
    )
):

    candidate_parts = (
        cell23_next_target_evidence.get(
            "target_parts"
        )
    )


    if isinstance(
        candidate_parts,
        list,
    ):
        cell23_raw_target_parts = copy.deepcopy(
            candidate_parts
        )


cell23_next_target_text_parts = []


for index, part in enumerate(
    cell23_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )


        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell23_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell23_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell23_next_target_text_parts
    )
)


cell23_next_target_text_available = bool(
    cell23_next_combined_target_text.strip()
)


# ============================================================
# 14) Build next Pilot workspace
# ============================================================

cell23_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell23_next_pilot_id,

    "golden_case_id":
        cell23_next_golden_case_id,

    "task_draft_id":
        cell23_next_task_draft_id,

    "target_evidence_id":
        cell23_next_target_evidence_id,

    "source_type":
        cell23_next_source_type,

    "stratum":
        cell23_next_stratum,

    "task_family":
        cell23_next_task_family,

    "question_archetype":
        cell23_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell23_next_target_text_parts
        ),

    "combined_target_text":
        cell23_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell23_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell23_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell23_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL23_FULL_572_BLOCK_STATUS,
}


cell23_next_workspace_ready = all([
    cell23_next_packet_valid,
    cell23_next_identity_valid,
    isinstance(
        cell23_next_golden_case,
        dict,
    ),
    isinstance(
        cell23_next_task_plan,
        dict,
    ),
    isinstance(
        cell23_next_target_evidence,
        dict,
    ),
    cell23_next_target_text_available,
])


# ============================================================
# 15) Update Human Review Registry
# ============================================================

cell23_human_review_registry = copy.deepcopy(
    cell22_human_review_registry
)


cell23_completed_cases = copy.deepcopy(
    cell23_human_review_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell23_completed_cases.append(
    copy.deepcopy(
        cell23_completed_human_review_case
    )
)


cell23_human_review_registry[
    "completed_human_review_cases"
] = cell23_completed_cases


cell23_human_review_registry[
    "active_workspace"
] = copy.deepcopy(
    cell23_next_pilot_workspace
)


cell23_human_review_registry[
    "remaining_pilot_count"
] = 20


# ============================================================
# 16) Persistent Registry payload
# ============================================================

cell23_persistent_registry_payload = copy.deepcopy(
    cell22_persisted_registry
)


cell23_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell23_completed_cases
)


cell23_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell23_next_pilot_workspace
)


cell23_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell23_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell23_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell23_persistent_registry_payload[
    "author_review"
][
    "cases_ready_for_review"
] = len(
    cell23_completed_cases
)


# ============================================================
# 17) Persist
# ============================================================

cell23_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell23_persistent_registry_payload
    )
)


cell23_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell23_persisted_active_case = (
    cell23_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell23_persistence_valid = all([
    len(
        cell23_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    2,

    cell23_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,

    cell23_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell23_next_pilot_id,

    cell23_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell23_next_pilot_id,
])


# ============================================================
# 18) Safety
# ============================================================

cell23_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell23_full_572_blocked = (
    cell23_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL23_FULL_572_BLOCK_STATUS
)


# ============================================================
# 19) Final Gate
# ============================================================

cell23_gate = all([
    len(
        cell23_failures
    )
    ==
    0,

    cell23_citation_accept_valid,

    cell23_queue_preconditions_valid,

    cell23_queue_advanced_valid,

    cell23_next_workspace_ready,

    cell23_persistence_valid,

    cell23_base_seed_unchanged,

    cell23_full_572_blocked,
])


# ============================================================
# 20) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + "
    "REGISTRY SYNC + NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL23_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell20_question_text,
)

print(
    "Answer:",
    cell20_expected_answer_text,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell23_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell23_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell23_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL23_COMPLETED_PILOT_ID
        not in
        cell23_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell23_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell23_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Remaining Pilots:",
    cell23_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell23_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Registry sync valid:",
    cell23_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell23_next_pilot_id,
)

print(
    "Golden Case:",
    cell23_next_golden_case_id,
)

print(
    "Task Draft:",
    cell23_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell23_next_target_evidence_id,
)

print(
    "Source type:",
    cell23_next_source_type,
)

print(
    "Stratum:",
    cell23_next_stratum,
)

print(
    "Task family:",
    cell23_next_task_family,
)

print(
    "Question archetype:",
    cell23_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell23_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell23_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell23_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell23_failures
)


print(
    "\nCODE CELL 23 GATE:",
    (
        "PASS"
        if cell23_gate
        else "FAIL"
    ),
)


assert cell23_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 23 before continuing."
)

=== PILOT COMPLETED + REGISTRY SYNC + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:083ad832854722e126c05d0b89d00cb1
Question: سواد مالی در اروپا چقدر مورد توجه قرار گرفته و چند درصد از اروپایی‌ها سواد مالی مناسبی دارند؟
Answer: بر اساس متن، تنها نیمی از اروپایی‌ها، یعنی حدود 50 درصد، سواد مالی مناسبی دارند. سواد مالی در اروپا مورد بررسی گسترده قرار گرفته است؛ در سال 2014 نظرسنجی جهانی سواد مالی یکی از ابزارهای مقایسه سطح سواد مالی کشورهای اروپایی بود و در بهار 2023 نیز کمیسیون اروپا نظرسنجی جامعی در هر 27 کشور عضو اتحادیه اروپا انجام داد. این سنجش جدید به دلیل پوشش کامل کشورهای عضو، اطلاعات قابل‌اتکاتری برای پژوهشگران و سیاست‌گذاران فراهم کرد.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 21
After: 20
Completed Pilot removed: True
Next Pilot: question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 24 — Question ACCEPT + Expected Answer Draft + Persistent Sync

سؤال این Pilot پس از Human Review پذیرفته شد:

**«داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟»**

در این مرحله:

- Question Draft ثبت و با تصمیم انسانی `ACCEPT` نهایی می‌شود؛
- Expected Answer فقط از Evidence همین Pilot ساخته می‌شود؛
- Answer برای Human Review متوقف می‌شود؛
- Human Review Registry روی دیسک Sync می‌شود؛
- Citation هنوز ساخته نمی‌شود؛
- Queue همچنان 20 باقی می‌ماند؛
- Full 572 Generation همچنان Blocked است.

</div>

In [50]:
# Code Cell 24 — REPLACEMENT

cell24_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell23_gate") is not True:
    cell24_failures.append(
        "CELL23_NOT_PASS"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell24_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL24_PILOT_ID = (
    cell23_next_pilot_id
)

CELL24_GOLDEN_CASE_ID = (
    cell23_next_golden_case_id
)

CELL24_TASK_DRAFT_ID = (
    cell23_next_task_draft_id
)

CELL24_TARGET_EVIDENCE_ID = (
    cell23_next_target_evidence_id
)

CELL24_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


cell24_identity_valid = all([
    isinstance(
        CELL24_PILOT_ID,
        str,
    ),

    isinstance(
        CELL24_GOLDEN_CASE_ID,
        str,
    ),

    isinstance(
        CELL24_TASK_DRAFT_ID,
        str,
    ),

    isinstance(
        CELL24_TARGET_EVIDENCE_ID,
        str,
    ),

    cell23_next_pilot_workspace.get(
        "workflow_state"
    )
    ==
    "QUESTION_DRAFT_PENDING",
])


# ============================================================
# 3) Start compact workflow session
# ============================================================

cell24_pilot_bundle = {
    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,
}


cell24_initial_session = (
    cell13_build_session(
        cell24_pilot_bundle
    )
)


cell24_initial_session_valid = (
    cell24_initial_session.get(
        "current_state"
    )
    ==
    "QUESTION_DRAFT_PENDING"
)


# ============================================================
# 4) Accepted Question
# ============================================================

CELL24_QUESTION_TEXT = (
    "داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها "
    "برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟"
)


cell24_question_text = (
    CELL24_QUESTION_TEXT.strip()
)


cell24_question_text_sha256 = (
    hashlib.sha256(
        cell24_question_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


cell24_question_draft_id = (
    cell12_question_id_function(
        pilot_case_id=
            CELL24_PILOT_ID,

        question_text_sha256=
            cell24_question_text_sha256,
    )
)


# ============================================================
# 5) Persian-safe normalization helpers
# ============================================================

def cell24_fa_surface(
    value,
):

    text = cell15_normalize_text(
        value
    )

    replacements = {
        "ۀ": "ه",
        "ة": "ه",
        "ؤ": "و",
        "ئ": "ی",
        "أ": "ا",
        "إ": "ا",
        "ٱ": "ا",
    }

    for old, new in replacements.items():
        text = text.replace(
            old,
            new,
        )

    text = text.replace(
        "\u200c",
        " ",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def cell24_has_any(
    text,
    terms,
):

    return any(
        term in text
        for term
        in terms
    )


cell24_normalized_question = (
    cell24_fa_surface(
        cell24_question_text
    )
)


cell24_normalized_evidence = (
    cell24_fa_surface(
        cell23_next_combined_target_text
    )
)


# ============================================================
# 6) Question grounding
# ============================================================

cell24_question_grounding_checks = {
    "saving":
        all([
            cell24_has_any(
                cell24_normalized_question,
                [
                    "پس انداز",
                    "پس‌انداز",
                ],
            ),

            cell24_has_any(
                cell24_normalized_evidence,
                [
                    "پس انداز",
                    "پس‌انداز",
                ],
            ),
        ]),

    "goal":
        (
            "هدف"
            in
            cell24_normalized_question
            and
            "هدف"
            in
            cell24_normalized_evidence
        ),

    "motivation":
        (
            "انگیز"
            in
            cell24_normalized_question
            and
            "انگیز"
            in
            cell24_normalized_evidence
        ),

    "money":
        (
            "پول"
            in
            cell24_normalized_question
            and
            "پول"
            in
            cell24_normalized_evidence
        ),

    "desired_outcome":
        (
            "خواسته"
            in
            cell24_normalized_question
            and
            "خواسته"
            in
            cell24_normalized_evidence
        ),
}


cell24_question_machine_pass = all(
    cell24_question_grounding_checks.values()
)


# ============================================================
# 7) Question Draft Record
# ============================================================

cell24_question_draft_record = {
    "record_version":
        "question_draft_record_v1",

    "question_draft_id":
        cell24_question_draft_id,

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "source_type":
        cell23_next_source_type,

    "stratum":
        cell23_next_stratum,

    "task_family":
        cell23_next_task_family,

    "question_archetype":
        cell23_next_question_archetype,

    "question_text":
        cell24_question_text,

    "question_text_sha256":
        cell24_question_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell24_question_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "ACCEPT",

    "external_knowledge_used":
        False,

    "full_572_generation":
        CELL24_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 8) Register Question
# ============================================================

cell24_question_review_session = (
    cell13_apply_event(
        cell24_initial_session,
        "QUESTION_DRAFT_REGISTERED",
        artifact=
            cell24_question_draft_record,
    )
)


CELL24_HUMAN_QUESTION_NOTE = (
    "Human Reviewer accepted the revised Question wording "
    "as a source-grounded and audience-appropriate formulation."
)


cell24_answer_pending_session = (
    cell13_apply_event(
        cell24_question_review_session,
        "QUESTION_ACCEPTED",
        human_note=
            CELL24_HUMAN_QUESTION_NOTE,
    )
)


cell24_question_accept_valid = (
    cell24_answer_pending_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_DRAFT_PENDING"
)


# ============================================================
# 9) Expected Answer Draft
# ============================================================

CELL24_EXPECTED_ANSWER_TEXT = (
    "داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد "
    "تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند "
    "و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را "
    "پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت "
    "بخرد، در حالی که دیگری همه پولش را خرج کرده بود."
)


cell24_expected_answer_text = (
    CELL24_EXPECTED_ANSWER_TEXT.strip()
)


cell24_expected_answer_text_sha256 = (
    hashlib.sha256(
        cell24_expected_answer_text.encode(
            "utf-8"
        )
    ).hexdigest()
)


# ============================================================
# 10) Expected Answer grounding
# ============================================================

cell24_normalized_answer = (
    cell24_fa_surface(
        cell24_expected_answer_text
    )
)


cell24_answer_grounding_checks = {
    "saving_goal":
        (
            "هدف"
            in
            cell24_normalized_evidence
            and
            "هدف"
            in
            cell24_normalized_answer
        ),

    "motivation":
        (
            "انگیز"
            in
            cell24_normalized_evidence
            and
            "انگیز"
            in
            cell24_normalized_answer
        ),

    "earn_more_money":
        (
            "پول بیشتر"
            in
            cell24_normalized_evidence
            and
            "پول بیشتر"
            in
            cell24_normalized_answer
        ),

    "reach_goal_sooner":
        (
            "زودتر"
            in
            cell24_normalized_evidence
            and
            "زودتر"
            in
            cell24_normalized_answer
        ),

    "buy_desired_things":
        (
            "چیزهایی"
            in
            cell24_normalized_evidence
            and
            "چیزهایی"
            in
            cell24_normalized_answer
        ),

    "spent_all_money":
        all([
            cell24_has_any(
                cell24_normalized_evidence,
                [
                    "همه پول",
                    "همه پول هایش",
                    "همه پولهایش",
                ],
            ),

            cell24_has_any(
                cell24_normalized_answer,
                [
                    "همه پول",
                    "همه پولش",
                ],
            ),
        ]),
}


cell24_answer_machine_pass = all(
    cell24_answer_grounding_checks.values()
)


# ============================================================
# 11) Expected Answer Draft ID
# ============================================================

cell24_answer_identity_payload = {
    "identity_version":
        "expected_answer_draft_identity_v1",

    "pilot_case_id":
        CELL24_PILOT_ID,

    "question_draft_id":
        cell24_question_draft_id,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "expected_answer_text_sha256":
        cell24_expected_answer_text_sha256,
}


cell24_expected_answer_draft_id = (
    "expected-answer-draft:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell24_answer_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 12) Expected Answer Record
# ============================================================

cell24_expected_answer_record = {
    "record_version":
        "expected_answer_draft_record_v1",

    "expected_answer_draft_id":
        cell24_expected_answer_draft_id,

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "question_draft_id":
        cell24_question_draft_id,

    "question_text":
        cell24_question_text,

    "expected_answer_text":
        cell24_expected_answer_text,

    "expected_answer_text_sha256":
        cell24_expected_answer_text_sha256,

    "source_evidence_policy":
        "CURRENT_VALIDATED_TARGET_EVIDENCE_ONLY",

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell24_answer_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "external_knowledge_used":
        False,

    "human_context_used_as_evidence":
        False,

    "full_572_generation":
        CELL24_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Expected Answer
# ============================================================

cell24_current_session = (
    cell13_apply_event(
        cell24_answer_pending_session,
        "EXPECTED_ANSWER_REGISTERED",
        artifact=
            cell24_expected_answer_record,
    )
)


cell24_answer_registered_valid = all([
    cell24_current_session.get(
        "current_state"
    )
    ==
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING",

    isinstance(
        cell24_current_session.get(
            "artifacts",
            {}
        ).get(
            "expected_answer"
        ),
        dict,
    ),

    cell24_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Human Review Case
# ============================================================

cell24_active_human_review_case = {
    "human_review_case_version":
        "human_review_case_v1",

    "pilot_case_id":
        CELL24_PILOT_ID,

    "golden_case_id":
        CELL24_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL24_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL24_TARGET_EVIDENCE_ID,

    "source_evidence":
        copy.deepcopy(
            cell23_next_target_text_parts
        ),

    "model_drafts":
        [
            {
                "revision":
                    1,

                "question_draft_id":
                    cell24_question_draft_id,

                "question_text":
                    cell24_question_text,

                "machine_validation":
                    (
                        "PASS"
                        if cell24_question_machine_pass
                        else "FAIL"
                    ),

                "human_decision":
                    "ACCEPT",
            }
        ],

    "human_reviews":
        [
            {
                "review_stage":
                    "QUESTION",

                "human_decision":
                    "ACCEPT",

                "human_note":
                    CELL24_HUMAN_QUESTION_NOTE,

                "human_context_evidence_status":
                    "NON_EVIDENCE",
            }
        ],

    "final_question_draft_id":
        cell24_question_draft_id,

    "expected_answer_review":
        {
            "expected_answer_draft_id":
                cell24_expected_answer_draft_id,

            "expected_answer_text":
                cell24_expected_answer_text,

            "machine_validation_status":
                cell24_expected_answer_record[
                    "machine_validation_status"
                ],

            "human_decision":
                "PENDING",
        },

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "final_status":
        "QUESTION_ACCEPTED_EXPECTED_ANSWER_PENDING_REVIEW",
}


# ============================================================
# 15) Update persistent Registry
# ============================================================

cell24_persistent_registry_payload = copy.deepcopy(
    cell23_persisted_registry
)


cell24_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell24_active_human_review_case
)


cell24_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell24_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL24_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell24_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell24_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell24_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell24_persisted_active_case = (
    cell24_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell24_persistence_valid = all([
    cell24_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL24_PILOT_ID,

    cell24_persisted_active_case.get(
        "final_question_draft_id"
    )
    ==
    cell24_question_draft_id,

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "expected_answer_draft_id"
    )
    ==
    cell24_expected_answer_draft_id,

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",

    cell24_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,
])


# ============================================================
# 17) Safety
# ============================================================

cell24_queue_still_20 = (
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20
)


cell24_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell24_full_572_blocked = (
    cell24_expected_answer_record[
        "full_572_generation"
    ]
    ==
    CELL24_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell24_gate = all([
    len(
        cell24_failures
    )
    ==
    0,

    cell24_identity_valid,

    cell24_initial_session_valid,

    cell24_question_machine_pass,

    cell24_question_accept_valid,

    cell24_answer_machine_pass,

    cell24_answer_registered_valid,

    cell24_persistence_valid,

    cell24_queue_still_20,

    cell24_base_seed_unchanged,

    cell24_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== SOURCE EVIDENCE ==="
)

print(
    cell23_next_combined_target_text
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell24_question_text
)

print(
    "Question Draft ID:",
    cell24_question_draft_id,
)

print(
    "Human Decision:",
    "ACCEPT",
)

print(
    "Question Machine PASS:",
    cell24_question_machine_pass,
)


print(
    "\n=== QUESTION GROUNDING ==="
)

for name, status in (
    cell24_question_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== EXPECTED ANSWER DRAFT ==="
)

print(
    cell24_expected_answer_text
)

print(
    "Expected Answer Draft ID:",
    cell24_expected_answer_draft_id,
)

print(
    "Machine PASS:",
    cell24_answer_machine_pass,
)


print(
    "\n=== ANSWER GROUNDING ==="
)

for name, status in (
    cell24_answer_grounding_checks.items()
):
    print(
        f"{name}:",
        status,
    )


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell24_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Expected Answer Decision:",
    "PENDING",
)

print(
    "Citation:",
    None,
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Active Pilot:",
    cell24_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persisted Question Machine Status:",
    (
        cell24_persisted_active_case.get(
            "model_drafts",
            [{}],
        )[-1].get(
            "machine_validation"
        )
    ),
)

print(
    "Persisted Answer Machine Status:",
    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "machine_validation_status"
    ),
)

print(
    "Persisted Answer Decision:",
    cell24_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell24_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 20:",
    cell24_queue_still_20,
)

print(
    "Base Seed unchanged:",
    cell24_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell24_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell24_failures
)


print(
    "\nCODE CELL 24 GATE:",
    (
        "PASS"
        if cell24_gate
        else "FAIL"
    ),
)


assert cell24_gate, (
    "Question ACCEPT / Expected Answer Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 24 "
    "before Expected Answer Human Review."
)

=== QUESTION ACCEPTED + EXPECTED ANSWER DRAFT + REGISTRY SYNC ===

=== SOURCE EVIDENCE ===
یکی از آن‌ها پس‌انداز می‌کند و دیگری همۀ پول‌هایش را خرج می‌کند تا روزی که به فروشگاهی بزرگ می‌روند و خواهر موجود کوچولو با پس‌اندازش چیزهایی می‌خرد که دوست دارد.
5.در راه رسیدن به هدف پس‌اندازی، بچه‌ها انگیزۀ به دست آوردن پول بیشتر برای رسیدن زودتر به خواسته‌شان را دارند و ممکن است خودشان بخواهند دست به کار شوند و پول پس‌انداز کنند.

=== ACCEPTED QUESTION ===
داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟
Question Draft ID: question-draft:d14395bbe4c2f16bfba208eadab41590
Human Decision: ACCEPT
Question Machine PASS: True

=== QUESTION GROUNDING ===
saving: True
goal: True
motivation: True
money: True
desired_outcome: True

=== EXPECTED ANSWER DRAFT ===
داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود،

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 25 — Expected Answer ACCEPT + Citation Draft + Persistent Sync

Expected Answer این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- تصمیم انسانی `ACCEPT` برای Expected Answer ثبت می‌شود؛
- Citation فقط از Target Evidence اعتبارسنجی‌شده ساخته می‌شود؛
- Document ID، Locator و محدوده‌های متنی از منبع موجود کپی می‌شوند و هیچ Locator جدیدی ساخته نمی‌شود؛
- Citation به‌صورت ماشینی اعتبارسنجی می‌شود؛
- Workflow در وضعیت `CITATION_HUMAN_REVIEW_PENDING` متوقف می‌شود؛
- Registry دائمی Human Review روی دیسک Sync می‌شود.

در این مرحله Pilot هنوز کامل نشده و Queue همچنان 20 باقی می‌ماند.

</div>

In [51]:
# Code Cell 25

cell25_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell24_gate") is not True:
    cell25_failures.append(
        "CELL24_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell24_current_session"
    ),
    dict,
):
    cell25_failures.append(
        "CELL24_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell24_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "EXPECTED_ANSWER_HUMAN_REVIEW_PENDING"
):
    cell25_failures.append(
        "EXPECTED_ANSWER_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell25_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot identity
# ============================================================

CELL25_PILOT_ID = (
    cell23_next_pilot_id
)

CELL25_GOLDEN_CASE_ID = (
    cell23_next_golden_case_id
)

CELL25_TASK_DRAFT_ID = (
    cell23_next_task_draft_id
)

CELL25_TARGET_EVIDENCE_ID = (
    cell23_next_target_evidence_id
)

CELL25_QUESTION_DRAFT_ID = (
    cell24_question_draft_id
)

CELL25_EXPECTED_ANSWER_DRAFT_ID = (
    cell24_expected_answer_draft_id
)

CELL25_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Expected Answer identity validation
# ============================================================

cell25_expected_answer_record = copy.deepcopy(
    cell24_current_session[
        "artifacts"
    ][
        "expected_answer"
    ]
)


cell25_expected_answer_identity_valid = all([
    cell25_expected_answer_record.get(
        "expected_answer_draft_id"
    )
    ==
    CELL25_EXPECTED_ANSWER_DRAFT_ID,

    cell25_expected_answer_record.get(
        "pilot_case_id"
    )
    ==
    CELL25_PILOT_ID,

    cell25_expected_answer_record.get(
        "question_draft_id"
    )
    ==
    CELL25_QUESTION_DRAFT_ID,

    cell25_expected_answer_record.get(
        "target_evidence_id"
    )
    ==
    CELL25_TARGET_EVIDENCE_ID,

    cell25_expected_answer_record.get(
        "machine_validation_status"
    )
    ==
    "PASS_PENDING_HUMAN_ADJUDICATION",
])


# ============================================================
# 4) Human ACCEPT for Expected Answer
# ============================================================

CELL25_HUMAN_ANSWER_NOTE = (
    "Human Reviewer accepted the Expected Answer "
    "after reviewing it against the validated source evidence."
)


cell25_answer_human_review = {
    "review_record_version":
        "human_expected_answer_review_record_v1",

    "pilot_case_id":
        CELL25_PILOT_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL25_HUMAN_ANSWER_NOTE,

    "human_context_used_as_evidence":
        False,

    "external_knowledge_used":
        False,
}


# ============================================================
# 5) Apply Expected Answer ACCEPT
# ============================================================

cell25_citation_pending_session = (
    cell13_apply_event(
        cell24_current_session,
        "EXPECTED_ANSWER_ACCEPTED",
        human_note=
            CELL25_HUMAN_ANSWER_NOTE,
    )
)


cell25_answer_accept_valid = all([
    cell25_citation_pending_session.get(
        "current_state"
    )
    ==
    "CITATION_MAPPING_PENDING",

    len(
        cell25_citation_pending_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    2,

    cell25_citation_pending_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "EXPECTED_ANSWER_ACCEPTED",

    cell25_citation_pending_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 6) Read validated Target Evidence parts
# ============================================================

cell25_target_parts = copy.deepcopy(
    cell23_next_target_evidence.get(
        "target_parts",
        [],
    )
)


cell25_target_parts_valid = all([
    isinstance(
        cell25_target_parts,
        list,
    ),

    len(
        cell25_target_parts
    )
    >
    0,
])


# ============================================================
# 7) Build Citation entries
# ============================================================

cell25_citation_entries = []


for index, part in enumerate(
    cell25_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    cell25_citation_entries.append({
        "citation_index":
            index,

        "target_evidence_id":
            CELL25_TARGET_EVIDENCE_ID,

        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "char_count":
            part.get(
                "char_count"
            ),

        "token_count":
            part.get(
                "token_count"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),

        "attribution":
            copy.deepcopy(
                part.get(
                    "attribution"
                )
            ),

        "locator_origin":
            "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    })


# ============================================================
# 8) Exact projection validation
# ============================================================

cell25_source_projection = [
    {
        "document_id":
            part.get(
                "document_id"
            ),

        "source_id":
            part.get(
                "source_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                part.get(
                    "locator",
                    {},
                )
            ),
    }

    for part
    in cell25_target_parts

    if isinstance(
        part,
        dict,
    )
]


cell25_citation_projection = [
    {
        "document_id":
            entry.get(
                "document_id"
            ),

        "source_id":
            entry.get(
                "source_id"
            ),

        "role":
            entry.get(
                "role"
            ),

        "char_start":
            entry.get(
                "char_start"
            ),

        "char_end":
            entry.get(
                "char_end"
            ),

        "locator":
            copy.deepcopy(
                entry.get(
                    "locator",
                    {},
                )
            ),
    }

    for entry
    in cell25_citation_entries
]


cell25_no_locator_fabrication = (
    cell25_source_projection
    ==
    cell25_citation_projection
)


# ============================================================
# 9) Validate Citation against displayed Evidence
# ============================================================

cell25_displayed_parts = copy.deepcopy(
    cell23_next_target_text_parts
)


cell25_displayed_by_document = {
    row.get(
        "document_id"
    ):
        row

    for row
    in cell25_displayed_parts

    if isinstance(
        row,
        dict,
    )
}


cell25_citation_validation_rows = []


for entry in cell25_citation_entries:

    document_id = entry.get(
        "document_id"
    )


    displayed = (
        cell25_displayed_by_document.get(
            document_id
        )
    )


    locator = (
        entry.get(
            "locator"
        )
        or
        {}
    )


    canonical_url = locator.get(
        "canonical_url"
    )


    url = locator.get(
        "url"
    )


    row_valid = all([
        isinstance(
            document_id,
            str,
        ),

        isinstance(
            displayed,
            dict,
        ),

        entry.get(
            "char_start"
        )
        ==
        displayed.get(
            "char_start"
        ),

        entry.get(
            "char_end"
        )
        ==
        displayed.get(
            "char_end"
        ),

        entry.get(
            "role"
        )
        ==
        displayed.get(
            "role"
        ),

        (
            canonical_url is None
            or
            (
                isinstance(
                    canonical_url,
                    str,
                )
                and
                canonical_url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        (
            url is None
            or
            (
                isinstance(
                    url,
                    str,
                )
                and
                url.startswith(
                    (
                        "http://",
                        "https://",
                    )
                )
            )
        ),

        entry.get(
            "locator_origin"
        )
        ==
        "COPIED_FROM_VALIDATED_TARGET_EVIDENCE",
    ])


    cell25_citation_validation_rows.append({
        "citation_index":
            entry.get(
                "citation_index"
            ),

        "document_id":
            document_id,

        "range_match":
            (
                isinstance(
                    displayed,
                    dict,
                )
                and
                entry.get(
                    "char_start"
                )
                ==
                displayed.get(
                    "char_start"
                )
                and
                entry.get(
                    "char_end"
                )
                ==
                displayed.get(
                    "char_end"
                )
            ),

        "role_match":
            (
                isinstance(
                    displayed,
                    dict,
                )
                and
                entry.get(
                    "role"
                )
                ==
                displayed.get(
                    "role"
                )
            ),

        "valid":
            row_valid,
    })


cell25_all_entries_valid = all([
    len(
        cell25_citation_entries
    )
    >
    0,

    len(
        cell25_citation_entries
    )
    ==
    len(
        cell25_displayed_parts
    ),

    all(
        row.get(
            "valid"
        )
        is True

        for row
        in cell25_citation_validation_rows
    ),
])


# ============================================================
# 10) Citation Mapping ID
# ============================================================

cell25_citation_identity_payload = {
    "identity_version":
        "citation_mapping_identity_v1",

    "pilot_case_id":
        CELL25_PILOT_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "citation_projection":
        copy.deepcopy(
            cell25_citation_projection
        ),
}


cell25_citation_mapping_id = (
    "citation-mapping:"
    +
    hashlib.sha256(
        cell10_canonical_json(
            cell25_citation_identity_payload
        ).encode(
            "utf-8"
        )
    ).hexdigest()[
        :32
    ]
)


# ============================================================
# 11) Citation Machine Validation
# ============================================================

cell25_citation_machine_validation = {
    "target_parts_present":
        cell25_target_parts_valid,

    "citation_entries_present":
        (
            len(
                cell25_citation_entries
            )
            >
            0
        ),

    "all_target_parts_covered":
        (
            len(
                cell25_citation_entries
            )
            ==
            len(
                cell25_displayed_parts
            )
        ),

    "all_entries_valid":
        cell25_all_entries_valid,

    "locator_fabrication_absent":
        cell25_no_locator_fabrication,

    "target_evidence_identity_match":
        all(
            entry.get(
                "target_evidence_id"
            )
            ==
            CELL25_TARGET_EVIDENCE_ID

            for entry
            in cell25_citation_entries
        ),

    "external_locator_lookup_used":
        False,
}


cell25_citation_machine_pass = all([
    cell25_citation_machine_validation[
        "target_parts_present"
    ],

    cell25_citation_machine_validation[
        "citation_entries_present"
    ],

    cell25_citation_machine_validation[
        "all_target_parts_covered"
    ],

    cell25_citation_machine_validation[
        "all_entries_valid"
    ],

    cell25_citation_machine_validation[
        "locator_fabrication_absent"
    ],

    cell25_citation_machine_validation[
        "target_evidence_identity_match"
    ],
])


# ============================================================
# 12) Citation Mapping Record
# ============================================================

cell25_citation_mapping_record = {
    "record_version":
        "citation_mapping_record_v1",

    "citation_mapping_id":
        cell25_citation_mapping_id,

    "pilot_case_id":
        CELL25_PILOT_ID,

    "golden_case_id":
        CELL25_GOLDEN_CASE_ID,

    "task_draft_id":
        CELL25_TASK_DRAFT_ID,

    "target_evidence_id":
        CELL25_TARGET_EVIDENCE_ID,

    "question_draft_id":
        CELL25_QUESTION_DRAFT_ID,

    "expected_answer_draft_id":
        CELL25_EXPECTED_ANSWER_DRAFT_ID,

    "citation_entries":
        copy.deepcopy(
            cell25_citation_entries
        ),

    "citation_scope":
        "VALIDATED_TARGET_EVIDENCE_ONLY",

    "locator_source":
        "EXISTING_TARGET_EVIDENCE_LOCATORS",

    "external_locator_lookup_used":
        False,

    "machine_validation_status":
        (
            "PASS_PENDING_HUMAN_ADJUDICATION"
            if cell25_citation_machine_pass
            else "FAIL_REQUIRES_REVISION"
        ),

    "human_adjudication_status":
        "PENDING",

    "full_572_generation":
        CELL25_FULL_572_BLOCK_STATUS,
}


# ============================================================
# 13) Register Citation
# ============================================================

cell25_current_session = (
    cell13_apply_event(
        cell25_citation_pending_session,
        "CITATION_MAPPING_REGISTERED",
        artifact=
            cell25_citation_mapping_record,
    )
)


cell25_citation_registered_valid = all([
    cell25_current_session.get(
        "current_state"
    )
    ==
    "CITATION_HUMAN_REVIEW_PENDING",

    isinstance(
        cell25_current_session.get(
            "artifacts",
            {}
        ).get(
            "citation_mapping"
        ),
        dict,
    ),

    cell25_current_session[
        "artifacts"
    ][
        "citation_mapping"
    ].get(
        "citation_mapping_id"
    )
    ==
    cell25_citation_mapping_id,

    cell25_current_session.get(
        "queue_advance_performed"
    )
    is False,
])


# ============================================================
# 14) Update Human Review active case
# ============================================================

cell25_active_human_review_case = copy.deepcopy(
    cell24_active_human_review_case
)


cell25_active_human_review_case[
    "expected_answer_review"
][
    "human_decision"
] = "ACCEPT"


cell25_active_human_review_case[
    "expected_answer_review"
][
    "human_review"
] = copy.deepcopy(
    cell25_answer_human_review
)


cell25_active_human_review_case[
    "citation_review"
] = {
    "citation_mapping_id":
        cell25_citation_mapping_id,

    "citation_entries":
        copy.deepcopy(
            cell25_citation_entries
        ),

    "machine_validation_status":
        cell25_citation_mapping_record[
            "machine_validation_status"
        ],

    "human_decision":
        "PENDING",
}


cell25_active_human_review_case[
    "final_status"
] = (
    "QUESTION_AND_ANSWER_ACCEPTED_"
    "CITATION_PENDING_REVIEW"
)


# ============================================================
# 15) Persistent Registry update
# ============================================================

cell25_persistent_registry_payload = copy.deepcopy(
    cell24_persisted_registry
)


cell25_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell25_active_human_review_case
)


cell25_persistent_registry_payload[
    "remaining_pilot_count"
] = 20


cell25_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        CELL25_PILOT_ID,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell23_remaining_pilot_ids
        ),

    "remaining_count":
        20,

    "continuation_state_sha256":
        cell23_continuation_state_sha256,
}


cell25_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell25_persistent_registry_payload
    )
)


# ============================================================
# 16) Read back Registry
# ============================================================

cell25_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell25_persisted_active_case = (
    cell25_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell25_persistence_valid = all([
    cell25_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    CELL25_PILOT_ID,

    cell25_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "ACCEPT",

    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "citation_mapping_id"
    )
    ==
    cell25_citation_mapping_id,

    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    )
    ==
    "PENDING",

    cell25_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    20,
])


# ============================================================
# 17) Safety
# ============================================================

cell25_queue_still_20 = (
    len(
        cell23_remaining_pilot_ids
    )
    ==
    20
)


cell25_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell25_full_572_blocked = (
    cell25_citation_mapping_record[
        "full_572_generation"
    ]
    ==
    CELL25_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell25_gate = all([
    len(
        cell25_failures
    )
    ==
    0,

    cell25_expected_answer_identity_valid,

    cell25_answer_accept_valid,

    cell25_target_parts_valid,

    cell25_no_locator_fabrication,

    cell25_all_entries_valid,

    cell25_citation_machine_pass,

    cell25_citation_registered_valid,

    cell25_persistence_valid,

    cell25_queue_still_20,

    cell25_base_seed_unchanged,

    cell25_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== ANSWER ACCEPTED + CITATION DRAFT "
    "+ REGISTRY SYNC ==="
)


print(
    "\n=== ACCEPTED QUESTION ==="
)

print(
    cell24_question_text
)


print(
    "\n=== ACCEPTED ANSWER ==="
)

print(
    cell24_expected_answer_text
)

print(
    "Human Answer Decision:",
    "ACCEPT",
)


print(
    "\n=== CITATION DRAFT ==="
)

print(
    "Citation Mapping ID:",
    cell25_citation_mapping_id,
)

print(
    "Citation entry count:",
    len(
        cell25_citation_entries
    ),
)


for entry in cell25_citation_entries:

    print(
        "\n--- CITATION ENTRY",
        entry.get(
            "citation_index"
        ),
        "---"
    )

    print(
        "Document:",
        entry.get(
            "document_id"
        ),
    )

    print(
        "Role:",
        entry.get(
            "role"
        ),
    )

    print(
        "Chars:",
        (
            entry.get(
                "char_start"
            ),
            entry.get(
                "char_end"
            ),
        ),
    )

    print(
        "Canonical URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "canonical_url"
            )
        ),
    )

    print(
        "URL:",
        (
            entry.get(
                "locator",
                {}
            ).get(
                "url"
            )
        ),
    )


print(
    "\n=== CITATION MACHINE VALIDATION ==="
)

for name, status in (
    cell25_citation_machine_validation.items()
):

    print(
        f"{name}:",
        status,
    )


print(
    "Machine PASS:",
    cell25_citation_machine_pass,
)

print(
    "No locator fabrication:",
    cell25_no_locator_fabrication,
)


print(
    "\n=== WORKFLOW ==="
)

print(
    "Current State:",
    cell25_current_session.get(
        "current_state"
    ),
)

print(
    "Question Decision:",
    "ACCEPT",
)

print(
    "Answer Decision:",
    "ACCEPT",
)

print(
    "Citation Decision:",
    "PENDING",
)


print(
    "\n=== PERSISTENCE ==="
)

print(
    "Registry:",
    cell21_registry_path,
)

print(
    "Persisted Answer Decision:",
    cell25_persisted_active_case.get(
        "expected_answer_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persisted Citation Decision:",
    cell25_persisted_active_case.get(
        "citation_review",
        {},
    ).get(
        "human_decision"
    ),
)

print(
    "Persistence valid:",
    cell25_persistence_valid,
)


print(
    "\n=== QUEUE / SAFETY ==="
)

print(
    "Queue still 20:",
    cell25_queue_still_20,
)

print(
    "Base Seed unchanged:",
    cell25_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell25_full_572_blocked,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell25_failures
)


print(
    "\nCODE CELL 25 GATE:",
    (
        "PASS"
        if cell25_gate
        else "FAIL"
    ),
)


assert cell25_gate, (
    "Expected Answer ACCEPT / Citation Draft / "
    "Registry Sync failed. Repair this SAME Code Cell 25 "
    "before Citation Human Review."
)

=== ANSWER ACCEPTED + CITATION DRAFT + REGISTRY SYNC ===

=== ACCEPTED QUESTION ===
داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟

=== ACCEPTED ANSWER ===
داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت بخرد، در حالی که دیگری همه پولش را خرج کرده بود.
Human Answer Decision: ACCEPT

=== CITATION DRAFT ===
Citation Mapping ID: citation-mapping:6cc191d209ce82b4cf39f79f2336cb6a
Citation entry count: 1

--- CITATION ENTRY 0 ---
Document: website:C185157D
Role: LOCAL_SENTENCE_WINDOW
Chars: (1607, 1942)
Canonical URL: https://www.fintelligence.ir/Blog/Detail/C185157D
URL: https://www.fintelligence.ir/Blog/Detail/C185157D/%D9%BE%D8%B3-%D8%A7%D9%86%D8%AF%D8%A7%D8%B2-%D8%AF%D8%B1%D8%B3%DB%8C-%D8%A8%D8%B1%D8%A7%DB%8C-%D8%A2%DB%8C%D9%86%D8%AF%D9%87

=== CIT

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

## Cell 26 — Citation ACCEPT، تکمیل Pilot، Queue Advance و آماده‌سازی Pilot بعدی

Citation این Pilot توسط Human Reviewer پذیرفته شد.

در این مرحله:

- Citation با تصمیم `ACCEPT` ثبت می‌شود؛
- Question + Expected Answer + Citation این Pilot به‌عنوان Human Review کامل بسته می‌شوند؛
- پرونده در `completed_human_review_cases` ذخیره می‌شود؛
- Author Final Review همچنان `NOT_STARTED` باقی می‌ماند؛
- Registry دائمی روی دیسک Sync می‌شود؛
- Queue از 20 به 19 کاهش پیدا می‌کند؛
- Pilot بعدی انتخاب و Evidence اصلی آن نمایش داده می‌شود.

Human Review Notebook جداگانه نیازی به بازنویسی ندارد و آخرین Registry JSON را هنگام اجرا می‌خواند.

Full 572 Generation همچنان Blocked است.

</div>

In [52]:
# Code Cell 26

cell26_failures = []


# ============================================================
# 1) Prerequisites
# ============================================================

if globals().get("cell25_gate") is not True:
    cell26_failures.append(
        "CELL25_NOT_PASS"
    )


if not isinstance(
    globals().get(
        "cell25_current_session"
    ),
    dict,
):
    cell26_failures.append(
        "CELL25_CURRENT_SESSION_MISSING"
    )


if (
    globals().get(
        "cell25_current_session",
        {},
    ).get(
        "current_state"
    )
    !=
    "CITATION_HUMAN_REVIEW_PENDING"
):
    cell26_failures.append(
        "CITATION_NOT_AT_HUMAN_REVIEW_GATE"
    )


if not callable(
    globals().get(
        "cell21_sync_human_review_registry"
    )
):
    cell26_failures.append(
        "REGISTRY_SYNC_FUNCTION_MISSING"
    )


# ============================================================
# 2) Current Pilot
# ============================================================

CELL26_COMPLETED_PILOT_ID = (
    cell23_next_pilot_id
)

CELL26_FULL_572_BLOCK_STATUS = (
    "BLOCKED_PENDING_FULL_PILOT_WORKFLOW_VALIDATION"
)


# ============================================================
# 3) Human Citation ACCEPT
# ============================================================

CELL26_HUMAN_CITATION_NOTE = (
    "Human Reviewer accepted the Citation Mapping "
    "after reviewing the validated source locator."
)


cell26_citation_human_review = {
    "review_record_version":
        "human_citation_review_record_v1",

    "pilot_case_id":
        CELL26_COMPLETED_PILOT_ID,

    "citation_mapping_id":
        cell25_citation_mapping_id,

    "target_evidence_id":
        cell23_next_target_evidence_id,

    "human_decision":
        "ACCEPT",

    "human_note":
        CELL26_HUMAN_CITATION_NOTE,

    "external_locator_lookup_used":
        False,
}


# ============================================================
# 4) Apply Citation ACCEPT
# ============================================================

cell26_pre_advance_session = (
    cell13_apply_event(
        cell25_current_session,
        "CITATION_ACCEPTED",
        human_note=
            CELL26_HUMAN_CITATION_NOTE,
    )
)


cell26_citation_accept_valid = all([
    cell26_pre_advance_session.get(
        "current_state"
    )
    ==
    "PILOT_COMPLETE_PENDING_QUEUE_ADVANCE",

    cell26_pre_advance_session.get(
        "queue_advance_permitted"
    )
    is True,

    cell26_pre_advance_session.get(
        "queue_advance_performed"
    )
    is False,

    len(
        cell26_pre_advance_session.get(
            "human_decisions",
            [],
        )
    )
    ==
    3,

    cell26_pre_advance_session[
        "human_decisions"
    ][
        -1
    ].get(
        "event"
    )
    ==
    "CITATION_ACCEPTED",
])


# ============================================================
# 5) Finalize Human Review Case
# ============================================================

cell26_completed_human_review_case = (
    copy.deepcopy(
        cell25_active_human_review_case
    )
)


cell26_completed_human_review_case[
    "citation_review"
][
    "human_decision"
] = "ACCEPT"


cell26_completed_human_review_case[
    "citation_review"
][
    "human_review"
] = copy.deepcopy(
    cell26_citation_human_review
)


cell26_completed_human_review_case[
    "author_final_review"
] = {
    "status":
        "NOT_STARTED",

    "decision":
        None,

    "notes":
        None,
}


cell26_completed_human_review_case[
    "final_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


cell26_completed_human_review_case_sha256 = (
    cell10_stable_sha(
        cell26_completed_human_review_case
    )
)


# ============================================================
# 6) Queue advancement
# ============================================================

cell26_parent_remaining_ids = copy.deepcopy(
    cell23_continuation_state[
        "remaining_pilot_ids"
    ]
)


cell26_parent_remaining_queue = copy.deepcopy(
    cell23_continuation_state[
        "remaining_pilot_queue"
    ]
)


cell26_queue_preconditions_valid = all([
    len(
        cell26_parent_remaining_ids
    )
    ==
    20,

    len(
        cell26_parent_remaining_queue
    )
    ==
    20,

    cell26_parent_remaining_ids[
        0
    ]
    ==
    CELL26_COMPLETED_PILOT_ID,

    cell26_citation_accept_valid,
])


if cell26_queue_preconditions_valid:

    cell26_remaining_pilot_ids = copy.deepcopy(
        cell26_parent_remaining_ids[
            1:
        ]
    )

    cell26_remaining_pilot_queue = copy.deepcopy(
        cell26_parent_remaining_queue[
            1:
        ]
    )

else:

    cell26_remaining_pilot_ids = []
    cell26_remaining_pilot_queue = []


cell26_next_pilot_id = (
    cell26_remaining_pilot_ids[
        0
    ]

    if cell26_remaining_pilot_ids

    else None
)


cell26_next_pilot_queue_entry = (
    copy.deepcopy(
        cell26_remaining_pilot_queue[
            0
        ]
    )

    if cell26_remaining_pilot_queue

    else None
)


cell26_queue_advanced_valid = all([
    len(
        cell26_remaining_pilot_ids
    )
    ==
    19,

    len(
        cell26_remaining_pilot_queue
    )
    ==
    19,

    CELL26_COMPLETED_PILOT_ID
    not in
    cell26_remaining_pilot_ids,

    isinstance(
        cell26_next_pilot_id,
        str,
    ),

    cell26_next_pilot_id
    !=
    CELL26_COMPLETED_PILOT_ID,
])


# ============================================================
# 7) Completed session
# ============================================================

cell26_completed_session = copy.deepcopy(
    cell26_pre_advance_session
)


cell26_completed_session[
    "current_state"
] = "PILOT_COMPLETE"


cell26_completed_session[
    "queue_advance_permitted"
] = False


cell26_completed_session[
    "queue_advance_performed"
] = True


cell26_completed_session[
    "completion_status"
] = (
    "HUMAN_REVIEW_COMPLETE_"
    "AUTHOR_FINAL_REVIEW_PENDING"
)


# ============================================================
# 8) New authoritative Continuation State
# ============================================================

cell26_continuation_state = {
    "state_version":
        "compact_pilot_continuation_state_v3",

    "authoritative_forward_state":
        True,

    "parent_state_sha256":
        cell23_continuation_state_sha256,

    "just_completed_pilot_id":
        CELL26_COMPLETED_PILOT_ID,

    "just_completed_case_sha256":
        cell26_completed_human_review_case_sha256,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_pilot_queue":
        copy.deepcopy(
            cell26_remaining_pilot_queue
        ),

    "remaining_count":
        19,

    "next_pilot_id":
        cell26_next_pilot_id,

    "next_pilot_queue_entry":
        copy.deepcopy(
            cell26_next_pilot_queue_entry
        ),

    "full_572_generation":
        CELL26_FULL_572_BLOCK_STATUS,
}


cell26_continuation_state_sha256 = (
    cell10_stable_sha(
        cell26_continuation_state
    )
)


# ============================================================
# 9) Resolve next Pilot packet
# ============================================================

cell26_pilot_by_id = (
    cell13_seed[
        "final_runtime"
    ][
        "question_drafting_pilot_by_id"
    ]
)


cell26_next_pilot_packet = copy.deepcopy(
    cell26_pilot_by_id.get(
        cell26_next_pilot_id
    )
)


cell26_next_packet_valid = isinstance(
    cell26_next_pilot_packet,
    dict,
)


# ============================================================
# 10) Resolve next Pilot identity chain
# ============================================================

cell26_next_golden_case_id = None
cell26_next_task_draft_id = None
cell26_next_target_evidence_id = None


if cell26_next_packet_valid:

    try:

        cell26_next_golden_case_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "golden_case",
            )
        )


        cell26_next_task_draft_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "task_draft",
            )
        )


        cell26_next_target_evidence_id = (
            cell13_exact_single_id(
                cell26_next_pilot_packet,
                "target_evidence",
            )
        )


    except Exception as exc:

        cell26_failures.append({
            "reason":
                "NEXT_ID_CHAIN_FAILED",

            "error":
                repr(
                    exc
                ),
        })


cell26_next_identity_valid = all([
    isinstance(
        cell26_next_golden_case_id,
        str,
    ),

    isinstance(
        cell26_next_task_draft_id,
        str,
    ),

    isinstance(
        cell26_next_target_evidence_id,
        str,
    ),
])


# ============================================================
# 11) Resolve support records
# ============================================================

cell26_next_golden_case = None
cell26_next_task_plan = None
cell26_next_target_evidence = None


if cell26_next_identity_valid:

    try:

        cell26_next_golden_case = copy.deepcopy(
            cell13_active_support[
                "golden_case_records_by_id"
            ][
                cell26_next_golden_case_id
            ]
        )


        cell26_next_task_plan = copy.deepcopy(
            cell13_find_unique_mapping_value(
                cell13_active_support.get(
                    "task_draft_plans",
                    {},
                ),
                "task_draft",
                cell26_next_task_draft_id,
            )[
                "value"
            ]
        )


        cell26_next_target_evidence = copy.deepcopy(
            cell13_find_unique_list_record(
                cell13_active_support.get(
                    "target_evidence_records",
                    [],
                ),
                "target_evidence_id",
                cell26_next_target_evidence_id,
            )
        )


    except Exception as exc:

        cell26_failures.append({
            "reason":
                "NEXT_SUPPORT_RESOLUTION_FAILED",

            "error":
                repr(
                    exc
                ),
        })


# ============================================================
# 12) Classification helper
# ============================================================

def cell26_scalar(
    mapping,
    names,
):

    match = cell14_find_first_scalar(
        mapping,
        names,
    )

    if isinstance(
        match,
        dict,
    ):
        return match.get(
            "value"
        )

    return None


cell26_next_source_type = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "source_type",
        "source_kind",
        "source",
    ],
)


cell26_next_stratum = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "stratum",
        "stratum_name",
        "stratum_id",
    ],
)


cell26_next_task_family = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "task_family",
    ],
)


cell26_next_question_archetype = cell26_scalar(
    cell26_next_pilot_packet,
    [
        "question_archetype",
    ],
)


# ============================================================
# 13) Extract next Evidence text
# ============================================================

cell26_raw_target_parts = []


if cell26_next_packet_valid:

    packet_parts = (
        cell26_next_pilot_packet.get(
            "target_text_parts"
        )
    )

    if isinstance(
        packet_parts,
        list,
    ):
        cell26_raw_target_parts = copy.deepcopy(
            packet_parts
        )


if (
    not cell26_raw_target_parts
    and
    isinstance(
        cell26_next_target_evidence,
        dict,
    )
):

    evidence_parts = (
        cell26_next_target_evidence.get(
            "target_parts"
        )
    )

    if isinstance(
        evidence_parts,
        list,
    ):
        cell26_raw_target_parts = copy.deepcopy(
            evidence_parts
        )


cell26_next_target_text_parts = []


for index, part in enumerate(
    cell26_raw_target_parts
):

    if not isinstance(
        part,
        dict,
    ):
        continue


    text_value = None


    for field_name in (
        "text",
        "target_text",
        "content",
        "evidence_text",
        "text_content",
    ):

        candidate = part.get(
            field_name
        )

        if (
            isinstance(
                candidate,
                str,
            )
            and
            candidate.strip()
        ):

            text_value = candidate.strip()
            break


    if text_value is None:
        continue


    cell26_next_target_text_parts.append({
        "part_index":
            index,

        "document_id":
            part.get(
                "document_id"
            ),

        "role":
            part.get(
                "role"
            ),

        "char_start":
            part.get(
                "char_start"
            ),

        "char_end":
            part.get(
                "char_end"
            ),

        "text":
            text_value,

        "text_sha256":
            hashlib.sha256(
                text_value.encode(
                    "utf-8"
                )
            ).hexdigest(),
    })


cell26_next_combined_target_text = (
    "\n\n".join(
        row[
            "text"
        ]

        for row
        in cell26_next_target_text_parts
    )
)


cell26_next_target_text_available = bool(
    cell26_next_combined_target_text.strip()
)


# ============================================================
# 14) Build next Pilot workspace
# ============================================================

cell26_next_pilot_workspace = {
    "workspace_version":
        "compact_human_review_workspace_v1",

    "status":
        "READY_FOR_QUESTION_DRAFT",

    "pilot_case_id":
        cell26_next_pilot_id,

    "golden_case_id":
        cell26_next_golden_case_id,

    "task_draft_id":
        cell26_next_task_draft_id,

    "target_evidence_id":
        cell26_next_target_evidence_id,

    "source_type":
        cell26_next_source_type,

    "stratum":
        cell26_next_stratum,

    "task_family":
        cell26_next_task_family,

    "question_archetype":
        cell26_next_question_archetype,

    "source_evidence":
        copy.deepcopy(
            cell26_next_target_text_parts
        ),

    "combined_target_text":
        cell26_next_combined_target_text,

    "golden_case":
        copy.deepcopy(
            cell26_next_golden_case
        ),

    "task_plan":
        copy.deepcopy(
            cell26_next_task_plan
        ),

    "target_evidence":
        copy.deepcopy(
            cell26_next_target_evidence
        ),

    "model_drafts":
        [],

    "human_reviews":
        [],

    "expected_answer_review":
        None,

    "citation_review":
        None,

    "author_final_review":
        {
            "status":
                "NOT_STARTED",

            "decision":
                None,

            "notes":
                None,
        },

    "workflow_state":
        "QUESTION_DRAFT_PENDING",

    "full_572_generation":
        CELL26_FULL_572_BLOCK_STATUS,
}


cell26_next_workspace_ready = all([
    cell26_next_packet_valid,

    cell26_next_identity_valid,

    isinstance(
        cell26_next_golden_case,
        dict,
    ),

    isinstance(
        cell26_next_task_plan,
        dict,
    ),

    isinstance(
        cell26_next_target_evidence,
        dict,
    ),

    cell26_next_target_text_available,
])


# ============================================================
# 15) Update persistent Registry
# ============================================================

cell26_completed_cases = copy.deepcopy(
    cell25_persisted_registry.get(
        "completed_human_review_cases",
        [],
    )
)


cell26_completed_cases.append(
    copy.deepcopy(
        cell26_completed_human_review_case
    )
)


cell26_persistent_registry_payload = copy.deepcopy(
    cell25_persisted_registry
)


cell26_persistent_registry_payload[
    "completed_human_review_cases"
] = copy.deepcopy(
    cell26_completed_cases
)


cell26_persistent_registry_payload[
    "active_workspace"
] = copy.deepcopy(
    cell26_next_pilot_workspace
)


cell26_persistent_registry_payload[
    "remaining_pilot_count"
] = 19


cell26_persistent_registry_payload[
    "continuation_state"
] = {
    "next_pilot_id":
        cell26_next_pilot_id,

    "remaining_pilot_ids":
        copy.deepcopy(
            cell26_remaining_pilot_ids
        ),

    "remaining_count":
        19,

    "continuation_state_sha256":
        cell26_continuation_state_sha256,
}


cell26_author_review = (
    cell26_persistent_registry_payload.get(
        "author_review"
    )
)


if not isinstance(
    cell26_author_review,
    dict,
):

    cell26_author_review = {
        "status":
            "NOT_STARTED",

        "reviewer":
            "AUTHOR",

        "cases_ready_for_review":
            0,

        "notes":
            None,
    }


cell26_author_review[
    "cases_ready_for_review"
] = len(
    cell26_completed_cases
)


cell26_persistent_registry_payload[
    "author_review"
] = cell26_author_review


# ============================================================
# 16) Sync Registry to disk
# ============================================================

cell26_registry_sync_result = (
    cell21_sync_human_review_registry(
        cell26_persistent_registry_payload
    )
)


cell26_persisted_registry = json.loads(
    cell21_registry_path.read_text(
        encoding="utf-8"
    )
)


cell26_persisted_active_case = (
    cell26_persisted_registry.get(
        "active_workspace"
    )
    or
    {}
)


cell26_persistence_valid = all([
    len(
        cell26_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    )
    ==
    len(
        cell26_completed_cases
    ),

    cell26_persisted_registry.get(
        "remaining_pilot_count"
    )
    ==
    19,

    cell26_persisted_active_case.get(
        "pilot_case_id"
    )
    ==
    cell26_next_pilot_id,

    cell26_persisted_registry.get(
        "continuation_state",
        {},
    ).get(
        "next_pilot_id"
    )
    ==
    cell26_next_pilot_id,

    cell26_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    )
    ==
    len(
        cell26_completed_cases
    ),
])


# ============================================================
# 17) Safety
# ============================================================

cell26_base_seed_unchanged = (
    cell10_stable_sha(
        cell13_seed
    )
    ==
    cell19_base_seed_sha_before
)


cell26_full_572_blocked = (
    cell26_continuation_state[
        "full_572_generation"
    ]
    ==
    CELL26_FULL_572_BLOCK_STATUS
)


# ============================================================
# 18) Final Gate
# ============================================================

cell26_gate = all([
    len(
        cell26_failures
    )
    ==
    0,

    cell26_citation_accept_valid,

    cell26_queue_preconditions_valid,

    cell26_queue_advanced_valid,

    cell26_next_workspace_ready,

    cell26_persistence_valid,

    cell26_base_seed_unchanged,

    cell26_full_572_blocked,
])


# ============================================================
# 19) Diagnostics
# ============================================================

print(
    "=== PILOT COMPLETED + REGISTRY SYNC "
    "+ NEXT PILOT ==="
)


print(
    "\n=== COMPLETED PILOT ==="
)

print(
    "Pilot:",
    CELL26_COMPLETED_PILOT_ID,
)

print(
    "Question:",
    cell24_question_text,
)

print(
    "Answer:",
    cell24_expected_answer_text,
)

print(
    "Citation:",
    "ACCEPT",
)

print(
    "Human Review status:",
    cell26_completed_human_review_case[
        "final_status"
    ],
)

print(
    "Author Final Review:",
    "NOT_STARTED",
)


print(
    "\n=== QUEUE ==="
)

print(
    "Before:",
    len(
        cell26_parent_remaining_ids
    ),
)

print(
    "After:",
    len(
        cell26_remaining_pilot_ids
    ),
)

print(
    "Completed Pilot removed:",
    (
        CELL26_COMPLETED_PILOT_ID
        not in
        cell26_remaining_pilot_ids
    ),
)

print(
    "Next Pilot:",
    cell26_next_pilot_id,
)


print(
    "\n=== PERSISTENT REGISTRY ==="
)

print(
    "Path:",
    cell21_registry_path,
)

print(
    "Completed Human Review cases:",
    len(
        cell26_persisted_registry.get(
            "completed_human_review_cases",
            [],
        )
    ),
)

print(
    "Cases ready for Author Review:",
    cell26_persisted_registry.get(
        "author_review",
        {},
    ).get(
        "cases_ready_for_review"
    ),
)

print(
    "Remaining Pilots:",
    cell26_persisted_registry.get(
        "remaining_pilot_count"
    ),
)

print(
    "Active Pilot:",
    cell26_persisted_active_case.get(
        "pilot_case_id"
    ),
)

print(
    "Persistence valid:",
    cell26_persistence_valid,
)


print(
    "\n=== NEXT PILOT IDENTITY ==="
)

print(
    "Pilot:",
    cell26_next_pilot_id,
)

print(
    "Golden Case:",
    cell26_next_golden_case_id,
)

print(
    "Task Draft:",
    cell26_next_task_draft_id,
)

print(
    "Target Evidence:",
    cell26_next_target_evidence_id,
)

print(
    "Source type:",
    cell26_next_source_type,
)

print(
    "Stratum:",
    cell26_next_stratum,
)

print(
    "Task family:",
    cell26_next_task_family,
)

print(
    "Question archetype:",
    cell26_next_question_archetype,
)


print(
    "\n=== NEXT PILOT SOURCE EVIDENCE ==="
)

print(
    cell26_next_combined_target_text
)


print(
    "\n=== SAFETY ==="
)

print(
    "Base Seed unchanged:",
    cell26_base_seed_unchanged,
)

print(
    "Full 572 blocked:",
    cell26_full_572_blocked,
)

print(
    "Author Notebook rewrite:",
    False,
)

print(
    "Commit / Push:",
    False,
)


print(
    "\n=== FAILURES ==="
)

print(
    cell26_failures
)


print(
    "\nCODE CELL 26 GATE:",
    (
        "PASS"
        if cell26_gate
        else "FAIL"
    ),
)


assert cell26_gate, (
    "Citation ACCEPT / Pilot completion / Queue advance / "
    "Registry sync / next Pilot preparation failed. "
    "Repair this SAME Code Cell 26 before continuing."
)

=== PILOT COMPLETED + REGISTRY SYNC + NEXT PILOT ===

=== COMPLETED PILOT ===
Pilot: question-pilot:b801c37bf6fc78b00fcc0c79b368ef7d
Question: داشتن هدف برای پس‌انداز چه تأثیری بر انگیزه بچه‌ها برای به دست آوردن پول و رسیدن به خواسته‌هایشان دارد؟
Answer: داشتن یک هدف برای پس‌انداز می‌تواند به بچه‌ها انگیزه بدهد تا برای رسیدن زودتر به خواسته‌شان پول بیشتری به دست بیاورند و آن را پس‌انداز کنند. در مثال متن نیز کسی که پولش را پس‌انداز کرده بود، توانست با پس‌اندازش چیزهایی را که دوست داشت بخرد، در حالی که دیگری همه پولش را خرج کرده بود.
Citation: ACCEPT
Human Review status: HUMAN_REVIEW_COMPLETE_AUTHOR_FINAL_REVIEW_PENDING
Author Final Review: NOT_STARTED

=== QUEUE ===
Before: 20
After: 19
Completed Pilot removed: True
Next Pilot: question-pilot:909b695797935246c92c7a66ebe0f773

=== PERSISTENT REGISTRY ===
Path: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/data/phase2/human_review/human_review_registry.json
Completed Human Review cases: 3
Cases ready for Autho